# **Tiny Text-Conditioned Avatar Diffusion from Scratch**

Questo notebook documenta l'intero sviluppo del progetto di Deep Learning, dalla preparazione dell'ambiente e del dataset fino all'addestramento e alla valutazione del modello per la generazione di avatar cartoon.

L'obiettivo è costruire una pipeline riproducibile in Google Colab, basata su un dataset strutturato, un tokenizer addestrato da zero, un text encoder transformer e una U-Net compatta per DDPM in pixel space.

---



## Setup iniziale e riproducibilità dell'ambiente

In questa sezione verifichiamo il runtime Colab, controlliamo la disponibilità della GPU, configuriamo lo storage persistente su Google Drive e prepariamo il repository GitHub del progetto.

Lo scopo di questo blocco è garantire che tutte le fasi successive vengano eseguite in un ambiente coerente, tracciabile e riproducibile.

In [ ]:
import os
import platform
import torch

# Verifica preliminare del runtime: versioni software, disponibilità CUDA e spazio disco.
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA disponibile:", torch.cuda.is_available())
print("Numero GPU:", torch.cuda.device_count())

assert torch.cuda.is_available(), (
    "GPU non disponibile. Seleziona Runtime > Change runtime type > GPU."
)

for i in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(i)
    print(
        f"GPU {i}:",
        properties.name,
        "- VRAM:",
        round(properties.total_memory / 1024**3, 2),
        "GB",
    )

disk = os.statvfs("/content")
free_gb = disk.f_bavail * disk.f_frsize / 1024**3
print("Spazio libero:", round(free_gb, 2), "GB")

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive

# Monta Google Drive per salvare dataset, checkpoint e risultati in modo persistente.
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

# Radice persistente comune: la release finale userà una sottocartella dedicata.
DRIVE_ROOT = Path("/content/drive/MyDrive/tiny-avatar-diffusion-storage")

persistent_directories = [
    DRIVE_ROOT / "datasets",
    DRIVE_ROOT / "checkpoints",
    DRIVE_ROOT / "runs",
    DRIVE_ROOT / "generated_samples",
    DRIVE_ROOT / "final_results",
]

for directory in persistent_directories:
    directory.mkdir(parents=True, exist_ok=True)

print("Storage persistente:", DRIVE_ROOT)
for directory in persistent_directories:
    print("-", directory)


## Configurazione di Git e del repository

Per mantenere il progetto versionato e facilmente sincronizzabile tra Colab e GitHub, installiamo gli strumenti necessari, configuriamo l'autenticazione e cloniamo il repository di lavoro.

In [ ]:
%%bash

# Installa GitHub CLI solo se non è già disponibile nel runtime.
if ! command -v gh >/dev/null 2>&1; then
  apt-get update -qq
  apt-get install -y gh
fi

gh --version
git --version

In [ ]:
import os
import subprocess
from google.colab import userdata

GITHUB_USERNAME = "domdam11"
REPOSITORY_NAME = "tiny-avatar-diffusion"

# Recupera il token GitHub dai secret di Colab.
github_token = userdata.get("GITHUB_TOKEN")

assert github_token, (
    "Secret GITHUB_TOKEN non trovato. "
    "Aggiungilo nella sezione Secrets di Colab."
)

os.environ["GH_TOKEN"] = github_token

# Verifica che il token sia visibile alla GitHub CLI e configura Git per usare gh.
subprocess.run(["gh", "auth", "status"], check=True)
subprocess.run(["gh", "auth", "setup-git"], check=True)

print("Autenticazione GitHub configurata.")

In [ ]:
from pathlib import Path
import subprocess

PROJECT_ROOT = Path("/content/tiny-avatar-diffusion")

# Clona il repository solo se non è già presente nel filesystem locale del runtime.
if PROJECT_ROOT.exists():
    print("Repository già presente:", PROJECT_ROOT)
else:
    subprocess.run(
        [
            "gh",
            "repo",
            "clone",
            f"{GITHUB_USERNAME}/{REPOSITORY_NAME}",
            str(PROJECT_ROOT),
        ],
        check=True,
    )

print("Repository:", PROJECT_ROOT)

## Struttura iniziale del progetto

Creiamo una struttura ordinata per configurazioni, dati, moduli sorgente, test, report e artefatti. Questo rende più semplice mantenere il progetto pulito e coerente durante tutto lo sviluppo.

In [ ]:
%%bash

cd /content/tiny-avatar-diffusion

mkdir -p \
  configs \
  data/raw \
  data/processed \
  environment \
  notebooks \
  outputs \
  report \
  src/data \
  src/models \
  src/diffusion \
  src/evaluation \
  src/training \
  src/utils \
  tests

touch data/raw/.gitkeep
touch data/processed/.gitkeep
touch outputs/.gitkeep

touch src/__init__.py
touch src/data/__init__.py
touch src/models/__init__.py
touch src/diffusion/__init__.py
touch src/evaluation/__init__.py
touch src/training/__init__.py
touch src/utils/__init__.py

find . -maxdepth 3 -type d | sort

In [ ]:
%%bash

cd /content/tiny-avatar-diffusion

cat > .gitignore <<'EOF'
# Python
__pycache__/
*.py[cod]
*.so
.pytest_cache/
.mypy_cache/

# Jupyter
.ipynb_checkpoints/

# Virtual environments
.venv/
venv/
env/
.env

# Raw and processed data
data/raw/*
!data/raw/.gitkeep
data/processed/*
!data/processed/.gitkeep

# Training artifacts
checkpoints
runs
wandb/
*.pth
*.pt
*.ckpt

# Generated outputs
outputs/*
!outputs/.gitkeep

# Local files
.DS_Store
Thumbs.db
*.log

# Secrets
kaggle.json
credentials.json
*.token
EOF

cat .gitignore

In [ ]:
%%bash

cd /content/tiny-avatar-diffusion

cat > README.md <<'EOF'
# Tiny Text-Conditioned Avatar Diffusion

University project for the Deep Learning course at Politecnico di Bari.

## Objective

Implement and train from scratch a compact text-conditioned pixel-space
DDPM that generates cartoon avatars from deterministic attribute captions.

## Main components

- Reproducible preprocessing pipeline
- Compositional train/validation/test/OOD split
- Tokenizer trained from the training split
- Transformer text encoder trained from scratch
- Compact time-conditioned U-Net
- Unconditional baseline
- Text-conditioned diffusion model
- Quality, conditioning, diversity and efficiency evaluation

## Constraints

The main model does not use:

- Pretrained diffusion checkpoints
- Stable Diffusion
- Pretrained text encoders
- CLIP, BERT or T5 embeddings
- Pretrained VAEs
- Diffusers training pipelines as black boxes

## Initial configuration

- Dataset: Google Cartoon Set 10k
- Image resolution: 32x32 RGB
- Model: pixel-space DDPM
- Execution environment: Google Colab Pro
EOF

cat README.md

## Dipendenze e tracciamento dell'ambiente

Definiamo esplicitamente le dipendenze del progetto, le installiamo in Colab e salviamo una fotografia completa dell'ambiente effettivamente utilizzato. Questa scelta è utile sia per la riproducibilità sia per la documentazione finale.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/tiny-avatar-diffusion")
REQUIREMENTS_PATH = PROJECT_ROOT / "requirements.txt"

requirements_text = """\
einops==0.8.2
PyYAML==6.0.3
tokenizers==0.23.2
numpy==2.1.3
pandas==2.2.3
Pillow==11.3.0
matplotlib==3.10.0
tqdm==4.67.3
scikit-learn==1.6.1
scikit-image==0.25.2
tensorboard==2.21.0
torchmetrics==1.9.0
torch-fidelity==0.4.0
pytest==8.4.2
"""

REQUIREMENTS_PATH.write_text(requirements_text, encoding="utf-8")

print("requirements.txt aggiornato:\n")
print(REQUIREMENTS_PATH.read_text(encoding="utf-8"))

In [ ]:
%%bash

cd /content/tiny-avatar-diffusion

# Installa le dipendenze dirette nelle versioni registrate.
python -m pip install -q -r requirements.txt

In [ ]:
from importlib.metadata import PackageNotFoundError, version

DIRECT_DEPENDENCIES = [
    "einops",
    "PyYAML",
    "tokenizers",
    "numpy",
    "pandas",
    "Pillow",
    "matplotlib",
    "tqdm",
    "scikit-learn",
    "scikit-image",
    "tensorboard",
    "torchmetrics",
    "torch-fidelity",
    "pytest",
]

installed_versions = {}
missing_dependencies = []

for distribution_name in DIRECT_DEPENDENCIES:
    try:
        installed_versions[distribution_name] = version(distribution_name)
    except PackageNotFoundError:
        missing_dependencies.append(distribution_name)

assert not missing_dependencies, (
    "Dipendenze mancanti dopo l'installazione: "
    f"{missing_dependencies}"
)

print("Dipendenze dirette verificate:")
for distribution_name, installed_version in installed_versions.items():
    print(f"- {distribution_name}=={installed_version}")

In [ ]:
from pathlib import Path
import platform
import subprocess
import torch
import torchvision

PROJECT_ROOT = Path("/content/tiny-avatar-diffusion")
ENVIRONMENT_DIR = PROJECT_ROOT / "environment"
ENVIRONMENT_DIR.mkdir(parents=True, exist_ok=True)

# Salva il freeze completo dell'ambiente Python del runtime corrente.
freeze_path = ENVIRONMENT_DIR / "colab-freeze.txt"
with freeze_path.open("w", encoding="utf-8") as output_file:
    subprocess.run(
        ["python", "-m", "pip", "freeze"],
        stdout=output_file,
        check=True,
        text=True,
    )

# Salva anche un riepilogo sintetico dell'hardware e delle librerie principali.
runtime_lines = [
    f"Python: {platform.python_version()}",
    f"PyTorch: {torch.__version__}",
    f"Torchvision: {torchvision.__version__}",
    f"CUDA runtime: {torch.version.cuda}",
    f"CUDA available: {torch.cuda.is_available()}",
    (
        f"GPU: {torch.cuda.get_device_name(0)}"
        if torch.cuda.is_available()
        else "GPU: None"
    ),
]

if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    runtime_lines.append(
        f"GPU memory GiB: {properties.total_memory / 1024**3:.2f}"
    )

runtime_text = "\n".join(runtime_lines) + "\n"
runtime_path = ENVIRONMENT_DIR / "runtime.txt"
runtime_path.write_text(runtime_text, encoding="utf-8")

# Genera uno script minimo per ricostruire l'ambiente in future sessioni Colab.
setup_script = """#!/usr/bin/env bash
set -euo pipefail

python -m pip install -r requirements.txt

python - <<'PY'
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA disponibile:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA non disponibile: selezionare un runtime GPU in Colab."
    )
PY
"""

setup_path = PROJECT_ROOT / "setup_env.sh"
setup_path.write_text(setup_script, encoding="utf-8")
setup_path.chmod(0o755)

print("Ambiente registrato correttamente.\n")
print(runtime_text)
print(f"Freeze completo: {freeze_path}")
print(f"Setup script: {setup_path}")

## Collegamento degli artefatti persistenti

Per evitare di perdere checkpoint e log alla chiusura del runtime, colleghiamo le directory di addestramento locali a cartelle persistenti su Google Drive tramite link simbolici.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/tiny-avatar-diffusion")
DRIVE_ROOT = Path("/content/drive/MyDrive/tiny-avatar-diffusion-storage")

# Solo checkpoint e runs sono linkati durante il lavoro; data/processed e outputs
# saranno copiati esplicitamente dalla cella di freeze finale.
links = {
    PROJECT_ROOT / "checkpoints": DRIVE_ROOT / "checkpoints",
    PROJECT_ROOT / "runs": DRIVE_ROOT / "runs",
}

for link, target in links.items():
    target.mkdir(parents=True, exist_ok=True)

    if link.is_symlink():
        if link.resolve() != target.resolve():
            link.unlink()
            link.symlink_to(target, target_is_directory=True)
    elif link.exists():
        if link.is_dir() and not any(link.iterdir()):
            link.rmdir()
            link.symlink_to(target, target_is_directory=True)
        else:
            raise RuntimeError(
                f"{link} esiste già e non è un collegamento vuoto."
            )
    else:
        link.symlink_to(target, target_is_directory=True)

    print(link, "->", target)


In [ ]:
%%bash

cd /content/tiny-avatar-diffusion

git config user.name "Domenico"
git config user.email "d.dambrosio6@studenti.poliba.it"

git config --get user.name
git config --get user.email

In [ ]:
%%bash

cd /content/tiny-avatar-diffusion

# Verifica informativa del branch e del remote configurati per il freeze finale.

git branch --show-current

git remote -v


A questo punto l'ambiente di lavoro è pronto: il runtime è stato verificato, lo storage persistente è configurato, il repository è stato inizializzato correttamente e le dipendenze sono state installate e tracciate. Nella sezione successiva passiamo alla preparazione del dataset e alla costruzione degli artefatti di preprocessing.

## Dataset raw, metadata e audit preliminare

In questa sezione scarichiamo e verifichiamo il dataset **Google Cartoon Set 10k**, estraiamo in modo controllato le immagini e i rispettivi file di attributi, costruiamo un unico file di metadata e analizziamo la distribuzione degli attributi.

L'audit statistico viene eseguito prima della definizione degli split, così da scegliere in modo documentato una combinazione di attributi appropriata per la successiva valutazione out-of-distribution.

In [ ]:
from pathlib import Path
import hashlib
import subprocess

# Cartella persistente su Drive in cui conserviamo l'archivio originale.
DATASET_STORAGE = Path(
    "/content/drive/MyDrive/"
    "tiny-avatar-diffusion-storage/datasets"
)
DATASET_STORAGE.mkdir(parents=True, exist_ok=True)

# Percorso locale dell'archivio compresso del dataset.
ARCHIVE_PATH = DATASET_STORAGE / "cartoonset10k.tgz"

# URL sorgente ufficiale del Cartoon Set 10k.
DATASET_URL = (
    "https://huggingface.co/datasets/"
    "cgarciae/cartoonset/resolve/1.0.0/"
    "data/cartoonset10k.tgz?download=true"
)

# Metadati attesi dell'archivio, usati per verificare integrità e completezza.
EXPECTED_SIZE_BYTES = 476_635_078
EXPECTED_SHA256 = (
    "902145ad5096af10ca9ee200b648e2a60"
    "e8b6b83dd2f4dc5356817988be8ff43"
)


def compute_sha256(path: Path) -> str:
    # Calcolo incrementale dell'hash per evitare di caricare
    # l'intero file in memoria.
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def archive_is_valid(path: Path) -> bool:
    # L'archivio è valido solo se esiste, ha la dimensione attesa
    # e coincide esattamente con l'hash di riferimento.
    if not path.is_file():
        return False

    if path.stat().st_size != EXPECTED_SIZE_BYTES:
        return False

    return compute_sha256(path) == EXPECTED_SHA256


if archive_is_valid(ARCHIVE_PATH):
    print("Archivio già presente e verificato: download saltato.")

else:
    if ARCHIVE_PATH.exists():
        current_size = ARCHIVE_PATH.stat().st_size

        # Un file parziale può essere ripreso con wget --continue.
        # Se invece ha dimensione attesa ma hash errato, oppure è più grande
        # del previsto, lo eliminiamo e riscarichiamo da zero.
        if current_size >= EXPECTED_SIZE_BYTES:
            print("Archivio esistente non valido: eliminazione.")
            ARCHIVE_PATH.unlink()

    print("Download del Cartoon Set 10k...")

    subprocess.run(
        [
            "wget",
            "--continue",
            "--tries=10",
            "--timeout=60",
            "--show-progress",
            "-O",
            str(ARCHIVE_PATH),
            DATASET_URL,
        ],
        check=True,
    )

# Verifica forte dell'archivio scaricato.
assert archive_is_valid(ARCHIVE_PATH), (
    "L’archivio scaricato non supera il controllo "
    "di dimensione e SHA-256."
)

# gzip -t controlla che lo stream compresso non sia corrotto.
subprocess.run(
    ["gzip", "-t", str(ARCHIVE_PATH)],
    check=True,
)

print("Archivio verificato correttamente.")
print("Dimensione:", ARCHIVE_PATH.stat().st_size, "byte")
print("SHA-256:", compute_sha256(ARCHIVE_PATH))

L'archivio viene validato sia tramite dimensione attesa sia tramite checksum SHA-256. In questo modo eventuali download incompleti o file corrotti vengono rilevati prima dell'estrazione e delle fasi di preprocessing successive.

In [ ]:
%%bash

# Ispezione rapida dell'archivio senza estrarlo:
# verifichiamo la struttura interna e i nomi dei primi file.
tar -tzf \
  /content/drive/MyDrive/tiny-avatar-diffusion-storage/datasets/cartoonset10k.tgz \
  | head -n 20

In [ ]:
from pathlib import Path
import shutil
import subprocess

ARCHIVE_PATH = Path(
    "/content/drive/MyDrive/"
    "tiny-avatar-diffusion-storage/"
    "datasets/cartoonset10k.tgz"
)

# Directory di lavoro locale in cui estraiamo il dataset raw.
DESTINATION = Path(
    "/content/tiny-avatar-diffusion/data/raw/cartoonset10k"
)


def raw_dataset_is_complete(destination: Path) -> bool:
    # Il dataset raw è considerato completo solo se contiene
    # esattamente 10.000 immagini PNG e 10.000 file CSV.
    if not destination.exists():
        return False

    png_count = sum(1 for _ in destination.rglob("*.png"))
    csv_count = sum(1 for _ in destination.rglob("*.csv"))

    return png_count == 10_000 and csv_count == 10_000


if raw_dataset_is_complete(DESTINATION):
    print("Dataset già estratto e completo.")
else:
    if DESTINATION.exists():
        # Se l'estrazione precedente è incompleta o incoerente,
        # ricostruiamo da zero la cartella raw.
        print("Estrazione incompleta: ricostruzione della cartella raw.")
        shutil.rmtree(DESTINATION)

    DESTINATION.mkdir(parents=True, exist_ok=True)

    subprocess.run(
        [
            "tar",
            "-xzf",
            str(ARCHIVE_PATH),
            "-C",
            str(DESTINATION),
        ],
        check=True,
    )

    # Dopo l'estrazione pretendiamo il numero esatto di file attesi.
    assert raw_dataset_is_complete(DESTINATION), (
        "L'estrazione non ha prodotto 10.000 PNG e 10.000 CSV."
    )

    print("Dataset estratto e verificato correttamente.")

print(f"Destinazione: {DESTINATION}")

In [ ]:
from pathlib import Path

dataset_root = Path(
    "/content/tiny-avatar-diffusion/"
    "data/raw/cartoonset10k"
)

# Indicizziamo separatamente immagini e CSV per una prima verifica strutturale.
images = sorted(dataset_root.rglob("*.png"))
csv_files = sorted(dataset_root.rglob("*.csv"))

print("Immagini PNG:", len(images))
print("File CSV:", len(csv_files))

print("\nPrime immagini:")
for path in images[:5]:
    print(path.relative_to(dataset_root))

print("\nPrimi CSV:")
for path in csv_files[:5]:
    print(path.relative_to(dataset_root))

In [ ]:
# Ogni avatar deve avere esattamente una coppia:
# immagine PNG + file CSV con lo stesso identificativo.
image_ids = {path.stem for path in images}
csv_ids = {path.stem for path in csv_files}

missing_csv = sorted(image_ids - csv_ids)
missing_images = sorted(csv_ids - image_ids)

print("Immagini senza CSV:", len(missing_csv))
print("CSV senza immagine:", len(missing_images))

# Controlli di integrità minimi del dataset raw.
assert len(images) == 10_000
assert len(csv_files) == 10_000
assert not missing_csv
assert not missing_images

print("Dataset completo e correttamente accoppiato.")

## Struttura dei file di attributi

Ogni immagine è associata a un file CSV che contiene 18 attributi discreti. Per ogni attributo sono disponibili il nome, il valore assegnato all'avatar e il numero totale di varianti previste nel dataset.

In [ ]:
from pathlib import Path
import pandas as pd

dataset_root = Path(
    "/content/tiny-avatar-diffusion/data/raw/cartoonset10k"
)

# Scegliamo un file CSV di esempio per capire il formato atomico
# delle annotazioni originali del dataset.
sample_csv_path = sorted(dataset_root.rglob("*.csv"))[0]

attributes = pd.read_csv(
    sample_csv_path,
    header=None,
    names=["attribute", "value", "num_variants"],
)

print(f"File: {sample_csv_path.name}")
display(attributes)

In [ ]:
%cd /content/tiny-avatar-diffusion

In [ ]:
%%writefile src/data/build_metadata.py
import argparse
import json
from pathlib import Path

import pandas as pd


# Numero atteso di varianti per ciascun attributo discreto.
EXPECTED_VARIANTS = {
    "eye_angle": 3,
    "eye_lashes": 2,
    "eye_lid": 2,
    "chin_length": 3,
    "eyebrow_weight": 2,
    "eyebrow_shape": 14,
    "eyebrow_thickness": 4,
    "face_shape": 7,
    "facial_hair": 15,
    "hair": 111,
    "eye_color": 5,
    "face_color": 11,
    "hair_color": 10,
    "glasses": 12,
    "glasses_color": 7,
    "eye_slant": 3,
    "eyebrow_width": 3,
    "eye_eyebrow_distance": 3,
}


def parse_single_metadata(csv_path: Path, dataset_root: Path) -> dict:
    # Ogni CSV descrive un singolo avatar come tabella verticale
    # attributo -> valore -> numero totale di varianti possibili.
    raw = pd.read_csv(
        csv_path,
        header=None,
        names=["attribute", "value", "num_variants"],
    )

    # Verifichiamo che gli attributi siano esattamente quelli attesi.
    observed_attributes = raw["attribute"].tolist()
    expected_attributes = list(EXPECTED_VARIANTS.keys())

    if observed_attributes != expected_attributes:
        raise ValueError(
            f"Ordine o contenuto attributi non valido in {csv_path.name}."
        )

    # Verifichiamo che il numero di varianti dichiarato nel CSV
    # coincida con quello atteso per ogni attributo.
    observed_num_variants = {
        row.attribute: int(row.num_variants)
        for row in raw.itertuples(index=False)
    }

    if observed_num_variants != EXPECTED_VARIANTS:
        raise ValueError(
            f"Numero varianti non coerente in {csv_path.name}."
        )

    image_id = csv_path.stem
    image_path = csv_path.with_suffix(".png")

    if not image_path.is_file():
        raise FileNotFoundError(
            f"Immagine mancante per {image_id}."
        )

    # Rappresentazione piatta: una riga per immagine,
    # una colonna per attributo.
    record = {
        "image_id": image_id,
        "image_path": str(image_path.relative_to(dataset_root)),
        "csv_path": str(csv_path.relative_to(dataset_root)),
    }

    for row in raw.itertuples(index=False):
        record[row.attribute] = int(row.value)

    return record


def build_metadata(dataset_root: Path) -> pd.DataFrame:
    # Costruiamo il metadata centralizzato iterando in modo deterministico
    # sui CSV ordinati lessicograficamente.
    csv_paths = sorted(dataset_root.rglob("*.csv"))

    if len(csv_paths) != 10_000:
        raise ValueError(
            f"Attesi 10.000 CSV, trovati {len(csv_paths)}."
        )

    records = [
        parse_single_metadata(csv_path, dataset_root)
        for csv_path in csv_paths
    ]

    metadata = pd.DataFrame(records)

    # Ordinamento canonico per riproducibilità.
    metadata = (
        metadata
        .sort_values("image_id")
        .reset_index(drop=True)
    )

    if metadata["image_id"].duplicated().any():
        raise ValueError("Sono presenti image_id duplicati.")

    return metadata


def build_summary(metadata: pd.DataFrame) -> dict:
    # Riepilogo semplice ma utile per i controlli successivi.
    return {
        "num_examples": int(len(metadata)),
        "num_attributes": int(len(EXPECTED_VARIANTS)),
        "columns": list(metadata.columns),
    }


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--dataset-root",
        type=Path,
        default=Path("data/raw/cartoonset10k"),
    )
    parser.add_argument(
        "--output",
        type=Path,
        default=Path("data/processed/metadata.csv"),
    )
    parser.add_argument(
        "--summary-output",
        type=Path,
        default=Path("data/processed/metadata_summary.json"),
    )

    return parser.parse_args()


def main() -> None:
    args = parse_args()

    args.output.parent.mkdir(parents=True, exist_ok=True)

    metadata = build_metadata(args.dataset_root)
    summary = build_summary(metadata)

    metadata.to_csv(args.output, index=False)

    with args.summary_output.open("w", encoding="utf-8") as file:
        json.dump(summary, file, indent=2, sort_keys=True)

    print(json.dumps(summary, indent=2, sort_keys=True))


if __name__ == "__main__":
    main()

In [ ]:
%%bash

# Generiamo il file metadata centralizzato che useremo
# in tutti i blocchi successivi della pipeline.
python -m src.data.build_metadata \
  --dataset-root data/raw/cartoonset10k \
  --output data/processed/metadata.csv \
  --summary-output data/processed/metadata_summary.json

In [ ]:
from pathlib import Path
import json
import pandas as pd

metadata_path = Path("data/processed/metadata.csv")
summary_path = Path("data/processed/metadata_summary.json")

metadata = pd.read_csv(metadata_path)

with summary_path.open("r", encoding="utf-8") as file:
    metadata_summary = json.load(file)

# Ispezione del metadata finale: struttura tabellare ampia,
# una riga per immagine e 18 attributi discreti.
display(metadata.head())

print("Numero esempi:", len(metadata))
print("Numero colonne:", len(metadata.columns))
print("Colonne:")
print(metadata.columns.tolist())

print("\nRiepilogo JSON:")
print(json.dumps(metadata_summary, indent=2, ensure_ascii=False))

In [ ]:
from src.data.build_metadata import EXPECTED_VARIANTS

# Per ogni attributo verifichiamo:
# 1) il numero di valori distinti osservati;
# 2) la frequenza minima e massima;
# 3) la coerenza con il numero di varianti attese.
attribute_rows = []

for attribute, expected_num_variants in EXPECTED_VARIANTS.items():
    counts = metadata[attribute].value_counts().sort_index()

    attribute_rows.append(
        {
            "expected_num_variants": expected_num_variants,
            "observed_num_variants": int(counts.size),
            "minimum_count": int(counts.min()),
            "maximum_count": int(counts.max()),
        }
    )

attribute_summary = pd.DataFrame(
    attribute_rows,
    index=EXPECTED_VARIANTS.keys(),
)

display(attribute_summary)

# Tutti gli attributi devono avere copertura completa.
assert (
    attribute_summary["expected_num_variants"]
    == attribute_summary["observed_num_variants"]
).all(), "Alcuni attributi non coprono tutte le varianti attese."

In [ ]:
# ==========================================================
# AUDIT DELLE COPPIE DI ATTRIBUTI PER IL TEST COMPOSIZIONALE OOD
# ==========================================================
#
# L'obiettivo non è ancora creare gli split, ma scegliere una coppia
# di attributi adatta a definire una combinazione da trattenere.
#
# Una buona coppia candidata dovrebbe avere:
# - tutte o quasi tutte le combinazioni teoricamente possibili presenti;
# - un numero sufficiente di esempi per combinazione;
# - frequenze relativamente omogenee, così che la combinazione OOD
#   scelta non sia troppo rara né troppo dominante.
#
# Nel blocco successivo verrà selezionata una combinazione specifica
# appartenente alla coppia migliore e verrà destinata interamente
# allo split test_ood.
# ==========================================================

# Elenco delle coppie di attributi da confrontare.
#
# Sono state incluse coppie con cardinalità differenti:
# - poche combinazioni e frequenze alte, ad esempio eye_color-chin_length;
# - molte combinazioni e frequenze più basse, ad esempio hair_color-face_shape;
# - una coppia potenzialmente sbilanciata, glasses-face_shape,
#   utile come termine di confronto.
candidate_pairs = [
    ("hair_color", "face_shape"),
    ("eye_color", "chin_length"),
    ("hair_color", "chin_length"),
    ("eye_color", "face_shape"),
    ("glasses", "face_shape"),
]

# pair_summaries conterrà una riga riassuntiva per ogni coppia:
# numero di combinazioni osservate e statistiche sui relativi conteggi.
pair_summaries = []

# pair_tables conserva invece la tabella completa di ogni coppia,
# per poterla visualizzare in dettaglio dopo il riepilogo generale.
pair_tables = {}


# Analizziamo una coppia di attributi alla volta.
for first_attribute, second_attribute in candidate_pairs:
    # Nome leggibile e stabile della coppia, usato come etichetta
    # nel riepilogo e come chiave del dizionario pair_tables.
    pair_name = f"{first_attribute}__{second_attribute}"

    # Raggruppiamo le 10.000 immagini per combinazione dei due attributi.
    #
    # Esempio per hair_color e face_shape:
    # una riga della tabella risultante rappresenta una combinazione
    # specifica, come hair_color=3 e face_shape=3.
    counts = (
        metadata
        .groupby([first_attribute, second_attribute])
        .size()
        .reset_index(name="count")
        # Ordinamento crescente: le combinazioni meno frequenti
        # compaiono per prime, rendendo più facile individuare
        # eventuali combinazioni troppo rare per lo split OOD.
        .sort_values("count")
        .reset_index(drop=True)
    )

    # Convertiamo il conteggio assoluto nella frequenza relativa
    # rispetto alle 10.000 immagini del dataset.
    #
    # Ad esempio, count=144 diventa frequency=0.0144,
    # cioè l'1,44% del dataset complessivo.
    counts["frequency"] = counts["count"] / len(metadata)

    # Numero teorico massimo di combinazioni possibili per questa coppia.
    #
    # Esempio:
    # hair_color ha 10 varianti e face_shape ne ha 7,
    # quindi ci aspettiamo al massimo 10 x 7 = 70 combinazioni.
    #
    # Questo valore è utile per confrontare il numero di combinazioni
    # realmente osservate con quello teoricamente disponibile.
    expected_combinations = (
        EXPECTED_VARIANTS[first_attribute]
        * EXPECTED_VARIANTS[second_attribute]
    )

    # Salviamo statistiche aggregate per confrontare velocemente le coppie.
    #
    # - observed_combinations: quante combinazioni sono effettivamente
    #   presenti nel dataset;
    # - minimum_count: numerosità della combinazione più rara;
    # - median_count: valore centrale dei conteggi, utile per scegliere
    #   una combinazione OOD rappresentativa;
    # - maximum_count: numerosità della combinazione più frequente.
    #
    # expected_combinations viene aggiunto esplicitamente: consente di
    # verificare se la copertura è completa, cioè se tutte le combinazioni
    # teoriche compaiono almeno una volta nel dataset.
    pair_summaries.append(
        {
            "pair": pair_name,
            "expected_combinations": float(expected_combinations),
            "observed_combinations": float(len(counts)),
            "coverage_complete": (
                len(counts) == expected_combinations
            ),
            "minimum_count": float(counts["count"].min()),
            "median_count": float(counts["count"].median()),
            "maximum_count": float(counts["count"].max()),
        }
    )

    # Standardizziamo i nomi delle prime due colonne:
    # questo rende confrontabili tutte le tabelle dettagliate,
    # anche quando gli attributi di origine hanno nomi diversi.
    #
    # In pratica, ogni tabella finale avrà sempre:
    # first_value, second_value, count, frequency.
    pair_tables[pair_name] = counts.rename(
        columns={
            first_attribute: "first_value",
            second_attribute: "second_value",
        }
    )


# Costruiamo una tabella comparativa compatta a partire dai riepiloghi.
#
# L'ordinamento crescente per mediana mette in alto le coppie che,
# in media, hanno meno esempi per combinazione. Non è una graduatoria
# automatica di qualità: la decisione finale deve considerare insieme
# copertura completa, numerosità minima e uniformità min/max.
pair_summary = (
    pd.DataFrame(pair_summaries)
    .sort_values(
        ["coverage_complete", "median_count"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

print(
    "Riepilogo delle coppie candidate per lo split OOD "
    "composizionale:"
)
display(pair_summary)


# Visualizziamo anche il dettaglio completo per ogni coppia.
#
# L'ordinamento già applicato per count consente di verificare:
# - se esistono combinazioni assenti;
# - se alcune combinazioni sono troppo rare;
# - se la distribuzione è abbastanza uniforme da sostenere
#   una valutazione OOD credibile.
for pair_name, table in pair_tables.items():
    print(f"\nDistribuzione combinazioni per {pair_name}")
    display(table)

## Test automatici dei metadata

I test seguenti verificano numero di esempi, unicità degli identificativi, assenza di valori mancanti, presenza di tutte le colonne attese, validità degli intervalli degli attributi ed esistenza dei file raw indicati nei metadata.

In [ ]:
%%writefile tests/test_metadata.py
from pathlib import Path

import pandas as pd

from src.data.build_metadata import EXPECTED_VARIANTS

# Percorso del file metadata centralizzato
METADATA_PATH = Path("data/processed/metadata.csv")

def load_metadata() -> pd.DataFrame:
    # Tutti i test partono dallo stesso file.
    # Se il file manca, il blocco precedente non è stato eseguito correttamente.
    assert METADATA_PATH.exists(), "Metadata non trovato."
    return pd.read_csv(METADATA_PATH)

def test_metadata_has_expected_number_of_examples() -> None:
    # Il Cartoon Set 10k deve contenere esattamente 10.000 esempi.
    metadata = load_metadata()
    assert len(metadata) == 10_000

def test_metadata_has_unique_image_ids() -> None:
    # Ogni immagine deve comparire una sola volta nel metadata.
    metadata = load_metadata()
    assert metadata["image_id"].nunique() == 10_000
    assert not metadata["image_id"].duplicated().any()

def test_all_expected_columns_are_present() -> None:
    # Le colonne attese includono gli identificativi base
    # più tutti i 18 attributi discreti del dataset.
    metadata = load_metadata()

    expected_columns = {
        "image_id",
        "image_path",
        "csv_path",
        *EXPECTED_VARIANTS.keys(),
    }

    assert set(metadata.columns) == expected_columns

def test_all_attribute_values_are_in_valid_range() -> None:
    # Ogni attributo discreto deve avere valori interi
    # compresi tra 0 e num_variants - 1.
    metadata = load_metadata()

    for attribute, num_variants in EXPECTED_VARIANTS.items():
        assert metadata[attribute].min() >= 0
        assert metadata[attribute].max() == num_variants - 1

def test_all_attributes_cover_all_expected_variants() -> None:
    # Verifichiamo che per ogni attributo siano presenti
    # tutte le varianti dichiarate nella specifica del dataset.
    metadata = load_metadata()

    for attribute, num_variants in EXPECTED_VARIANTS.items():
        observed_values = sorted(metadata[attribute].unique().tolist())
        expected_values = list(range(num_variants))
        assert observed_values == expected_values

def test_all_referenced_files_exist() -> None:
    # I path salvati nel metadata devono puntare a file reali
    # dentro la directory raw del dataset.
    metadata = load_metadata()
    dataset_root = Path("data/raw/cartoonset10k")

    for row in metadata.itertuples(index=False):
        assert (dataset_root / row.image_path).is_file()
        assert (dataset_root / row.csv_path).is_file()

In [ ]:
%cd /content/tiny-avatar-diffusion

# Eseguiamo i test automatici del blocco metadata.
!python -m pytest -q tests/test_metadata.py

## Verifica visiva delle annotazioni

Dopo avere verificato struttura, completezza e validità numerica di `metadata.csv`, controlliamo che alcune annotazioni corrispondano a differenze visive osservabili nelle immagini reali.

Per ciascun attributo selezionato mostriamo campioni reali raggruppati per variante. Questo controllo è descrittivo e serve a validare qualitativamente il collegamento tra i valori categorici estratti dai file CSV e le caratteristiche visive che verranno poi espresse nelle caption testuali.

La visualizzazione non è una valutazione del modello generativo: viene eseguita prima della definizione degli split e prima dell'addestramento.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image


def generate_contact_sheet(
    attribute: str,
    metadata_path: Path,
    dataset_root: Path,
    samples_per_variant: int = 6,
    seed: int = 42,
) -> None:
    # Visualizza campioni reali raggruppati per variante
    metadata = pd.read_csv(metadata_path)
    dataset_root = Path(dataset_root)

    # Ogni contact sheet richiede l'attributo scelto
    # e il percorso relativo dell'immagine nel dataset raw.
    required_columns = {attribute, "image_path"}
    missing_columns = required_columns - set(metadata.columns)

    assert not missing_columns, (
        f"Colonne mancanti in metadata.csv: "
        f"{sorted(missing_columns)}"
    )

    # Rimuoviamo eventuali valori mancanti e ordiniamo le varianti
    # per ottenere una visualizzazione riproducibile.
    variants = sorted(
        metadata[attribute].dropna().unique()
    )

    assert variants, (
        f"Nessuna variante valida trovata per '{attribute}'."
    )

    num_variants = len(variants)

    figure, axes = plt.subplots(
        nrows=num_variants,
        ncols=samples_per_variant,
        figsize=(
            samples_per_variant * 1.7,
            num_variants * 1.8,
        ),
        squeeze=False,
    )

    figure.suptitle(
        f"Dataset reale — varianti di '{attribute}'",
        fontsize=16,
        y=0.995,
    )

    for row_index, variant in enumerate(variants):
        # Selezioniamo esclusivamente immagini reali associate
        # alla variante corrente dell'attributo.
        variant_rows = metadata.loc[
            metadata[attribute] == variant
        ]

        # Evita errori se una variante possiede meno esempi
        # del numero richiesto per riga.
        sample_count = min(
            samples_per_variant,
            len(variant_rows),
        )

        sampled_rows = variant_rows.sample(
            n=sample_count,
            random_state=seed + row_index,
            replace=False,
        )

        for column_index, axis in enumerate(axes[row_index]):
            axis.axis("off")

            # Lasciamo vuote le celle in eccesso quando una variante
            # contiene meno di `samples_per_variant` immagini.
            if column_index >= sample_count:
                continue

            row = sampled_rows.iloc[column_index]

            # I path presenti in metadata.csv sono relativi
            # alla directory radice del dataset raw.
            image_path = dataset_root / row["image_path"]

            assert image_path.is_file(), (
                f"Immagine non trovata: {image_path}"
            )

            with Image.open(image_path) as image:
                axis.imshow(image.convert("RGB"))

            # Il titolo compare una sola volta a sinistra
            # e identifica la variante rappresentata dalla riga.
            if column_index == 0:
                axis.set_title(
                    f"Variante {variant}",
                    loc="left",
                    fontsize=11,
                    fontweight="bold",
                    pad=8,
                )

    plt.tight_layout(rect=(0, 0, 1, 0.98))
    plt.show()


# ---------------------------------------------------------
# Percorsi generati nelle celle precedenti di questo blocco
# ---------------------------------------------------------

metadata_file = Path("data/processed/metadata.csv")
dataset_root = Path("data/raw/cartoonset10k")

assert metadata_file.is_file(), (
    f"Metadata non trovato: {metadata_file}"
)

assert dataset_root.is_dir(), (
    f"Directory dataset non trovata: {dataset_root}"
)


# Attributi direttamente osservabili e rilevanti per le caption
# condizionali e per l'analisi successiva dello split OOD.
attributes_to_inspect = [
    "hair_color",
    "eye_color",
    "face_shape",
]


# Ogni griglia contiene soltanto immagini reali del dataset,
# raggruppate per valore dell'attributo selezionato.
for attribute in attributes_to_inspect:
    print(f"Verifica visiva delle varianti di: {attribute}")

    generate_contact_sheet(
        attribute=attribute,
        metadata_path=metadata_file,
        dataset_root=dataset_root,
        samples_per_variant=6,
        seed=42,
    )

## Audit delle distribuzioni

Prima di definire gli split train, validation, test e out-of-distribution, analizziamo le distribuzioni marginali degli attributi e le frequenze di alcune coppie candidate.

L'obiettivo è identificare una coppia di attributi con tutte le combinazioni osservate e con frequenze sufficienti per costruire uno split OOD significativo senza compromettere la stabilità dell'addestramento.

In [ ]:
%%writefile src/data/audit_metadata.py
import argparse
import json
from pathlib import Path

import pandas as pd

from src.data.build_metadata import EXPECTED_VARIANTS


# ==========================================================
# COPPIE DI ATTRIBUTI DA VALUTARE PER LO SPLIT OOD
# ==========================================================
#
# Ogni coppia viene analizzata per verificare:
# - quante combinazioni di valori sono effettivamente presenti;
# - quanti esempi appartengono a ciascuna combinazione;
# - se esiste una combinazione sufficientemente rappresentativa
#   da poter essere trattenuta per una valutazione OOD composizionale.
#
# La scelta definitiva della coppia e della combinazione OOD
# verrà effettuata nel blocco successivo.
CANDIDATE_PAIRS = [
    ("hair_color", "face_shape"),
    ("eye_color", "chin_length"),
    ("hair_color", "chin_length"),
    ("eye_color", "face_shape"),
    ("glasses", "face_shape"),
]


def build_attribute_counts(
    metadata: pd.DataFrame,
) -> pd.DataFrame:
    # Costruisce le distribuzioni marginali di tutti gli attributi.

    # Il risultato contiene una riga per ogni coppia
    # attributo-valore osservata, con conteggio assoluto e frequenza
    # relativa rispetto al numero totale di esempi.

    # Ogni tabella prodotta nel ciclo descrive un singolo attributo;
    # alla fine le uniamo in un unico DataFrame long-format.
    tables = []

    # EXPECTED_VARIANTS definisce la lista canonica dei 18 attributi.
    # Iterare su questa struttura mantiene l'audit coerente con
    # la validazione usata durante la costruzione dei metadata.
    for attribute in EXPECTED_VARIANTS:
        # Conta quante immagini assumono ciascun valore dell'attributo.
        #
        # sort_index() ordina i valori discreti in senso numerico,
        # così 0, 1, 2, ... compaiono nel loro ordine naturale.
        counts = (
            metadata[attribute]
            .value_counts()
            .sort_index()
            .rename_axis("value")
            .reset_index(name="count")
        )

        # Aggiungiamo il nome dell'attributo come prima colonna,
        # così tutte le distribuzioni possono convivere nello stesso CSV.
        counts.insert(0, "attribute", attribute)

        # La frequenza relativa consente il confronto fra attributi
        # anche se in futuro l'audit verrà applicato a dataset di dimensioni diverse.
        counts["frequency"] = counts["count"] / len(metadata)

        tables.append(counts)

    # ignore_index=True produce un indice consecutivo e non conserva
    # gli indici locali generati per ogni singolo attributo.
    return pd.concat(tables, ignore_index=True)


def build_pair_counts(
    metadata: pd.DataFrame,
) -> pd.DataFrame:
    # Costruisce le distribuzioni congiunte delle coppie candidate OOD.

    # Per ogni coppia vengono restituiti i valori dei due attributi,
    # il numero di esempi che condividono quella combinazione
    # e la frequenza relativa sul dataset.

    # Ogni elemento di tables conterrà le combinazioni osservate
    # per una delle coppie definite in CANDIDATE_PAIRS.
    tables = []

    for first, second in CANDIDATE_PAIRS:
        # groupby conta il numero di righe per ogni combinazione reale
        # dei due attributi, ad esempio hair_color=3 e face_shape=3.
        #
        # observed=True evita di produrre combinazioni non osservate
        # nel caso in cui le colonne vengano trattate come categoriche.
        counts = (
            metadata
            .groupby([first, second], observed=True)
            .size()
            .reset_index(name="count")
        )

        # Etichetta leggibile della coppia, utile per filtrare le righe
        # dopo la concatenazione di tutte le tabelle.
        counts.insert(0, "pair", f"{first}__{second}")

        # Uniformiamo i nomi delle colonne in tutte le tabelle:
        # ciò permette di confrontare coppie diverse nello stesso formato.
        #
        # Esempio:
        # hair_color e face_shape diventano first_value e second_value.
        counts = counts.rename(
            columns={
                first: "first_value",
                second: "second_value",
            }
        )

        # Frequenza relativa della singola combinazione sul totale dataset.
        counts["frequency"] = counts["count"] / len(metadata)

        tables.append(counts)

    return pd.concat(tables, ignore_index=True)


def main() -> None:
    # ======================================================
    # INTERFACCIA A RIGA DI COMANDO
    # ======================================================
    #
    # I valori di default consentono l'esecuzione standard:
    # python -m src.data.audit_metadata
    #
    # Gli argomenti permettono comunque di applicare lo stesso audit
    # a metadata o directory di output differenti.
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--metadata",
        type=Path,
        default=Path("data/processed/metadata.csv"),
        help="Percorso del metadata centralizzato in formato CSV.",
    )
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=Path("data/processed/audit"),
        help="Directory in cui salvare gli artefatti dell'audit.",
    )

    args = parser.parse_args()

    # La cartella viene creata in modo idempotente:
    # nessun errore se esiste già da un'esecuzione precedente.
    args.output_dir.mkdir(parents=True, exist_ok=True)

    # Carichiamo il metadata una sola volta e lo riutilizziamo
    # per tutte le elaborazioni seguenti.
    metadata = pd.read_csv(args.metadata)

    # Produciamo i due artefatti tabellari dell'audit:
    # distribuzioni marginali e distribuzioni congiunte.
    attribute_counts = build_attribute_counts(metadata)
    pair_counts = build_pair_counts(metadata)

    # CSV long-format delle frequenze per valore di attributo.
    # È utile sia per visualizzazioni sia per verifiche automatiche.
    attribute_counts.to_csv(
        args.output_dir / "attribute_counts.csv",
        index=False,
    )

    # CSV delle frequenze per combinazione delle coppie candidate OOD.
    pair_counts.to_csv(
        args.output_dir / "candidate_pair_counts.csv",
        index=False,
    )

    # ======================================================
    # RIEPILOGO JSON
    # ======================================================
    #
    # Il JSON conserva statistiche aggregate leggere, facilmente
    # ispezionabili senza dover rileggere l'intero CSV metadata.
    summary = {
        "num_examples": int(len(metadata)),
        "attributes": {},
        "candidate_pairs": {},
    }

    # Per ciascun attributo salviamo cardinalità attesa e osservata,
    # più la frequenza minima e massima tra i suoi valori.
    for attribute, num_variants in EXPECTED_VARIANTS.items():
        counts = metadata[attribute].value_counts()

        summary["attributes"][attribute] = {
            # Numero di categorie dichiarate dalla specifica del dataset.
            "expected_num_variants": num_variants,

            # Numero di categorie effettivamente presenti nel metadata.
            "observed_num_variants": int(counts.size),

            # Un minimo basso può segnalare una categoria rara,
            # utile da conoscere prima di definire split stratificati.
            "minimum_count": int(counts.min()),

            # Un massimo alto, confrontato con il minimo, aiuta a valutare
            # l'eventuale sbilanciamento della distribuzione.
            "maximum_count": int(counts.max()),
        }

    # Per ogni coppia candidata calcoliamo statistiche sulle sole
    # combinazioni realmente osservate nel dataset.
    for first, second in CANDIDATE_PAIRS:
        pair_name = f"{first}__{second}"

        # Filtriamo la tabella cumulativa per ottenere solo le righe
        # associate alla coppia che stiamo riassumendo.
        selected = pair_counts[
            pair_counts["pair"] == pair_name
        ]

        summary["candidate_pairs"][pair_name] = {
            # Numero di combinazioni effettivamente presenti.
            "observed_combinations": int(len(selected)),

            # Conteggio della combinazione più rara.
            "minimum_count": int(selected["count"].min()),

            # Conteggio della combinazione più frequente.
            "maximum_count": int(selected["count"].max()),

            # Valore centrale: è utile per scegliere una combinazione OOD
            # rappresentativa, evitando valori estremamente rari o comuni.
            "median_count": float(selected["count"].median()),
        }

    # Salviamo un report JSON stabile e leggibile.
    summary_path = args.output_dir / "audit_summary.json"

    with summary_path.open("w", encoding="utf-8") as file:
        json.dump(
            summary,
            file,
            indent=2,
            sort_keys=True,
        )

    # Stampiamo lo stesso riepilogo per avere un feedback immediato
    # quando lo script viene eseguito da notebook o terminale.
    print(json.dumps(summary, indent=2, sort_keys=True))


# Il blocco viene eseguito soltanto quando il file è invocato come modulo
# o script principale, non quando le funzioni vengono importate nei test.
if __name__ == "__main__":
    main()

In [ ]:
%%bash

cd /content/tiny-avatar-diffusion

# Avviamo il modulo di audit sui metadata centralizzati.
#
# Lo script produce:
# - attribute_counts.csv: distribuzioni marginali dei 18 attributi;
# - candidate_pair_counts.csv: frequenze delle combinazioni
#   per le coppie candidate allo split OOD;
# - audit_summary.json: riepilogo compatto delle statistiche principali.
python -m src.data.audit_metadata \
  --metadata data/processed/metadata.csv \
  --output-dir data/processed/audit

In [ ]:
import json
from pathlib import Path

import pandas as pd


# Percorso del riepilogo JSON generato dallo script di audit.
# Questo file contiene statistiche aggregate, più leggere da leggere
# rispetto alle tabelle CSV complete.
summary_path = Path(
    "data/processed/audit/audit_summary.json"
)

# Carichiamo il report JSON in memoria.
with summary_path.open("r", encoding="utf-8") as file:
    audit_summary = json.load(file)

# La sezione "attributes" ha una struttura:
# attributo -> statistiche.
#
# La trasposizione (.T) trasforma i nomi degli attributi
# nell'indice delle righe, rendendo la tabella più leggibile.
attribute_summary = pd.DataFrame(
    audit_summary["attributes"]
).T

# La sezione "candidate_pairs" contiene invece le statistiche
# aggregate per ciascuna coppia di attributi valutata per il test OOD.
#
# Anche qui usiamo .T per ottenere una riga per coppia.
pair_summary = pd.DataFrame(
    audit_summary["candidate_pairs"]
).T

print("Distribuzioni marginali degli attributi:")

# Ordine alfabetico stabile degli attributi:
# rende l'output riproducibile e più semplice da confrontare
# dopo eventuali modifiche alla pipeline.
display(attribute_summary.sort_index())

print("Coppie candidate per lo split OOD:")

# Ordiniamo in ordine decrescente di mediana dei conteggi.
#
# Una mediana alta indica che, in media, le combinazioni della coppia
# hanno più esempi disponibili. La scelta finale deve comunque tenere
# conto anche di copertura completa e di frequenze min/max.
display(
    pair_summary.sort_values(
        "median_count",
        ascending=False,
    )
)

In [ ]:
from pathlib import Path

import pandas as pd


# Carichiamo la tabella dettagliata prodotta dall'audit.
#
# Ogni riga descrive una combinazione osservata per una coppia:
# - pair: nome della coppia, ad esempio hair_color__face_shape;
# - first_value e second_value: valori che identificano la combinazione;
# - count: numero assoluto di immagini con quella combinazione;
# - frequency: frequenza relativa rispetto all'intero dataset.
pair_counts = pd.read_csv(
    Path(
        "data/processed/audit/"
        "candidate_pair_counts.csv"
    )
)

# Analizziamo una coppia alla volta per poter controllare
# visivamente la distribuzione completa delle sue combinazioni.
for pair_name in pair_counts["pair"].unique():
    print(f"\n{pair_name}")

    # Filtriamo esclusivamente le righe associate alla coppia corrente.
    #
    # L'ordinamento crescente mette all'inizio le combinazioni meno
    # frequenti: è utile per individuare rapidamente combinazioni troppo
    # rare, che non sarebbero adatte a essere trattenute per test_ood.
    display(
        pair_counts[
            pair_counts["pair"] == pair_name
        ]
        .sort_values("count")
        .reset_index(drop=True)
    )

Dall'audit preliminare emerge che la coppia `hair_color`–`face_shape` è particolarmente adatta alla successiva definizione dello split OOD: presenta tutte le 70 combinazioni teoricamente possibili, con frequenze comprese tra 115 e 171 esempi per combinazione. Tale distribuzione è relativamente uniforme e consente di trattenere una combinazione specifica per la valutazione composizionale senza introdurre casi estremamente rari.

La scelta definitiva della combinazione OOD e la generazione riproducibile degli split verranno eseguite nel blocco successivo.

## Split train, validation, test e test OOD

In questa sezione costruiamo una suddivisione riproducibile del dataset in quattro sottoinsiemi:

- `train`: utilizzato per l'ottimizzazione dei parametri;
- `validation`: utilizzato per il monitoraggio e la selezione del checkpoint;
- `test_ordinary`: utilizzato per valutare la qualità generativa su combinazioni osservate durante l'addestramento;
- `test_ood`: utilizzato per valutare la generalizzazione composizionale su una combinazione di attributi deliberatamente esclusa dal training.

Lo split OOD viene costruito trattenendo una specifica combinazione tra `hair_color` e `face_shape`. I due valori individuali restano presenti nel training, ma la loro co-occorrenza selezionata non viene mai mostrata al modello.

## Configurazione dello split

Dall'audit preliminare del blocco precedente, la coppia `hair_color`–`face_shape` risulta adatta alla costruzione dello split OOD perché tutte le 70 combinazioni possibili sono presenti con frequenze sufficienti e relativamente uniformi.

La combinazione trattenuta viene scelta come quella con frequenza più vicina alla mediana della coppia candidata; in caso di parità viene applicato un ordinamento lessicografico dei valori. Con questo protocollo viene selezionata la combinazione `hair_color=3` e `face_shape=3`, composta da 144 esempi.

In [ ]:
%%bash

# Creiamo la directory delle configurazioni se non esiste già.
mkdir -p configs

# La configurazione separa le scelte sperimentali dal codice:
# seed, combinazione OOD, proporzioni degli split e attributi di stratificazione.
cat > configs/data_split.yaml <<'EOF'
seed: 42

compositional_ood:
  first_attribute: hair_color
  first_value: 3
  second_attribute: face_shape
  second_value: 3
  expected_count: 144

  # Documentiamo il protocollo usato nell'audit per selezionare
  # una combinazione rappresentativa e non eccessivamente rara.
  selection_protocol:
    candidate_pair: hair_color__face_shape
    criterion: closest_to_median_combination_count
    median_count: 143.5
    tie_break: lexicographically_smallest_attribute_values

ordinary_split:
  train_ratio: 0.8
  validation_ratio: 0.1
  test_ratio: 0.1

  # La distribuzione delle combinazioni osservate viene mantenuta
  # nei tre split ordinari tramite stratificazione congiunta.
  stratify_by:
    - hair_color
    - face_shape
EOF

cat configs/data_split.yaml

## Costruzione degli split

Prima viene isolata l’intera combinazione OOD; tutti gli esempi rimanenti
vengono poi suddivisi in train, validation e test ordinario con
stratificazione sulla coppia `hair_color`–`face_shape`.

La pipeline salva sia il metadata completo con lo split assegnato sia file
separati per ciascun sottoinsieme, un riepilogo JSON e un checksum delle
assegnazioni.

In [ ]:
%%writefile src/data/build_splits.py
import argparse
import hashlib
import json
from pathlib import Path

import pandas as pd
import yaml
from sklearn.model_selection import train_test_split

from src.data.build_metadata import EXPECTED_VARIANTS


VALID_SPLITS = (
    "train",
    "validation",
    "test_ordinary",
    "test_ood",
)


def compute_sha256(path: Path) -> str:
    # Calcola il checksum SHA-256 di un file
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def load_split_config(config_path: Path) -> dict:
    # Carica e valida la configurazione degli split
    with config_path.open("r", encoding="utf-8") as file:
        config = yaml.safe_load(file)

    required_sections = {
        "seed",
        "compositional_ood",
        "ordinary_split",
    }

    missing_sections = required_sections - set(config)

    if missing_sections:
        raise ValueError(
            "Sezioni mancanti nella configurazione: "
            f"{sorted(missing_sections)}"
        )

    ratios = config["ordinary_split"]

    ratio_sum = (
        float(ratios["train_ratio"])
        + float(ratios["validation_ratio"])
        + float(ratios["test_ratio"])
    )

    if abs(ratio_sum - 1.0) > 1e-9:
        raise ValueError(
            "Le proporzioni degli split ordinari "
            f"devono sommare a 1, non {ratio_sum}."
        )

    return config


def validate_metadata(
    metadata: pd.DataFrame,
    config: dict,
) -> None:
    # Verifica che i metadata contengano le colonne necessarie
    ood_config = config["compositional_ood"]

    required_columns = {
        "image_id",
        "image_path",
        ood_config["first_attribute"],
        ood_config["second_attribute"],
    }

    missing_columns = (
        required_columns - set(metadata.columns)
    )

    if missing_columns:
        raise ValueError(
            "Colonne mancanti nel metadata: "
            f"{sorted(missing_columns)}"
        )

    if metadata.empty:
        raise ValueError("Il metadata è vuoto.")

    if metadata["image_id"].isna().any():
        raise ValueError("Sono presenti image_id mancanti.")

    if metadata["image_id"].duplicated().any():
        duplicate_ids = metadata.loc[
            metadata["image_id"].duplicated(),
            "image_id",
        ].tolist()

        raise ValueError(
            "Sono presenti image_id duplicati: "
            f"{duplicate_ids[:5]}"
        )


def make_stratum(
    dataframe: pd.DataFrame,
    columns: list[str],
) -> pd.Series:
    # Costruisce un'etichetta congiunta per la stratificazione
    return dataframe[columns].astype(str).agg(
        "__".join,
        axis=1,
    )


def assign_splits(
    metadata: pd.DataFrame,
    config: dict,
) -> pd.DataFrame:
    # Assegna ogni esempio a train, validation, test ordinario o OOD
    validate_metadata(metadata, config)

    # L'ordinamento canonico evita che l'ordine originario del CSV
    # influenzi le assegnazioni riproducibili.
    metadata = (
        metadata
        .sort_values("image_id")
        .reset_index(drop=True)
        .copy()
    )

    seed = int(config["seed"])
    ood_config = config["compositional_ood"]
    ordinary_config = config["ordinary_split"]

    first_attribute = ood_config["first_attribute"]
    second_attribute = ood_config["second_attribute"]

    first_value = int(ood_config["first_value"])
    second_value = int(ood_config["second_value"])

    expected_ood_count = int(
        ood_config["expected_count"]
    )

    # La combinazione selezionata viene sottratta integralmente
    # dagli split ordinari e costituisce test_ood.
    ood_mask = (
        (metadata[first_attribute] == first_value)
        & (metadata[second_attribute] == second_value)
    )

    ood = metadata.loc[ood_mask].copy()
    ordinary = metadata.loc[~ood_mask].copy()

    if len(ood) != expected_ood_count:
        raise ValueError(
            "Numero inatteso di esempi OOD: "
            f"trovati {len(ood)}, "
            f"attesi {expected_ood_count}."
        )

    if ordinary.empty:
        raise ValueError("Lo split ordinario è vuoto.")

    stratify_columns = list(
        ordinary_config["stratify_by"]
    )

    # Ogni strato corrisponde a una combinazione osservata,
    # ad esempio hair_color=2 e face_shape=5.
    ordinary["_stratum"] = make_stratum(
        ordinary,
        stratify_columns,
    )

    train_ratio = float(
        ordinary_config["train_ratio"]
    )
    validation_ratio = float(
        ordinary_config["validation_ratio"]
    )
    test_ratio = float(
        ordinary_config["test_ratio"]
    )

    temporary_ratio = validation_ratio + test_ratio

    # Primo split: train contro la porzione temporanea
    # che verrà ulteriormente separata in validation e test ordinario.
    train, temporary = train_test_split(
        ordinary,
        train_size=train_ratio,
        test_size=temporary_ratio,
        random_state=seed,
        shuffle=True,
        stratify=ordinary["_stratum"],
    )

    relative_test_ratio = (
        test_ratio / temporary_ratio
    )

    # Secondo split: conserviamo la stessa stratificazione
    # anche nella separazione validation/test ordinario.
    validation, test_ordinary = train_test_split(
        temporary,
        test_size=relative_test_ratio,
        random_state=seed,
        shuffle=True,
        stratify=temporary["_stratum"],
    )

    split_frames = []

    for split_name, frame in (
        ("train", train),
        ("validation", validation),
        ("test_ordinary", test_ordinary),
        ("test_ood", ood),
    ):
        current = frame.drop(
            columns=["_stratum"],
            errors="ignore",
        ).copy()

        # Inseriamo lo split accanto all'identificativo dell'immagine
        # per rendere il file risultante più leggibile.
        current.insert(1, "split", split_name)

        split_frames.append(current)

    result = pd.concat(
        split_frames,
        ignore_index=True,
    )

    result = (
        result
        .sort_values("image_id")
        .reset_index(drop=True)
    )

    # Controlli di integrità: ogni immagine deve comparire una sola volta
    # e deve appartenere a uno degli split previsti.
    if len(result) != len(metadata):
        raise RuntimeError(
            "Il numero finale di esempi non "
            "coincide con il metadata originale."
        )

    if result["image_id"].duplicated().any():
        raise RuntimeError(
            "Gli split contengono image_id duplicati."
        )

    if result["split"].isna().any():
        raise RuntimeError(
            "Alcuni esempi non hanno uno split."
        )

    return result


def validate_assignments(
    assigned: pd.DataFrame,
    config: dict,
) -> None:
    # Verifica la separazione OOD e la coerenza degli split
    found_splits = set(
        assigned["split"].unique()
    )

    if found_splits != set(VALID_SPLITS):
        raise ValueError(
            f"Split trovati: {found_splits}; "
            f"attesi: {set(VALID_SPLITS)}."
        )

    ood_config = config["compositional_ood"]

    first_attribute = ood_config["first_attribute"]
    second_attribute = ood_config["second_attribute"]

    first_value = int(ood_config["first_value"])
    second_value = int(ood_config["second_value"])

    combination_mask = (
        (assigned[first_attribute] == first_value)
        & (assigned[second_attribute] == second_value)
    )

    # La combinazione trattenuta non deve comparire in train,
    # validation o test ordinario.
    combination_splits = set(
        assigned.loc[
            combination_mask,
            "split",
        ].unique()
    )

    if combination_splits != {"test_ood"}:
        raise ValueError(
            "La combinazione OOD compare negli "
            f"split sbagliati: {combination_splits}"
        )

    test_ood = assigned[
        assigned["split"] == "test_ood"
    ]

    if not combination_mask.loc[
        test_ood.index
    ].all():
        raise ValueError(
            "test_ood contiene esempi estranei "
            "alla combinazione selezionata."
        )

    train = assigned[
        assigned["split"] == "train"
    ]

    # I singoli valori trattenuti devono comunque essere osservati
    # nel training: l'obiettivo è testare composizione, non novità assoluta.
    if first_value not in set(
        train[first_attribute]
    ):
        raise ValueError(
            f"{first_attribute}={first_value} "
            "non è presente nel training."
        )

    if second_value not in set(
        train[second_attribute]
    ):
        raise ValueError(
            f"{second_attribute}={second_value} "
            "non è presente nel training."
        )


def build_summary(
    assigned: pd.DataFrame,
    config: dict,
    assignments_path: Path,
) -> dict:
    # Costruisce il riepilogo JSON degli split prodotti
    ood_config = config["compositional_ood"]

    first_attribute = ood_config["first_attribute"]
    second_attribute = ood_config["second_attribute"]

    first_value = int(ood_config["first_value"])
    second_value = int(ood_config["second_value"])

    counts = (
        assigned["split"]
        .value_counts()
        .reindex(VALID_SPLITS)
    )

    train = assigned[
        assigned["split"] == "train"
    ]

    attribute_coverage = {}

    # Verifichiamo che il trattenimento della combinazione OOD
    # non abbia eliminato alcun valore individuale dal training.
    for attribute in EXPECTED_VARIANTS:
        expected_values = list(
            range(EXPECTED_VARIANTS[attribute])
        )

        observed_values = sorted(
            int(value)
            for value in train[attribute].unique()
        )

        attribute_coverage[attribute] = {
            "expected_values": expected_values,
            "train_values": observed_values,
            "complete": (
                expected_values == observed_values
            ),
        }

    summary = {
        "seed": int(config["seed"]),
        "num_examples": int(len(assigned)),
        "counts": {
            split_name: int(counts[split_name])
            for split_name in VALID_SPLITS
        },
        "compositional_ood": {
            "first_attribute": first_attribute,
            "first_value": first_value,
            "second_attribute": second_attribute,
            "second_value": second_value,
            "count": int(counts["test_ood"]),
        },
        "selected_values_in_training": {
            f"{first_attribute}={first_value}": int(
                (
                    train[first_attribute]
                    == first_value
                ).sum()
            ),
            f"{second_attribute}={second_value}": int(
                (
                    train[second_attribute]
                    == second_value
                ).sum()
            ),
        },
        "training_attribute_coverage": (
            attribute_coverage
        ),
        # Il checksum identifica in modo univoco le assegnazioni
        # e consente di verificare che non siano cambiate.
        "assignments_sha256": compute_sha256(
            assignments_path
        ),
    }

    return summary


def save_splits(
    assigned: pd.DataFrame,
    output_dir: Path,
    config: dict,
) -> dict:
    # Salva gli artefatti degli split e il relativo riepilogo
    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    full_path = (
        output_dir / "metadata_with_splits.csv"
    )

    assignments_path = (
        output_dir / "split_assignments.csv"
    )

    # File completo: utile per tutte le fasi successive della pipeline.
    assigned.to_csv(
        full_path,
        index=False,
    )

    # File minimale: utile per confronti, versionamento e checksum.
    assigned[
        ["image_id", "split"]
    ].to_csv(
        assignments_path,
        index=False,
    )

    # Salviamo anche una vista separata per ogni split.
    for split_name in VALID_SPLITS:
        split_path = (
            output_dir / f"{split_name}.csv"
        )

        assigned[
            assigned["split"] == split_name
        ].to_csv(
            split_path,
            index=False,
        )

    summary = build_summary(
        assigned=assigned,
        config=config,
        assignments_path=assignments_path,
    )

    summary_path = (
        output_dir / "split_summary.json"
    )

    with summary_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            summary,
            file,
            indent=2,
            sort_keys=True,
        )

    return summary


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--metadata",
        type=Path,
        default=Path(
            "data/processed/metadata.csv"
        ),
    )

    parser.add_argument(
        "--config",
        type=Path,
        default=Path(
            "configs/data_split.yaml"
        ),
    )

    parser.add_argument(
        "--output-dir",
        type=Path,
        default=Path(
            "data/processed/splits"
        ),
    )

    return parser.parse_args()


def main() -> None:
    args = parse_args()

    metadata = pd.read_csv(args.metadata)
    config = load_split_config(args.config)

    assigned = assign_splits(
        metadata=metadata,
        config=config,
    )

    validate_assignments(
        assigned=assigned,
        config=config,
    )

    summary = save_splits(
        assigned=assigned,
        output_dir=args.output_dir,
        config=config,
    )

    print(
        json.dumps(
            summary,
            indent=2,
            sort_keys=True,
        )
    )


if __name__ == "__main__":
    main()

In [ ]:
%%bash

# Eseguiamo la costruzione effettiva degli split a partire
# dai metadata centralizzati e dalla configurazione appena salvata.
python -m src.data.build_splits \
  --metadata data/processed/metadata.csv \
  --config configs/data_split.yaml \
  --output-dir data/processed/splits

## Verifica degli artefatti prodotti

Verifichiamo ora le dimensioni dei quattro split, l’unicità della
combinazione contenuta in `test_ood`, la presenza separata dei due valori
nel training e il checksum delle assegnazioni finali.

In [ ]:
import json
from pathlib import Path

import pandas as pd


splits_root = Path("data/processed/splits")

# Il file completo contiene metadata e split assegnato a ogni immagine.
assigned = pd.read_csv(
    splits_root / "metadata_with_splits.csv"
)

# Il summary conserva conteggi, copertura del training e checksum.
with (splits_root / "split_summary.json").open(
    "r",
    encoding="utf-8",
) as file:
    split_summary = json.load(file)

print("Conteggi:")
print(
    assigned["split"]
    .value_counts()
    .reindex(
        [
            "train",
            "validation",
            "test_ordinary",
            "test_ood",
        ]
    )
)

print("\nCombinazioni presenti in test_ood:")

# test_ood deve contenere esclusivamente hair_color=3 e face_shape=3.
display(
    assigned.loc[
        assigned["split"] == "test_ood",
        ["hair_color", "face_shape"],
    ]
    .value_counts()
    .reset_index(name="count")
)

train = assigned[
    assigned["split"] == "train"
]

print("\nPresenza separata nel training:")
print(
    "hair_color=3:",
    int((train["hair_color"] == 3).sum()),
)
print(
    "face_shape=3:",
    int((train["face_shape"] == 3).sum()),
)

print("\nSHA-256 assegnazioni:")
print(split_summary["assignments_sha256"])

In [ ]:
%%writefile tests/test_splits.py
from pathlib import Path

import pandas as pd
import pandas.testing as pdt

from src.data.build_splits import (
    VALID_SPLITS,
    assign_splits,
    load_split_config,
    validate_assignments,
)


METADATA_PATH = Path("data/processed/metadata.csv")
CONFIG_PATH = Path("configs/data_split.yaml")
SPLITS_PATH = Path(
    "data/processed/splits/metadata_with_splits.csv"
)


def load_inputs() -> tuple[pd.DataFrame, dict, pd.DataFrame]:
    # Carica gli input condivisi dai test
    assert METADATA_PATH.exists()
    assert CONFIG_PATH.exists()
    assert SPLITS_PATH.exists()

    metadata = pd.read_csv(METADATA_PATH)
    config = load_split_config(CONFIG_PATH)
    assigned = pd.read_csv(SPLITS_PATH)

    return metadata, config, assigned


def test_total_number_of_examples() -> None:
    # Lo split deve contenere tutti i 10.000 esempi
    metadata, _, assigned = load_inputs()

    assert len(metadata) == 10_000
    assert len(assigned) == 10_000


def test_unique_assignments() -> None:
    # Ogni immagine deve essere assegnata una sola volta
    _, _, assigned = load_inputs()

    assert assigned["image_id"].nunique() == 10_000
    assert not assigned["image_id"].duplicated().any()


def test_expected_split_names() -> None:
    # Devono essere presenti esattamente i quattro split previsti
    _, _, assigned = load_inputs()

    assert set(assigned["split"].unique()) == set(
        VALID_SPLITS
    )


def test_expected_split_sizes() -> None:
    # Le cardinalità attese rendono lo split facilmente verificabile
    _, _, assigned = load_inputs()

    counts = assigned["split"].value_counts()

    assert int(counts["train"]) == 7_884
    assert int(counts["validation"]) == 986
    assert int(counts["test_ordinary"]) == 986
    assert int(counts["test_ood"]) == 144


def test_ood_combination_is_held_out() -> None:
    # La combinazione OOD non deve comparire negli split ordinari
    _, config, assigned = load_inputs()

    validate_assignments(
        assigned=assigned,
        config=config,
    )

    ood_mask = (
        (assigned["hair_color"] == 3)
        & (assigned["face_shape"] == 3)
    )

    assert (
        assigned.loc[ood_mask, "split"]
        == "test_ood"
    ).all()

    ordinary_mask = assigned["split"] != "test_ood"

    assert not (
        (
            assigned.loc[
                ordinary_mask,
                "hair_color",
            ]
            == 3
        )
        & (
            assigned.loc[
                ordinary_mask,
                "face_shape",
            ]
            == 3
        )
    ).any()


def test_ood_contains_only_selected_combination() -> None:
    # test_ood non deve includere esempi non appartenenti alla coppia
    _, _, assigned = load_inputs()

    test_ood = assigned[
        assigned["split"] == "test_ood"
    ]

    assert (test_ood["hair_color"] == 3).all()
    assert (test_ood["face_shape"] == 3).all()


def test_individual_values_remain_in_training() -> None:
    # I valori singoli devono restare osservabili durante il training
    _, _, assigned = load_inputs()

    train = assigned[
        assigned["split"] == "train"
    ]

    assert (train["hair_color"] == 3).any()
    assert (train["face_shape"] == 3).any()


def test_split_is_deterministic() -> None:
    # Stesso input e stesso seed devono produrre lo stesso split
    metadata, config, _ = load_inputs()

    first_assignment = assign_splits(
        metadata=metadata,
        config=config,
    )

    second_assignment = assign_splits(
        metadata=metadata,
        config=config,
    )

    pdt.assert_frame_equal(
        first_assignment,
        second_assignment,
        check_like=False,
    )

In [ ]:
%cd /content/tiny-avatar-diffusion

!python -m pytest -q tests/test_metadata.py tests/test_splits.py

In [ ]:
import pandas as pd

# Il summary registra per ogni attributo i valori attesi
# e quelli effettivamente presenti nel solo training set.
coverage = pd.DataFrame(
    split_summary["training_attribute_coverage"]
).T

display(coverage)

# Lo split OOD è valido solo se tutti gli attributi mantengono
# copertura completa dei propri valori nel training.
assert coverage["complete"].all(), (
    "Alcuni attributi non hanno copertura completa "
    "nel training."
)

print(
    "Tutti gli attributi mantengono copertura completa "
    "nel training."
)

Gli split sono stati costruiti in modo riproducibile con seed 42.
La combinazione `hair_color=3` e `face_shape=3` è presente esclusivamente
in `test_ood`, mentre i due valori restano individualmente osservabili nel
training. Le verifiche automatiche confermano completezza, assenza di
sovrapposizioni, coerenza OOD, copertura degli attributi e determinismo
delle assegnazioni.

## Captioning e tokenizer

In questo blocco convertiamo gli attributi discreti degli avatar in caption
testuali deterministiche e costruiamo il tokenizer che verrà usato dal
modello di text conditioning.

Le caption sono generate direttamente dai metadata: la stessa combinazione
di attributi produce sempre la stessa descrizione. Il tokenizer viene
addestrato soltanto sulle caption di training, così validation e test non
contribuiscono alla definizione del vocabolario.

## Configurazione testuale

La configurazione definisce il template delle caption, l’ordine degli
attributi descritti e i token speciali del tokenizer. L’ordine è fisso per
rendere le caption coerenti e riproducibili tra tutte le esecuzioni.

In [ ]:
%%bash

cd /content/tiny-avatar-diffusion

# La configurazione mantiene separati dal codice sia il template
# delle caption sia i dettagli del tokenizer.
cat > configs/text_conditioning.yaml <<'EOF'
caption:
  prefix: "a cartoon avatar with"

  # Gli attributi sono elencati in ordine fisso:
  # la stessa riga di metadata produrrà sempre la stessa caption.
  attributes:
    - key: face_shape
      phrase: "face shape variant"
    - key: face_color
      phrase: "face color variant"
    - key: hair
      phrase: "hairstyle variant"
    - key: hair_color
      phrase: "hair color variant"
    - key: eye_color
      phrase: "eye color variant"
    - key: chin_length
      phrase: "chin length variant"
    - key: facial_hair
      phrase: "facial hair variant"
    - key: glasses
      phrase: "glasses variant"

tokenizer:
  model: word_level
  pre_tokenizer: whitespace
  min_frequency: 1

  # Token riservati per parole sconosciute, padding e delimitazione
  # delle sequenze in ingresso al text encoder.
  special_tokens:
    unk: "[UNK]"
    pad: "[PAD]"
    sos: "[SOS]"
    eos: "[EOS]"
EOF

cat configs/text_conditioning.yaml

## Generazione delle caption

La pipeline legge `metadata_with_splits.csv`, controlla che gli attributi
richiesti siano disponibili e genera una caption per ogni immagine.

Gli artefatti salvati mantengono i quattro split già definiti: il file
completo con caption, un CSV per ciascuno split e un riepilogo JSON con
statistiche e checksum.

In [ ]:
%%writefile src/data/build_captions.py
import argparse
import hashlib
import json
from pathlib import Path
from typing import Mapping

import pandas as pd
import yaml

from src.data.build_metadata import EXPECTED_VARIANTS
from src.data.build_splits import VALID_SPLITS


def compute_sha256(path: Path) -> str:
    # Calcola il checksum SHA-256 di un file
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def load_text_config(config_path: Path) -> dict:
    # Carica e controlla la configurazione delle caption
    with config_path.open("r", encoding="utf-8") as file:
        config = yaml.safe_load(file)

    if "caption" not in config or "tokenizer" not in config:
        raise ValueError(
            "Sezioni 'caption' o 'tokenizer' mancanti."
        )

    caption_config = config["caption"]

    if not caption_config.get("prefix"):
        raise ValueError("Prefix della caption mancante.")

    attributes = caption_config.get("attributes", [])

    if not attributes:
        raise ValueError("Nessun attributo configurato.")

    keys = [item["key"] for item in attributes]

    if len(keys) != len(set(keys)):
        raise ValueError(
            "Attributi duplicati nella configurazione."
        )

    unknown_attributes = sorted(
        set(keys) - set(EXPECTED_VARIANTS)
    )

    if unknown_attributes:
        raise ValueError(
            f"Attributi sconosciuti: {unknown_attributes}"
        )

    return config


def generate_caption(
    row: Mapping,
    caption_config: dict,
) -> str:
    # Genera una caption deterministica a partire dagli attributi
    prefix = caption_config["prefix"].strip()
    fragments = []

    for item in caption_config["attributes"]:
        attribute = item["key"]
        phrase = item["phrase"].strip()

        if attribute not in row:
            raise KeyError(
                f"Attributo mancante: {attribute}"
            )

        value = int(row[attribute])

        if not (
            0 <= value < EXPECTED_VARIANTS[attribute]
        ):
            raise ValueError(
                f"Valore {value} non valido per {attribute}."
            )

        # Il valore numerico mantiene il collegamento esplicito
        # tra testo e categoria originale del metadata.
        fragments.append(f"{phrase} {value}")

    # L'ultima congiunzione rende la caption grammaticalmente coerente
    # senza introdurre variazioni casuali nel template.
    if len(fragments) == 1:
        description = fragments[0]
    else:
        description = (
            ", ".join(fragments[:-1])
            + ", and "
            + fragments[-1]
        )

    return f"{prefix} {description}"


def build_captions(
    assigned: pd.DataFrame,
    config: dict,
) -> pd.DataFrame:
    # Aggiunge una caption a ogni riga del metadata con split
    required_columns = {
        "image_id",
        "image_path",
        "split",
    }

    caption_attributes = {
        item["key"]
        for item in config["caption"]["attributes"]
    }

    required_columns |= caption_attributes

    missing_columns = (
        required_columns - set(assigned.columns)
    )

    if missing_columns:
        raise ValueError(
            f"Colonne mancanti: {sorted(missing_columns)}"
        )

    if assigned["image_id"].duplicated().any():
        raise ValueError("Sono presenti image_id duplicati.")

    # L'ordinamento per image_id mantiene l'output stabile
    # anche se il file di input viene riordinato.
    output = (
        assigned
        .sort_values("image_id")
        .reset_index(drop=True)
        .copy()
    )

    output["caption"] = [
        generate_caption(
            row=row,
            caption_config=config["caption"],
        )
        for row in output.to_dict(orient="records")
    ]

    if (
        output["caption"].isna().any()
        or output["caption"].str.strip().eq("").any()
    ):
        raise RuntimeError(
            "Generate caption mancanti o vuote."
        )

    return output


def save_captions(
    captioned: pd.DataFrame,
    output_dir: Path,
    config: dict,
) -> dict:
    # Salva caption complete, CSV per split e riepilogo JSON
    output_dir.mkdir(parents=True, exist_ok=True)

    full_path = output_dir / "metadata_with_captions.csv"

    # File centrale da usare in seguito per creare dataset e dataloader.
    captioned.to_csv(full_path, index=False)

    split_counts = {}

    for split_name in VALID_SPLITS:
        current = captioned[
            captioned["split"] == split_name
        ].copy()

        current.to_csv(
            output_dir / f"{split_name}.csv",
            index=False,
        )

        split_counts[split_name] = int(len(current))

    # La lunghezza whitespace è un controllo semplice sul template.
    # In questo progetto ogni caption ha struttura e lunghezza costanti.
    caption_lengths = (
        captioned["caption"]
        .str.split()
        .str.len()
    )

    summary = {
        "num_examples": int(len(captioned)),
        "split_counts": split_counts,
        "caption_attributes": [
            item["key"]
            for item in config["caption"]["attributes"]
        ],
        "caption_prefix": config["caption"]["prefix"],
        "minimum_whitespace_length": int(
            caption_lengths.min()
        ),
        "maximum_whitespace_length": int(
            caption_lengths.max()
        ),
        "num_unique_captions": int(
            captioned["caption"].nunique()
        ),
        "captions_sha256": compute_sha256(full_path),
    }

    summary_path = output_dir / "caption_summary.json"

    with summary_path.open("w", encoding="utf-8") as file:
        json.dump(
            summary,
            file,
            indent=2,
            sort_keys=True,
        )

    return summary


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--splits",
        type=Path,
        default=Path(
            "data/processed/splits/"
            "metadata_with_splits.csv"
        ),
    )
    parser.add_argument(
        "--config",
        type=Path,
        default=Path(
            "configs/text_conditioning.yaml"
        ),
    )
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=Path("data/processed/captions"),
    )

    return parser.parse_args()


def main() -> None:
    args = parse_args()

    assigned = pd.read_csv(args.splits)
    config = load_text_config(args.config)

    captioned = build_captions(
        assigned=assigned,
        config=config,
    )

    summary = save_captions(
        captioned=captioned,
        output_dir=args.output_dir,
        config=config,
    )

    print(
        json.dumps(
            summary,
            indent=2,
            sort_keys=True,
        )
    )


if __name__ == "__main__":
    main()

In [ ]:
%%bash

cd /content/tiny-avatar-diffusion

# Generiamo le caption preservando le assegnazioni degli split
# create nel blocco precedente.
python -m src.data.build_captions \
  --splits data/processed/splits/metadata_with_splits.csv \
  --config configs/text_conditioning.yaml \
  --output-dir data/processed/captions

## Verifica delle caption

Controlliamo il riepilogo prodotto, la coerenza delle cardinalità tra split
e alcuni esempi di descrizione. Le caption devono essere non vuote,
deterministiche e allineate agli attributi presenti nei metadata.

In [ ]:
import json
from pathlib import Path

import pandas as pd

captions_root = Path("data/processed/captions")

captioned = pd.read_csv(
    captions_root / "metadata_with_captions.csv"
)

with (captions_root / "caption_summary.json").open(
    "r",
    encoding="utf-8",
) as file:
    caption_summary = json.load(file)

print(json.dumps(
    caption_summary,
    indent=2,
    ensure_ascii=False,
))

# Mostriamo gli attributi che guidano la caption insieme
# al testo generato: è un controllo diretto sulla corrispondenza.
display(
    captioned[
        [
            "image_id",
            "split",
            "hair_color",
            "face_shape",
            "caption",
        ]
    ].sample(
        n=10,
        random_state=42,
    )
)

## Addestramento del tokenizer

Il tokenizer word-level viene addestrato usando esclusivamente le caption dello split train. I token speciali vengono inseriti nel vocabolario esplicitamente, mentre le sequenze testuali effettivamente usate nella pipeline corrispondono alla tokenizzazione diretta delle caption; in seguito risultano tutte della stessa lunghezza e quindi compatibili con batching e padding uniforme.

In [ ]:
%%writefile src/data/build_tokenizer.py

import argparse
import hashlib
import json
from pathlib import Path

import pandas as pd
import yaml
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import TemplateProcessing
from tokenizers.trainers import WordLevelTrainer


def compute_sha256(path: Path) -> str:
    # Calcola lo SHA-256 di un file binario
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def load_text_config(config_path: Path) -> dict:
    # Carica e valida la configurazione per caption e tokenizer
    with Path(config_path).open(
        "r",
        encoding="utf-8",
    ) as file:
        config = yaml.safe_load(file)

    required_sections = {"caption", "tokenizer"}

    missing_sections = required_sections - set(config)
    if missing_sections:
        raise ValueError(
            "Sezioni mancanti nella configurazione: "
            f"{sorted(missing_sections)}"
        )

    tokenizer_config = config["tokenizer"]

    required_tokenizer_keys = {
        "model",
        "pre_tokenizer",
        "min_frequency",
        "special_tokens",
    }

    missing_tokenizer_keys = (
        required_tokenizer_keys
        - set(tokenizer_config)
    )

    if missing_tokenizer_keys:
        raise ValueError(
            "Parametri tokenizer mancanti: "
            f"{sorted(missing_tokenizer_keys)}"
        )

    if tokenizer_config["model"] != "word_level":
        raise ValueError(
            "È supportato solo tokenizer.model='word_level'."
        )

    if tokenizer_config["pre_tokenizer"] != "whitespace":
        raise ValueError(
            "È supportato solo "
            "tokenizer.pre_tokenizer='whitespace'."
        )

    if int(tokenizer_config["min_frequency"]) < 1:
        raise ValueError(
            "tokenizer.min_frequency deve essere almeno 1."
        )

    special_tokens = tokenizer_config["special_tokens"]

    required_special_tokens = {
        "unk",
        "pad",
        "sos",
        "eos",
    }

    missing_special_tokens = (
        required_special_tokens - set(special_tokens)
    )

    if missing_special_tokens:
        raise ValueError(
            "Token speciali mancanti: "
            f"{sorted(missing_special_tokens)}"
        )

    token_values = list(special_tokens.values())

    if len(token_values) != len(set(token_values)):
        raise ValueError(
            "I token speciali devono essere tutti distinti."
        )

    return config


def build_tokenizer(
    train_captions: list[str],
    config: dict,
) -> tuple[Tokenizer, dict]:
    # Addestra il tokenizer esclusivamente sulle caption train,
    # poi applica SOS, EOS, truncation e padding.
    if not train_captions:
        raise ValueError(
            "Nessuna caption di training disponibile."
        )

    tokenizer_config = config["tokenizer"]
    special_tokens = tokenizer_config["special_tokens"]

    ordered_special_tokens = [
        special_tokens["unk"],
        special_tokens["pad"],
        special_tokens["sos"],
        special_tokens["eos"],
    ]

    tokenizer = Tokenizer(
        WordLevel(
            unk_token=special_tokens["unk"]
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = WordLevelTrainer(
        min_frequency=int(
            tokenizer_config["min_frequency"]
        ),
        special_tokens=ordered_special_tokens,
        show_progress=True,
    )

    # Il vocabolario viene costruito solo dal training split:
    # validation, test ordinary e OOD non contribuiscono.
    tokenizer.train_from_iterator(
        train_captions,
        trainer=trainer,
        length=len(train_captions),
    )

    special_token_ids = {
        name: tokenizer.token_to_id(token)
        for name, token in special_tokens.items()
    }

    missing_ids = [
        name
        for name, token_id in special_token_ids.items()
        if token_id is None
    ]

    if missing_ids:
        raise RuntimeError(
            "Token speciali senza ID: "
            f"{missing_ids}"
        )

    # Calcoliamo la massima lunghezza della sola caption,
    # prima di aggiungere i delimitatori SOS ed EOS.
    raw_lengths = [
        len(
            tokenizer.encode(
                caption,
                add_special_tokens=False,
            ).ids
        )
        for caption in train_captions
    ]

    max_train_length_without_boundaries = max(
        raw_lengths
    )

    # Ogni sequenza avrà: [SOS] + caption + [EOS].
    maximum_sequence_length = (
        max_train_length_without_boundaries + 2
    )

    # Aggiunge automaticamente SOS e EOS a ogni caption.
    tokenizer.post_processor = TemplateProcessing(
        single=(
            f"{special_tokens['sos']} "
            f"$A "
            f"{special_tokens['eos']}"
        ),
        special_tokens=[
            (
                special_tokens["sos"],
                special_token_ids["sos"],
            ),
            (
                special_tokens["eos"],
                special_token_ids["eos"],
            ),
        ],
    )

    # Impone una lunghezza fissa per i batch del Transformer.
    tokenizer.enable_truncation(
        max_length=maximum_sequence_length
    )

    tokenizer.enable_padding(
        length=maximum_sequence_length,
        pad_id=special_token_ids["pad"],
        pad_token=special_tokens["pad"],
    )

    return tokenizer, {
        "max_train_length_without_boundaries": (
            max_train_length_without_boundaries
        ),
        "maximum_sequence_length": (
            maximum_sequence_length
        ),
        "special_token_ids": special_token_ids,
    }


def build_summary(
    tokenizer: Tokenizer,
    captioned: pd.DataFrame,
    train_captions: list[str],
    special_tokens: dict,
    tokenizer_info: dict,
) -> dict:
    # Crea un summary verificabile degli artefatti tokenizer
    encoded_train = tokenizer.encode_batch(
        train_captions
    )

    encoded_lengths = {
        len(encoding.ids)
        for encoding in encoded_train
    }

    if encoded_lengths != {
        tokenizer_info["maximum_sequence_length"]
    }:
        raise RuntimeError(
            "Le sequenze train non hanno lunghezza fissa. "
            f"Osservate: {sorted(encoded_lengths)}"
        )

    unknown_token_id = tokenizer_info[
        "special_token_ids"
    ]["unk"]

    unknown_token_counts = {}

    for split_name, split_frame in captioned.groupby(
        "split"
    ):
        encodings = tokenizer.encode_batch(
            split_frame["caption"]
            .astype(str)
            .tolist()
        )

        unknown_token_counts[split_name] = sum(
            encoding.ids.count(unknown_token_id)
            for encoding in encodings
        )

    return {
        "tokenizer_model": "WordLevel",
        "pre_tokenizer": "Whitespace",
        "num_train_captions": int(
            len(train_captions)
        ),
        "training_examples": int(
            len(train_captions)
        ),
        "vocabulary_size": int(
            tokenizer.get_vocab_size()
        ),
        "special_tokens": special_tokens,
        "special_token_ids": {
            name: int(token_id)
            for name, token_id in tokenizer_info[
                "special_token_ids"
            ].items()
        },
        "max_train_length_without_boundaries": int(
            tokenizer_info[
                "max_train_length_without_boundaries"
            ]
        ),
        "maximum_sequence_length": int(
            tokenizer_info["maximum_sequence_length"]
        ),
        # Chiave mantenuta per compatibilità con i test
        # e il DataLoader già costruiti nel progetto.
        "sequence_length_with_boundaries": int(
            tokenizer_info["maximum_sequence_length"]
        ),
        "unknown_token_counts": {
            split_name: int(count)
            for split_name, count
            in unknown_token_counts.items()
        },
    }


def main() -> None:
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--captions",
        type=Path,
        default=Path(
            "data/processed/captions/"
            "metadata_with_captions.csv"
        ),
    )

    parser.add_argument(
        "--config",
        type=Path,
        default=Path(
            "configs/text_conditioning.yaml"
        ),
    )

    parser.add_argument(
        "--output-dir",
        type=Path,
        default=Path(
            "data/processed/tokenizer"
        ),
    )

    args = parser.parse_args()

    if not args.captions.is_file():
        raise FileNotFoundError(
            f"File caption non trovato: {args.captions}"
        )

    captioned = pd.read_csv(args.captions)

    required_columns = {
        "image_id",
        "split",
        "caption",
    }

    missing_columns = (
        required_columns - set(captioned.columns)
    )

    if missing_columns:
        raise ValueError(
            "Colonne mancanti nel file caption: "
            f"{sorted(missing_columns)}"
        )

    config = load_text_config(args.config)

    train_captions = (
        captioned.loc[
            captioned["split"] == "train",
            "caption",
        ]
        .astype(str)
        .tolist()
    )

    tokenizer, tokenizer_info = build_tokenizer(
        train_captions=train_captions,
        config=config,
    )

    args.output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    tokenizer_path = (
        args.output_dir / "tokenizer.json"
    )

    # Il salvataggio avviene soltanto dopo aver configurato
    # post-processing, truncation e padding.
    tokenizer.save(str(tokenizer_path))

    special_tokens = config["tokenizer"][
        "special_tokens"
    ]

    summary = build_summary(
        tokenizer=tokenizer,
        captioned=captioned,
        train_captions=train_captions,
        special_tokens=special_tokens,
        tokenizer_info=tokenizer_info,
    )

    summary["tokenizer_sha256"] = compute_sha256(
        tokenizer_path
    )

    summary_path = (
        args.output_dir / "tokenizer_summary.json"
    )

    with summary_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            summary,
            file,
            indent=2,
            sort_keys=True,
        )

    print(
        json.dumps(
            summary,
            indent=2,
            sort_keys=True,
        )
    )


if __name__ == "__main__":
    main()

In [ ]:
%cd /content/tiny-avatar-diffusion

!python -m src.data.build_tokenizer \
  --captions data/processed/captions/metadata_with_captions.csv \
  --config configs/text_conditioning.yaml \
  --output-dir data/processed/tokenizer

## Verifica di encoding e decoding

Verifichiamo che una caption venga trasformata in token IDs, delimitata con
i token speciali e decodificata correttamente. Questo controllo è utile
prima di collegare il tokenizer al dataset PyTorch e al text encoder.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from tokenizers import Tokenizer

tokenizer_root = Path("data/processed/tokenizer")

tokenizer = Tokenizer.from_file(
    str(tokenizer_root / "tokenizer.json")
)

with (tokenizer_root / "tokenizer_summary.json").open(
    "r",
    encoding="utf-8",
) as file:
    tokenizer_summary = json.load(file)

print(json.dumps(
    tokenizer_summary,
    indent=2,
    ensure_ascii=False,
))

# Prendiamo una caption qualunque per controllare direttamente
# come il tokenizer separa testo, punteggiatura e valori numerici.
example_caption = captioned.iloc[0]["caption"]

encoding = tokenizer.encode(example_caption)

print("Caption:")
print(example_caption)

print("\nToken:")
print(encoding.tokens)

print("\nToken ID:")
print(encoding.ids)

# Senza padding ogni posizione è valida, quindi la mask contiene solo 1.
# Gli 0 compariranno dopo il padding nel collate function.
print("\nAttention mask:")
print(encoding.attention_mask)

print("\nLunghezza:")
print(len(encoding.ids))

In [ ]:
# Verifichiamo tutte le caption dello split compositional OOD.
# Devono essere codificabili senza [UNK], perché i singoli valori
# hair_color=3 e face_shape=3 restano presenti nel training.
ood_captions = captioned.loc[
    captioned["split"] == "test_ood",
    "caption",
].tolist()

ood_encodings = tokenizer.encode_batch(ood_captions)

has_unk = any(
    "[UNK]" in encoding.tokens
    for encoding in ood_encodings
)

print("Numero caption OOD:", len(ood_captions))
print("Presenza di [UNK] nello split OOD:", has_unk)

assert not has_unk, (
    "Almeno una caption OOD contiene [UNK]."
)

In [ ]:
%%writefile tests/test_captions_and_tokenizer.py
from pathlib import Path

import pandas as pd
import re
from tokenizers import Tokenizer

from src.data.build_captions import (
    generate_caption,
    load_text_config,
)
from src.data.build_metadata import EXPECTED_VARIANTS
from src.data.build_splits import VALID_SPLITS


CONFIG_PATH = Path("configs/text_conditioning.yaml")
CAPTIONS_PATH = Path(
    "data/processed/captions/metadata_with_captions.csv"
)
TOKENIZER_PATH = Path(
    "data/processed/tokenizer/tokenizer.json"
)


def load_inputs() -> tuple[dict, pd.DataFrame, Tokenizer]:
    # Carica gli artefatti creati nel blocco captioning
    assert CONFIG_PATH.exists()
    assert CAPTIONS_PATH.exists()
    assert TOKENIZER_PATH.exists()

    config = load_text_config(CONFIG_PATH)
    captioned = pd.read_csv(CAPTIONS_PATH)
    tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

    return config, captioned, tokenizer


def test_caption_dataset_has_expected_size() -> None:
    # Le caption devono coprire tutti gli esempi del dataset
    _, captioned, _ = load_inputs()

    assert len(captioned) == 10_000
    assert captioned["image_id"].nunique() == 10_000


def test_caption_dataset_preserves_splits() -> None:
    # Le assegnazioni split devono restare invariate
    _, captioned, _ = load_inputs()

    assert set(captioned["split"].unique()) == set(
        VALID_SPLITS
    )


def test_captions_are_not_empty() -> None:
    # Ogni esempio deve avere una caption testuale non vuota
    _, captioned, _ = load_inputs()

    assert captioned["caption"].notna().all()
    assert captioned["caption"].str.strip().ne("").all()


def test_caption_generation_is_deterministic() -> None:
    # La stessa riga di metadata deve generare la stessa caption
    config, captioned, _ = load_inputs()

    row = captioned.iloc[0].to_dict()

    first_caption = generate_caption(
        row=row,
        caption_config=config["caption"],
    )
    second_caption = generate_caption(
        row=row,
        caption_config=config["caption"],
    )

    assert first_caption == second_caption
    assert first_caption == row["caption"]


def test_caption_values_are_in_expected_ranges() -> None:
    # Le caption si basano su valori discreti validi
    _, captioned, _ = load_inputs()

    for attribute, num_variants in EXPECTED_VARIANTS.items():
        assert captioned[attribute].min() >= 0
        assert captioned[attribute].max() < num_variants


def test_tokenizer_contains_special_tokens() -> None:
    # Tutti i token speciali devono avere un ID assegnato
    config, _, tokenizer = load_inputs()

    for token in config["tokenizer"]["special_tokens"].values():
        assert tokenizer.token_to_id(token) is not None


def test_train_captions_roundtrip_without_unknown_tokens() -> None:
    # Le caption train devono essere rappresentabili senza [UNK].
    # Il tokenizer restituisce anche [SOS], [EOS] e padding:
    # per confrontare il testo originale consideriamo solo i token della caption.
    config, captioned, tokenizer = load_inputs()

    special = config["tokenizer"]["special_tokens"]
    special_tokens = {
        special["sos"],
        special["eos"],
        special["pad"],
    }
    unk_token = special["unk"]

    train_captions = captioned.loc[
        captioned["split"] == "train",
        "caption",
    ].tolist()

    for caption in train_captions[:100]:
        encoding = tokenizer.encode(caption)

        # Il vocabolario è addestrato sulle sole caption di training:
        # nessuna caption train deve produrre token sconosciuti.
        assert unk_token not in encoding.tokens

        # Rimuoviamo i token tecnici aggiunti dal post-processing
        # prima del confronto con la tokenizzazione attesa della caption.
        caption_tokens = [
            token
            for token in encoding.tokens
            if token not in special_tokens
        ]

        assert caption_tokens == re.findall(r"\w+|[^\w\s]", caption)

In [ ]:
%cd /content/tiny-avatar-diffusion

# Eseguiamo i test dei blocchi precedenti e quelli specifici
# per caption deterministiche e tokenizer.
!python -m pytest -q \
  tests/test_metadata.py \
  tests/test_splits.py \
  tests/test_captions_and_tokenizer.py

Le 10.000 immagini dispongono ora di caption testuali deterministiche,
allineate agli split train, validation, test ordinario e test OOD. Il
tokenizer word-level è stato addestrato solo sulle caption del training,
con token speciali espliciti per unknown, padding, inizio e fine sequenza.

Gli artefatti prodotti sono pronti per il blocco successivo: dataset PyTorch,
collate function, text encoder e modello di diffusione condizionale.

# Diffusione DDPM e pipeline dati

In questo blocco implementiamo il processo forward di un Denoising Diffusion
Probabilistic Model (DDPM) e costruiamo dataset/dataloader PyTorch pronti per
il training condizionale.

Le immagini vengono ridimensionate a 32x32 e normalizzate in [-1, 1].
Le caption vengono convertite in token IDs tramite il tokenizer del blocco 4;
gli attributi discreti restano inoltre disponibili come metadati di controllo.

In [ ]:
%cd /content/tiny-avatar-diffusion

## Configurazione DDPM

Usiamo 1.000 timestep con beta schedule lineare. Il modello verrà addestrato
a predire il rumore gaussiano epsilon aggiunto all'immagine originale.

La normalizzazione in [-1, 1] è coerente con il processo diffusivo e con
l'output che il modello dovrà ricostruire durante il sampling.

In [ ]:
%%bash

cd /content/tiny-avatar-diffusion

# I parametri del processo diffusivo sono mantenuti in un file separato:
# questo permette di cambiare schedule senza modificare l'implementazione.
cat > configs/diffusion.yaml <<'EOF'
diffusion:
  num_timesteps: 1000

  beta_schedule:
    type: linear
    beta_start: 0.0001
    beta_end: 0.02

  # Il modello predice direttamente il rumore aggiunto all'immagine.
  prediction_type: epsilon

data:
  image_size: 32
  image_channels: 3

  normalization:
    minimum: -1.0
    maximum: 1.0
EOF

cat configs/diffusion.yaml

## Schedule e forward diffusion

Per ogni timestep \(t\), il processo forward produce una versione rumorosa
dell'immagine originale:

\[
x_t =
\sqrt{\bar{\alpha}_t} \, x_0 +
\sqrt{1 - \bar{\alpha}_t} \, \epsilon,
\qquad
\epsilon \sim \mathcal{N}(0, I)
\]

I coefficienti vengono calcolati una sola volta e registrati come buffer del
modulo PyTorch: si spostano automaticamente fra CPU e GPU, ma non sono
parametri ottimizzabili.

In [ ]:
%%writefile src/diffusion/schedule.py
from pathlib import Path
from typing import Optional

import torch
from torch import Tensor, nn
import yaml


def load_diffusion_config(
    config_path: Path,
) -> dict:
    # Carica e valida la configurazione DDPM
    with config_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        config = yaml.safe_load(file)

    if "diffusion" not in config:
        raise ValueError(
            "Sezione 'diffusion' mancante."
        )

    if "data" not in config:
        raise ValueError(
            "Sezione 'data' mancante."
        )

    diffusion = config["diffusion"]
    data = config["data"]

    required_diffusion = {
        "num_timesteps",
        "beta_schedule",
        "prediction_type",
    }

    missing = required_diffusion - set(diffusion)

    if missing:
        raise ValueError(
            "Parametri diffusion mancanti: "
            f"{sorted(missing)}"
        )

    schedule = diffusion["beta_schedule"]

    if schedule.get("type") != "linear":
        raise ValueError(
            "In questa implementazione iniziale "
            "è supportato solo lo schedule lineare."
        )

    beta_start = float(schedule["beta_start"])
    beta_end = float(schedule["beta_end"])

    num_timesteps = int(diffusion["num_timesteps"])

    if num_timesteps < 2:
        raise ValueError(
            "num_timesteps deve essere almeno 2."
        )

    if not (
        0.0 < beta_start < beta_end < 1.0
    ):
        raise ValueError(
            "È richiesto "
            "0 < beta_start < beta_end < 1."
        )

    if diffusion["prediction_type"] != "epsilon":
        raise ValueError(
            "Il progetto usa epsilon prediction."
        )

    if int(data["image_size"]) != 32:
        raise ValueError(
            "La configurazione iniziale richiede "
            "immagini 32x32."
        )

    if int(data["image_channels"]) != 3:
        raise ValueError(
            "Il dataset deve essere RGB."
        )

    normalization = data["normalization"]

    if (
        float(normalization["minimum"]) != -1.0
        or float(normalization["maximum"]) != 1.0
    ):
        raise ValueError(
            "Le immagini devono essere "
            "normalizzate in [-1, 1]."
        )

    return config


def extract(
    values: Tensor,
    timesteps: Tensor,
    reference: Tensor,
) -> Tensor:
    # Seleziona un coefficiente per elemento del batch e lo rimodella
    # per il broadcasting con un tensore immagine [B, C, H, W].

    if timesteps.dtype != torch.long:
        raise TypeError(
            "timesteps deve avere dtype torch.long."
        )

    if timesteps.ndim != 1:
        raise ValueError(
            "timesteps deve avere shape [batch]."
        )

    if reference.ndim < 2:
        raise ValueError(
            "reference deve avere almeno 2 dimensioni."
        )

    if timesteps.shape[0] != reference.shape[0]:
        raise ValueError(
            "La dimensione batch dei timestep "
            "non coincide con quella del tensore."
        )

    if values.device != reference.device:
        raise ValueError(
            "Schedule e tensore sono su device "
            "diversi. Eseguire schedule.to(device)."
        )

    if timesteps.device != reference.device:
        raise ValueError(
            "Timestep e tensore sono su device diversi."
        )

    if timesteps.numel() > 0:
        minimum = int(timesteps.min().item())
        maximum = int(timesteps.max().item())

        if minimum < 0:
            raise IndexError(
                "I timestep non possono essere negativi."
            )

        if maximum >= values.shape[0]:
            raise IndexError(
                f"Timestep {maximum} fuori intervallo. "
                f"Massimo consentito: "
                f"{values.shape[0] - 1}."
            )

    gathered = values.gather(
        dim=0,
        index=timesteps,
    )

    output_shape = (
        timesteps.shape[0],
        *([1] * (reference.ndim - 1)),
    )

    return (
        gathered
        .reshape(output_shape)
        .to(dtype=reference.dtype)
    )


class DDPMSchedule(nn.Module):
    # Coefficienti del processo forward e della posteriore DDPM.

    # I coefficienti sono buffer, non parametri apprendibili: servono
    # per costruire x_t e per il sampling inverso, ma non devono ricevere
    # gradienti né essere aggiornati dall'optimizer.

    def __init__(
        self,
        num_timesteps: int = 1000,
        beta_start: float = 0.0001,
        beta_end: float = 0.02,
    ) -> None:
        super().__init__()

        if num_timesteps < 2:
            raise ValueError(
                "num_timesteps deve essere almeno 2."
            )

        if not (
            0.0 < beta_start < beta_end < 1.0
        ):
            raise ValueError(
                "È richiesto "
                "0 < beta_start < beta_end < 1."
            )

        self.num_timesteps = int(num_timesteps)
        self.beta_start = float(beta_start)
        self.beta_end = float(beta_end)

        # Il calcolo iniziale in float64 limita gli errori di accumulo
        # nel prodotto cumulativo delle alpha; i buffer finali sono float32.
        betas = torch.linspace(
            beta_start,
            beta_end,
            num_timesteps,
            dtype=torch.float64,
        )

        alphas = 1.0 - betas

        alphas_cumprod = torch.cumprod(
            alphas,
            dim=0,
        )

        alphas_cumprod_previous = torch.cat(
            [
                torch.ones(
                    1,
                    dtype=torch.float64,
                ),
                alphas_cumprod[:-1],
            ],
            dim=0,
        )

        sqrt_alphas_cumprod = torch.sqrt(
            alphas_cumprod
        )

        sqrt_one_minus_alphas_cumprod = torch.sqrt(
            1.0 - alphas_cumprod
        )

        sqrt_recip_alphas = torch.sqrt(
            1.0 / alphas
        )

        sqrt_recip_alphas_cumprod = torch.sqrt(
            1.0 / alphas_cumprod
        )

        sqrt_recipm1_alphas_cumprod = torch.sqrt(
            1.0 / alphas_cumprod - 1.0
        )

        posterior_variance = (
            betas
            * (1.0 - alphas_cumprod_previous)
            / (1.0 - alphas_cumprod)
        )

        # A t=0 la varianza posteriore è zero. Il clamp permette
        # di calcolarne il log in modo numericamente stabile.
        posterior_log_variance_clipped = torch.log(
            posterior_variance.clamp(min=1e-20)
        )

        posterior_mean_coefficient_1 = (
            betas
            * torch.sqrt(alphas_cumprod_previous)
            / (1.0 - alphas_cumprod)
        )

        posterior_mean_coefficient_2 = (
            (1.0 - alphas_cumprod_previous)
            * torch.sqrt(alphas)
            / (1.0 - alphas_cumprod)
        )

        buffers = {
            "betas": betas,
            "alphas": alphas,
            "alphas_cumprod": alphas_cumprod,
            "alphas_cumprod_previous": (
                alphas_cumprod_previous
            ),
            "sqrt_alphas_cumprod": (
                sqrt_alphas_cumprod
            ),
            "sqrt_one_minus_alphas_cumprod": (
                sqrt_one_minus_alphas_cumprod
            ),
            "sqrt_recip_alphas": sqrt_recip_alphas,
            "sqrt_recip_alphas_cumprod": (
                sqrt_recip_alphas_cumprod
            ),
            "sqrt_recipm1_alphas_cumprod": (
                sqrt_recipm1_alphas_cumprod
            ),
            "posterior_variance": posterior_variance,
            "posterior_log_variance_clipped": (
                posterior_log_variance_clipped
            ),
            "posterior_mean_coefficient_1": (
                posterior_mean_coefficient_1
            ),
            "posterior_mean_coefficient_2": (
                posterior_mean_coefficient_2
            ),
        }

        for name, value in buffers.items():
            self.register_buffer(
                name,
                value.to(torch.float32),
                persistent=True,
            )

    @classmethod
    def from_config(
        cls,
        config: dict,
    ) -> "DDPMSchedule":
        diffusion = config["diffusion"]
        beta_schedule = diffusion["beta_schedule"]

        return cls(
            num_timesteps=int(
                diffusion["num_timesteps"]
            ),
            beta_start=float(
                beta_schedule["beta_start"]
            ),
            beta_end=float(
                beta_schedule["beta_end"]
            ),
        )

    def q_sample(
        self,
        x_start: Tensor,
        timesteps: Tensor,
        noise: Optional[Tensor] = None,
        generator: Optional[torch.Generator] = None,
    ) -> tuple[Tensor, Tensor]:
        # Campiona x_t dalla distribuzione q(x_t | x_0).

        # Se noise non viene fornito, viene generato rumore gaussiano;
        # il generator opzionale rende il test o il sampling riproducibile.
        if not x_start.is_floating_point():
            raise TypeError(
                "x_start deve essere floating point."
            )

        if noise is not None and generator is not None:
            raise ValueError(
                "Specificare noise oppure generator, "
                "non entrambi."
            )

        if noise is None:
            if generator is None:
                noise = torch.randn_like(x_start)
            else:
                noise = torch.randn(
                    x_start.shape,
                    dtype=x_start.dtype,
                    device=x_start.device,
                    generator=generator,
                )

        if noise.shape != x_start.shape:
            raise ValueError(
                "noise e x_start devono avere "
                "la stessa shape."
            )

        if noise.device != x_start.device:
            raise ValueError(
                "noise e x_start devono essere "
                "sullo stesso device."
            )

        signal_scale = extract(
            self.sqrt_alphas_cumprod,
            timesteps,
            x_start,
        )

        noise_scale = extract(
            self.sqrt_one_minus_alphas_cumprod,
            timesteps,
            x_start,
        )

        x_t = (
            signal_scale * x_start
            + noise_scale * noise
        )

        return x_t, noise

    def predict_start_from_noise(
        self,
        x_t: Tensor,
        timesteps: Tensor,
        noise: Tensor,
    ) -> Tensor:
        # Ricostruisce x_0 usando x_t e il rumore epsilon stimato.

        # Questa è l'inversione algebrica del forward process e sarà
        # utilizzata durante il sampling, una volta sostituito noise
        # con la previsione del modello.
        if x_t.shape != noise.shape:
            raise ValueError(
                "x_t e noise devono avere la stessa shape."
            )

        if not x_t.is_floating_point():
            raise TypeError(
                "x_t deve essere floating point."
            )

        if not noise.is_floating_point():
            raise TypeError(
                "noise deve essere floating point."
            )

        scale_x_t = extract(
            self.sqrt_recip_alphas_cumprod,
            timesteps,
            x_t,
        )

        scale_noise = extract(
            self.sqrt_recipm1_alphas_cumprod,
            timesteps,
            x_t,
        )

        x_start = (
            scale_x_t * x_t
            - scale_noise * noise
        )

        return x_start

    def q_posterior(
        self,
        x_start: Tensor,
        x_t: Tensor,
        timesteps: Tensor,
    ) -> tuple[Tensor, Tensor, Tensor]:
        # Calcola i parametri della posteriore q(x_{t-1} | x_t, x_0)
        if x_start.shape != x_t.shape:
            raise ValueError(
                "x_start e x_t devono avere la stessa shape."
            )

        coefficient_1 = extract(
            self.posterior_mean_coefficient_1,
            timesteps,
            x_t,
        )

        coefficient_2 = extract(
            self.posterior_mean_coefficient_2,
            timesteps,
            x_t,
        )

        posterior_mean = (
            coefficient_1 * x_start
            + coefficient_2 * x_t
        )

        posterior_variance = extract(
            self.posterior_variance,
            timesteps,
            x_t,
        )

        posterior_log_variance = extract(
            self.posterior_log_variance_clipped,
            timesteps,
            x_t,
        )

        return (
            posterior_mean,
            posterior_variance,
            posterior_log_variance,
        )

    def reverse_step(
        self,
        noisyimages: Tensor,
        predicted_noise: Tensor,
        timesteps: Tensor,
        generator: Optional[torch.Generator] = None,
        clip_denoised: bool = True,
    ) -> tuple[Tensor, Tensor]:
        # Esegue un singolo passo del reverse process DDPM.

        # Restituisce:
        # - x_{t-1}
        # - x_0 ricostruito a partire da x_t e dal rumore predetto
        if noisyimages.shape != predicted_noise.shape:
            raise ValueError(
                "noisyimages e predicted_noise devono "
                "avere la stessa shape."
            )

        if timesteps.dtype != torch.long:
            raise TypeError(
                "timesteps deve avere dtype torch.long."
            )

        predicted_x_start = self.predict_start_from_noise(
            x_t=noisyimages,
            timesteps=timesteps,
            noise=predicted_noise,
        )

        if clip_denoised:
            predicted_x_start = predicted_x_start.clamp(
                min=-1.0,
                max=1.0,
            )

        (
            posterior_mean,
            posterior_variance,
            _posterior_log_variance,
        ) = self.q_posterior(
            x_start=predicted_x_start,
            x_t=noisyimages,
            timesteps=timesteps,
        )

        if generator is None:
            noise = torch.randn_like(noisyimages)
        else:
            noise = torch.randn(
                noisyimages.shape,
                dtype=noisyimages.dtype,
                device=noisyimages.device,
                generator=generator,
            )

        nonzero_mask = (
            timesteps.ne(0)
            .to(dtype=noisyimages.dtype)
            .reshape(
                noisyimages.shape[0],
                *([1] * (noisyimages.ndim - 1)),
            )
        )

        x_previous = (
            posterior_mean
            + nonzero_mask
            * torch.sqrt(posterior_variance)
            * noise
        )

        return x_previous, predicted_x_start

## Test dello schedule DDPM

I test verificano i coefficienti, le shape, la monotonicità della quantità
di segnale residuo, il forward noising, l'inversione analitica con rumore
noto e i casi di errore più importanti.

Questi controlli sono fondamentali: un errore piccolo nei coefficienti della
diffusione renderebbe il training apparentemente eseguibile, ma invalido.

In [ ]:
%%writefile tests/test_diffusion_schedule.py
from pathlib import Path

import pytest
import torch

from src.diffusion.schedule import (
    DDPMSchedule,
    extract,
    load_diffusion_config,
)


CONFIG_PATH = Path("configs/diffusion.yaml")


def build_schedule() -> DDPMSchedule:
    config = load_diffusion_config(CONFIG_PATH)

    return DDPMSchedule.from_config(config)


def test_configuration() -> None:
    # La configurazione deve rispettare le ipotesi del progetto
    config = load_diffusion_config(CONFIG_PATH)

    assert (
        config["diffusion"]["num_timesteps"] == 1000
    )
    assert (
        config["diffusion"]["prediction_type"]
        == "epsilon"
    )
    assert config["data"]["image_size"] == 32
    assert config["data"]["image_channels"] == 3


def test_schedule_has_no_trainable_parameters() -> None:
    # Lo schedule è fisso e non deve aggiungere parametri all'optimizer
    schedule = build_schedule()

    assert sum(
        parameter.numel()
        for parameter in schedule.parameters()
    ) == 0


def test_schedule_shapes() -> None:
    # Ogni coefficiente deve contenere un valore per timestep
    schedule = build_schedule()

    expected_shape = (1000,)

    assert schedule.betas.shape == expected_shape
    assert schedule.alphas.shape == expected_shape
    assert (
        schedule.alphas_cumprod.shape
        == expected_shape
    )
    assert (
        schedule.posterior_variance.shape
        == expected_shape
    )


def test_beta_schedule_endpoints() -> None:
    # Il beta schedule deve rispettare gli estremi configurati
    schedule = build_schedule()

    torch.testing.assert_close(
        schedule.betas[0],
        torch.tensor(0.0001),
        rtol=0.0,
        atol=1e-8,
    )

    torch.testing.assert_close(
        schedule.betas[-1],
        torch.tensor(0.02),
        rtol=0.0,
        atol=1e-8,
    )


def test_beta_schedule_is_increasing() -> None:
    # Il rumore aggiunto a ogni passaggio cresce con il timestep
    schedule = build_schedule()

    assert torch.all(
        schedule.betas[1:] > schedule.betas[:-1]
    )


def test_alpha_cumprod_is_decreasing() -> None:
    # La quota di segnale originale deve diminuire nel tempo
    schedule = build_schedule()

    assert torch.all(schedule.alphas_cumprod > 0)
    assert torch.all(schedule.alphas_cumprod <= 1)

    assert torch.all(
        schedule.alphas_cumprod[1:]
        < schedule.alphas_cumprod[:-1]
    )

    assert schedule.alphas_cumprod[-1] < 1e-4


def test_previous_alpha_cumprod() -> None:
    # Alpha-bar precedente deve essere shiftata di un timestep
    schedule = build_schedule()

    torch.testing.assert_close(
        schedule.alphas_cumprod_previous[0],
        torch.tensor(1.0),
    )

    torch.testing.assert_close(
        schedule.alphas_cumprod_previous[1:],
        schedule.alphas_cumprod[:-1],
    )


def test_q_sample_matches_closed_form() -> None:
    # q_sample deve coincidere con la formula DDPM esplicita
    schedule = build_schedule()

    x_start = torch.linspace(
        -1.0,
        1.0,
        steps=2 * 3 * 8 * 8,
    ).reshape(2, 3, 8, 8)

    noise = torch.linspace(
        1.0,
        -1.0,
        steps=2 * 3 * 8 * 8,
    ).reshape(2, 3, 8, 8)

    timesteps = torch.tensor(
        [0, 999],
        dtype=torch.long,
    )

    x_t, returned_noise = schedule.q_sample(
        x_start=x_start,
        timesteps=timesteps,
        noise=noise,
    )

    expected = (
        schedule.sqrt_alphas_cumprod[timesteps]
        .reshape(2, 1, 1, 1)
        * x_start
        + schedule.sqrt_one_minus_alphas_cumprod[
            timesteps
        ]
        .reshape(2, 1, 1, 1)
        * noise
    )

    torch.testing.assert_close(x_t, expected)
    torch.testing.assert_close(
        returned_noise,
        noise,
    )


def test_predict_start_inverts_q_sample() -> None:
    # Con rumore noto, l'inversione deve ricostruire x_0
    schedule = build_schedule()

    generator = torch.Generator().manual_seed(42)

    x_start = (
        torch.rand(
            4,
            3,
            16,
            16,
            generator=generator,
        )
        * 2.0
        - 1.0
    )

    noise = torch.randn(
        x_start.shape,
        generator=generator,
    )

    timesteps = torch.tensor(
        [0, 50, 500, 999],
        dtype=torch.long,
    )

    x_t, _ = schedule.q_sample(
        x_start=x_start,
        timesteps=timesteps,
        noise=noise,
    )

    reconstructed = schedule.predict_start_from_noise(
        x_t=x_t,
        timesteps=timesteps,
        noise=noise,
    )

    torch.testing.assert_close(
        reconstructed,
        x_start,
        rtol=1e-3,
        atol=2e-4,
    )


def test_q_sample_preserves_shape_dtype() -> None:
    # Il forward process mantiene shape e dtype delle immagini
    schedule = build_schedule()

    x_start = torch.zeros(
        5,
        3,
        32,
        32,
        dtype=torch.float32,
    )

    timesteps = torch.tensor(
        [0, 100, 300, 700, 999],
        dtype=torch.long,
    )

    x_t, noise = schedule.q_sample(
        x_start=x_start,
        timesteps=timesteps,
    )

    assert x_t.shape == x_start.shape
    assert noise.shape == x_start.shape
    assert x_t.dtype == x_start.dtype
    assert noise.dtype == x_start.dtype

    assert torch.isfinite(x_t).all()
    assert torch.isfinite(noise).all()


def test_q_sample_is_reproducible() -> None:
    # Un generator con lo stesso seed deve produrre lo stesso rumore
    schedule = build_schedule()

    x_start = torch.zeros(2, 3, 8, 8)

    timesteps = torch.tensor(
        [10, 900],
        dtype=torch.long,
    )

    first_generator = (
        torch.Generator().manual_seed(123)
    )
    second_generator = (
        torch.Generator().manual_seed(123)
    )

    first, first_noise = schedule.q_sample(
        x_start=x_start,
        timesteps=timesteps,
        generator=first_generator,
    )

    second, second_noise = schedule.q_sample(
        x_start=x_start,
        timesteps=timesteps,
        generator=second_generator,
    )

    torch.testing.assert_close(first, second)
    torch.testing.assert_close(
        first_noise,
        second_noise,
    )


def test_q_sample_preserves_gradients() -> None:
    # q_sample deve preservare il grafo autograd
    schedule = build_schedule()

    x_start = torch.randn(
        2,
        3,
        8,
        8,
        requires_grad=True,
    )

    noise = torch.randn_like(x_start)

    timesteps = torch.tensor(
        [100, 900],
        dtype=torch.long,
    )

    x_t, _ = schedule.q_sample(
        x_start=x_start,
        timesteps=timesteps,
        noise=noise,
    )

    x_t.square().mean().backward()

    assert x_start.grad is not None
    assert torch.isfinite(x_start.grad).all()


def test_posterior_at_first_timestep() -> None:
    # A t=0 la posteriore collassa sull'immagine originale
    schedule = build_schedule()

    x_start = torch.randn(2, 3, 8, 8)
    x_t = torch.randn_like(x_start)

    timesteps = torch.zeros(
        2,
        dtype=torch.long,
    )

    mean, variance, log_variance = (
        schedule.q_posterior(
            x_start=x_start,
            x_t=x_t,
            timesteps=timesteps,
        )
    )

    torch.testing.assert_close(
        mean,
        x_start,
        rtol=1e-4,
        atol=1e-5,
    )

    torch.testing.assert_close(
        variance,
        torch.zeros_like(variance),
        atol=1e-8,
        rtol=0.0,
    )

    assert torch.isfinite(log_variance).all()


def test_invalid_timestep_dtype() -> None:
    # I timestep devono essere indici interi
    schedule = build_schedule()

    x_start = torch.zeros(2, 3, 8, 8)

    timesteps = torch.tensor([0.0, 1.0])

    with pytest.raises(TypeError):
        schedule.q_sample(
            x_start=x_start,
            timesteps=timesteps,
        )


def test_invalid_timestep_range() -> None:
    # Gli indici validi sono compresi tra 0 e T-1
    schedule = build_schedule()

    x_start = torch.zeros(2, 3, 8, 8)

    timesteps = torch.tensor(
        [0, 1000],
        dtype=torch.long,
    )

    with pytest.raises(IndexError):
        schedule.q_sample(
            x_start=x_start,
            timesteps=timesteps,
        )


def test_extract_broadcast_shape() -> None:
    # extract deve produrre coefficienti broadcastabili sulle immagini
    values = torch.arange(
        10,
        dtype=torch.float32,
    )

    reference = torch.zeros(3, 4, 5, 6)

    timesteps = torch.tensor(
        [1, 4, 8],
        dtype=torch.long,
    )

    extracted = extract(
        values=values,
        timesteps=timesteps,
        reference=reference,
    )

    assert extracted.shape == (3, 1, 1, 1)

    torch.testing.assert_close(
        extracted.flatten(),
        torch.tensor([1.0, 4.0, 8.0]),
    )


@pytest.mark.skipif(
    not torch.cuda.is_available(),
    reason="CUDA non disponibile",
)
def test_schedule_on_cuda() -> None:
    # I buffer devono seguire il modulo sul device CUDA
    schedule = build_schedule().to("cuda")

    x_start = torch.randn(
        2,
        3,
        8,
        8,
        device="cuda",
    )

    timesteps = torch.tensor(
        [0, 999],
        dtype=torch.long,
        device="cuda",
    )

    x_t, noise = schedule.q_sample(
        x_start=x_start,
        timesteps=timesteps,
    )

    assert x_t.is_cuda
    assert noise.is_cuda
    assert torch.isfinite(x_t).all()

In [ ]:
%cd /content/tiny-avatar-diffusion

!python -m pytest -q tests/test_diffusion_schedule.py

## Ispezione dei coefficienti

Questa cella non addestra nulla: serve a verificare che, al termine del
processo forward, il segnale originale sia quasi completamente sostituito
dal rumore. È un controllo quantitativo complementare ai test unitari.

In [ ]:
from pathlib import Path

from src.diffusion.schedule import (
    DDPMSchedule,
    load_diffusion_config,
)


config = load_diffusion_config(
    Path("configs/diffusion.yaml")
)

schedule = DDPMSchedule.from_config(config)

print("Timestep:", schedule.num_timesteps)
print("Beta iniziale:", schedule.betas[0].item())
print("Beta finale:", schedule.betas[-1].item())

print(
    "Alpha-bar iniziale:",
    schedule.alphas_cumprod[0].item(),
)

print(
    "Alpha-bar finale:",
    schedule.alphas_cumprod[-1].item(),
)

print(
    "Signal scale finale:",
    schedule.sqrt_alphas_cumprod[-1].item(),
)

print(
    "Noise scale finale:",
    schedule.sqrt_one_minus_alphas_cumprod[-1].item(),
)

print(
    "Posterior variance t=0:",
    schedule.posterior_variance[0].item(),
)

# Tutti i coefficienti devono restare finiti:
# il log della varianza a t=0 è gestito tramite clamp.
print(
    "Valori finiti:",
    all(
        tensor.isfinite().all().item()
        for _, tensor in schedule.named_buffers()
    ),
)

## Controllo della normalizzazione immagini

Prima di inserire le immagini nel processo DDPM, controlliamo un esempio
reale del dataset: deve essere RGB, avere shape `[1, 3, 32, 32]` dopo
l'aggiunta della dimensione batch e assumere valori nell'intervallo [-1, 1].

In [ ]:
from pathlib import Path

import pandas as pd
from PIL import Image
from torchvision import transforms


captioned = pd.read_csv(
    "data/processed/captions/"
    "metadata_with_captions.csv"
)

dataset_root = Path("data/raw/cartoonset10k")

example = captioned.iloc[0]

image_path = (
    dataset_root / example["image_path"]
)

image = Image.open(image_path).convert("RGB")

image_transform = transforms.Compose(
    [
        transforms.Resize(
            (32, 32),
            antialias=True,
        ),
        transforms.ToTensor(),
        # ToTensor produce valori in [0, 1]; DDPM usa [-1, 1].
        transforms.Lambda(
            lambda tensor: tensor * 2.0 - 1.0
        ),
    ]
)

x_start = image_transform(image).unsqueeze(0)

print("Percorso:", image_path)
print("Shape:", x_start.shape)
print("Minimo:", x_start.min().item())
print("Massimo:", x_start.max().item())
print("Caption:", example["caption"])

## Visualizzazione del forward diffusion

Visualizziamo la stessa immagine pulita a timestep progressivamente più alti.

Viene usata la stessa realizzazione di rumore per tutti i timestep: in questo
modo, le differenze visibili dipendono soltanto dai coefficienti DDPM e dalla
progressiva riduzione del segnale originale, non da campioni di rumore diversi.

In [ ]:
import matplotlib.pyplot as plt
import torch


# I timestep coprono l'intera catena DDPM: immagine quasi pulita,
# rumore leggero, rumore intermedio, rumore dominante e rumore quasi puro.
timesteps = torch.tensor(
    [0, 49, 199, 499, 999],
    dtype=torch.long,
)

# q_sample opera per elemento del batch: replichiamo la stessa immagine
# una volta per ciascun timestep da visualizzare.
x_batch = x_start.repeat(
    len(timesteps),
    1,
    1,
    1,
)

generator = torch.Generator().manual_seed(42)

# Lo stesso epsilon è associato a ogni replica di x_start.
# Questo isola l'effetto del timestep nel confronto visivo.
shared_noise = torch.randn(
    x_start.shape,
    generator=generator,
).repeat(
    len(timesteps),
    1,
    1,
    1,
)

x_noisy, returned_noise = schedule.q_sample(
    x_start=x_batch,
    timesteps=timesteps,
    noise=shared_noise,
)

# Verifiche minime sul contratto del forward process.
assert x_noisy.shape == x_batch.shape
assert torch.equal(returned_noise, shared_noise)
assert torch.isfinite(x_noisy).all()

images_to_show = torch.cat(
    [
        x_start,
        x_noisy,
    ],
    dim=0,
)

titles = [
    "Clean x0",
    "t = 0",
    "t = 49",
    "t = 199",
    "t = 499",
    "t = 999",
]

figure, axes = plt.subplots(
    1,
    len(titles),
    figsize=(15, 3),
)

for axis, tensor, title in zip(
    axes,
    images_to_show,
    titles,
):
    # Il tensore del DDPM è in [-1, 1]; imshow richiede tipicamente [0, 1].
    display_image = (
        (tensor + 1.0) / 2.0
    ).clamp(0.0, 1.0)

    axis.imshow(
        display_image
        .permute(1, 2, 0)
        .cpu()
        .numpy()
    )

    axis.set_title(title)
    axis.axis("off")

plt.tight_layout()
plt.show()

## Dataset e DataLoader

Il dataset legge il file con caption e split, carica le immagini al bisogno,
precalcola la tokenizzazione delle caption e restituisce un dizionario con:

- immagine normalizzata;
- token IDs e attention mask;
- tutti gli attributi del metadata;
- gli otto attributi usati per il conditioning;
- identificativi e informazioni di tracciabilità.

Poiché le caption hanno tutte la stessa struttura, dopo l'aggiunta di
`[SOS]` e `[EOS]` la sequenza è resa uniforme tramite padding a lunghezza 44.

In [ ]:
%%bash

cd /content/tiny-avatar-diffusion

cat > configs/data_loader.yaml <<'EOF'
data:
  dataset_root: data/raw/cartoonset10k
  captions_path: data/processed/captions/metadata_with_captions.csv
  tokenizer_path: data/processed/tokenizer/tokenizer.json

  image_size: 32

  expected_split_counts:
    train: 7884
    validation: 986
    test_ordinary: 986
    test_ood: 144

  # Gli attributi sono mantenuti anche separatamente dai token
  # per analisi, controlli e valutazioni condizionali successive.
  conditioning_attributes:
    - face_shape
    - face_color
    - hair
    - hair_color
    - eye_color
    - chin_length
    - facial_hair
    - glasses

loader:
  train_batch_size: 64
  evaluation_batch_size: 64

  num_workers: 2
  pin_memory: true
  persistent_workers: true
  prefetch_factor: 2

  # Drop-last è applicato solo al train, così ogni batch di training
  # ha dimensione costante; gli split di valutazione restano completi.
  drop_last: true
  seed: 42
EOF

cat configs/data_loader.yaml

In [ ]:
%%writefile src/data/avatar_dataset.py
import random
from pathlib import Path
from typing import Optional, Sequence

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tokenizers import Tokenizer
from torch import Tensor
from torch.utils.data import DataLoader, Dataset

from src.data.build_metadata import EXPECTED_VARIANTS
from src.data.build_splits import VALID_SPLITS


def image_to_tensor(
    image: Image.Image,
    image_size: int,
) -> Tensor:
    # Converte un'immagine PIL RGB in un tensore CHW in [-1, 1]
    if image_size < 1:
        raise ValueError(
            "image_size deve essere positivo."
        )

    image = image.convert("RGB")
    image = image.resize(
        (image_size, image_size),
        resample=Image.Resampling.BILINEAR,
    )

    array = np.asarray(
        image,
        dtype=np.float32,
    )

    # HWC [0, 255] -> CHW [-1, 1]
    tensor = torch.from_numpy(
        array.transpose(2, 0, 1)
    )

    return tensor.div(127.5).sub(1.0)


class CartoonAvatarDataset(Dataset):
    # Dataset per immagini avatar e caption condizionali.

    # Le caption sono tokenizzate durante l'inizializzazione: questo evita
    # di ripetere la tokenizzazione ad ogni chiamata di __getitem__.


    def __init__(
        self,
        dataset_root: Path,
        captions_path: Path,
        tokenizer_path: Path,
        split: str,
        image_size: int,
        expected_count: Optional[int] = None,
        conditioning_attributes: Optional[
            Sequence[str]
        ] = None,
        validate_paths: bool = True,
    ) -> None:
        if split not in VALID_SPLITS:
            raise ValueError(
                f"Split non valido: {split}."
            )

        self.dataset_root = dataset_root.resolve()
        self.captions_path = captions_path
        self.tokenizer_path = tokenizer_path
        self.split = split
        self.image_size = int(image_size)

        if not self.dataset_root.is_dir():
            raise FileNotFoundError(
                f"Dataset root non trovata: "
                f"{self.dataset_root}"
            )

        if not self.captions_path.is_file():
            raise FileNotFoundError(
                f"File caption non trovato: "
                f"{self.captions_path}"
            )

        if not self.tokenizer_path.is_file():
            raise FileNotFoundError(
                f"Tokenizer non trovato: "
                f"{self.tokenizer_path}"
            )

        metadata = pd.read_csv(
            self.captions_path,
            dtype={
                "image_id": "string",
                "image_path": "string",
                "caption": "string",
                "split": "string",
            },
        )

        required_columns = {
            "image_id",
            "image_path",
            "caption",
            "split",
            *EXPECTED_VARIANTS.keys(),
        }

        missing_columns = (
            required_columns - set(metadata.columns)
        )

        if missing_columns:
            raise ValueError(
                "Colonne mancanti nel metadata: "
                f"{sorted(missing_columns)}"
            )

        if metadata["image_id"].duplicated().any():
            raise ValueError(
                "Il metadata contiene image_id duplicati."
            )

        available_splits = set(
            metadata["split"].unique()
        )

        if available_splits != set(VALID_SPLITS):
            raise ValueError(
                "Split trovati nel metadata: "
                f"{available_splits}; attesi: "
                f"{set(VALID_SPLITS)}."
            )

        # L'ordine canonico stabilizza dataset, test e campionamento.
        metadata = (
            metadata[
                metadata["split"] == split
            ]
            .sort_values("image_id")
            .reset_index(drop=True)
        )

        if metadata.empty:
            raise ValueError(
                f"Lo split {split} è vuoto."
            )

        if (
            expected_count is not None
            and len(metadata) != expected_count
        ):
            raise ValueError(
                f"Lo split {split} contiene "
                f"{len(metadata)} esempi; "
                f"attesi {expected_count}."
            )

        if metadata["caption"].isna().any():
            raise ValueError(
                "Sono presenti caption mancanti."
            )

        if (
            metadata["caption"]
            .str.strip()
            .eq("")
            .any()
        ):
            raise ValueError(
                "Sono presenti caption vuote."
            )

        if conditioning_attributes is None:
            conditioning_attributes = list(
                EXPECTED_VARIANTS
            )

        unknown_attributes = sorted(
            set(conditioning_attributes)
            - set(EXPECTED_VARIANTS)
        )

        if unknown_attributes:
            raise ValueError(
                "Attributi sconosciuti: "
                f"{unknown_attributes}"
            )

        self.conditioning_attributes = tuple(
            conditioning_attributes
        )

        self.all_attribute_names = tuple(
            EXPECTED_VARIANTS.keys()
        )

        image_paths = []

        for relative_path in metadata["image_path"]:
            absolute_path = (
                self.dataset_root
                / str(relative_path)
            ).resolve()

            # Protezione contro percorsi che escano dalla root del dataset.
            if (
                absolute_path != self.dataset_root
                and self.dataset_root
                not in absolute_path.parents
            ):
                raise ValueError(
                    "Percorso immagine esterno "
                    "al dataset root: "
                    f"{relative_path}"
                )

            image_paths.append(absolute_path)

        if validate_paths:
            missing_paths = [
                path
                for path in image_paths
                if not path.is_file()
            ]

            if missing_paths:
                raise FileNotFoundError(
                    f"Mancano {len(missing_paths)} "
                    "immagini. Primo percorso: "
                    f"{missing_paths[0]}"
                )

        tokenizer = Tokenizer.from_file(
            str(self.tokenizer_path)
        )

        captions = (
            metadata["caption"]
            .astype(str)
            .tolist()
        )

        encodings = tokenizer.encode_batch(captions)

        encoded_lengths = {
            len(encoding.ids)
            for encoding in encodings
        }

        mask_lengths = {
            len(encoding.attention_mask)
            for encoding in encodings
        }

        if len(encoded_lengths) != 1:
            raise ValueError(
                "Le sequenze tokenizzate non hanno "
                "lunghezza uniforme."
            )

        if encoded_lengths != mask_lengths:
            raise ValueError(
                "Token ID e attention mask hanno "
                "lunghezze differenti."
            )

        self.sequence_length = next(
            iter(encoded_lengths)
        )

        self.input_ids = torch.tensor(
            [
                encoding.ids
                for encoding in encodings
            ],
            dtype=torch.long,
        )

        self.attention_masks = torch.tensor(
            [
                encoding.attention_mask
                for encoding in encodings
            ],
            dtype=torch.long,
        )

        self.all_attributes = torch.tensor(
            metadata[
                list(self.all_attribute_names)
            ].to_numpy(dtype=np.int64),
            dtype=torch.long,
        )

        self.conditioning_values = torch.tensor(
            metadata[
                list(self.conditioning_attributes)
            ].to_numpy(dtype=np.int64),
            dtype=torch.long,
        )

        self.image_ids = (
            metadata["image_id"]
            .astype(str)
            .tolist()
        )

        self.image_paths = image_paths
        self.captions = captions

    def __len__(self) -> int:
        return len(self.image_ids)

    def __getitem__(
        self,
        index: int,
    ) -> dict:
        image_path = self.image_paths[index]

        with Image.open(image_path) as image:
            image_tensor = image_to_tensor(
                image=image,
                image_size=self.image_size,
            )

        if image_tensor.shape != (
            3,
            self.image_size,
            self.image_size,
        ):
            raise RuntimeError(
                "Shape immagine inattesa: "
                f"{tuple(image_tensor.shape)}"
            )

        if not torch.isfinite(image_tensor).all():
            raise RuntimeError(
                f"Valori non finiti in {image_path}"
            )

        if (
            image_tensor.min() < -1.0
            or image_tensor.max() > 1.0
        ):
            raise RuntimeError(
                "Immagine fuori dall'intervallo "
                "[-1, 1]."
            )

        return {
            "image": image_tensor,
            "input_ids": self.input_ids[index],
            "attention_mask": (
                self.attention_masks[index]
            ),
            "attributes": self.all_attributes[index],
            "conditioning_values": (
                self.conditioning_values[index]
            ),
            "image_id": self.image_ids[index],
            "caption": self.captions[index],
            "image_path": str(image_path),
            "split": self.split,
        }


def seed_worker(worker_id: int) -> None:
    # Inizializza NumPy e random in ogni worker del DataLoader
    del worker_id

    worker_seed = (
        torch.initial_seed() % 2**32
    )

    np.random.seed(worker_seed)
    random.seed(worker_seed)


def build_dataloader(
    dataset: CartoonAvatarDataset,
    batch_size: int,
    shuffle: bool,
    num_workers: int,
    pin_memory: bool,
    persistent_workers: bool,
    prefetch_factor: int,
    drop_last: bool,
    seed: int,
) -> DataLoader:
    # Costruisce un DataLoader riproducibile per uno split
    if batch_size < 1:
        raise ValueError(
            "batch_size deve essere positivo."
        )

    if num_workers < 0:
        raise ValueError(
            "num_workers non può essere negativo."
        )

    generator = torch.Generator()
    generator.manual_seed(seed)

    loader_arguments = {
        "dataset": dataset,
        "batch_size": batch_size,
        "shuffle": shuffle,
        "num_workers": num_workers,
        "pin_memory": bool(
            pin_memory
            and torch.cuda.is_available()
        ),
        "drop_last": drop_last,
        "worker_init_fn": seed_worker,
        "generator": generator,
    }

    # PyTorch non accetta questi argomenti con num_workers=0.
    if num_workers > 0:
        loader_arguments.update(
            {
                "persistent_workers": (
                    persistent_workers
                ),
                "prefetch_factor": (
                    prefetch_factor
                ),
            }
        )

    return DataLoader(**loader_arguments)


def build_datasets(
    config: dict,
) -> dict[str, CartoonAvatarDataset]:
    # Costruisce un dataset per ogni split del protocollo
    data_config = config["data"]

    dataset_root = Path(
        data_config["dataset_root"]
    )

    captions_path = Path(
        data_config["captions_path"]
    )

    tokenizer_path = Path(
        data_config["tokenizer_path"]
    )

    image_size = int(data_config["image_size"])

    expected_counts = data_config[
        "expected_split_counts"
    ]

    conditioning_attributes = data_config[
        "conditioning_attributes"
    ]

    datasets = {}

    for split_name in VALID_SPLITS:
        datasets[split_name] = CartoonAvatarDataset(
            dataset_root=dataset_root,
            captions_path=captions_path,
            tokenizer_path=tokenizer_path,
            split=split_name,
            image_size=image_size,
            expected_count=int(
                expected_counts[split_name]
            ),
            conditioning_attributes=(
                conditioning_attributes
            ),
        )

    return datasets


def build_dataloaders(
    datasets: dict[str, CartoonAvatarDataset],
    config: dict,
) -> dict[str, DataLoader]:
    # Costruisce un DataLoader per ogni split
    loader_config = config["loader"]

    loaders = {}

    for split_name in VALID_SPLITS:
        is_training = split_name == "train"

        batch_size = int(
            loader_config[
                "train_batch_size"
                if is_training
                else "evaluation_batch_size"
            ]
        )

        loaders[split_name] = build_dataloader(
            dataset=datasets[split_name],
            batch_size=batch_size,
            shuffle=is_training,
            num_workers=int(
                loader_config["num_workers"]
            ),
            pin_memory=bool(
                loader_config["pin_memory"]
            ),
            persistent_workers=bool(
                loader_config["persistent_workers"]
            ),
            prefetch_factor=int(
                loader_config["prefetch_factor"]
            ),
            # I batch evaluation non devono perdere esempi.
            drop_last=bool(
                loader_config["drop_last"]
            )
            if is_training
            else False,
            seed=int(loader_config["seed"]),
        )

    return loaders

## Test del dataset e dei dataloader

Verifichiamo cardinalità, forme, intervalli di normalizzazione, tokenizzazione,
attributi di conditioning e comportamento dei batch. L'obiettivo è intercettare
errori di integrazione prima del training del modello.

In [ ]:
%%writefile tests/test_avatar_dataset.py
import json
from pathlib import Path

import pytest
import torch
import yaml
from PIL import Image

from src.data.avatar_dataset import (
    build_dataloader,
    build_dataloaders,
    build_datasets,
    image_to_tensor,
)
from src.data.build_splits import VALID_SPLITS

CONFIG_PATH = Path("configs/data_loader.yaml")


def load_config() -> dict:
    with CONFIG_PATH.open("r", encoding="utf-8") as file:
        return yaml.safe_load(file)


def load_expected_sequence_length() -> int:
    summary_path = Path(
        "data/processed/tokenizer/tokenizer_summary.json"
    )
    assert summary_path.exists(), (
        f"Tokenizer summary non trovato: {summary_path}"
    )

    with summary_path.open("r", encoding="utf-8") as file:
        summary = json.load(file)

    return int(summary["sequence_length_with_boundaries"])


@pytest.fixture(scope="module")
def config() -> dict:
    return load_config()


@pytest.fixture(scope="module")
def datasets(config: dict) -> dict:
    return build_datasets(config)


def test_all_splits_are_built(datasets: dict) -> None:
    assert set(datasets) == set(VALID_SPLITS)


def test_expected_split_sizes(
    datasets: dict,
    config: dict,
) -> None:
    expected_counts = config["data"]["expected_split_counts"]
    for split_name in VALID_SPLITS:
        assert len(datasets[split_name]) == int(
            expected_counts[split_name]
        )


def test_conditioning_attributes_are_consistent(
    datasets: dict,
    config: dict,
) -> None:
    expected_attributes = config["data"][
        "conditioning_attributes"
    ]
    dataset = datasets["train"]

    assert dataset.conditioning_attributes == tuple(
        expected_attributes
    )
    assert dataset.conditioning_values.shape[1] == len(
        expected_attributes
    )


def test_caption_tokens_have_uniform_length(
    datasets: dict,
) -> None:
    lengths = {
        dataset.sequence_length
        for dataset in datasets.values()
    }

    assert len(lengths) == 1
    assert next(iter(lengths)) == load_expected_sequence_length()


def test_images_are_normalized(
    datasets: dict,
) -> None:
    indices = [0, len(datasets["train"]) // 2, -1]
    for index in indices:
        image = datasets["train"][index]["image"]
        assert torch.isfinite(image).all()
        assert image.min() >= -1.0
        assert image.max() <= 1.0


def test_validation_loader_keeps_all_examples(
    datasets: dict,
    config: dict,
) -> None:
    loaders = build_dataloaders(
        datasets=datasets,
        config=config,
    )
    validation_count = sum(
        batch["image"].shape[0]
        for batch in loaders["validation"]
    )
    assert validation_count == len(datasets["validation"])


def test_train_dataloader_batch(
    datasets: dict,
    config: dict,
) -> None:
    loader_config = config["loader"]

    loader = build_dataloader(
        dataset=datasets["train"],
        batch_size=64,
        shuffle=True,
        num_workers=0,
        pin_memory=bool(loader_config["pin_memory"]),
        persistent_workers=False,
        prefetch_factor=2,
        drop_last=True,
        seed=42,
    )

    batch = next(iter(loader))
    expected_sequence_length = (
        load_expected_sequence_length()
    )

    assert batch["image"].shape == (64, 3, 32, 32)
    assert batch["input_ids"].shape == (
        64,
        expected_sequence_length,
    )
    assert batch["attention_mask"].shape == (
        64,
        expected_sequence_length,
    )
    assert batch["attributes"].shape == (64, 18)
    assert batch["conditioning_values"].shape == (64, 8)

    assert batch["image"].dtype == torch.float32
    assert batch["input_ids"].dtype == torch.long
    assert batch["attention_mask"].dtype == torch.long


def test_evaluation_loader_keeps_final_batch(
    datasets: dict,
    config: dict,
) -> None:
    loaders = build_dataloaders(
        datasets=datasets,
        config=config,
    )
    expected_total = len(datasets["test_ood"])
    observed_total = sum(
        batch["image"].shape[0]
        for batch in loaders["test_ood"]
    )
    assert observed_total == expected_total


def test_train_sample_shapes_and_types(
    datasets: dict,
) -> None:
    sample = datasets["train"][0]

    assert sample["image"].shape == (3, 32, 32)
    assert sample["image"].dtype == torch.float32

    assert sample["input_ids"].ndim == 1
    assert sample["input_ids"].dtype == torch.long

    assert (
        sample["attention_mask"].shape
        == sample["input_ids"].shape
    )
    assert sample["attention_mask"].dtype == torch.long

    assert sample["attributes"].shape == (18,)
    assert sample["conditioning_values"].shape == (8,)
    assert sample["split"] == "train"


def test_image_to_tensor_rejects_invalid_size() -> None:
    image = Image.new("RGB", (8, 8))
    with pytest.raises(ValueError):
        image_to_tensor(image=image, image_size=0)

In [ ]:
%cd /content/tiny-avatar-diffusion

import json
import pandas as pd
from pathlib import Path
from tokenizers import Tokenizer

tokenizer_root = Path("data/processed/tokenizer")
captions_path = Path("data/processed/captions/metadata_with_captions.csv")

tokenizer = Tokenizer.from_file(str(tokenizer_root / "tokenizer.json"))

with (tokenizer_root / "tokenizer_summary.json").open("r", encoding="utf-8") as f:
    tokenizer_summary = json.load(f)

captioned = pd.read_csv(captions_path)
train_captions = captioned.loc[captioned["split"] == "train", "caption"].astype(str).tolist()

lengths = sorted({len(tokenizer.encode(c).ids) for c in train_captions})
max_len = max(len(tokenizer.encode(c).ids) for c in train_captions)

print("Lunghezze osservate nel tokenizer:", lengths[:10], "...", lengths[-10:])
print("Massima lunghezza reale senza SOS/EOS:", max_len)
print("Summary max_train_length_without_boundaries:", tokenizer_summary["max_train_length_without_boundaries"])
print("Summary sequence_length_with_boundaries:", tokenizer_summary["sequence_length_with_boundaries"])

example = train_captions[0]
enc = tokenizer.encode(example)
print("\nCaption esempio:")
print(example)
print("\nToken:")
print(enc.tokens)
print("Numero token reali:", len(enc.ids))

In [ ]:
%cd /content/tiny-avatar-diffusion

!python -m pytest -q \
  tests/test_metadata.py \
  tests/test_splits.py \
  tests/test_captions_and_tokenizer.py \
  tests/test_diffusion_schedule.py \
  tests/test_avatar_dataset.py

## Ispezione di un batch

L'ispezione mostra il contratto effettivo tra pipeline dati e modello:
immagini `[B, 3, 32, 32]`, token IDs `[B, 44]`, attention mask `[B, 44]`
e vettori di attributi. Questa è l'ultima verifica manuale prima di
costruire text encoder e U-Net condizionale.

In [ ]:
from pathlib import Path
import yaml

from src.data.avatar_dataset import (
    build_dataloaders,
    build_datasets,
)

config_path = Path("configs/data_loader.yaml")

with config_path.open("r", encoding="utf-8") as file:
    data_config = yaml.safe_load(file)

datasets = build_datasets(data_config)

dataloaders = build_dataloaders(
    datasets=datasets,
    config=data_config,
)

for split_name, dataset in datasets.items():
    print(
        split_name,
        "dataset:",
        len(dataset),
        "batch:",
        len(dataloaders[split_name]),
    )

In [ ]:
batch = next(
    iter(dataloaders["train"])
)

print("Chiavi:")
print(batch.keys())

print("\nShape:")
print("image:", batch["image"].shape)
print("input_ids:", batch["input_ids"].shape)
print(
    "attention_mask:",
    batch["attention_mask"].shape,
)
print(
    "attributes:",
    batch["attributes"].shape,
)
print(
    "conditioning_values:",
    batch[
        "conditioning_values"
    ].shape,
)

print("\nDtype:")
print("image:", batch["image"].dtype)
print(
    "input_ids:",
    batch["input_ids"].dtype,
)
print(
    "attention_mask:",
    batch["attention_mask"].dtype,
)

print("\nIntervallo immagini:")
print("min:", batch["image"].min().item())
print("max:", batch["image"].max().item())

print("\nPrimo elemento:")
print("ID:", batch["image_id"][0])
print("Split:", batch["split"][0])
print("Caption:", batch["caption"][0])
print(
    "Token:",
    batch["input_ids"][0].tolist(),
)
print(
    "Mask:",
    batch[
        "attention_mask"
    ][0].tolist(),
)

In [ ]:
import matplotlib.pyplot as plt

from torchvision.utils import make_grid


images = batch["image"][:16]

# Il dataset usa [-1, 1]; matplotlib richiede valori visualizzabili in [0, 1].
display_images = (
    (images + 1.0) / 2.0
).clamp(0.0, 1.0)

grid = make_grid(
    display_images,
    nrow=4,
    padding=2,
)

plt.figure(figsize=(8, 8))

plt.imshow(
    grid
    .permute(1, 2, 0)
    .cpu()
    .numpy()
)

plt.axis("off")
plt.title(
    "Training batch — 32×32 RGB"
)
plt.show()

## Visualizzazione del forward noising

Osserviamo la stessa immagine a timestep progressivamente più alti.
All'aumentare di \(t\), la componente informativa dell'avatar viene
gradualmente sostituita da rumore gaussiano.

In [ ]:
import matplotlib.pyplot as plt
import torch


x_start = batch["image"][0:1]

timesteps_to_show = [
    0,
    100,
    300,
    600,
    999,
]

generator = torch.Generator().manual_seed(42)

# Usiamo lo stesso rumore per tutti i timestep: cambia solo il peso
# attribuito a segnale e rumore, rendendo il confronto più leggibile.
fixed_noise = torch.randn(
    x_start.shape,
    generator=generator,
)

fig, axes = plt.subplots(
    1,
    len(timesteps_to_show) + 1,
    figsize=(18, 4),
)

def to_display_image(tensor: torch.Tensor):
    # Converte un tensore CHW in [-1, 1] in immagine HWC [0, 1]
    return (
        tensor
        .detach()
        .cpu()
        .squeeze(0)
        .permute(1, 2, 0)
        .add(1.0)
        .div(2.0)
        .clamp(0.0, 1.0)
        .numpy()
    )

axes[0].imshow(to_display_image(x_start))
axes[0].set_title("x0")
axes[0].axis("off")

for axis, timestep in zip(
    axes[1:],
    timesteps_to_show,
):
    timestep_tensor = torch.tensor(
        [timestep],
        dtype=torch.long,
    )

    x_t, _ = schedule.q_sample(
        x_start=x_start,
        timesteps=timestep_tensor,
        noise=fixed_noise,
    )

    axis.imshow(to_display_image(x_t))
    axis.set_title(f"t = {timestep}")
    axis.axis("off")

plt.tight_layout()
plt.show()

Il processo forward DDPM è implementato e testato: lo schedule lineare
produce immagini progressivamente più rumorose e i coefficienti della
posteriore sono disponibili per il successivo sampling inverso.

La pipeline dati restituisce batch coerenti con il training condizionale:
immagini RGB normalizzate in [-1, 1], token IDs e attention mask di lunghezza
44, attributi completi e valori di conditioning. Il blocco successivo può
implementare text encoder, embedding temporale e U-Net condizionale.

# Modelli: Text Encoder, U-Net condizionata e Reverse Sampler

In questo blocco costruiamo i tre componenti che trasformano
un prompt testuale e rumore gaussiano in una previsione di rumore:

1. **TextTransformerEncoder**: converte `input_ids` e `attention_mask`
   in rappresentazioni contestualizzate `[B, L, D]`;
2. **ConditionalUNet**: predice il rumore aggiunto all'immagine,
   condizionandosi su timestep e testo tramite cross-attention;
3. **ReverseSampler**: esegue il processo inverso DDPM, con supporto
   a classifier-free guidance (CFG).

Il Blocco 6 verifica contratti, shape, masking, gradienti, checkpoint
e interoperabilità tra componenti. Il Blocco 7 userà poi questa
infrastruttura per i tiny-overfit e le verifiche preliminari di training.

## Contratto dimensionale

Per un batch di immagini RGB a 32×32 pixel:

- immagini rumorose: `x_t` con shape `[B, 3, 32, 32]`;
- timestep: `t` con shape `[B]`, dtype `torch.long`;
- token IDs: `[B, 44]`;
- attention mask: `[B, 44]`;
- text context: `[B, 44, 128]`;
- rumore predetto dalla U-Net: `[B, 3, 32, 32]`.

Il text encoder mantiene esplicitamente a zero le posizioni di padding.
La U-Net usa la mask nella cross-attention, quindi token mascherati non
devono influenzare la previsione.

In [ ]:
from pathlib import Path

import torch

PROJECT_ROOT = Path("/content/tiny-avatar-diffusion")

if PROJECT_ROOT.exists():
    %cd /content/tiny-avatar-diffusion

assert Path("configs").exists(), "Directory configs/ non trovata."
assert Path("src").exists(), "Directory src/ non trovata."
assert Path("tests").exists(), "Directory tests/ non trovata."

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Device selezionato: {device}")

if device.type == "cuda":
    properties = torch.cuda.get_device_properties(0)

    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(
        "VRAM totale:",
        f"{properties.total_memory / 1024**3:.2f} GiB",
    )
    print(
        "BF16 supportato:",
        torch.cuda.is_bf16_supported(),
    )

## Text encoder Transformer

Il text encoder usa embedding token, positional encoding sinusoidale e
blocchi Transformer pre-norm. L'output resta a livello di token, perché
la cross-attention della U-Net deve poter selezionare dinamicamente le
parti del prompt rilevanti per ciascuna posizione spaziale dell'immagine.

La configurazione deve restare coerente con il tokenizer: vocabolario,
ID di padding e lunghezza massima sono un unico contratto.

In [ ]:
%%writefile src/models/text_encoder.py
from pathlib import Path

import torch
import yaml
from torch import Tensor, nn


def load_text_encoder_config(config_path: Path) -> dict:
    # Carica il file YAML della configurazione.
    with Path(config_path).open("r", encoding="utf-8") as file:
        config = yaml.safe_load(file)

    # Verifica la presenza della sezione principale.
    if "text_encoder" not in config:
        raise ValueError("Sezione 'text_encoder' mancante.")

    encoder_config = config["text_encoder"]

    # Chiavi minime richieste dal modello.
    required_keys = {
        "vocab_size",
        "max_sequence_length",
        "pad_token_id",
        "d_model",
        "num_heads",
        "num_layers",
        "feedforward_dim",
        "dropout",
        "positional_encoding",
        "normalization",
        "activation",
    }

    missing_keys = required_keys - set(encoder_config)
    if missing_keys:
        raise ValueError(
            "Parametri text encoder mancanti: "
            f"{sorted(missing_keys)}"
        )

    # Controlli di validità di base.
    if int(encoder_config["vocab_size"]) <= 0:
        raise ValueError("vocab_size deve essere positivo.")

    if int(encoder_config["max_sequence_length"]) <= 0:
        raise ValueError("max_sequence_length deve essere positivo.")

    if int(encoder_config["pad_token_id"]) < 0:
        raise ValueError("pad_token_id non può essere negativo.")

    if int(encoder_config["d_model"]) <= 0:
        raise ValueError("d_model deve essere positivo.")

    if int(encoder_config["num_heads"]) <= 0:
        raise ValueError("num_heads deve essere positivo.")

    if int(encoder_config["num_layers"]) <= 0:
        raise ValueError("num_layers deve essere positivo.")

    if int(encoder_config["feedforward_dim"]) <= 0:
        raise ValueError("feedforward_dim deve essere positivo.")

    if float(encoder_config["dropout"]) < 0.0 or float(encoder_config["dropout"]) >= 1.0:
        raise ValueError("dropout deve stare in [0, 1).")

    if int(encoder_config["d_model"]) % int(encoder_config["num_heads"]) != 0:
        raise ValueError("d_model deve essere divisibile per num_heads.")

    if str(encoder_config["positional_encoding"]) != "sinusoidal":
        raise ValueError("È supportato solo positional_encoding='sinusoidal'.")

    if str(encoder_config["normalization"]) != "pre_norm":
        raise ValueError("È supportato solo normalization='pre_norm'.")

    if str(encoder_config["activation"]) != "gelu":
        raise ValueError("È supportato solo activation='gelu'.")

    return config


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_sequence_length: int) -> None:
        super().__init__()

        if int(d_model) <= 0:
            raise ValueError("d_model deve essere positivo.")
        if int(max_sequence_length) <= 0:
            raise ValueError("max_sequence_length deve essere positivo.")

        # Costruisce una matrice fissa di positional encoding sinusoidale.
        position = torch.arange(max_sequence_length, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-torch.log(torch.tensor(10000.0, dtype=torch.float32)) / d_model)
        )

        positional_encoding = torch.zeros(
            max_sequence_length,
            d_model,
            dtype=torch.float32,
        )
        positional_encoding[:, 0::2] = torch.sin(position * div_term)
        positional_encoding[:, 1::2] = torch.cos(position * div_term[: positional_encoding[:, 1::2].shape[1]])

        # Buffer: si sposta con il modello ma non è un parametro addestrabile.
        self.register_buffer(
            "positional_encoding",
            positional_encoding.unsqueeze(0),
            persistent=False,
        )

    def forward(self, token_embeddings: Tensor) -> Tensor:
        if token_embeddings.ndim != 3:
            raise ValueError(
                "token_embeddings deve avere shape [batch, sequence_length, d_model]."
            )

        sequence_length = token_embeddings.shape[1]
        if sequence_length > self.positional_encoding.shape[1]:
            raise ValueError("sequence_length supera il massimo supportato.")

        return token_embeddings + self.positional_encoding[:, :sequence_length, :]


class TextTransformerEncoder(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        max_sequence_length: int,
        pad_token_id: int,
        d_model: int,
        num_heads: int,
        num_layers: int,
        feedforward_dim: int,
        dropout: float,
        positional_encoding: str,
        normalization: str,
        activation: str,
    ) -> None:
        super().__init__()

        # Attributi esposti anche ad altri componenti della pipeline.
        self.vocab_size = int(vocab_size)
        self.max_sequence_length = int(max_sequence_length)
        self.pad_token_id = int(pad_token_id)
        self.d_model = int(d_model)

        # Embedding dei token.
        self.token_embedding = nn.Embedding(
            num_embeddings=self.vocab_size,
            embedding_dim=self.d_model,
            padding_idx=self.pad_token_id,
        )

        self.position_encoding = SinusoidalPositionalEncoding(
            d_model=self.d_model,
            max_sequence_length=self.max_sequence_length,
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.d_model,
            nhead=int(num_heads),
            dim_feedforward=int(feedforward_dim),
            dropout=float(dropout),
            activation="gelu",
            batch_first=True,
            norm_first=True,  # pre-norm
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer=encoder_layer,
            num_layers=int(num_layers),
            enable_nested_tensor=False,
        )

        self.final_dropout = nn.Dropout(float(dropout))

    @classmethod
    def from_config(cls, config: dict) -> "TextTransformerEncoder":
        encoder_config = config["text_encoder"]
        return cls(
            vocab_size=int(encoder_config["vocab_size"]),
            max_sequence_length=int(encoder_config["max_sequence_length"]),
            pad_token_id=int(encoder_config["pad_token_id"]),
            d_model=int(encoder_config["d_model"]),
            num_heads=int(encoder_config["num_heads"]),
            num_layers=int(encoder_config["num_layers"]),
            feedforward_dim=int(encoder_config["feedforward_dim"]),
            dropout=float(encoder_config["dropout"]),
            positional_encoding=str(encoder_config["positional_encoding"]),
            normalization=str(encoder_config["normalization"]),
            activation=str(encoder_config["activation"]),
        )

    def forward(
        self,
        input_ids: Tensor,
        attention_mask: Tensor,
    ) -> Tensor:
        # Controlli dimensionali e di tipo.
        if input_ids.ndim != 2:
            raise ValueError("input_ids deve avere shape [batch, sequence_length].")

        if attention_mask.ndim != 2:
            raise ValueError("attention_mask deve avere shape [batch, sequence_length].")

        if input_ids.shape != attention_mask.shape:
            raise ValueError("input_ids e attention_mask devono avere la stessa shape.")

        if input_ids.dtype != torch.long:
            raise TypeError("input_ids deve avere dtype torch.long.")

        if attention_mask.dtype not in (torch.long, torch.bool, torch.int32, torch.int64):
            raise TypeError("attention_mask deve avere dtype intero o booleano.")

        if input_ids.shape[1] > self.max_sequence_length:
            raise ValueError(
                "La sequenza supera max_sequence_length: "
                f"{input_ids.shape[1]} > {self.max_sequence_length}"
            )

        # Forziamo la mask a formato intero 0/1.
        attention_mask = attention_mask.long()

        if not torch.all((attention_mask == 0) | (attention_mask == 1)):
            raise ValueError("attention_mask deve contenere solo 0 e 1.")

        # Embedding + positional encoding.
        hidden_states = self.token_embedding(input_ids)
        hidden_states = self.position_encoding(hidden_states)
        hidden_states = self.final_dropout(hidden_states)

        # In PyTorch il padding mask usa True sulle posizioni da ignorare.
        key_padding_mask = attention_mask == 0

        # Output token-level per la cross-attention della U-Net.
        encoded = self.encoder(
            hidden_states,
            src_key_padding_mask=key_padding_mask,
        )

        # Azzera esplicitamente le posizioni di padding nell'output finale.
        encoded = encoded * attention_mask.unsqueeze(-1).to(encoded.dtype)

        return encoded

In [ ]:
%%writefile tests/test_text_encoder.py
from pathlib import Path

import pytest
import torch

from src.models.text_encoder import (
    TextTransformerEncoder,
    load_text_encoder_config,
)

CONFIG_PATH = Path("configs/text_encoder.yaml")


def test_load_text_encoder_config() -> None:
    config = load_text_encoder_config(CONFIG_PATH)

    assert "text_encoder" in config
    assert int(config["text_encoder"]["vocab_size"]) > 0
    assert int(config["text_encoder"]["max_sequence_length"]) > 0
    assert int(config["text_encoder"]["pad_token_id"]) >= 0


def test_text_encoder_output_shape_and_padding() -> None:
    config = load_text_encoder_config(CONFIG_PATH)
    model = TextTransformerEncoder.from_config(config)

    sequence_length = int(config["text_encoder"]["max_sequence_length"])
    d_model = int(config["text_encoder"]["d_model"])
    pad_token_id = int(config["text_encoder"]["pad_token_id"])

    input_ids = torch.full((2, sequence_length), fill_value=pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((2, sequence_length), dtype=torch.long)

    input_ids[0, :5] = torch.tensor([2, 9, 12, 11, 3], dtype=torch.long)
    attention_mask[0, :5] = 1

    input_ids[1, :4] = torch.tensor([2, 14, 7, 3], dtype=torch.long)
    attention_mask[1, :4] = 1

    output = model(input_ids=input_ids, attention_mask=attention_mask)

    assert output.shape == (2, sequence_length, d_model)
    assert torch.isfinite(output).all()

    padded_positions = attention_mask == 0
    assert torch.allclose(
        output[padded_positions],
        torch.zeros_like(output[padded_positions]),
        atol=1e-6,
    )


def test_text_encoder_rejects_wrong_dtype() -> None:
    config = load_text_encoder_config(CONFIG_PATH)
    model = TextTransformerEncoder.from_config(config)

    sequence_length = int(config["text_encoder"]["max_sequence_length"])

    # input_ids deve essere torch.long.
    with pytest.raises(TypeError, match="input_ids deve avere dtype torch.long."):
        model(
            input_ids=torch.zeros((1, sequence_length), dtype=torch.float32),
            attention_mask=torch.ones((1, sequence_length), dtype=torch.long),
        )

In [ ]:
%%writefile configs/text_encoder.yaml
# Configurazione del Transformer text encoder.

text_encoder:
  # Contratto con il tokenizer.
  vocab_size: 132
  max_sequence_length: 44
  pad_token_id: 1

  # Architettura Transformer.
  d_model: 128
  num_heads: 4
  num_layers: 4
  feedforward_dim: 512
  dropout: 0.10

  # Scelte implementative esplicite.
  positional_encoding: sinusoidal
  normalization: pre_norm
  activation: gelu

In [ ]:
from pathlib import Path

import torch
from tokenizers import Tokenizer

from src.models.text_encoder import (
    TextTransformerEncoder,
    load_text_encoder_config,
)


# Percorso della configurazione coerente con il nuovo loader.
TEXT_CONFIG_PATH = Path("configs/text_encoder.yaml")

# Tokenizer addestrato soltanto sulle caption dello split train.
TOKENIZER_PATH = Path("data/processed/tokenizer/tokenizer.json")


# Carica e valida la configurazione YAML.
text_config = load_text_encoder_config(TEXT_CONFIG_PATH)

# Estrae la sezione dedicata al text encoder.
encoder_config = text_config["text_encoder"]


# Carica il tokenizer WordLevel precedentemente costruito.
tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))


# Verifica il contratto tokenizer <-> text encoder.
assert tokenizer.get_vocab_size() == encoder_config["vocab_size"]
assert encoder_config["max_sequence_length"] == 44
assert encoder_config["pad_token_id"] == 1


# Crea il Transformer e lo sposta sul device selezionato.
text_encoder = TextTransformerEncoder.from_config(
    text_config
).to(device)

# Modalità evaluation: disabilita dropout durante controlli e sampling.
text_encoder.eval()


# Conteggio dei parametri per documentare la dimensione del modello.
parameter_count = sum(
    parameter.numel()
    for parameter in text_encoder.parameters()
)


print("Text encoder pronto.")
print(f"Vocab size: {encoder_config['vocab_size']}")
print(
    "Max sequence length:",
    encoder_config["max_sequence_length"],
)
print(f"Context dimension: {encoder_config['d_model']}")
print(f"Parametri: {parameter_count:,}")
print(
    "Memoria parametri FP32:",
    f"{parameter_count * 4 / 1024**2:.2f} MiB",
)

## U-Net condizionata

La U-Net riceve un'immagine rumorosa `x_t` e il timestep `t`; predice il rumore `epsilon_theta(x_t, t, c)`. Il conditioning testuale `c` viene inserito mediante cross-attention nei livelli indicati dalla configurazione e nel bottleneck.

Manteniamo anche la variante unconditional della stessa architettura: viene addestrata come baseline generativa separata, senza text encoder né context. Nel protocollo finale il sampler conditional usa `guidance_scale = 1.0`, perché il training non introduce conditioning dropout per classifier-free guidance.


In [ ]:
%%writefile src/models/conditional_unet.py

from __future__ import annotations

import math
from pathlib import Path
from typing import Optional

import torch
import torch.nn.functional as F
import yaml
from torch import Tensor, nn


def load_unet_config(config_path: Path) -> dict:
    # Carica e valida la configurazione della U-Net

    with Path(config_path).open("r", encoding="utf-8") as file:
        config = yaml.safe_load(file)

    if "unet" not in config:
        raise ValueError("Sezione 'unet' mancante.")

    unet_config = config["unet"]

    required_keys = {
        "image_channels",
        "output_channels",
        "base_channels",
        "channel_multipliers",
        "num_res_blocks",
        "normalization_groups",
        "dropout",
        "time_embedding_dim",
        "conditional",
        "context_dim",
        "attention_levels",
        "attention_heads",
    }

    missing_keys = required_keys - set(unet_config)

    if missing_keys:
        raise ValueError(
            "Parametri U-Net mancanti: "
            f"{sorted(missing_keys)}"
        )

    image_channels = int(unet_config["image_channels"])
    output_channels = int(unet_config["output_channels"])
    base_channels = int(unet_config["base_channels"])

    channel_multipliers = [
        int(value)
        for value in unet_config["channel_multipliers"]
    ]

    num_res_blocks = int(unet_config["num_res_blocks"])
    normalization_groups = int(
        unet_config["normalization_groups"]
    )
    time_embedding_dim = int(
        unet_config["time_embedding_dim"]
    )
    context_dim = int(unet_config["context_dim"])
    attention_heads = int(unet_config["attention_heads"])

    attention_levels = [
        int(value)
        for value in unet_config["attention_levels"]
    ]

    if image_channels < 1:
        raise ValueError("image_channels deve essere positivo.")

    if output_channels < 1:
        raise ValueError("output_channels deve essere positivo.")

    if base_channels < 1:
        raise ValueError("base_channels deve essere positivo.")

    if not channel_multipliers:
        raise ValueError(
            "channel_multipliers non può essere vuoto."
        )

    if any(value < 1 for value in channel_multipliers):
        raise ValueError(
            "channel_multipliers deve contenere valori positivi."
        )

    if num_res_blocks < 1:
        raise ValueError(
            "num_res_blocks deve essere positivo."
        )

    if normalization_groups < 1:
        raise ValueError(
            "normalization_groups deve essere positivo."
        )

    if time_embedding_dim < 1:
        raise ValueError(
            "time_embedding_dim deve essere positivo."
        )

    if context_dim < 1:
        raise ValueError("context_dim deve essere positivo.")

    if attention_heads < 1:
        raise ValueError(
            "attention_heads deve essere positivo."
        )

    if not 0.0 <= float(unet_config["dropout"]) < 1.0:
        raise ValueError("dropout deve appartenere a [0, 1).")

    channels = [
        base_channels * multiplier
        for multiplier in channel_multipliers
    ]

    for channel_count in channels:
        if channel_count % normalization_groups != 0:
            raise ValueError(
                f"{channel_count} canali non divisibili per "
                f"{normalization_groups} gruppi."
            )

    invalid_attention_levels = [
        level
        for level in attention_levels
        if level < 0 or level >= len(channels)
    ]

    if invalid_attention_levels:
        raise ValueError(
            "Livelli attention non validi: "
            f"{invalid_attention_levels}"
        )

    for level in attention_levels:
        if channels[level] % attention_heads != 0:
            raise ValueError(
                f"I canali del livello {level} non sono "
                "divisibili per attention_heads."
            )

    return config


class SinusoidalTimeEmbedding(nn.Module):
    # Embedding sinusoidale dei timestep DDPM

    def __init__(
        self,
        embedding_dim: int,
        max_period: float = 10_000.0,
    ) -> None:
        super().__init__()

        self.embedding_dim = int(embedding_dim)
        self.max_period = float(max_period)

    def forward(self, timesteps: Tensor) -> Tensor:
        if timesteps.ndim != 1:
            raise ValueError(
                "timesteps deve avere shape [batch]."
            )

        half_dim = self.embedding_dim // 2

        if half_dim == 0:
            return timesteps.float().unsqueeze(1)

        frequencies = torch.exp(
            -math.log(self.max_period)
            * torch.arange(
                half_dim,
                device=timesteps.device,
                dtype=torch.float32,
            )
            / max(half_dim - 1, 1)
        )

        arguments = (
            timesteps.float().unsqueeze(1)
            * frequencies.unsqueeze(0)
        )

        embedding = torch.cat(
            [
                torch.sin(arguments),
                torch.cos(arguments),
            ],
            dim=1,
        )

        if embedding.shape[1] < self.embedding_dim:
            embedding = F.pad(embedding, (0, 1))

        return embedding


class ResidualBlock(nn.Module):
    # Blocco residuale con iniezione dell'embedding temporale.

    # Il timestep viene proiettato sui canali e sommato come bias
    # spazialmente condiviso dopo la prima convoluzione.
    def __init__(
        self,
        input_channels: int,
        output_channels: int,
        time_embedding_dim: int,
        normalization_groups: int,
        dropout: float,
    ) -> None:
        super().__init__()

        self.input_normalization = nn.GroupNorm(
            normalization_groups,
            input_channels,
        )

        self.input_convolution = nn.Conv2d(
            input_channels,
            output_channels,
            kernel_size=3,
            padding=1,
        )

        self.time_projection = nn.Sequential(
            nn.SiLU(),
            nn.Linear(
                time_embedding_dim,
                output_channels,
            ),
        )

        self.output_normalization = nn.GroupNorm(
            normalization_groups,
            output_channels,
        )

        self.dropout = nn.Dropout(dropout)

        self.output_convolution = nn.Conv2d(
            output_channels,
            output_channels,
            kernel_size=3,
            padding=1,
        )

        if input_channels == output_channels:
            self.residual_projection = nn.Identity()
        else:
            self.residual_projection = nn.Conv2d(
                input_channels,
                output_channels,
                kernel_size=1,
            )

    def forward(
        self,
        hidden_states: Tensor,
        time_embedding: Tensor,
    ) -> Tensor:
        residual = self.residual_projection(hidden_states)

        hidden_states = self.input_normalization(
            hidden_states
        )
        hidden_states = F.silu(hidden_states)
        hidden_states = self.input_convolution(
            hidden_states
        )

        time_bias = self.time_projection(
            time_embedding
        ).unsqueeze(-1).unsqueeze(-1)

        hidden_states = hidden_states + time_bias

        hidden_states = self.output_normalization(
            hidden_states
        )
        hidden_states = F.silu(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.output_convolution(
            hidden_states
        )

        return hidden_states + residual


class CrossAttention2D(nn.Module):
    # Cross-attention tra token spaziali dell'immagine e token testuali.

    # Il context è l'output token-level del Transformer, non un vettore
    # globale: ogni posizione spaziale può quindi pesare diversamente gli
    # attributi testuali della caption.
    def __init__(
        self,
        channels: int,
        context_dim: int,
        attention_heads: int,
        normalization_groups: int,
        dropout: float,
    ) -> None:
        super().__init__()

        if channels % attention_heads != 0:
            raise ValueError(
                "channels deve essere divisibile per attention_heads."
            )

        self.channels = int(channels)
        self.context_dim = int(context_dim)
        self.attention_heads = int(attention_heads)
        self.head_dim = self.channels // self.attention_heads
        self.scale = self.head_dim ** -0.5

        self.normalization = nn.GroupNorm(
            normalization_groups,
            self.channels,
        )

        self.query_projection = nn.Linear(
            self.channels,
            self.channels,
        )

        self.key_projection = nn.Linear(
            self.context_dim,
            self.channels,
        )

        self.value_projection = nn.Linear(
            self.context_dim,
            self.channels,
        )

        self.output_projection = nn.Linear(
            self.channels,
            self.channels,
        )

        self.output_dropout = nn.Dropout(dropout)

    def forward(
        self,
        hidden_states: Tensor,
        context: Tensor,
        attention_mask: Tensor,
    ) -> Tensor:
        if hidden_states.ndim != 4:
            raise ValueError(
                "hidden_states deve avere shape "
                "[batch, channels, height, width]."
            )

        if context.ndim != 3:
            raise ValueError(
                "context deve avere shape "
                "[batch, sequence, context_dim]."
            )

        if attention_mask.ndim != 2:
            raise ValueError(
                "attention_mask deve avere shape "
                "[batch, sequence]."
            )

        batch_size, channels, height, width = (
            hidden_states.shape
        )

        if context.shape[0] != batch_size:
            raise ValueError(
                "Batch incoerente tra immagini e context."
            )

        if context.shape[2] != self.context_dim:
            raise ValueError("context_dim non compatibile.")

        if attention_mask.shape != context.shape[:2]:
            raise ValueError(
                "attention_mask non compatibile con context."
            )

        valid_tokens = attention_mask.to(torch.bool)

        if not valid_tokens.any(dim=1).all():
            raise ValueError(
                "Ogni caption deve contenere almeno un token valido."
            )

        residual = hidden_states

        normalized_states = self.normalization(
            hidden_states
        )

        image_tokens = normalized_states.flatten(
            start_dim=2
        ).transpose(1, 2)

        query = self.query_projection(image_tokens)
        key = self.key_projection(context)
        value = self.value_projection(context)

        query = query.reshape(
            batch_size,
            height * width,
            self.attention_heads,
            self.head_dim,
        ).transpose(1, 2)

        key = key.reshape(
            batch_size,
            context.shape[1],
            self.attention_heads,
            self.head_dim,
        ).transpose(1, 2)

        value = value.reshape(
            batch_size,
            context.shape[1],
            self.attention_heads,
            self.head_dim,
        ).transpose(1, 2)

        attention_scores = torch.matmul(
            query.float(),
            key.float().transpose(-2, -1),
        ) * self.scale

        token_mask = valid_tokens.unsqueeze(1).unsqueeze(2)

        attention_scores = attention_scores.masked_fill(
            ~token_mask,
            torch.finfo(attention_scores.dtype).min,
        )

        attention_probabilities = torch.softmax(
            attention_scores,
            dim=-1,
        ).to(dtype=value.dtype)

        attended = torch.matmul(
            attention_probabilities,
            value,
        )

        attended = attended.transpose(1, 2).contiguous()
        attended = attended.reshape(
            batch_size,
            height * width,
            channels,
        )

        attended = self.output_projection(attended)
        attended = self.output_dropout(attended)

        attended = attended.transpose(1, 2).reshape(
            batch_size,
            channels,
            height,
            width,
        )

        return residual + attended


class Downsample2D(nn.Module):
    # Riduce altezza e larghezza di un fattore due

    def __init__(self, channels: int) -> None:
        super().__init__()

        self.convolution = nn.Conv2d(
            channels,
            channels,
            kernel_size=3,
            stride=2,
            padding=1,
        )

    def forward(self, hidden_states: Tensor) -> Tensor:
        return self.convolution(hidden_states)


class Upsample2D(nn.Module):
    # Aumenta altezza e larghezza di un fattore due

    def __init__(
        self,
        input_channels: int,
        output_channels: int,
    ) -> None:
        super().__init__()

        self.convolution = nn.Conv2d(
            input_channels,
            output_channels,
            kernel_size=3,
            padding=1,
        )

    def forward(self, hidden_states: Tensor) -> Tensor:
        hidden_states = F.interpolate(
            hidden_states,
            scale_factor=2.0,
            mode="nearest",
        )

        return self.convolution(hidden_states)


class DownBlock(nn.Module):
    # Livello encoder della U-Net

    def __init__(
        self,
        input_channels: int,
        output_channels: int,
        time_embedding_dim: int,
        num_res_blocks: int,
        normalization_groups: int,
        dropout: float,
        use_attention: bool,
        context_dim: int,
        attention_heads: int,
        add_downsample: bool,
    ) -> None:
        super().__init__()

        blocks = []

        for block_index in range(num_res_blocks):
            blocks.append(
                ResidualBlock(
                    input_channels=(
                        input_channels
                        if block_index == 0
                        else output_channels
                    ),
                    output_channels=output_channels,
                    time_embedding_dim=time_embedding_dim,
                    normalization_groups=normalization_groups,
                    dropout=dropout,
                )
            )

        self.residual_blocks = nn.ModuleList(blocks)

        self.attention = (
            CrossAttention2D(
                channels=output_channels,
                context_dim=context_dim,
                attention_heads=attention_heads,
                normalization_groups=normalization_groups,
                dropout=dropout,
            )
            if use_attention
            else None
        )

        self.downsample = (
            Downsample2D(output_channels)
            if add_downsample
            else None
        )

    def forward(
        self,
        hidden_states: Tensor,
        time_embedding: Tensor,
        context: Optional[Tensor],
        attention_mask: Optional[Tensor],
    ) -> tuple[Tensor, Tensor]:
        for block in self.residual_blocks:
            hidden_states = block(
                hidden_states,
                time_embedding,
            )

        if self.attention is not None:
            if context is None or attention_mask is None:
                raise ValueError(
                    "Context e attention_mask richiesti "
                    "nei livelli condizionati."
                )

            hidden_states = self.attention(
                hidden_states,
                context,
                attention_mask,
            )

        skip_connection = hidden_states

        if self.downsample is not None:
            hidden_states = self.downsample(hidden_states)

        return hidden_states, skip_connection


class UpBlock(nn.Module):
    # Livello decoder della U-Net con concatenazione skip

    def __init__(
        self,
        input_channels: int,
        skip_channels: int,
        output_channels: int,
        time_embedding_dim: int,
        num_res_blocks: int,
        normalization_groups: int,
        dropout: float,
        use_attention: bool,
        context_dim: int,
        attention_heads: int,
        add_upsample: bool,
        upsample_channels: Optional[int],
    ) -> None:
        super().__init__()

        blocks = []

        for block_index in range(num_res_blocks):
            blocks.append(
                ResidualBlock(
                    input_channels=(
                        input_channels + skip_channels
                        if block_index == 0
                        else output_channels
                    ),
                    output_channels=output_channels,
                    time_embedding_dim=time_embedding_dim,
                    normalization_groups=normalization_groups,
                    dropout=dropout,
                )
            )

        self.residual_blocks = nn.ModuleList(blocks)

        self.attention = (
            CrossAttention2D(
                channels=output_channels,
                context_dim=context_dim,
                attention_heads=attention_heads,
                normalization_groups=normalization_groups,
                dropout=dropout,
            )
            if use_attention
            else None
        )

        self.upsample = (
            Upsample2D(
                input_channels=output_channels,
                output_channels=int(upsample_channels),
            )
            if add_upsample
            else None
        )

    def forward(
        self,
        hidden_states: Tensor,
        skip_connection: Tensor,
        time_embedding: Tensor,
        context: Optional[Tensor],
        attention_mask: Optional[Tensor],
    ) -> Tensor:
        if hidden_states.shape[-2:] != skip_connection.shape[-2:]:
            raise ValueError(
                "Risoluzioni incompatibili tra decoder e skip connection."
            )

        hidden_states = torch.cat(
            [hidden_states, skip_connection],
            dim=1,
        )

        for block in self.residual_blocks:
            hidden_states = block(
                hidden_states,
                time_embedding,
            )

        if self.attention is not None:
            if context is None or attention_mask is None:
                raise ValueError(
                    "Context e attention_mask richiesti "
                    "nei livelli condizionati."
                )

            hidden_states = self.attention(
                hidden_states,
                context,
                attention_mask,
            )

        if self.upsample is not None:
            hidden_states = self.upsample(hidden_states)

        return hidden_states


class MiddleBlock(nn.Module):
    # Blocco centrale alla minima risoluzione spaziale

    def __init__(
        self,
        channels: int,
        time_embedding_dim: int,
        normalization_groups: int,
        dropout: float,
        use_attention: bool,
        context_dim: int,
        attention_heads: int,
    ) -> None:
        super().__init__()

        self.block1 = ResidualBlock(
            input_channels=channels,
            output_channels=channels,
            time_embedding_dim=time_embedding_dim,
            normalization_groups=normalization_groups,
            dropout=dropout,
        )

        self.attention = (
            CrossAttention2D(
                channels=channels,
                context_dim=context_dim,
                attention_heads=attention_heads,
                normalization_groups=normalization_groups,
                dropout=dropout,
            )
            if use_attention
            else None
        )

        self.block2 = ResidualBlock(
            input_channels=channels,
            output_channels=channels,
            time_embedding_dim=time_embedding_dim,
            normalization_groups=normalization_groups,
            dropout=dropout,
        )

    def forward(
        self,
        hidden_states: Tensor,
        time_embedding: Tensor,
        context: Optional[Tensor],
        attention_mask: Optional[Tensor],
    ) -> Tensor:
        hidden_states = self.block1(
            hidden_states,
            time_embedding,
        )

        if self.attention is not None:
            if context is None or attention_mask is None:
                raise ValueError(
                    "Context e attention_mask richiesti "
                    "nel middle block condizionato."
                )

            hidden_states = self.attention(
                hidden_states,
                context,
                attention_mask,
            )

        return self.block2(
            hidden_states,
            time_embedding,
        )


class ConditionalUNet(nn.Module):
    # U-Net DDPM compatta, condizionata opzionalmente dal testo.

    # La struttura usa una piramide a più risoluzioni, collegamenti skip
    # encoder-decoder e cross-attention nei livelli indicati dal file YAML.
    def __init__(
        self,
        image_channels: int,
        output_channels: int,
        base_channels: int,
        channel_multipliers: list[int],
        num_res_blocks: int,
        normalization_groups: int,
        dropout: float,
        time_embedding_dim: int,
        conditional: bool,
        context_dim: Optional[int],
        attention_levels: list[int],
        attention_heads: int,
    ) -> None:
        super().__init__()

        self.image_channels = int(image_channels)
        self.output_channels = int(output_channels)
        self.base_channels = int(base_channels)

        self.channel_multipliers = [
            int(value)
            for value in channel_multipliers
        ]

        self.num_res_blocks = int(num_res_blocks)
        self.normalization_groups = int(
            normalization_groups
        )
        self.time_embedding_dim = int(
            time_embedding_dim
        )
        self.conditional = bool(conditional)

        self.context_dim = (
            None
            if context_dim is None
            else int(context_dim)
        )

        self.attention_levels = [
            int(value)
            for value in attention_levels
        ]

        self.attention_heads = int(attention_heads)

        if self.conditional and self.context_dim is None:
            raise ValueError(
                "context_dim richiesto se conditional=True."
            )

        self.channels = [
            self.base_channels * multiplier
            for multiplier in self.channel_multipliers
        ]

        self.input_convolution = nn.Conv2d(
            self.image_channels,
            self.channels[0],
            kernel_size=3,
            padding=1,
        )

        self.time_embedding = SinusoidalTimeEmbedding(
            self.time_embedding_dim
        )

        self.time_mlp = nn.Sequential(
            nn.Linear(
                self.time_embedding_dim,
                self.time_embedding_dim,
            ),
            nn.SiLU(),
            nn.Linear(
                self.time_embedding_dim,
                self.time_embedding_dim,
            ),
        )

        down_blocks = []

        for level, output_channels_at_level in enumerate(
            self.channels
        ):
            input_channels_at_level = (
                self.channels[level - 1]
                if level > 0
                else self.channels[0]
            )

            down_blocks.append(
                DownBlock(
                    input_channels=input_channels_at_level,
                    output_channels=output_channels_at_level,
                    time_embedding_dim=self.time_embedding_dim,
                    num_res_blocks=self.num_res_blocks,
                    normalization_groups=self.normalization_groups,
                    dropout=float(dropout),
                    use_attention=(
                        self.conditional
                        and level in self.attention_levels
                    ),
                    context_dim=(
                        int(self.context_dim)
                        if self.conditional
                        else 1
                    ),
                    attention_heads=self.attention_heads,
                    add_downsample=(
                        level < len(self.channels) - 1
                    ),
                )
            )

        self.down_blocks = nn.ModuleList(down_blocks)

        deepest_level = len(self.channels) - 1

        self.middle_block = MiddleBlock(
            channels=self.channels[-1],
            time_embedding_dim=self.time_embedding_dim,
            normalization_groups=self.normalization_groups,
            dropout=float(dropout),
            use_attention=(
                self.conditional
                and deepest_level in self.attention_levels
            ),
            context_dim=(
                int(self.context_dim)
                if self.conditional
                else 1
            ),
            attention_heads=self.attention_heads,
        )

        up_blocks = []
        current_channels = self.channels[-1]

        for level in reversed(range(len(self.channels))):
            output_channels_at_level = self.channels[level]

            add_upsample = level > 0

            up_blocks.append(
                UpBlock(
                    input_channels=current_channels,
                    skip_channels=self.channels[level],
                    output_channels=output_channels_at_level,
                    time_embedding_dim=self.time_embedding_dim,
                    num_res_blocks=self.num_res_blocks,
                    normalization_groups=self.normalization_groups,
                    dropout=float(dropout),
                    use_attention=(
                        self.conditional
                        and level in self.attention_levels
                    ),
                    context_dim=(
                        int(self.context_dim)
                        if self.conditional
                        else 1
                    ),
                    attention_heads=self.attention_heads,
                    add_upsample=add_upsample,
                    upsample_channels=(
                        self.channels[level - 1]
                        if add_upsample
                        else None
                    ),
                )
            )

            current_channels = (
                self.channels[level - 1]
                if add_upsample
                else output_channels_at_level
            )

        self.up_blocks = nn.ModuleList(up_blocks)

        self.output_normalization = nn.GroupNorm(
            self.normalization_groups,
            self.channels[0],
        )

        self.output_convolution = nn.Conv2d(
            self.channels[0],
            self.output_channels,
            kernel_size=3,
            padding=1,
        )

    @classmethod
    def from_config(
        cls,
        config: dict,
        conditional: Optional[bool] = None,
    ) -> "ConditionalUNet":
        unet_config = config["unet"]

        effective_conditional = (
            bool(unet_config["conditional"])
            if conditional is None
            else bool(conditional)
        )

        return cls(
            image_channels=int(
                unet_config["image_channels"]
            ),
            output_channels=int(
                unet_config["output_channels"]
            ),
            base_channels=int(
                unet_config["base_channels"]
            ),
            channel_multipliers=[
                int(value)
                for value
                in unet_config["channel_multipliers"]
            ],
            num_res_blocks=int(
                unet_config["num_res_blocks"]
            ),
            normalization_groups=int(
                unet_config["normalization_groups"]
            ),
            dropout=float(unet_config["dropout"]),
            time_embedding_dim=int(
                unet_config["time_embedding_dim"]
            ),
            conditional=effective_conditional,
            context_dim=(
                int(unet_config["context_dim"])
                if effective_conditional
                else None
            ),
            attention_levels=[
                int(value)
                for value
                in unet_config["attention_levels"]
            ],
            attention_heads=int(
                unet_config["attention_heads"]
            ),
        )

    def forward(
        self,
        noisyimages: Tensor,
        timesteps: Tensor,
        context: Optional[Tensor],
        attention_mask: Optional[Tensor],
    ) -> Tensor:
        if noisyimages.ndim != 4:
            raise ValueError(
                "noisyimages deve avere shape "
                "[batch, channels, height, width]."
            )

        if noisyimages.shape[1] != self.image_channels:
            raise ValueError(
                "Numero di canali immagine non compatibile."
            )

        if timesteps.ndim != 1:
            raise ValueError(
                "timesteps deve avere shape [batch]."
            )

        if noisyimages.shape[0] != timesteps.shape[0]:
            raise ValueError(
                "Batch incoerente tra immagini e timestep."
            )

        if self.conditional:
            if context is None or attention_mask is None:
                raise ValueError(
                    "Context e attention_mask richiesti "
                    "in modalità conditional."
                )

            if context.ndim != 3:
                raise ValueError(
                    "context deve avere shape "
                    "[batch, sequence, context_dim]."
                )

            if attention_mask.ndim != 2:
                raise ValueError(
                    "attention_mask deve avere shape "
                    "[batch, sequence]."
                )

            if context.shape[0] != noisyimages.shape[0]:
                raise ValueError(
                    "Batch incoerente tra immagini e testo."
                )

            if context.shape[2] != self.context_dim:
                raise ValueError(
                    "Dimensione context non compatibile."
                )

            if attention_mask.shape != context.shape[:2]:
                raise ValueError(
                    "attention_mask non compatibile con context."
                )

        else:
            context = None
            attention_mask = None

        time_embedding = self.time_embedding(timesteps)
        time_embedding = self.time_mlp(time_embedding)

        hidden_states = self.input_convolution(noisyimages)

        skip_connections = []

        for down_block in self.down_blocks:
            hidden_states, skip_connection = down_block(
                hidden_states=hidden_states,
                time_embedding=time_embedding,
                context=context,
                attention_mask=attention_mask,
            )

            skip_connections.append(skip_connection)

        hidden_states = self.middle_block(
            hidden_states=hidden_states,
            time_embedding=time_embedding,
            context=context,
            attention_mask=attention_mask,
        )

        for up_block, skip_connection in zip(
            self.up_blocks,
            reversed(skip_connections),
        ):
            hidden_states = up_block(
                hidden_states=hidden_states,
                skip_connection=skip_connection,
                time_embedding=time_embedding,
                context=context,
                attention_mask=attention_mask,
            )

        hidden_states = self.output_normalization(
            hidden_states
        )
        hidden_states = F.silu(hidden_states)

        predicted_noise = self.output_convolution(
            hidden_states
        )

        if predicted_noise.shape != noisyimages.shape:
            raise RuntimeError(
                "La U-Net non ha preservato la shape dell'immagine: "
                f"{tuple(predicted_noise.shape)} invece di "
                f"{tuple(noisyimages.shape)}."
            )

        return predicted_noise

In [ ]:
%%writefile tests/test_conditional_unet.py

from pathlib import Path

import pytest
import torch

from src.models.conditional_unet import (
    ConditionalUNet,
    load_unet_config,
)


CONFIG_PATH = Path("configs/unet.yaml")


@pytest.fixture(scope="module")
def config() -> dict:
    return load_unet_config(CONFIG_PATH)


@pytest.fixture(scope="module")
def conditional_unet(config: dict) -> ConditionalUNet:
    torch.manual_seed(2026)

    model = ConditionalUNet.from_config(
        config,
        conditional=True,
    )

    model.eval()

    return model


@pytest.fixture(scope="module")
def unconditional_unet(config: dict) -> ConditionalUNet:
    torch.manual_seed(2026)

    model = ConditionalUNet.from_config(
        config,
        conditional=False,
    )

    model.eval()

    return model


def build_conditional_inputs(
    batch_size: int = 2,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    # Costruisce un batch sintetico compatibile con le configurazioni
    # ufficiali del progetto.

    torch.manual_seed(2026)

    noisyimages = torch.randn(
        batch_size,
        3,
        32,
        32,
    )

    timesteps = torch.tensor(
        [10, 750][:batch_size],
        dtype=torch.long,
    )

    context = torch.randn(
        batch_size,
        44,
        128,
    )

    attention_mask = torch.zeros(
        batch_size,
        44,
        dtype=torch.long,
    )

    attention_mask[:, :8] = 1

    return (
        noisyimages,
        timesteps,
        context,
        attention_mask,
    )


def test_load_unet_config(config: dict) -> None:
    unet_config = config["unet"]

    assert unet_config["image_channels"] == 3
    assert unet_config["output_channels"] == 3
    assert unet_config["base_channels"] == 64

    assert unet_config["channel_multipliers"] == [
        1,
        2,
        4,
    ]

    assert unet_config["num_res_blocks"] == 2
    assert unet_config["time_embedding_dim"] == 256
    assert unet_config["context_dim"] == 128
    assert unet_config["attention_levels"] == [1, 2]
    assert unet_config["attention_heads"] == 4


def test_conditional_unet_is_a_real_multiresolution_unet(
    conditional_unet: ConditionalUNet,
) -> None:
    # Verifica strutturale: la configurazione deve governare davvero
    # encoder, decoder, downsampling, upsampling e skip connection.

    assert len(conditional_unet.channels) == 3
    assert conditional_unet.channels == [64, 128, 256]

    assert len(conditional_unet.down_blocks) == 3
    assert len(conditional_unet.up_blocks) == 3

    # I primi due livelli encoder riducono la risoluzione.
    assert conditional_unet.down_blocks[0].downsample is not None
    assert conditional_unet.down_blocks[1].downsample is not None
    assert conditional_unet.down_blocks[2].downsample is None

    # I primi due blocchi decoder, ordinati dal livello profondo
    # verso quello superficiale, riportano alla risoluzione iniziale.
    assert conditional_unet.up_blocks[0].upsample is not None
    assert conditional_unet.up_blocks[1].upsample is not None
    assert conditional_unet.up_blocks[2].upsample is None

    # I blocchi residuali configurati sono realmente presenti.
    for down_block in conditional_unet.down_blocks:
        assert len(down_block.residual_blocks) == 2

    for up_block in conditional_unet.up_blocks:
        assert len(up_block.residual_blocks) == 2

    # attention_levels=[1, 2]: nessuna attention al livello 0,
    # attention ai due livelli più profondi.
    assert conditional_unet.down_blocks[0].attention is None
    assert conditional_unet.down_blocks[1].attention is not None
    assert conditional_unet.down_blocks[2].attention is not None

    assert conditional_unet.middle_block.attention is not None

    # L'ordine del decoder è: livello 2, livello 1, livello 0.
    assert conditional_unet.up_blocks[0].attention is not None
    assert conditional_unet.up_blocks[1].attention is not None
    assert conditional_unet.up_blocks[2].attention is None


def test_conditional_forward_preserves_image_shape(
    conditional_unet: ConditionalUNet,
) -> None:
    (
        noisyimages,
        timesteps,
        context,
        attention_mask,
    ) = build_conditional_inputs()

    with torch.inference_mode():
        predicted_noise = conditional_unet(
            noisyimages=noisyimages,
            timesteps=timesteps,
            context=context,
            attention_mask=attention_mask,
        )

    assert predicted_noise.shape == noisyimages.shape
    assert torch.isfinite(predicted_noise).all()


def test_unconditional_forward_preserves_image_shape(
    unconditional_unet: ConditionalUNet,
) -> None:
    torch.manual_seed(2026)

    noisyimages = torch.randn(2, 3, 32, 32)

    timesteps = torch.tensor(
        [10, 750],
        dtype=torch.long,
    )

    with torch.inference_mode():
        predicted_noise = unconditional_unet(
            noisyimages=noisyimages,
            timesteps=timesteps,
            context=None,
            attention_mask=None,
        )

    assert predicted_noise.shape == noisyimages.shape
    assert torch.isfinite(predicted_noise).all()


def test_conditional_model_requires_text_context(
    conditional_unet: ConditionalUNet,
) -> None:
    noisyimages = torch.randn(1, 3, 32, 32)
    timesteps = torch.tensor([10], dtype=torch.long)

    with pytest.raises(ValueError, match="Context"):
        conditional_unet(
            noisyimages=noisyimages,
            timesteps=timesteps,
            context=None,
            attention_mask=None,
        )


def test_unet_has_meaningful_capacity(
    conditional_unet: ConditionalUNet,
    unconditional_unet: ConditionalUNet,
) -> None:
    # Evita una nuova regressione alla rete piatta da circa 0.3 M
    # parametri: il denoiser deve avere capacità adeguata al dataset.

    conditional_parameters = sum(
        parameter.numel()
        for parameter in conditional_unet.parameters()
    )

    unconditional_parameters = sum(
        parameter.numel()
        for parameter in unconditional_unet.parameters()
    )

    assert conditional_parameters > 5_000_000
    assert unconditional_parameters > 5_000_000

    # La cross-attention rende la versione conditional più grande.
    assert conditional_parameters > unconditional_parameters

In [ ]:
%%bash
set -euo pipefail

cat > configs/unet.yaml <<'EOF'
unet:
  # Canali immagine in input e output.
  image_channels: 3
  output_channels: 3

  # Larghezza base della U-Net compatta.
  base_channels: 64

  # Campi mantenuti in configurazione per coerenza progettuale.
  channel_multipliers:
    - 1
    - 2
    - 4

  num_res_blocks: 2
  normalization_groups: 8
  dropout: 0.10

  # Embedding temporale DDPM.
  time_embedding_dim: 256

  # Conditioning testuale.
  conditional: true
  context_dim: 128
  attention_levels:
    - 1
    - 2
  attention_heads: 4
EOF

echo "Configurazione U-Net:"
cat configs/unet.yaml

In [ ]:
from pathlib import Path
import importlib
import torch
import src.models.conditional_unet as conditional_unet_module

ConditionalUNet = conditional_unet_module.ConditionalUNet
load_unet_config = conditional_unet_module.load_unet_config

UNET_CONFIG_PATH = Path("configs/unet.yaml")
unet_config = load_unet_config(UNET_CONFIG_PATH)

# Istanzia la U-Net condizionata dal testo.
conditional_unet = ConditionalUNet.from_config(
    unet_config,
    conditional=True,
).to(device)

# Istanzia la variante unconditional, utile come baseline
# Il protocollo finale usa guidance_scale=1.0, quindi non applica
# classifier-free guidance durante il campionamento.
unconditional_unet = ConditionalUNet.from_config(
    unet_config,
    conditional=False,
).to(device)

# Conta i parametri dei due modelli.
conditional_parameters = sum(
    parameter.numel()
    for parameter in conditional_unet.parameters()
)

unconditional_parameters = sum(
    parameter.numel()
    for parameter in unconditional_unet.parameters()
)

print("U-Net pronta.")
print(f"Parametri conditional: {conditional_parameters:,}")
print(f"Parametri unconditional: {unconditional_parameters:,}")
print(
    "Parametri specifici del conditioning:",
    f"{conditional_parameters - unconditional_parameters:,}",
)
print(
    "Memoria conditional FP32:",
    f"{conditional_parameters * 4 / 1024**2:.2f} MiB",
)

## Smoke test: Text Encoder → U-Net

Questa verifica non misura ancora la capacità di apprendimento.
Controlla invece che:

- tokenizer, text encoder e U-Net rispettino le dimensioni concordate;
- tutti gli output siano finiti;
- la U-Net riceva correttamente il contesto testuale;
- i gradienti attraversino entrambi i modelli.

In [ ]:
import torch

torch.manual_seed(2026)

batch_size = 2
image_size = 32
sequence_length = 44

input_ids = torch.tensor(
    [
        [2, 9, 12, 11, 20, 3] + [1] * (sequence_length - 6),
        [2, 14, 7, 8, 31, 3] + [1] * (sequence_length - 6),
    ],
    dtype=torch.long,
    device=device,
)

attention_mask = (
    input_ids != encoder_config["pad_token_id"]
).long()

noisyimages = torch.randn(
    batch_size,
    3,
    image_size,
    image_size,
    device=device,
)

timesteps = torch.tensor(
    [10, 900],
    dtype=torch.long,
    device=device,
)

text_encoder.train()
conditional_unet.train()

context = text_encoder(
    input_ids=input_ids,
    attention_mask=attention_mask,
)

predicted_noise = conditional_unet(
    noisyimages=noisyimages,
    timesteps=timesteps,
    context=context,
    attention_mask=attention_mask,
)

assert context.shape == (
    batch_size,
    sequence_length,
    encoder_config["d_model"],
)

assert predicted_noise.shape == noisyimages.shape
assert torch.isfinite(context).all()
assert torch.isfinite(predicted_noise).all()

loss = predicted_noise.square().mean()

text_encoder.zero_grad(set_to_none=True)
conditional_unet.zero_grad(set_to_none=True)

loss.backward()

text_gradients = [
    parameter.grad
    for parameter in text_encoder.parameters()
    if parameter.grad is not None
]

unet_gradients = [
    parameter.grad
    for parameter in conditional_unet.parameters()
    if parameter.grad is not None
]

assert text_gradients, "Nessun gradiente nel text encoder."
assert unet_gradients, "Nessun gradiente nella U-Net."

assert all(
    torch.isfinite(gradient).all()
    for gradient in text_gradients
)

assert all(
    torch.isfinite(gradient).all()
    for gradient in unet_gradients
)

print("Smoke test Text Encoder -> U-Net superato.")
print(f"Context shape: {tuple(context.shape)}")
print(
    "Predicted noise shape:",
    tuple(predicted_noise.shape),
)
print(f"Loss sintetica: {loss.item():.6f}")

## Reverse sampler DDPM

Il reverse sampler appartiene al Blocco 6 perché è codice di inferenza del modello: non è una verifica di training.

Per il modello conditional, il sampler calcola una sola volta il contesto dal text encoder prima del ciclo di denoising. A ogni timestep la U-Net usa quel contesto nella cross-attention, predice il rumore e la schedule applica la transizione da `x_t` a `x_{t-1}`. Al termine restituisce l'immagine finale in `[-1, 1]`.

Il sampler non aggiorna pesi e deve eseguire l'inferenza in `torch.inference_mode()`.


In [ ]:
%%bash

set -euo pipefail

cat > configs/sampling.yaml <<'EOF'
sampling:
  num_inference_steps: 1000
  guidance_scale: 1.0
  clip_denoised: true
EOF

echo "Configurazione sampling:"
cat configs/sampling.yaml

In [ ]:
%%writefile src/diffusion/reverse_sampler.py
from __future__ import annotations

from pathlib import Path
from typing import Optional

import torch
import yaml
from torch import Tensor, nn

from src.diffusion.schedule import DDPMSchedule


def load_sampling_config(
    config_path: Path,
) -> dict:
    with Path(config_path).open(
        "r",
        encoding="utf-8",
    ) as file:
        config = yaml.safe_load(file)

    if "sampling" not in config:
        raise ValueError(
            "Sezione 'sampling' mancante."
        )

    sampling_config = config["sampling"]

    required_keys = {
        "guidance_scale",
        "clip_denoised",
    }

    missing_keys = required_keys - set(sampling_config)

    if missing_keys:
        raise ValueError(
            "Parametri sampling mancanti: "
            f"{sorted(missing_keys)}"
        )

    guidance_scale = float(
        sampling_config["guidance_scale"]
    )

    if guidance_scale < 0.0:
        raise ValueError(
            "guidance_scale non può essere negativo."
        )

    if not isinstance(
        sampling_config["clip_denoised"],
        bool,
    ):
        raise TypeError(
            "clip_denoised deve essere booleano."
        )

    return config


class ReverseSampler:
    # Sampler DDPM condizionato dal testo.

    # Responsabilità:
    # - costruire il contesto testuale condizionato;
    # - costruire il contesto unconditional per CFG;
    # - interrogare la U-Net ad ogni timestep;
    # - delegare alla schedule il reverse step DDPM.

    def __init__(
        self,
        conditional_unet: nn.Module,
        text_encoder: nn.Module,
        schedule: DDPMSchedule,
        guidance_scale: float = 1.0,
        clip_denoised: bool = True,
        device: Optional[torch.device] = None,
    ) -> None:
        self.conditional_unet = conditional_unet
        self.text_encoder = text_encoder
        self.schedule = schedule

        self.guidance_scale = float(guidance_scale)
        self.clip_denoised = bool(clip_denoised)

        if self.guidance_scale < 0.0:
            raise ValueError(
                "guidance_scale non può essere negativo."
            )

        if device is None:
            try:
                device = next(
                    conditional_unet.parameters()
                ).device
            except StopIteration:
                device = torch.device("cpu")

        self.device = torch.device(device)

        self.conditional_unet.to(self.device)
        self.text_encoder.to(self.device)
        self.schedule.to(self.device)

        pad_token_id = getattr(
            self.text_encoder,
            "pad_token_id",
            None,
        )

        if pad_token_id is None:
            raise AttributeError(
                "Il text encoder deve esporre "
                "pad_token_id."
            )

        self.pad_token_id = int(pad_token_id)

    @classmethod
    def from_config(
        cls,
        conditional_unet: nn.Module,
        text_encoder: nn.Module,
        schedule: DDPMSchedule,
        config: dict,
        device: Optional[torch.device] = None,
    ) -> "ReverseSampler":
        sampling_config = config["sampling"]

        return cls(
            conditional_unet=conditional_unet,
            text_encoder=text_encoder,
            schedule=schedule,
            guidance_scale=float(
                sampling_config["guidance_scale"]
            ),
            clip_denoised=bool(
                sampling_config["clip_denoised"]
            ),
            device=device,
        )

    def _validate_tokens(
        self,
        input_ids: Tensor,
        attention_mask: Tensor,
    ) -> None:
        if input_ids.ndim != 2:
            raise ValueError(
                "input_ids deve avere shape "
                "[batch, sequence_length]."
            )

        if attention_mask.ndim != 2:
            raise ValueError(
                "attention_mask deve avere shape "
                "[batch, sequence_length]."
            )

        if input_ids.shape != attention_mask.shape:
            raise ValueError(
                "input_ids e attention_mask devono "
                "avere la stessa shape."
            )

        if input_ids.dtype != torch.long:
            raise TypeError(
                "input_ids deve avere dtype torch.long."
            )

        if attention_mask.dtype != torch.long:
            raise TypeError(
                "attention_mask deve avere dtype torch.long."
            )

    def _build_unconditional_inputs(
        self,
        input_ids: Tensor,
    ) -> tuple[Tensor, Tensor]:
        # Costruisce la condizione nulla per la CFG:
        # tutti PAD, mask a zero.
        unconditional_ids = torch.full_like(
            input_ids,
            fill_value=self.pad_token_id,
        )

        unconditional_mask = torch.zeros_like(
            input_ids,
            dtype=torch.long,
        )

        return unconditional_ids, unconditional_mask

    def _predict_noise(
        self,
        noisyimages: Tensor,
        timesteps: Tensor,
        conditional_context: Tensor,
        conditional_mask: Tensor,
        unconditional_context: Optional[Tensor] = None,
        unconditional_mask: Optional[Tensor] = None,
        guidance_scale: float = 1.0,
    ) -> Tensor:
        conditional_noise = self.conditional_unet(
            noisyimages=noisyimages,
            timesteps=timesteps,
            context=conditional_context,
            attention_mask=conditional_mask,
        )

        if guidance_scale == 1.0:
            return conditional_noise

        if (
            unconditional_context is None
            or unconditional_mask is None
        ):
            raise ValueError(
                "CFG richiede context e mask unconditional."
            )

        unconditional_noise = self.conditional_unet(
            noisyimages=noisyimages,
            timesteps=timesteps,
            context=unconditional_context,
            attention_mask=unconditional_mask,
        )

        return unconditional_noise + guidance_scale * (
            conditional_noise - unconditional_noise
        )

    def _p_sample(
        self,
        x_t: Tensor,
        timesteps: Tensor,
        predicted_noise: Tensor,
        generator: Optional[torch.Generator] = None,
    ) -> tuple[Tensor, Tensor]:
        # Deleghiamo il reverse step alla schedule per avere
        # una sola implementazione matematica condivisa.
        return self.schedule.reverse_step(
            noisyimages=x_t,
            predicted_noise=predicted_noise,
            timesteps=timesteps,
            generator=generator,
            clip_denoised=self.clip_denoised,
        )

    @torch.inference_mode()
    def sample(
        self,
        input_ids: Tensor,
        attention_mask: Tensor,
        image_shape: tuple[int, int, int],
        generator: Optional[torch.Generator] = None,
        guidance_scale: Optional[float] = None,
        return_intermediates: bool = False,
        record_every: Optional[int] = None,
    ):
        self._validate_tokens(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        if len(image_shape) != 3:
            raise ValueError(
                "image_shape deve essere "
                "(channels, height, width)."
            )

        channels, height, width = image_shape

        if min(channels, height, width) < 1:
            raise ValueError(
                "Le dimensioni immagine devono essere positive."
            )

        if height % 4 != 0 or width % 4 != 0:
            raise ValueError(
                "Altezza e larghezza devono essere "
                "divisibili per 4."
            )

        if guidance_scale is None:
            guidance_scale = self.guidance_scale

        guidance_scale = float(guidance_scale)

        if guidance_scale < 0.0:
            raise ValueError(
                "guidance_scale non può essere negativo."
            )

        input_ids = input_ids.to(
            device=self.device,
            dtype=torch.long,
        )

        attention_mask = attention_mask.to(
            device=self.device,
            dtype=torch.long,
        )

        batch_size = input_ids.shape[0]

        self.conditional_unet.eval()
        self.text_encoder.eval()

        conditional_context = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        unconditional_ids, unconditional_mask = (
            self._build_unconditional_inputs(
                input_ids=input_ids
            )
        )

        unconditional_context = self.text_encoder(
            input_ids=unconditional_ids,
            attention_mask=unconditional_mask,
        )

        x_t = torch.randn(
            batch_size,
            channels,
            height,
            width,
            device=self.device,
            dtype=torch.float32,
            generator=generator,
        )

        intermediates = []

        for timestep in reversed(
            range(self.schedule.num_timesteps)
        ):
            timesteps = torch.full(
                (batch_size,),
                fill_value=timestep,
                device=self.device,
                dtype=torch.long,
            )

            predicted_noise = self._predict_noise(
                noisyimages=x_t,
                timesteps=timesteps,
                conditional_context=conditional_context,
                conditional_mask=attention_mask,
                unconditional_context=unconditional_context,
                unconditional_mask=unconditional_mask,
                guidance_scale=guidance_scale,
            )

            x_t, predicted_x_start = self._p_sample(
                x_t=x_t,
                timesteps=timesteps,
                predicted_noise=predicted_noise,
                generator=generator,
            )

            should_record = (
                return_intermediates
                and (
                    record_every is None
                    or timestep
                    == self.schedule.num_timesteps - 1
                    or timestep == 0
                    or timestep % record_every == 0
                )
            )

            if should_record:
                intermediates.append(
                    (
                        timestep,
                        predicted_x_start.detach().cpu(),
                    )
                )

        x_t = x_t.clamp(min=-1.0, max=1.0)

        if return_intermediates:
            return x_t, intermediates

        return x_t

In [ ]:
%%writefile tests/test_reverse_sampler.py
from pathlib import Path

import pytest
import torch

from src.diffusion.reverse_sampler import (
    ReverseSampler,
    load_sampling_config,
)


class DummyTextEncoder(torch.nn.Module):
    # Text encoder minimo per testare solo il sampler.

    pad_token_id = 1

    def forward(
        self,
        input_ids,
        attention_mask,
    ):
        batch_size, sequence_length = input_ids.shape

        return torch.zeros(
            batch_size,
            sequence_length,
            8,
            device=input_ids.device,
        )


class DummyUNet(torch.nn.Module):
    # Restituisce rumore nullo: il test verifica l'orchestrazione,
    # non la qualità del modello.

    def __init__(self):
        super().__init__()
        self.dummy_parameter = torch.nn.Parameter(
            torch.zeros(1)
        )

    def forward(
        self,
        noisyimages,
        timesteps,
        context,
        attention_mask,
    ):
        return torch.zeros_like(noisyimages)


class DummySchedule(torch.nn.Module):
    # Espone lo stesso contratto minimo richiesto dal sampler reale.

    def __init__(self):
        super().__init__()
        self.num_timesteps = 4

    def reverse_step(
        self,
        noisyimages,
        predicted_noise,
        timesteps,
        generator=None,
        clip_denoised=True,
    ):
        # Con rumore predetto nullo, il test può limitarsi a restituire
        # la stessa immagine e una stima grezza di x_0.
        predicted_x_start = noisyimages.clamp(-1.0, 1.0)

        return noisyimages, predicted_x_start


@pytest.fixture()
def sampling_config_file(tmp_path: Path) -> Path:
    config_path = tmp_path / "sampling.yaml"

    config_path.write_text(
        """
sampling:
  num_inference_steps: 1000
  guidance_scale: 1.0
  clip_denoised: true
""".strip(),
        encoding="utf-8",
    )

    return config_path


def test_load_sampling_config_ok(
    sampling_config_file: Path,
) -> None:
    config = load_sampling_config(
        sampling_config_file
    )

    assert config["sampling"]["guidance_scale"] == 1.0
    assert config["sampling"]["clip_denoised"] is True


def test_load_sampling_config_requires_sampling_section(
    tmp_path: Path,
) -> None:
    config_path = tmp_path / "invalid.yaml"
    config_path.write_text(
        "foo: 1\n",
        encoding="utf-8",
    )

    with pytest.raises(
        ValueError,
        match="Sezione 'sampling' mancante.",
    ):
        load_sampling_config(config_path)


def test_sampler_generates_tensor_with_expected_shape() -> None:
    text_encoder = DummyTextEncoder()
    unet = DummyUNet()
    schedule = DummySchedule()

    sampler = ReverseSampler(
        conditional_unet=unet,
        text_encoder=text_encoder,
        schedule=schedule,
        guidance_scale=1.0,
        clip_denoised=True,
        device=torch.device("cpu"),
    )

    input_ids = torch.tensor(
        [
            [2, 3, 4, 1],
            [5, 6, 1, 1],
        ],
        dtype=torch.long,
    )

    attention_mask = torch.tensor(
        [
            [1, 1, 1, 0],
            [1, 1, 0, 0],
        ],
        dtype=torch.long,
    )

    generator = torch.Generator().manual_seed(7)

    samples = sampler.sample(
        input_ids=input_ids,
        attention_mask=attention_mask,
        image_shape=(3, 32, 32),
        generator=generator,
    )

    assert samples.shape == (2, 3, 32, 32)
    assert samples.dtype == torch.float32
    assert torch.all(samples <= 1.0)
    assert torch.all(samples >= -1.0)


def test_sampler_can_return_intermediates() -> None:
    text_encoder = DummyTextEncoder()
    unet = DummyUNet()
    schedule = DummySchedule()

    sampler = ReverseSampler(
        conditional_unet=unet,
        text_encoder=text_encoder,
        schedule=schedule,
        guidance_scale=1.0,
        clip_denoised=True,
        device=torch.device("cpu"),
    )

    input_ids = torch.tensor(
        [[2, 3, 4, 1]],
        dtype=torch.long,
    )

    attention_mask = torch.tensor(
        [[1, 1, 1, 0]],
        dtype=torch.long,
    )

    generator = torch.Generator().manual_seed(11)

    final_sample, intermediates = sampler.sample(
        input_ids=input_ids,
        attention_mask=attention_mask,
        image_shape=(3, 32, 32),
        generator=generator,
        return_intermediates=True,
        record_every=1,
    )

    assert final_sample.shape == (1, 3, 32, 32)
    assert len(intermediates) == schedule.num_timesteps

    timestep_0, image_0 = intermediates[0]
    assert isinstance(timestep_0, int)
    assert image_0.shape == (1, 3, 32, 32)


def test_sampler_validates_input_dtypes() -> None:
    text_encoder = DummyTextEncoder()
    unet = DummyUNet()
    schedule = DummySchedule()

    sampler = ReverseSampler(
        conditional_unet=unet,
        text_encoder=text_encoder,
        schedule=schedule,
        device=torch.device("cpu"),
    )

    with pytest.raises(
        TypeError,
        match="input_ids deve avere dtype torch.long.",
    ):
        sampler.sample(
            input_ids=torch.zeros(1, 4),
            attention_mask=torch.ones(
                1,
                4,
                dtype=torch.long,
            ),
            image_shape=(3, 32, 32),
        )


def test_sampler_rejects_negative_guidance_scale() -> None:
    text_encoder = DummyTextEncoder()
    unet = DummyUNet()
    schedule = DummySchedule()

    with pytest.raises(
        ValueError,
        match="guidance_scale non può essere negativo.",
    ):
        ReverseSampler(
            conditional_unet=unet,
            text_encoder=text_encoder,
            schedule=schedule,
            guidance_scale=-1.0,
        )

## Test di regressione

I test unitari verificano proprietà locali, mentre il test integrato
verifica che il percorso completo testo → context → U-Net produca
output finiti e differenziabili.

Non eseguiamo ancora un vero sampling con la U-Net inizializzata:
un output casuale è normale prima dell'addestramento e non sarebbe
un criterio di qualità. Il primo test qualitativo del sampling sarà
appropriato dopo il tiny overfit del Blocco 7.

In [ ]:
%cd /content/tiny-avatar-diffusion

!python -m pytest -q \
  tests/test_text_encoder.py \
  tests/test_conditional_unet.py \
  tests/test_reverse_sampler.py

Da questo punto in avanti il criterio non sarà più soltanto
“l’implementazione è corretta?”, ma soprattutto
“il sistema riesce davvero ad apprendere e a ricostruire un micro-dataset
controllato?”.

# Tiny overfit e verifiche preliminari

In questo blocco verifichiamo che la pipeline diffusiva sia davvero
addestrabile su un sottoinsieme molto piccolo e controllato del dataset.

L'obiettivo non è ancora la qualità generativa finale, ma la conferma che
text encoder, U-Net condizionata, schedule DDPM, backward, optimizer,
checkpointing e mixed precision lavorino correttamente insieme durante
un training minimo ma reale.

In [ ]:
%%bash
set -euo pipefail

cat > configs/overfit.yaml <<'EOF'
overfit:
  seed: 2026
  subset_size: 32
  batch_size: 32
  max_steps: 2000

  learning_rate:
    unet: 0.0002
    text_encoder: 0.0001

  weight_decay: 0.0001
  gradient_clip_norm: 1.0

  amp:
    enabled: true
    dtype: float16

  evaluation:
    seed: 3107
    every_steps: 100

  checkpoint:
    every_steps: 250
    cold_resume_step: 500
    directory: checkpoints/tiny_overfit

  logging:
    every_steps: 25
    history_path: outputs/tiny_overfit/history.csv
EOF

echo "Configurazione tiny overfit:"
cat configs/overfit.yaml

In [ ]:
!mkdir -p src/training
!touch src/training/__init__.py

## Implementazione del trainer di tiny overfit

Definiamo un trainer dedicato, volutamente minimale, che lavora su un batch
fisso e controllato. Questo trainer serve a verificare la capacità della
pipeline di ridurre la loss su un micro-dataset, con supporto a AMP,
gradient clipping, evaluation periodica, checkpoint e resume.

In [ ]:
%%writefile src/training/tiny_overfit.py

import csv
import math
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import yaml

from torch import nn, Tensor

from src.diffusion.schedule import (
    DDPMSchedule,
)
from src.utils.checkpoint import (
    load_training_checkpoint,
    save_training_checkpoint,
)


def load_overfit_config(
    path: Path,
) -> dict:
    # Carica la configurazione dedicata al tiny overfit
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as file:
        config = yaml.safe_load(file)

    if "overfit" not in config:
        raise ValueError(
            "Sezione overfit mancante."
        )

    return config


def set_training_seed(
    seed: int,
) -> None:
    # Rende il training riproducibile
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


class TinyOverfitTrainer:

    def __init__(
        self,
        conditional_unet: nn.Module,
        text_encoder: nn.Module,
        schedule: DDPMSchedule,
        optimizer: torch.optim.Optimizer,
        fixed_batch: dict,
        device: torch.device,
        config: dict,
    ) -> None:
        # Il tiny overfit è pensato per GPU
        if device.type != "cuda":
            raise ValueError(
                "Il tiny-overfit richiede CUDA."
            )

        self.conditional_unet = (
            conditional_unet
        )
        self.text_encoder = text_encoder
        self.schedule = schedule
        self.optimizer = optimizer
        self.fixed_batch = fixed_batch
        self.device = device
        self.config = config

        overfit_config = config["overfit"]

        self.gradient_clip_norm = float(
            overfit_config[
                "gradient_clip_norm"
            ]
        )

        self.amp_enabled = bool(
            overfit_config["amp"]["enabled"]
        )

        self.scaler = torch.amp.GradScaler(
            "cuda",
            enabled=self.amp_enabled,
        )

        self.train_generator = (
            torch.Generator(
                device=device
            ).manual_seed(
                int(overfit_config["seed"])
            )
        )

        self.evaluation_seed = int(
            overfit_config["evaluation"][
                "seed"
            ]
        )

        self.step = 0
        self.best_evaluation_loss = math.inf
        self.history = []

        self.parameters = [
            *self.conditional_unet.parameters(),
            *self.text_encoder.parameters(),
        ]

    def _prepare_batch(
        self,
    ) -> tuple[Tensor, Tensor, Tensor]:
        # Sposta il batch fisso sul device di training
        images = self.fixed_batch[
            "image"
        ].to(
            self.device,
            non_blocking=True,
        )

        input_ids = self.fixed_batch[
            "input_ids"
        ].to(
            self.device,
            non_blocking=True,
        )

        attention_mask = self.fixed_batch[
            "attention_mask"
        ].to(
            self.device,
            non_blocking=True,
        )

        return (
            images,
            input_ids,
            attention_mask,
        )

    def _build_noisy_batch(
        self,
        images: Tensor,
        generator: torch.Generator,
    ) -> tuple[Tensor, Tensor, Tensor]:
        # Campiona i timestep e costruisce x_t con il rumore target
        timesteps = torch.randint(
            low=0,
            high=self.schedule.num_timesteps,
            size=(images.shape[0],),
            device=self.device,
            dtype=torch.long,
            generator=generator,
        )

        noisyimages, target_noise = (
            self.schedule.q_sample(
                x_start=images,
                timesteps=timesteps,
                generator=generator,
            )
        )

        return (
            noisyimages,
            target_noise,
            timesteps,
        )

    def _predict_noise(
        self,
        noisyimages,
        timesteps,
        input_ids,
        attention_mask,
    ):
        # Verifica minima sulla correttezza del conditioning
        if attention_mask is None:
            raise ValueError(
                "_predict_noise ha ricevuto attention_mask=None."
            )

        if attention_mask.ndim != 2:
            raise ValueError(
                "attention_mask deve avere shape "
                "[batch, sequence_length], ma ha "
                f"shape {tuple(attention_mask.shape)}."
            )

        if input_ids.shape != attention_mask.shape:
            raise ValueError(
                "input_ids e attention_mask devono avere "
                "la stessa shape: "
                f"{tuple(input_ids.shape)} contro "
                f"{tuple(attention_mask.shape)}."
            )

        text_context = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        return self.conditional_unet(
            noisyimages=noisyimages,
            timesteps=timesteps,
            context=text_context,
            attention_mask=attention_mask,
        )

    def train_step(
        self,
    ) -> dict:
        # Esegue un singolo passo di training sul batch fisso
        self.conditional_unet.train()
        self.text_encoder.train()

        (
            images,
            input_ids,
            attention_mask,
        ) = self._prepare_batch()

        (
            noisyimages,
            target_noise,
            timesteps,
        ) = self._build_noisy_batch(
            images=images,
            generator=self.train_generator,
        )

        self.optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=self.amp_enabled,
        ):
            predicted_noise = (
                self._predict_noise(
                    noisyimages=noisyimages,
                    timesteps=timesteps,
                    input_ids=input_ids,
                    attention_mask=(
                        attention_mask
                    ),
                )
            )

            loss = F.mse_loss(
                predicted_noise.float(),
                target_noise.float(),
            )

        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"Loss non finita allo step "
                f"{self.step}: {loss.item()}"
            )

        self.scaler.scale(
            loss
        ).backward()

        self.scaler.unscale_(
            self.optimizer
        )

        gradient_norm = (
            torch.nn.utils.clip_grad_norm_(
                self.parameters,
                max_norm=(
                    self.gradient_clip_norm
                ),
                error_if_nonfinite=True,
            )
        )

        self.scaler.step(
            self.optimizer
        )
        self.scaler.update()

        self.step += 1

        return {
            "step": self.step,
            "train_loss": float(
                loss.detach().item()
            ),
            "gradient_norm": float(
                gradient_norm.detach().item()
            ),
            "amp_scale": float(
                self.scaler.get_scale()
            ),
        }

    @torch.inference_mode()
    def evaluate(
        self,
    ) -> float:
        # Valutazione deterministica con seed fisso
        self.conditional_unet.eval()
        self.text_encoder.eval()

        (
            images,
            input_ids,
            attention_mask,
        ) = self._prepare_batch()

        generator = (
            torch.Generator(
                device=self.device
            ).manual_seed(
                self.evaluation_seed
            )
        )

        (
            noisyimages,
            target_noise,
            timesteps,
        ) = self._build_noisy_batch(
            images=images,
            generator=generator,
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=self.amp_enabled,
        ):
            predicted_noise = (
                self._predict_noise(
                    noisyimages=noisyimages,
                    timesteps=timesteps,
                    input_ids=input_ids,
                    attention_mask=(
                        attention_mask
                    ),
                )
            )

            loss = F.mse_loss(
                predicted_noise.float(),
                target_noise.float(),
            )

        return float(loss.item())

    def save(
        self,
        path: Path,
    ) -> None:
        # Salva lo stato completo del tiny overfit
        save_training_checkpoint(
            path=path,
            models={
                "conditional_unet": (
                    self.conditional_unet
                ),
                "text_encoder": (
                    self.text_encoder
                ),
            },
            optimizer=self.optimizer,
            scaler=self.scaler,
            step=self.step,
            metadata={
                "best_evaluation_loss": (
                    self.best_evaluation_loss
                ),
                "history": self.history,
            },
            configs=self.config,
            generators={
                "training": (
                    self.train_generator
                )
            },
        )

    def load(
        self,
        path: Path,
    ) -> dict:
        # Ripristina un run precedente
        checkpoint = (
            load_training_checkpoint(
                path=path,
                models={
                    "conditional_unet": (
                        self.conditional_unet
                    ),
                    "text_encoder": (
                        self.text_encoder
                    ),
                },
                optimizer=self.optimizer,
                scaler=self.scaler,
                generators={
                    "training": (
                        self.train_generator
                    )
                },
                map_location=self.device,
                restore_rng=True,
            )
        )

        self.step = int(
            checkpoint["step"]
        )

        metadata = checkpoint[
            "metadata"
        ]

        self.best_evaluation_loss = float(
            metadata.get(
                "best_evaluation_loss",
                math.inf,
            )
        )

        self.history = list(
            metadata.get(
                "history",
                [],
            )
        )

        return checkpoint

    def write_history(
        self,
        path: Path,
    ) -> None:
        # Esporta la history su CSV
        path = Path(path)
        path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        if not self.history:
            return

        keys = [
            "step",
            "train_loss",
            "evaluation_loss",
            "gradient_norm",
            "amp_scale",
        ]

        with path.open(
            "w",
            newline="",
            encoding="utf-8",
        ) as file:
            writer = csv.DictWriter(
                file,
                fieldnames=keys,
            )

            writer.writeheader()

            for row in self.history:
                writer.writerow(
                    {
                        key: row.get(
                            key,
                            "",
                        )
                        for key in keys
                    }
                )

    def fit(
        self,
        max_steps: int,
        evaluation_every: int,
        checkpoint_every: int,
        checkpoint_directory: Path,
        history_path: Path,
        logging_every: int,
    ) -> None:
        # Loop principale di tiny overfit
        checkpoint_directory = Path(
            checkpoint_directory
        )

        checkpoint_directory.mkdir(
            parents=True,
            exist_ok=True,
        )

        latest_path = (
            checkpoint_directory
            / "latest.pt"
        )

        best_path = (
            checkpoint_directory
            / "best.pt"
        )

        while self.step < max_steps:
            metrics = self.train_step()

            evaluation_loss = None

            if (
                self.step == 1
                or self.step
                % evaluation_every == 0
            ):
                evaluation_loss = (
                    self.evaluate()
                )

                metrics[
                    "evaluation_loss"
                ] = evaluation_loss

                if (
                    evaluation_loss
                    < self.best_evaluation_loss
                ):
                    self.best_evaluation_loss = (
                        evaluation_loss
                    )

                    self.history.append(
                        metrics.copy()
                    )

                    self.save(
                        best_path
                    )

                    self.history.pop()

            self.history.append(metrics)

            if (
                self.step == 1
                or self.step
                % logging_every == 0
            ):
                print(
                    f"step={self.step:05d} "
                    f"train={metrics['train_loss']:.6f} "
                    f"eval="
                    f"{evaluation_loss if evaluation_loss is not None else '—'} "
                    f"grad={metrics['gradient_norm']:.4f} "
                    f"scale={metrics['amp_scale']:.1f}"
                )

            if (
                self.step
                % checkpoint_every == 0
                or self.step == max_steps
            ):
                self.save(
                    latest_path
                )

            self.write_history(
                history_path
            )

        print(
            "Training terminato. "
            f"Step={self.step}, "
            f"best_eval="
            f"{self.best_evaluation_loss:.6f}"
        )

## Test del tiny overfit trainer

Prima del run effettivo verifichiamo che il trainer sia in grado di:

- eseguire almeno un train step con loss finita;
- eseguire evaluation deterministica;
- salvare e ricaricare correttamente un checkpoint;
- mantenere step e history dopo il resume.

In [ ]:
%%writefile tests/test_tiny_overfit.py

from pathlib import Path

import pytest
import torch

from src.diffusion.schedule import (
    DDPMSchedule,
)
from src.training.tiny_overfit import (
    TinyOverfitTrainer,
)


class DummyTextEncoder(torch.nn.Module):

    pad_token_id = 1

    def __init__(self):
        super().__init__()
        self.embedding = torch.nn.Embedding(
            32,
            8,
            padding_idx=1,
        )

    def forward(
        self,
        input_ids,
        attention_mask,
    ):
        hidden_states = self.embedding(
            input_ids
        )
        return hidden_states * attention_mask.unsqueeze(-1)


class DummyConditionalUNet(torch.nn.Module):

    def __init__(self):
        super().__init__()
        self.convolution = torch.nn.Conv2d(
            3,
            3,
            kernel_size=1,
        )
        self.context_projection = torch.nn.Linear(
            8,
            3,
        )

    def forward(
        self,
        noisyimages,
        timesteps,
        context,
        attention_mask,
    ):
        pooled_context = (
            context.sum(dim=1)
            / attention_mask.sum(
                dim=1,
                keepdim=True,
            ).clamp(min=1)
        )

        bias = self.context_projection(
            pooled_context
        ).unsqueeze(-1).unsqueeze(-1)

        return self.convolution(
            noisyimages
        ) + bias


@pytest.mark.skipif(
    not torch.cuda.is_available(),
    reason="CUDA non disponibile",
)
def test_tiny_overfit_step_eval_and_checkpoint(
    tmp_path,
):
    device = torch.device("cuda")

    fixed_batch = {
        "image": torch.randn(
            4,
            3,
            32,
            32,
        ),
        "input_ids": torch.randint(
            0,
            16,
            (4, 6),
            dtype=torch.long,
        ),
        "attention_mask": torch.ones(
            4,
            6,
            dtype=torch.long,
        ),
    }

    conditional_unet = (
        DummyConditionalUNet().to(device)
    )
    text_encoder = DummyTextEncoder().to(device)

    schedule = DDPMSchedule(
        num_timesteps=20,
        beta_start=0.0001,
        beta_end=0.02,
    ).to(device)

    optimizer = torch.optim.AdamW(
        [
            {
                "params": conditional_unet.parameters(),
                "lr": 1e-3,
            },
            {
                "params": text_encoder.parameters(),
                "lr": 1e-3,
            },
        ]
    )

    config = {
        "overfit": {
            "seed": 2026,
            "gradient_clip_norm": 1.0,
            "amp": {
                "enabled": True,
                "dtype": "float16",
            },
            "evaluation": {
                "seed": 3107,
            },
        }
    }

    trainer = TinyOverfitTrainer(
        conditional_unet=conditional_unet,
        text_encoder=text_encoder,
        schedule=schedule,
        optimizer=optimizer,
        fixed_batch=fixed_batch,
        device=device,
        config=config,
    )

    metrics = trainer.train_step()

    assert metrics["step"] == 1
    assert torch.isfinite(
        torch.tensor(
            metrics["train_loss"],
            device=device,
        )
    )

    evaluation_loss = trainer.evaluate()

    assert torch.isfinite(
        torch.tensor(
            evaluation_loss,
            device=device,
        )
    )

    trainer.history.append(
        {
            **metrics,
            "evaluation_loss": evaluation_loss,
        }
    )

    checkpoint_path = (
        tmp_path / "tiny_overfit.pt"
    )

    trainer.save(checkpoint_path)

    assert checkpoint_path.exists()

    restored_unet = (
        DummyConditionalUNet().to(device)
    )
    restored_text_encoder = (
        DummyTextEncoder().to(device)
    )

    restored_schedule = DDPMSchedule(
        num_timesteps=20,
        beta_start=0.0001,
        beta_end=0.02,
    ).to(device)

    restored_optimizer = torch.optim.AdamW(
        [
            {
                "params": restored_unet.parameters(),
                "lr": 1e-3,
            },
            {
                "params": restored_text_encoder.parameters(),
                "lr": 1e-3,
            },
        ]
    )

    restored_trainer = TinyOverfitTrainer(
        conditional_unet=restored_unet,
        text_encoder=restored_text_encoder,
        schedule=restored_schedule,
        optimizer=restored_optimizer,
        fixed_batch=fixed_batch,
        device=device,
        config=config,
    )

    restored_trainer.load(checkpoint_path)

    assert restored_trainer.step == trainer.step
    assert len(restored_trainer.history) == len(
        trainer.history
    )

In [ ]:
%cd /content/tiny-avatar-diffusion

!python -m pytest -q \
  tests/test_sampler.py \
  tests/test_checkpoint.py \
  tests/test_tiny_overfit.py

Una volta validato il trainer, il tiny overfit può essere lanciato sul batch
fisso del progetto. Il criterio di successo è osservare una discesa chiara
della training loss e la corretta produzione di checkpoint e history CSV.

In [ ]:
import torch

conditional_unet.train()
text_encoder.train()

optimizer = torch.optim.AdamW(
    [
        {
            "params": conditional_unet.parameters(),
            "lr": 1e-3,
        },
        {
            "params": text_encoder.parameters(),
            "lr": 1e-3,
        },
    ],
    betas=(0.9, 0.999),
    weight_decay=1e-4,
)

print("Optimizer creato correttamente.")
print("Tipo:", optimizer.__class__.__name__)
print("Gruppi di parametri:", len(optimizer.param_groups))
print("LR gruppo 0:", optimizer.param_groups[0]["lr"])
print("LR gruppo 1:", optimizer.param_groups[1]["lr"])

In [ ]:
import yaml
from pathlib import Path

from src.data.avatar_dataset import build_dataloaders, build_datasets


config_path = Path("configs/data_loader.yaml")

with config_path.open("r", encoding="utf-8") as file:
    data_config = yaml.safe_load(file)

datasets = build_datasets(data_config)
dataloaders = build_dataloaders(datasets=datasets, config=data_config)

fixed_batch = next(iter(dataloaders["train"]))

print("fixed_batch creato correttamente.")
print("Chiavi:", fixed_batch.keys())
print("image:", fixed_batch["image"].shape, fixed_batch["image"].dtype)
print("input_ids:", fixed_batch["input_ids"].shape, fixed_batch["input_ids"].dtype)
print("attention_mask:", fixed_batch["attention_mask"].shape, fixed_batch["attention_mask"].dtype)

In [ ]:
# Esempio di lancio reale del tiny overfit.
# Questa cella va eseguita solo quando modelli, schedule, optimizer
# e fixed_batch del progetto sono già stati istanziati.

import shutil
from pathlib import Path

import torch

from src.diffusion.schedule import (
    DDPMSchedule,
    load_diffusion_config,
)
from src.models.conditional_unet import (
    ConditionalUNet,
    load_unet_config,
)
from src.models.text_encoder import (
    TextTransformerEncoder,
    load_text_encoder_config,
)
from src.training.tiny_overfit import (
    TinyOverfitTrainer,
    load_overfit_config,
    set_training_seed,
)


# ---------------------------------------------------------
# 0. Controlli sugli input già costruiti nel notebook
# ---------------------------------------------------------

assert "fixed_batch" in globals(), (
    "fixed_batch non definito: esegui prima la cella "
    "che costruisce il batch fisso per il tiny-overfit."
)

assert "device" in globals(), "device non definito."

required_batch_keys = {
    "image",
    "input_ids",
    "attention_mask",
}

missing_batch_keys = required_batch_keys - set(fixed_batch)

assert not missing_batch_keys, (
    "fixed_batch non contiene le chiavi richieste: "
    f"{sorted(missing_batch_keys)}"
)


# ---------------------------------------------------------
# 1. Configurazioni e inizializzazione riproducibile
# ---------------------------------------------------------

overfit_config = load_overfit_config(
    Path("configs/overfit.yaml")
)

unet_config = load_unet_config(
    Path("configs/unet.yaml")
)

text_config = load_text_encoder_config(
    Path("configs/text_encoder.yaml")
)

diffusion_config = load_diffusion_config(
    Path("configs/diffusion.yaml")
)

overfit_settings = overfit_config["overfit"]

set_training_seed(
    int(overfit_settings["seed"])
)


# ---------------------------------------------------------
# 2. Ricostruzione pulita di modelli, schedule e optimizer
# ---------------------------------------------------------

conditional_unet = ConditionalUNet.from_config(
    unet_config,
    conditional=True,
).to(device)

text_encoder = TextTransformerEncoder.from_config(
    text_config,
).to(device)

schedule = DDPMSchedule.from_config(
    diffusion_config,
).to(device)

optimizer = torch.optim.AdamW(
    [
        {
            "params": conditional_unet.parameters(),
            "lr": float(
                overfit_settings["learning_rate"]["unet"]
            ),
        },
        {
            "params": text_encoder.parameters(),
            "lr": float(
                overfit_settings["learning_rate"]["text_encoder"]
            ),
        },
    ],
    weight_decay=float(
        overfit_settings["weight_decay"]
    ),
)

unet_parameters = sum(
    parameter.numel()
    for parameter in conditional_unet.parameters()
)

text_parameters = sum(
    parameter.numel()
    for parameter in text_encoder.parameters()
)

assert unet_parameters > 5_000_000, (
    "La U-Net ricostruita è troppo piccola: "
    "verifica la cella di src/models/conditional_unet.py."
)

print(
    f"Parametri U-Net: {unet_parameters:,}"
)
print(
    f"Parametri text encoder: {text_parameters:,}"
)
print(
    "Learning rate U-Net: "
    f"{overfit_settings['learning_rate']['unet']}"
)
print(
    "Learning rate text encoder: "
    f"{overfit_settings['learning_rate']['text_encoder']}"
)


# ---------------------------------------------------------
# 3. Pulizia degli artefatti
# ---------------------------------------------------------

checkpoint_directory = Path(
    overfit_settings["checkpoint"]["directory"]
)

history_path = Path(
    overfit_settings["logging"]["history_path"]
)

if checkpoint_directory.exists():
    shutil.rmtree(checkpoint_directory)
    print(
        "Checkpoint tiny-overfit precedente eliminato: "
        f"{checkpoint_directory}"
    )

if history_path.exists():
    history_path.unlink()
    print(
        "History tiny-overfit precedente eliminata: "
        f"{history_path}"
    )

checkpoint_directory.mkdir(
    parents=True,
    exist_ok=True,
)

history_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------
# 4. Tiny-overfit
# ---------------------------------------------------------

tiny_overfit_trainer = TinyOverfitTrainer(
    conditional_unet=conditional_unet,
    text_encoder=text_encoder,
    schedule=schedule,
    optimizer=optimizer,
    fixed_batch=fixed_batch,
    device=device,
    config=overfit_config,
)

initial_loss = float(tiny_overfit_trainer.evaluate())

assert torch.isfinite(
    torch.tensor(initial_loss)
), "Loss iniziale non finita."

print("--- AVVIO TINY-OVERFIT CON U-NET MULTIRISOLUZIONE ---")
print(f"Device: {device}")
print(
    "Batch fisso: "
    f"{tuple(fixed_batch['image'].shape)}"
)
print(f"Loss iniziale: {initial_loss:.6f}")

tiny_overfit_trainer.fit(
    max_steps=int(overfit_settings["max_steps"]),
    evaluation_every=int(
        overfit_settings["evaluation"]["every_steps"]
    ),
    checkpoint_every=int(
        overfit_settings["checkpoint"]["every_steps"]
    ),
    checkpoint_directory=checkpoint_directory,
    history_path=history_path,
    logging_every=int(
        overfit_settings["logging"]["every_steps"]
    ),
)

In [ ]:
# ---------------------------------------------------------
# Verifica finale del tiny-overfit con U-Net multirisoluzione
# ---------------------------------------------------------

import math
from pathlib import Path

import pandas as pd
import torch


overfit_settings = overfit_config["overfit"]

checkpoint_directory = Path(
    overfit_settings["checkpoint"]["directory"]
)

history_path = Path(
    overfit_settings["logging"]["history_path"]
)

latest_checkpoint = checkpoint_directory / "latest.pt"
best_checkpoint = checkpoint_directory / "best.pt"

assert tiny_overfit_trainer.step == int(
    overfit_settings["max_steps"]
), (
    f"Step finale inatteso: {tiny_overfit_trainer.step}"
)

assert latest_checkpoint.exists(), (
    f"Checkpoint latest mancante: {latest_checkpoint}"
)

assert best_checkpoint.exists(), (
    f"Checkpoint best mancante: {best_checkpoint}"
)

assert history_path.exists(), (
    f"History mancante: {history_path}"
)

history = pd.read_csv(history_path)

required_columns = {
    "step",
    "train_loss",
    "evaluation_loss",
    "gradient_norm",
    "amp_scale",
}

missing_columns = required_columns - set(history.columns)

assert not missing_columns, (
    "Colonne mancanti nella history: "
    f"{sorted(missing_columns)}"
)

assert len(history) == tiny_overfit_trainer.step
assert history["step"].is_monotonic_increasing
assert history["step"].is_unique

assert history["train_loss"].notna().all()
assert history["train_loss"].map(math.isfinite).all()
assert history["gradient_norm"].map(math.isfinite).all()

evaluations = history.dropna(
    subset=["evaluation_loss"]
).copy()

assert not evaluations.empty, (
    "Nessuna evaluation loss registrata."
)

assert evaluations["evaluation_loss"].map(
    math.isfinite
).all()

best_history_loss = float(
    evaluations["evaluation_loss"].min()
)

final_evaluation_loss = float(
    tiny_overfit_trainer.evaluate()
)

assert math.isfinite(final_evaluation_loss)
assert math.isfinite(tiny_overfit_trainer.best_evaluation_loss)

# Il test deve dimostrare apprendimento effettivo sul batch fisso.
assert best_history_loss < initial_loss, (
    "Il tiny-overfit non ha ridotto la loss rispetto "
    "all'inizializzazione."
)

assert tiny_overfit_trainer.best_evaluation_loss <= initial_loss

print("--- VERIFICA TINY-OVERFIT SUPERATA ---")
print(f"Step completati: {tiny_overfit_trainer.step}")
print(f"Loss iniziale: {initial_loss:.6f}")
print(f"Loss finale ricalcolata: {final_evaluation_loss:.6f}")
print(
    "Migliore evaluation loss: "
    f"{tiny_overfit_trainer.best_evaluation_loss:.6f}"
)
print(f"Checkpoint latest: {latest_checkpoint}")
print(f"Checkpoint best: {best_checkpoint}")
print(f"History: {history_path}")

In [ ]:
# ---------------------------------------------------------
# Test completi prima del training ufficiale
# ---------------------------------------------------------

from pathlib import Path
import compileall
import subprocess
import sys


PROJECT_ROOT = Path("/content/tiny-avatar-diffusion")

assert PROJECT_ROOT.exists(), (
    f"Project root non trovata: {PROJECT_ROOT}"
)

assert Path.cwd().resolve() == PROJECT_ROOT.resolve(), (
    "Directory corrente errata: "
    f"{Path.cwd().resolve()}"
)

# Controllo sintattico di tutti i moduli Python del progetto.
compiled = compileall.compile_dir(
    PROJECT_ROOT / "src",
    quiet=1,
)

assert compiled, (
    "Compilazione Python fallita: controlla i file in src/."
)

# Esegue l'intera suite, non soltanto i test della U-Net.
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

assert result.returncode == 0, (
    "Almeno un test non è superato."
)

print("--- AUDIT COMPLETO SUPERATO ---")

In [ ]:
%cd /content/tiny-avatar-diffusion

!python -m pytest -q

# Trainer generale + smoke test

In questo blocco generalizziamo l’infrastruttura di training oltre il caso di tiny-overfit, introducendo un trainer riutilizzabile sia per la U-Net condizionata sia per la baseline unconditional.

Prima dell’avvio controllato eseguiamo una verifica finale di coerenza su configurazioni, artefatti, tokenizer, split, suite di test e stato del repository. Successivamente implementiamo il trainer generale, lo validiamo con test unitari dedicati e ne verifichiamo il comportamento con uno smoke test breve ma completo.

## Verifica finale pre-Avvio controllato

Prima di introdurre il trainer generale verifichiamo che l’intero progetto sia in uno stato coerente e riproducibile. In particolare controlliamo la compilazione dei moduli Python, la coerenza tra caption e attributi di conditioning, l’allineamento tra tokenizer e text encoder, la correttezza degli split, il superamento della suite di test e lo stato pulito del repository Git.

In [ ]:
from pathlib import Path
import compileall
import json
import subprocess
import sys

import pandas as pd
import yaml

PROJECT_ROOT = Path("/content/tiny-avatar-diffusion")

assert Path.cwd().resolve() == PROJECT_ROOT.resolve(), (
    f"Directory corrente errata: {Path.cwd()}"
)

# ---------------------------------------------------------
# 1. Compilazione statica dei moduli Python
# ---------------------------------------------------------

src_compiles = compileall.compile_dir(
    PROJECT_ROOT / "src",
    quiet=1,
)

tests_compile = compileall.compile_dir(
    PROJECT_ROOT / "tests",
    quiet=1,
)

assert src_compiles, "Compilazione della cartella src fallita."
assert tests_compile, "Compilazione della cartella tests fallita."

print("✓ Compilazione Python completata")

# ---------------------------------------------------------
# 2. Risoluzione robusta dei file di configurazione
# ---------------------------------------------------------

# Preferiamo configs/data.yaml, ma se nel repository corrente
# è ancora presente solo data_loader.yaml usiamo quello.
data_config_candidates = [
    PROJECT_ROOT / "configs/data.yaml",
    PROJECT_ROOT / "configs/data_loader.yaml",
]

data_config_path = None
for candidate in data_config_candidates:
    if candidate.exists():
        data_config_path = candidate
        break

assert data_config_path is not None, (
    "Nessun file trovato tra configs/data.yaml "
    "e configs/data_loader.yaml."
)

with (PROJECT_ROOT / "configs/text_conditioning.yaml").open(
    "r",
    encoding="utf-8",
) as file:
    text_conditioning_config = yaml.safe_load(file)

with data_config_path.open(
    "r",
    encoding="utf-8",
) as file:
    data_config = yaml.safe_load(file)

with (PROJECT_ROOT / "configs/data_split.yaml").open(
    "r",
    encoding="utf-8",
) as file:
    split_config = yaml.safe_load(file)

with (PROJECT_ROOT / "configs/text_encoder.yaml").open(
    "r",
    encoding="utf-8",
) as file:
    text_encoder_config = yaml.safe_load(file)

with (
    PROJECT_ROOT
    / "data/processed/captions/caption_summary.json"
).open("r", encoding="utf-8") as file:
    caption_summary = json.load(file)

with (
    PROJECT_ROOT
    / "data/processed/tokenizer/tokenizer_summary.json"
).open("r", encoding="utf-8") as file:
    tokenizer_summary = json.load(file)

with (
    PROJECT_ROOT
    / "data/processed/splits/split_summary.json"
).open("r", encoding="utf-8") as file:
    split_summary = json.load(file)

print(f"✓ Config data usata: {data_config_path.name}")

# ---------------------------------------------------------
# 3. Coerenza caption e conditioning
# ---------------------------------------------------------

caption_attributes = [
    item["key"]
    for item in text_conditioning_config["caption"]["attributes"]
]

dataset_conditioning_attributes = data_config["data"][
    "conditioning_attributes"
]

assert caption_attributes == dataset_conditioning_attributes, (
    "Gli attributi delle caption non coincidono con quelli "
    "del dataset.\n"
    f"Caption: {caption_attributes}\n"
    f"Dataset: {dataset_conditioning_attributes}"
)

assert caption_summary["caption_attributes"] == caption_attributes

print("✓ Caption e attributi di conditioning coerenti")

# ---------------------------------------------------------
# 4. Coerenza tokenizer e Transformer
# ---------------------------------------------------------

encoder_config = text_encoder_config["text_encoder"]

assert (
    tokenizer_summary["vocabulary_size"]
    == encoder_config["vocab_size"]
), "Vocabulary size incoerente."

assert (
    tokenizer_summary["sequence_length_with_boundaries"]
    == encoder_config["max_sequence_length"]
), "Lunghezza massima incoerente."

assert (
    tokenizer_summary["special_token_ids"]["pad"]
    == encoder_config["pad_token_id"]
), "PAD token ID incoerente."

print("✓ Tokenizer e Transformer coerenti")
print(f"  Vocabulary size: {tokenizer_summary['vocabulary_size']}")
print(
    "  Sequence length:",
    tokenizer_summary["sequence_length_with_boundaries"],
)

# ---------------------------------------------------------
# 5. Coerenza degli split
# ---------------------------------------------------------

split_counts = split_summary["counts"]

assert sum(split_counts.values()) == 10_000

expected_split_counts = {
    split_name: int(count)
    for split_name, count
    in data_config["data"]["expected_split_counts"].items()
}

actual_split_counts = {
    split_name: int(count)
    for split_name, count
    in split_summary["counts"].items()
}

assert actual_split_counts == expected_split_counts, (
    "Conteggi degli split incoerenti.\n"
    f"Attesi: {expected_split_counts}\n"
    f"Ottenuti: {actual_split_counts}"
)

assignments_path = (
    PROJECT_ROOT
    / "data/processed/splits/metadata_with_splits.csv"
)

assignments = pd.read_csv(assignments_path)

assert len(assignments) == sum(expected_split_counts.values()), (
    "Il numero complessivo delle assegnazioni non coincide "
    "con la somma dei conteggi configurati."
)

print("Conteggi degli split coerenti:")
for split_name, count in expected_split_counts.items():
    print(f"- {split_name}: {count}")

# ---------------------------------------------------------
# 6. Suite completa dei test
# ---------------------------------------------------------

subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=PROJECT_ROOT,
    check=True,
)

print("✓ Suite completa superata")

# ---------------------------------------------------------
# 7. Stato Git
# ---------------------------------------------------------

# In fase di sviluppo del notebook non blocchiamo l'esecuzione
# se il repository contiene modifiche locali: mostriamo un warning.
git_status = subprocess.run(
    ["git", "status", "--porcelain"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

if git_status:
    print("\n[WARNING] Repository non pulito:")
    print(git_status)
    print(
        "\nContinuo comunque: lo stato Git sporco non implica "
        "necessariamente un errore del progetto."
    )
else:
    print("✓ Repository Git pulito")

print()
print("PRE-AVVIO CONTROLLATO: CONTROLLI TECNICI SUPERATI")

## Implementazione del Trainer generale

Definiamo ora un trainer unificato per il training diffusion, progettato per funzionare sia in modalità condizionata sia in modalità unconditional. La classe gestisce preparazione dei batch, diffusione forward, prediction del rumore, evaluation su validation set, checkpointing, history CSV e ripristino dello stato di training.

In [ ]:
%%writefile src/training/trainer.py
import csv
import math
import random
from pathlib import Path
from typing import Optional

import numpy as np
import torch
import torch.nn.functional as F
import yaml
from torch import Tensor, nn
from torch.utils.data import DataLoader

from src.diffusion.schedule import DDPMSchedule
from src.utils.checkpoint import load_training_checkpoint, save_training_checkpoint


def load_training_config(path: Path) -> dict:
    with Path(path).open("r", encoding="utf-8") as file:
        config = yaml.safe_load(file)

    if "training" not in config:
        raise ValueError("Sezione 'training' mancante nel file di configurazione.")

    return config


def set_training_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


class Trainer:
    def __init__(
        self,
        conditional_unet: nn.Module,
        schedule: DDPMSchedule,
        optimizer: torch.optim.Optimizer,
        train_dataloader: DataLoader,
        device: torch.device,
        config: dict,
        text_encoder: Optional[nn.Module] = None,
        validation_dataloader: Optional[DataLoader] = None,
    ) -> None:
        self.conditional_unet = conditional_unet
        self.text_encoder = text_encoder
        self.schedule = schedule
        self.optimizer = optimizer
        self.train_dataloader = train_dataloader
        self.validation_dataloader = validation_dataloader
        self.device = device
        self.config = config

        training_config = config["training"]

        # Configurazione numerica del training
        self.gradient_clip_norm = float(
            training_config.get("gradient_clip_norm", 1.0)
        )

        # AMP attivo solo quando richiesto e disponibile
        requested_amp = bool(
            training_config.get("amp", {}).get("enabled", False)
        )

        self.device_type = self.device.type
        self.amp_enabled = requested_amp and self.device_type == "cuda"
        self.amp_dtype = torch.float16

        self.scaler = torch.amp.GradScaler(
            self.device_type,
            enabled=self.amp_enabled,
        )

        seed = int(training_config.get("seed", 42))
        self.train_generator = torch.Generator(
            device=self.device
        ).manual_seed(seed)

        self.evaluation_seed = int(
            training_config.get("evaluation", {}).get("seed", 3107)
        )

        self.step = 0
        self.epoch = 0
        self.best_evaluation_loss = math.inf
        self.history: list[dict] = []

        self.parameters = list(self.conditional_unet.parameters())

        # Il trainer è conditional se è presente il text encoder
        if self.text_encoder is not None:
            self.parameters.extend(self.text_encoder.parameters())

    @property
    def is_conditional(self) -> bool:
        return self.text_encoder is not None

    def prepare_batch(self, batch: dict) -> tuple[Tensor, Tensor, Tensor]:
        images = batch["image"].to(self.device, non_blocking=True)
        input_ids = batch["input_ids"].to(self.device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(
            self.device,
            non_blocking=True,
        )

        return images, input_ids, attention_mask

    def build_noisy_batch(
        self,
        images: Tensor,
        generator: torch.Generator,
    ) -> tuple[Tensor, Tensor, Tensor]:
        timesteps = torch.randint(
            low=0,
            high=self.schedule.num_timesteps,
            size=(images.shape[0],),
            device=self.device,
            dtype=torch.long,
            generator=generator,
        )

        noisyimages, target_noise = self.schedule.q_sample(
            x_start=images,
            timesteps=timesteps,
            generator=generator,
        )

        return noisyimages, target_noise, timesteps

    def predict_noise(
        self,
        noisyimages: Tensor,
        timesteps: Tensor,
        input_ids: Tensor,
        attention_mask: Tensor,
    ) -> Tensor:
        # In unconditional la U-Net viene invocata senza context
        if not self.is_conditional:
            return self.conditional_unet(
                noisyimages=noisyimages,
                timesteps=timesteps,
                context=None,
                attention_mask=None,
            )

        text_context = self.text_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        return self.conditional_unet(
            noisyimages=noisyimages,
            timesteps=timesteps,
            context=text_context,
            attention_mask=attention_mask,
        )

    def train_step(self, batch: dict) -> dict:
        self.conditional_unet.train()

        if self.text_encoder is not None:
            self.text_encoder.train()

        images, input_ids, attention_mask = self.prepare_batch(batch)

        noisyimages, target_noise, timesteps = self.build_noisy_batch(
            images=images,
            generator=self.train_generator,
        )

        self.optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type=self.device_type,
            dtype=self.amp_dtype,
            enabled=self.amp_enabled,
        ):
            predicted_noise = self.predict_noise(
                noisyimages=noisyimages,
                timesteps=timesteps,
                input_ids=input_ids,
                attention_mask=attention_mask,
            )

            loss = F.mse_loss(
                predicted_noise.float(),
                target_noise.float(),
            )

        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"Loss non finita allo step {self.step}: {loss.item()}"
            )

        self.scaler.scale(loss).backward()
        self.scaler.unscale_(self.optimizer)

        gradient_norm = torch.nn.utils.clip_grad_norm_(
            self.parameters,
            max_norm=self.gradient_clip_norm,
            error_if_nonfinite=True,
        )

        self.scaler.step(self.optimizer)
        self.scaler.update()

        self.step += 1

        return {
            "step": self.step,
            "epoch": self.epoch,
            "train_loss": float(loss.detach().item()),
            "gradient_norm": float(gradient_norm.detach().item()),
            "amp_scale": float(self.scaler.get_scale()),
        }

    @torch.inference_mode()
    def evaluate(self) -> float:
        if self.validation_dataloader is None:
            raise RuntimeError(
                "evaluate() chiamata senza validation_dataloader."
            )

        self.conditional_unet.eval()

        if self.text_encoder is not None:
            self.text_encoder.eval()

        total_loss = 0.0
        batches = 0

        generator = torch.Generator(
            device=self.device
        ).manual_seed(self.evaluation_seed)

        for batch in self.validation_dataloader:
            images, input_ids, attention_mask = self.prepare_batch(batch)

            noisyimages, target_noise, timesteps = self.build_noisy_batch(
                images=images,
                generator=generator,
            )

            with torch.autocast(
                device_type=self.device_type,
                dtype=self.amp_dtype,
                enabled=self.amp_enabled,
            ):
                predicted_noise = self.predict_noise(
                    noisyimages=noisyimages,
                    timesteps=timesteps,
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                )

                loss = F.mse_loss(
                    predicted_noise.float(),
                    target_noise.float(),
                )

            if not torch.isfinite(loss):
                raise FloatingPointError(
                    "Loss non finita durante evaluation."
                )

            total_loss += float(loss.item())
            batches += 1

        # La validation loss è mediata su tutto il validation set
        return total_loss / max(1, batches)

    def save(self, path: Path) -> None:
        models = {"conditional_unet": self.conditional_unet}

        if self.text_encoder is not None:
            models["text_encoder"] = self.text_encoder

        save_training_checkpoint(
            path=path,
            models=models,
            optimizer=self.optimizer,
            scaler=self.scaler,
            step=self.step,
            epoch=self.epoch,
            metadata={
                "best_evaluation_loss": self.best_evaluation_loss,
                "history": self.history,
                "conditional": self.is_conditional,
            },
            configs=self.config,
            generators={"training": self.train_generator},
        )

    def load(self, path: Path) -> None:
        models = {"conditional_unet": self.conditional_unet}

        if self.text_encoder is not None:
            models["text_encoder"] = self.text_encoder

        checkpoint = load_training_checkpoint(
            path=path,
            models=models,
            optimizer=self.optimizer,
            scaler=self.scaler,
            map_location=self.device,
            generators={"training": self.train_generator},
        )

        self.step = int(checkpoint["step"])
        self.epoch = int(checkpoint["epoch"])

        metadata = checkpoint.get("metadata", {})
        self.best_evaluation_loss = float(
            metadata.get("best_evaluation_loss", math.inf)
        )
        self.history = list(metadata.get("history", []))

    def write_history(self, path: Path) -> None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)

        keys = [
            "step",
            "epoch",
            "train_loss",
            "evaluation_loss",
            "gradient_norm",
            "amp_scale",
        ]

        with path.open("w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=keys)
            writer.writeheader()

            for row in self.history:
                writer.writerow({key: row.get(key) for key in keys})

    def fit(
        self,
        max_steps: int,
        evaluation_every: int,
        checkpoint_every: int,
        checkpoint_directory: Path,
        history_path: Path,
        logging_every: int,
    ) -> None:
        checkpoint_directory = Path(checkpoint_directory)
        checkpoint_directory.mkdir(parents=True, exist_ok=True)

        latest_path = checkpoint_directory / "latest.pt"
        best_path = checkpoint_directory / "best.pt"

        train_iterator = iter(self.train_dataloader)

        while self.step < max_steps:
            try:
                batch = next(train_iterator)

            except StopIteration:
                self.epoch += 1
                train_iterator = iter(self.train_dataloader)
                batch = next(train_iterator)

            metrics = self.train_step(batch)
            evaluation_loss = None

            should_evaluate = (
                self.validation_dataloader is not None
                and evaluation_every > 0
                and (self.step == 1 or self.step % evaluation_every == 0)
            )

            if should_evaluate:
                evaluation_loss = self.evaluate()
                metrics["evaluation_loss"] = evaluation_loss

                if evaluation_loss < self.best_evaluation_loss:
                    self.best_evaluation_loss = evaluation_loss
                    self.history.append(metrics.copy())
                    self.save(best_path)
                    self.history.pop()

            self.history.append(metrics)

            if self.step == 1 or self.step % logging_every == 0:
                evaluation_string = (
                    f"{evaluation_loss:.6f}"
                    if evaluation_loss is not None
                    else "-"
                )

                print(
                    f"Epoch {self.epoch} "
                    f"Step {self.step:05d} "
                    f"Train {metrics['train_loss']:.6f} "
                    f"Eval {evaluation_string} "
                    f"Grad {metrics['gradient_norm']:.4f}"
                )

            # latest.pt salva lo stato corrente, best.pt il miglior modello
            if self.step % checkpoint_every == 0 or self.step == max_steps:
                self.save(latest_path)
                self.write_history(history_path)

        print(
            f"Training terminato. "
            f"Step={self.step}, "
            f"Epoch={self.epoch}, "
            f"Best eval={self.best_evaluation_loss:.6f}"
        )

## Test unitari del Trainer

Verifichiamo il nuovo trainer con test dedicati su tre aspetti essenziali: un passo di training condizionato, un passo di training unconditional e il corretto salvataggio/ripristino da checkpoint. In questo modo validiamo subito sia la flessibilità dell'interfaccia sia la robustezza del ciclo di training.

In [ ]:
%%writefile tests/test_trainer.py
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from src.diffusion.schedule import DDPMSchedule
from src.training.trainer import Trainer


class DummyUNet(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.conv = torch.nn.Conv2d(3, 3, kernel_size=1)

    def forward(
        self,
        noisyimages,
        timesteps,
        context,
        attention_mask,
    ):
        del timesteps, context, attention_mask
        return self.conv(noisyimages)


class DummyTextEncoder(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.embedding = torch.nn.Embedding(10, 16)

    def forward(self, input_ids, attention_mask):
        del attention_mask
        return self.embedding(input_ids)


# Dataset sintetico minimo per test veloci e deterministici
def build_loader() -> DataLoader:
    images = torch.randn(8, 3, 32, 32)
    input_ids = torch.randint(0, 10, (8, 5))
    attention_mask = torch.ones(8, 5, dtype=torch.long)

    dataset = [
        {
            "image": images[index],
            "input_ids": input_ids[index],
            "attention_mask": attention_mask[index],
        }
        for index in range(8)
    ]

    return DataLoader(dataset, batch_size=4, shuffle=False)


def build_config() -> dict:
    return {
        "training": {
            "seed": 42,
            "gradient_clip_norm": 1.0,
            "amp": {"enabled": False},
            "evaluation": {"seed": 3107},
        }
    }


# Verifica del percorso conditional
def test_conditional_trainer_runs_one_step() -> None:
    unet = DummyUNet()
    text_encoder = DummyTextEncoder()
    schedule = DDPMSchedule(num_timesteps=10)
    loader = build_loader()

    optimizer = torch.optim.AdamW(
        list(unet.parameters()) + list(text_encoder.parameters())
    )

    trainer = Trainer(
        conditional_unet=unet,
        text_encoder=text_encoder,
        schedule=schedule,
        optimizer=optimizer,
        train_dataloader=loader,
        validation_dataloader=loader,
        device=torch.device("cpu"),
        config=build_config(),
    )

    metrics = trainer.train_step(next(iter(loader)))

    assert trainer.is_conditional
    assert trainer.step == 1
    assert torch.isfinite(torch.tensor(metrics["train_loss"]))
    assert torch.isfinite(torch.tensor(metrics["gradient_norm"]))
    assert torch.isfinite(torch.tensor(trainer.evaluate()))


# Verifica del percorso unconditional
def test_unconditional_trainer_runs_one_step() -> None:
    unet = DummyUNet()
    schedule = DDPMSchedule(num_timesteps=10)
    loader = build_loader()

    optimizer = torch.optim.AdamW(unet.parameters())

    trainer = Trainer(
        conditional_unet=unet,
        text_encoder=None,
        schedule=schedule,
        optimizer=optimizer,
        train_dataloader=loader,
        validation_dataloader=loader,
        device=torch.device("cpu"),
        config=build_config(),
    )

    metrics = trainer.train_step(next(iter(loader)))

    assert not trainer.is_conditional
    assert trainer.step == 1
    assert torch.isfinite(torch.tensor(metrics["train_loss"]))
    assert torch.isfinite(torch.tensor(metrics["gradient_norm"]))
    assert torch.isfinite(torch.tensor(trainer.evaluate()))


# Verifica di save/load con ripristino dello stato optimizer
def test_trainer_checkpoint_resume(tmp_path: Path) -> None:
    loader = build_loader()
    config = build_config()
    schedule = DDPMSchedule(num_timesteps=10)

    unet = DummyUNet()
    text_encoder = DummyTextEncoder()

    optimizer = torch.optim.AdamW(
        list(unet.parameters()) + list(text_encoder.parameters())
    )

    trainer = Trainer(
        conditional_unet=unet,
        text_encoder=text_encoder,
        schedule=schedule,
        optimizer=optimizer,
        train_dataloader=loader,
        validation_dataloader=loader,
        device=torch.device("cpu"),
        config=config,
    )

    trainer.train_step(next(iter(loader)))
    trainer.best_evaluation_loss = trainer.evaluate()

    checkpoint_path = tmp_path / "checkpoint.pt"
    trainer.save(checkpoint_path)

    resumed_unet = DummyUNet()
    resumed_text_encoder = DummyTextEncoder()

    resumed_optimizer = torch.optim.AdamW(
        list(resumed_unet.parameters())
        + list(resumed_text_encoder.parameters())
    )

    resumed_trainer = Trainer(
        conditional_unet=resumed_unet,
        text_encoder=resumed_text_encoder,
        schedule=DDPMSchedule(num_timesteps=10),
        optimizer=resumed_optimizer,
        train_dataloader=loader,
        validation_dataloader=loader,
        device=torch.device("cpu"),
        config=config,
    )

    resumed_trainer.load(checkpoint_path)

    assert resumed_trainer.step == trainer.step
    assert resumed_trainer.best_evaluation_loss == trainer.best_evaluation_loss
    assert len(resumed_optimizer.state) > 0

### Esecuzione dei test del Trainer

Eseguiamo i test dedicati al trainer per verificare immediatamente il corretto comportamento delle modalità conditional e unconditional, oltre al ripristino da checkpoint.

In [ ]:
!python -m pytest -q tests/test_trainer.py

## Configurazione dello smoke test

Prepariamo una configurazione di training molto breve, pensata per validare rapidamente l'intera pipeline del trainer generale senza eseguire ancora l'addestramento completo. Lo smoke test usa un numero ridotto di step, evaluation frequente, checkpoint ravvicinati e logging completo della history.

In [ ]:
%%writefile configs/train_smoke.yaml
training:
  seed: 2026

  max_steps: 20
  batch_size: 16

  learning_rate:
    unet: 0.0002
    text_encoder: 0.0001

  weight_decay: 0.0001
  gradient_clip_norm: 1.0

  amp:
    enabled: true

  evaluation:
    seed: 3107
    every_steps: 10

  checkpoint:
    every_steps: 10
    directory: checkpoints/train_smoke

  logging:
    every_steps: 1
    history_path: outputs/train_smoke/history.csv

## Esecuzione dello smoke test

Avviamo ora uno smoke test controllato del trainer generale in modalità condizionata. L'obiettivo non è ottenere convergenza, ma verificare che training, evaluation, salvataggio checkpoint e scrittura della history funzionino correttamente in un flusso end-to-end realistico.

In [ ]:
import math
import shutil
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from src.diffusion.schedule import DDPMSchedule, load_diffusion_config
from src.models.conditional_unet import ConditionalUNet, load_unet_config
from src.models.text_encoder import (
    TextTransformerEncoder,
    load_text_encoder_config,
)
from src.training.trainer import (
    Trainer,
    load_training_config,
    set_training_seed,
)

# ---------------------------------------------------------
# 0. Configurazione e controlli iniziali
# ---------------------------------------------------------

assert "train" in datasets, "Split train non disponibile."
assert "validation" in datasets, "Split validation non disponibile."

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_config = load_training_config(Path("configs/train_smoke.yaml"))
unet_config = load_unet_config(Path("configs/unet.yaml"))
text_config = load_text_encoder_config(Path("configs/text_encoder.yaml"))
diffusion_config = load_diffusion_config(Path("configs/diffusion.yaml"))

training_config = train_config["training"]

required_keys = {
    "seed",
    "batch_size",
    "max_steps",
    "learning_rate",
    "weight_decay",
    "evaluation",
    "checkpoint",
    "logging",
}

missing_keys = required_keys - set(training_config)
assert not missing_keys, (
    "Chiavi mancanti in configs/train_smoke.yaml: "
    f"{sorted(missing_keys)}"
)

assert "unet" in training_config["learning_rate"]
assert "text_encoder" in training_config["learning_rate"]
assert "every_steps" in training_config["evaluation"]
assert "every_steps" in training_config["checkpoint"]
assert "directory" in training_config["checkpoint"]
assert "every_steps" in training_config["logging"]
assert "history_path" in training_config["logging"]

set_training_seed(int(training_config["seed"]))

batch_size = int(training_config["batch_size"])
max_steps = int(training_config["max_steps"])

assert batch_size > 0
assert max_steps > 0
assert len(datasets["train"]) >= batch_size, (
    "Il train split è più piccolo del batch size."
)
assert len(datasets["validation"]) > 0, (
    "Il validation split è vuoto."
)

# ---------------------------------------------------------
# 1. DataLoader
# ---------------------------------------------------------

train_loader = DataLoader(
    datasets["train"],
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    drop_last=True,
)

validation_loader = DataLoader(
    datasets["validation"],
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    drop_last=False,
)

assert len(train_loader) > 0
assert len(validation_loader) > 0

# ---------------------------------------------------------
# 2. Artefatti puliti
# ---------------------------------------------------------

smoke_checkpoint_dir = Path(
    training_config["checkpoint"]["directory"]
)

smoke_history_path = Path(
    training_config["logging"]["history_path"]
)

if smoke_checkpoint_dir.exists():
    shutil.rmtree(smoke_checkpoint_dir)

if smoke_history_path.exists():
    smoke_history_path.unlink()

smoke_checkpoint_dir.mkdir(parents=True, exist_ok=True)
smoke_history_path.parent.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 3. Modelli, schedule e optimizer
# ---------------------------------------------------------

smoke_unet = ConditionalUNet.from_config(
    unet_config,
    conditional=True,
).to(device)

smoke_text_encoder = TextTransformerEncoder.from_config(
    text_config,
).to(device)

smoke_schedule = DDPMSchedule.from_config(
    diffusion_config,
).to(device)

smoke_optimizer = torch.optim.AdamW(
    [
        {
            "params": smoke_unet.parameters(),
            "lr": float(
                training_config["learning_rate"]["unet"]
            ),
        },
        {
            "params": smoke_text_encoder.parameters(),
            "lr": float(
                training_config["learning_rate"]["text_encoder"]
            ),
        },
    ],
    weight_decay=float(training_config["weight_decay"]),
)

assert sum(
    parameter.numel()
    for parameter in smoke_unet.parameters()
) > 0

assert sum(
    parameter.numel()
    for parameter in smoke_text_encoder.parameters()
) > 0

assert len(smoke_optimizer.param_groups) == 2

# ---------------------------------------------------------
# 4. Trainer
# ---------------------------------------------------------

smoke_trainer = Trainer(
    conditional_unet=smoke_unet,
    text_encoder=smoke_text_encoder,
    schedule=smoke_schedule,
    optimizer=smoke_optimizer,
    train_dataloader=train_loader,
    validation_dataloader=validation_loader,
    device=device,
    config=train_config,
)

print(f"Device: {device}")
print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(validation_loader)}")
print(f"Batch size: {batch_size}")
print(f"Smoke steps: {max_steps}")

# ---------------------------------------------------------
# 5. Smoke test
# ---------------------------------------------------------

initial_smoke_loss = float(smoke_trainer.evaluate())
print(f"Initial validation loss: {initial_smoke_loss:.6f}")

smoke_trainer.fit(
    max_steps=max_steps,
    evaluation_every=int(
        training_config["evaluation"]["every_steps"]
    ),
    checkpoint_every=int(
        training_config["checkpoint"]["every_steps"]
    ),
    checkpoint_directory=smoke_checkpoint_dir,
    history_path=smoke_history_path,
    logging_every=int(
        training_config["logging"]["every_steps"]
    ),
)

final_smoke_loss = float(smoke_trainer.evaluate())

recorded_evaluations = [
    float(row["evaluation_loss"])
    for row in smoke_trainer.history
    if row.get("evaluation_loss") is not None
]

assert recorded_evaluations, (
    "Nessuna evaluation è stata registrata."
)

expected_best_loss = min(recorded_evaluations)

print(f"Final validation loss: {final_smoke_loss:.6f}")
print(
    "Best validation loss registrata: "
    f"{smoke_trainer.best_evaluation_loss:.6f}"
)

assert smoke_trainer.step == max_steps
assert math.isfinite(initial_smoke_loss)
assert math.isfinite(final_smoke_loss)
assert math.isfinite(smoke_trainer.best_evaluation_loss)

assert (smoke_checkpoint_dir / "latest.pt").exists()
assert (smoke_checkpoint_dir / "best.pt").exists()
assert smoke_history_path.exists()

assert math.isclose(
    smoke_trainer.best_evaluation_loss,
    expected_best_loss,
    rel_tol=1e-7,
    abs_tol=1e-8,
), (
    "best_evaluation_loss non coincide con il minimo "
    "delle evaluation registrate."
)

print("Smoke test del Trainer generale superato.")

## Verifica finale degli artefatti smoke

Concludiamo il blocco verificando la consistenza degli artefatti prodotti dallo smoke test: checkpoint `latest.pt`, checkpoint `best.pt` e file `history.csv`. Controlliamo inoltre che la best evaluation loss salvata coincida effettivamente con il minimo osservato nella history.

In [ ]:
import math
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.diffusion.schedule import (
    DDPMSchedule,
    load_diffusion_config,
)
from src.models.conditional_unet import (
    ConditionalUNet,
    load_unet_config,
)
from src.models.text_encoder import (
    TextTransformerEncoder,
    load_text_encoder_config,
)
from src.training.trainer import (
    Trainer,
    load_training_config,
    set_training_seed,
)

# ---------------------------------------------------------
# 0. Percorsi degli artefatti attesi
# ---------------------------------------------------------

train_config = load_training_config(Path("configs/train_smoke.yaml"))
training_config = train_config["training"]

checkpoint_dir = Path(training_config["checkpoint"]["directory"])
history_path = Path(training_config["logging"]["history_path"])

latest_checkpoint = checkpoint_dir / "latest.pt"
best_checkpoint = checkpoint_dir / "best.pt"

SMOKE_TRAINING_STEPS = int(training_config["max_steps"])

for path in [latest_checkpoint, best_checkpoint, history_path]:
    assert path.exists(), f"Artefatto mancante: {path}"
    print(f"{path} ({path.stat().st_size / 1024**2:.2f} MiB)")

# ---------------------------------------------------------
# 1. Verifica della history CSV
# ---------------------------------------------------------

history = pd.read_csv(history_path)

required_columns = {
    "step",
    "epoch",
    "train_loss",
    "evaluation_loss",
    "gradient_norm",
    "amp_scale",
}

assert len(history) == SMOKE_TRAINING_STEPS, (
    "Numero righe history inatteso: "
    f"{len(history)} invece di {SMOKE_TRAINING_STEPS}."
)

missing_columns = required_columns - set(history.columns)
assert not missing_columns, (
    "Colonne mancanti nella history: "
    f"{sorted(missing_columns)}"
)

assert history["step"].iloc[0] == 1
assert int(history["step"].iloc[-1]) == SMOKE_TRAINING_STEPS
assert history["step"].is_monotonic_increasing
assert history["step"].is_unique
assert history["train_loss"].notna().all()

evaluated = history.dropna(subset=["evaluation_loss"]).copy()

assert not evaluated.empty, (
    "Nessuna evaluation_loss valida trovata nella history."
)

assert evaluated["evaluation_loss"].map(math.isfinite).all()

best_row = evaluated.loc[evaluated["evaluation_loss"].idxmin()]
best_step_from_history = int(best_row["step"])
best_loss_from_history = float(best_row["evaluation_loss"])

print(f"Numero righe history: {len(history)}")
print(f"Numero evaluation registrate: {len(evaluated)}")
print(
    "Miglior evaluation_loss in history: "
    f"{best_loss_from_history:.6f} "
    f"(step {best_step_from_history})"
)

# ---------------------------------------------------------
# 2. Ricostruzione del trainer
# ---------------------------------------------------------

assert "train" in datasets, "Split train non disponibile."
assert "validation" in datasets, "Split validation non disponibile."

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

set_training_seed(int(training_config["seed"]))

batch_size = int(training_config["batch_size"])

train_loader = DataLoader(
    datasets["train"],
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    drop_last=True,
)

validation_loader = DataLoader(
    datasets["validation"],
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    drop_last=False,
)

unet_config = load_unet_config(Path("configs/unet.yaml"))
text_config = load_text_encoder_config(
    Path("configs/text_encoder.yaml")
)
diffusion_config = load_diffusion_config(
    Path("configs/diffusion.yaml")
)


def build_trainer():
    conditional_unet = ConditionalUNet.from_config(
        unet_config,
        conditional=True,
    ).to(device)

    text_encoder = TextTransformerEncoder.from_config(
        text_config,
    ).to(device)

    schedule = DDPMSchedule.from_config(
        diffusion_config,
    ).to(device)

    optimizer = torch.optim.AdamW(
        [
            {
                "params": conditional_unet.parameters(),
                "lr": float(
                    training_config["learning_rate"]["unet"]
                ),
            },
            {
                "params": text_encoder.parameters(),
                "lr": float(
                    training_config["learning_rate"]["text_encoder"]
                ),
            },
        ],
        weight_decay=float(training_config["weight_decay"]),
    )

    return Trainer(
        conditional_unet=conditional_unet,
        text_encoder=text_encoder,
        schedule=schedule,
        optimizer=optimizer,
        train_dataloader=train_loader,
        validation_dataloader=validation_loader,
        device=device,
        config=train_config,
    )


# ---------------------------------------------------------
# 3. Verifica di latest.pt
# ---------------------------------------------------------

latest_trainer = build_trainer()
latest_trainer.load(latest_checkpoint)

latest_eval = float(latest_trainer.evaluate())

assert latest_trainer.step == SMOKE_TRAINING_STEPS
assert len(latest_trainer.history) == SMOKE_TRAINING_STEPS
assert math.isfinite(latest_trainer.best_evaluation_loss)
assert math.isfinite(latest_eval)

assert math.isclose(
    latest_trainer.best_evaluation_loss,
    best_loss_from_history,
    rel_tol=1e-7,
    abs_tol=1e-8,
), (
    "Il best_evaluation_loss in latest.pt non coincide "
    "con il minimo della history."
)

print(f"Step latest.pt: {latest_trainer.step}")
print(
    "Best validation loss da latest.pt: "
    f"{latest_trainer.best_evaluation_loss:.6f}"
)
print(
    "Validation loss ricalcolata da latest.pt: "
    f"{latest_eval:.6f}"
)

# ---------------------------------------------------------
# 4. Verifica di best.pt
# ---------------------------------------------------------

best_trainer = build_trainer()
best_trainer.load(best_checkpoint)

best_eval = float(best_trainer.evaluate())

assert best_trainer.step == best_step_from_history, (
    "Lo step di best.pt non coincide con il miglior step "
    "ricavato dalla history."
)
assert len(best_trainer.history) == best_step_from_history
assert math.isfinite(best_trainer.best_evaluation_loss)
assert math.isfinite(best_eval)

assert math.isclose(
    best_trainer.best_evaluation_loss,
    best_loss_from_history,
    rel_tol=1e-7,
    abs_tol=1e-8,
), (
    "Il best_evaluation_loss in best.pt non coincide "
    "con il minimo della history."
)

print(f"Step best.pt: {best_trainer.step}")
print(
    "Best validation loss da best.pt: "
    f"{best_trainer.best_evaluation_loss:.6f}"
)
print(
    "Validation loss ricalcolata da best.pt: "
    f"{best_eval:.6f}"
)

print("Verifica finale degli artefatti smoke completata con successo.")

Il Trainer generale è stato implementato, testato e validato tramite smoke test completo con verifica finale degli artefatti. L’infrastruttura di training è ora pronta per passare al blocco successivo dedicato al training completo dei modelli conditional e unconditional.

# Training conditional e baseline unconditional

In questo blocco addestriamo i due modelli che verranno confrontati nella fase di evaluation: il modello diffusion condizionato dal testo e la baseline neural unconditional.

Per garantire un esperimento pulito, i due training ufficiali partono da nuovi pesi inizializzati con seed fissato e sovrascrivono esplicitamente checkpoint e history dei run ufficiali precedenti. Entrambi usano configurazioni versionate, validation periodica, checkpoint `latest.pt` e `best.pt`, logging completo e verifica finale di integrità degli artefatti prodotti.


## Training completo conditional

Avviamo il training completo del modello conditional composto da U-Net condizionata e Text Transformer Encoder. Il modello riceve l'immagine rumorosa, il timestep e il contesto testuale associato alla caption, mentre la validation loss viene calcolata periodicamente sul validation split con rumore deterministico.

In [ ]:
%%writefile configs/train.yaml
training:
  seed: 2026

  max_steps: 10000
  batch_size: 64

  learning_rate:
    unet: 0.0002
    text_encoder: 0.0001

  weight_decay: 0.0001
  gradient_clip_norm: 1.0

  amp:
    enabled: true
    dtype: float16

  evaluation:
    seed: 3107
    every_steps: 500

  checkpoint:
    every_steps: 1000
    directory: checkpoints/train

  logging:
    every_steps: 100
    history_path: outputs/train/history.csv

### Avvio pulito del training conditional

La cella ricostruisce modelli, schedule, optimizer e dataloader a partire dalle configurazioni versionate. Prima dell'addestramento elimina esclusivamente gli artefatti del run conditional ufficiale precedente e avvia un training pulito da zero con seed controllato. In questo modo `best.pt`, `latest.pt` e la history finale appartengono tutti alla stessa esecuzione.


In [ ]:
# ---------------------------------------------------------
# Training ufficiale: DDPM text-conditioned con vera U-Net
# ---------------------------------------------------------

import math
import shutil
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from src.diffusion.schedule import (
    DDPMSchedule,
    load_diffusion_config,
)
from src.models.conditional_unet import (
    ConditionalUNet,
    load_unet_config,
)
from src.models.text_encoder import (
    TextTransformerEncoder,
    load_text_encoder_config,
)
from src.training.trainer import (
    Trainer,
    load_training_config,
    set_training_seed,
)


# ---------------------------------------------------------
# 0. Configurazioni e controlli preliminari
# ---------------------------------------------------------

assert "train" in datasets, "Split train non disponibile."
assert "validation" in datasets, "Split validation non disponibile."

assert len(datasets["train"]) > 0, "Split train vuoto."
assert len(datasets["validation"]) > 0, "Split validation vuoto."

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert device.type == "cuda", (
    "Il training ufficiale richiede una GPU CUDA."
)

train_config = load_training_config(
    Path("configs/train.yaml")
)

unet_config = load_unet_config(
    Path("configs/unet.yaml")
)

text_config = load_text_encoder_config(
    Path("configs/text_encoder.yaml")
)

diffusion_config = load_diffusion_config(
    Path("configs/diffusion.yaml")
)

training_settings = train_config["training"]

set_training_seed(int(training_settings["seed"]))

max_steps = int(training_settings["max_steps"])
batch_size = int(training_settings["batch_size"])

evaluation_every = int(
    training_settings["evaluation"]["every_steps"]
)

checkpoint_every = int(
    training_settings["checkpoint"]["every_steps"]
)

logging_every = int(
    training_settings["logging"]["every_steps"]
)

assert max_steps > 0
assert batch_size > 0
assert evaluation_every > 0
assert checkpoint_every > 0
assert logging_every > 0

assert len(datasets["train"]) >= batch_size, (
    "Train split più piccolo del batch size."
)


# ---------------------------------------------------------
# 1. DataLoader
# ---------------------------------------------------------

loader_kwargs = {
    "num_workers": 0,
    "pin_memory": True,
}

conditional_train_loader = DataLoader(
    datasets["train"],
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    **loader_kwargs,
)

conditional_validation_loader = DataLoader(
    datasets["validation"],
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    **loader_kwargs,
)

assert len(conditional_train_loader) > 0
assert len(conditional_validation_loader) > 0


# ---------------------------------------------------------
# 2. Nuovi modelli, schedule e optimizer
# ---------------------------------------------------------

conditional_unet = ConditionalUNet.from_config(
    unet_config,
    conditional=True,
).to(device)

conditional_text_encoder = TextTransformerEncoder.from_config(
    text_config,
).to(device)

conditional_schedule = DDPMSchedule.from_config(
    diffusion_config,
).to(device)

conditional_optimizer = torch.optim.AdamW(
    [
        {
            "params": conditional_unet.parameters(),
            "lr": float(
                training_settings["learning_rate"]["unet"]
            ),
        },
        {
            "params": conditional_text_encoder.parameters(),
            "lr": float(
                training_settings["learning_rate"][
                    "text_encoder"
                ]
            ),
        },
    ],
    weight_decay=float(
        training_settings["weight_decay"]
    ),
)

unet_parameters = sum(
    parameter.numel()
    for parameter in conditional_unet.parameters()
)

text_parameters = sum(
    parameter.numel()
    for parameter in conditional_text_encoder.parameters()
)

assert unet_parameters > 5_000_000, (
    "La U-Net è troppo piccola: verifica "
    "src/models/conditional_unet.py."
)

print(f"Parametri U-Net: {unet_parameters:,}")
print(f"Parametri text encoder: {text_parameters:,}")


# ---------------------------------------------------------
# 3. Sovrascrittura artefatti conditional precedenti
# ---------------------------------------------------------

conditional_checkpoint_dir = Path(
    training_settings["checkpoint"]["directory"]
)

conditional_history_path = Path(
    training_settings["logging"]["history_path"]
)

if conditional_checkpoint_dir.exists():
    shutil.rmtree(conditional_checkpoint_dir)
    print(
        "Checkpoint conditional precedenti eliminati: "
        f"{conditional_checkpoint_dir}"
    )

if conditional_history_path.exists():
    conditional_history_path.unlink()
    print(
        "History conditional precedente eliminata: "
        f"{conditional_history_path}"
    )

conditional_checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True,
)

conditional_history_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------
# 4. Training da zero
# ---------------------------------------------------------

conditional_trainer = Trainer(
    conditional_unet=conditional_unet,
    text_encoder=conditional_text_encoder,
    schedule=conditional_schedule,
    optimizer=conditional_optimizer,
    train_dataloader=conditional_train_loader,
    validation_dataloader=conditional_validation_loader,
    device=device,
    config=train_config,
)

initial_validation_loss = float(
    conditional_trainer.evaluate()
)

assert math.isfinite(initial_validation_loss)

print("--- AVVIO TRAINING CONDITIONAL ---")
print(f"Device: {device}")
print(f"Batch size: {batch_size}")
print(
    "Batch train per epoch: "
    f"{len(conditional_train_loader)}"
)
print(
    "Batch validation: "
    f"{len(conditional_validation_loader)}"
)
print(f"Step target: {max_steps}")
print(
    "Validation loss iniziale: "
    f"{initial_validation_loss:.6f}"
)

conditional_trainer.fit(
    max_steps=max_steps,
    evaluation_every=evaluation_every,
    checkpoint_every=checkpoint_every,
    checkpoint_directory=conditional_checkpoint_dir,
    history_path=conditional_history_path,
    logging_every=logging_every,
)

final_validation_loss = float(
    conditional_trainer.evaluate()
)

latest_checkpoint = (
    conditional_checkpoint_dir / "latest.pt"
)

best_checkpoint = (
    conditional_checkpoint_dir / "best.pt"
)

assert conditional_trainer.step == max_steps
assert math.isfinite(final_validation_loss)
assert math.isfinite(
    conditional_trainer.best_evaluation_loss
)

assert latest_checkpoint.exists()
assert best_checkpoint.exists()
assert conditional_history_path.exists()

print("--- TRAINING CONDITIONAL COMPLETATO ---")
print(f"Step finale: {conditional_trainer.step}")
print(
    "Validation loss finale: "
    f"{final_validation_loss:.6f}"
)
print(
    "Migliore validation loss: "
    f"{conditional_trainer.best_evaluation_loss:.6f}"
)
print(f"Checkpoint migliore: {best_checkpoint}")

In [ ]:
# ---------------------------------------------------------
# Verifica riproducibile del miglior checkpoint conditional
# ---------------------------------------------------------

import math
from pathlib import Path

import pandas as pd
import torch

from src.diffusion.schedule import DDPMSchedule
from src.models.conditional_unet import ConditionalUNet
from src.models.text_encoder import TextTransformerEncoder
from src.training.trainer import Trainer, set_training_seed


conditional_latest_path = (
    conditional_checkpoint_dir / "latest.pt"
)

conditional_best_path = (
    conditional_checkpoint_dir / "best.pt"
)

assert conditional_latest_path.exists()
assert conditional_best_path.exists()
assert conditional_history_path.exists()

conditional_history = pd.read_csv(
    conditional_history_path
)

assert len(conditional_history) == max_steps
assert conditional_history["step"].is_unique
assert conditional_history["step"].is_monotonic_increasing

evaluations = conditional_history.dropna(
    subset=["evaluation_loss"]
).copy()

assert not evaluations.empty
assert evaluations["evaluation_loss"].map(
    math.isfinite
).all()

best_history_row = evaluations.loc[
    evaluations["evaluation_loss"].idxmin()
]

best_step_from_history = int(
    best_history_row["step"]
)

best_loss_from_history = float(
    best_history_row["evaluation_loss"]
)

# Ricostruzione completa: non riutilizziamo gli oggetti
# rimasti in memoria dal training appena terminato.
set_training_seed(int(training_settings["seed"]))

reloaded_conditional_unet = ConditionalUNet.from_config(
    unet_config,
    conditional=True,
).to(device)

reloaded_conditional_text_encoder = (
    TextTransformerEncoder.from_config(
        text_config
    ).to(device)
)

reloaded_conditional_schedule = (
    DDPMSchedule.from_config(
        diffusion_config
    ).to(device)
)

reloaded_conditional_optimizer = torch.optim.AdamW(
    [
        {
            "params": reloaded_conditional_unet.parameters(),
            "lr": float(
                training_settings["learning_rate"]["unet"]
            ),
        },
        {
            "params": (
                reloaded_conditional_text_encoder.parameters()
            ),
            "lr": float(
                training_settings["learning_rate"][
                    "text_encoder"
                ]
            ),
        },
    ],
    weight_decay=float(
        training_settings["weight_decay"]
    ),
)

reloaded_conditional_trainer = Trainer(
    conditional_unet=reloaded_conditional_unet,
    text_encoder=reloaded_conditional_text_encoder,
    schedule=reloaded_conditional_schedule,
    optimizer=reloaded_conditional_optimizer,
    train_dataloader=conditional_train_loader,
    validation_dataloader=conditional_validation_loader,
    device=device,
    config=train_config,
)

reloaded_conditional_trainer.load(
    conditional_best_path
)

reloaded_best_loss = float(
    reloaded_conditional_trainer.evaluate()
)

assert reloaded_conditional_trainer.step == best_step_from_history
assert math.isfinite(reloaded_best_loss)

torch.testing.assert_close(
    torch.tensor(reloaded_best_loss),
    torch.tensor(best_loss_from_history),
    rtol=0.0,
    atol=1e-6,
)

print("--- CHECKPOINT CONDITIONAL VERIFICATO ---")
print(f"Step best checkpoint: {best_step_from_history}")
print(
    "Best loss dalla history: "
    f"{best_loss_from_history:.6f}"
)
print(
    "Best loss ricalcolata: "
    f"{reloaded_best_loss:.6f}"
)
print(f"Checkpoint: {conditional_best_path}")

### Verifica degli artefatti conditional

Verifichiamo la coerenza della history e dei checkpoint prodotti dal training conditional. In particolare controlliamo il numero di step, la presenza delle colonne richieste, la validità numerica delle loss e la corrispondenza tra la migliore evaluation loss nella history e quella memorizzata nei checkpoint.

In [ ]:
import math
from pathlib import Path

import pandas as pd
import torch

# ---------------------------------------------------------
# 0. Percorsi degli artefatti conditional
# ---------------------------------------------------------

checkpoint_dir = Path("checkpoints/train")
history_path = Path("outputs/train/history.csv")

latest_path = checkpoint_dir / "latest.pt"
best_path = checkpoint_dir / "best.pt"

EXPECTED_MAX_STEPS = 10_000

for path in [latest_path, best_path, history_path]:
    assert path.exists(), f"Artefatto mancante: {path}"
    assert path.stat().st_size > 0, f"Artefatto vuoto: {path}"

print("Artefatti conditional trovati:")
for path in [latest_path, best_path, history_path]:
    print(f"- {path} ({path.stat().st_size / 1024**2:.2f} MiB)")

# ---------------------------------------------------------
# 1. Verifica della history
# ---------------------------------------------------------

history = pd.read_csv(history_path)

required_columns = {
    "step",
    "epoch",
    "train_loss",
    "evaluation_loss",
    "gradient_norm",
    "amp_scale",
}

missing_columns = required_columns - set(history.columns)

assert not missing_columns, (
    "Colonne mancanti nella history: "
    f"{sorted(missing_columns)}"
)

assert len(history) == EXPECTED_MAX_STEPS, (
    f"History con {len(history)} righe, "
    f"attese {EXPECTED_MAX_STEPS}."
)

assert int(history["step"].iloc[0]) == 1
assert int(history["step"].iloc[-1]) == EXPECTED_MAX_STEPS
assert history["step"].is_monotonic_increasing
assert history["step"].is_unique

assert history["train_loss"].notna().all()
assert history["train_loss"].map(math.isfinite).all()

evaluated = history.dropna(subset=["evaluation_loss"]).copy()

assert not evaluated.empty, (
    "Nessuna evaluation registrata nella history."
)

assert evaluated["evaluation_loss"].map(math.isfinite).all()

best_row = evaluated.loc[
    evaluated["evaluation_loss"].idxmin()
]

best_step_from_history = int(best_row["step"])
best_loss_from_history = float(
    best_row["evaluation_loss"]
)

# ---------------------------------------------------------
# 2. Verifica dei checkpoint serializzati
# ---------------------------------------------------------

latest_checkpoint = torch.load(
    latest_path,
    map_location="cpu",
    weights_only=False,
)

best_checkpoint = torch.load(
    best_path,
    map_location="cpu",
    weights_only=False,
)

required_checkpoint_keys = {
    "format_version",
    "step",
    "epoch",
    "models",
    "optimizer",
    "scaler",
    "rng_state",
    "generator_states",
    "metadata",
    "configs",
}

for checkpoint_name, checkpoint in {
    "latest.pt": latest_checkpoint,
    "best.pt": best_checkpoint,
}.items():
    assert isinstance(checkpoint, dict), (
        f"{checkpoint_name} non contiene un dizionario."
    )

    missing_keys = required_checkpoint_keys - set(checkpoint)

    assert not missing_keys, (
        f"Chiavi mancanti in {checkpoint_name}: "
        f"{sorted(missing_keys)}"
    )

    assert "conditional_unet" in checkpoint["models"]
    assert "text_encoder" in checkpoint["models"]
    assert isinstance(checkpoint["metadata"], dict)

assert int(latest_checkpoint["step"]) == EXPECTED_MAX_STEPS

assert int(best_checkpoint["step"]) == best_step_from_history, (
    "Lo step in best.pt non coincide con il miglior step "
    "ricavato dalla history."
)

latest_best_loss = float(
    latest_checkpoint["metadata"]["best_evaluation_loss"]
)

best_best_loss = float(
    best_checkpoint["metadata"]["best_evaluation_loss"]
)

assert math.isfinite(latest_best_loss)
assert math.isfinite(best_best_loss)

assert math.isclose(
    latest_best_loss,
    best_loss_from_history,
    rel_tol=1e-7,
    abs_tol=1e-8,
), (
    "La best evaluation loss nei metadata di latest.pt "
    "non coincide con il minimo della history."
)

assert math.isclose(
    best_best_loss,
    best_loss_from_history,
    rel_tol=1e-7,
    abs_tol=1e-8,
), (
    "La best evaluation loss nei metadata di best.pt "
    "non coincide con il minimo della history."
)

print("--- VERIFICA TRAINING CONDITIONAL SUPERATA ---")
print(f"Righe history: {len(history)}")
print(f"Numero evaluation: {len(evaluated)}")
print(f"Step latest.pt: {int(latest_checkpoint['step'])}")
print(f"Step best.pt: {int(best_checkpoint['step'])}")
print(f"Best step dalla history: {best_step_from_history}")
print(f"Best validation loss: {best_loss_from_history:.6f}")
print(f"Latest checkpoint: {latest_path}")
print(f"Best checkpoint: {best_path}")
print(f"History CSV: {history_path}")

## Baseline neural unconditional

Costruiamo ora la baseline unconditional usando la stessa architettura U-Net e la stessa schedule diffusion, ma senza Text Transformer Encoder e senza cross-attention condizionata dal testo. Questa baseline permette di misurare sperimentalmente il contributo del conditioning testuale durante la fase di evaluation.

In [ ]:
%%writefile configs/train_unconditional.yaml
training:
  seed: 2026

  max_steps: 10000
  batch_size: 64

  learning_rate:
    unet: 0.0002

  weight_decay: 0.0001
  gradient_clip_norm: 1.0

  amp:
    enabled: true
    dtype: float16

  evaluation:
    seed: 3107
    every_steps: 500

  checkpoint:
    every_steps: 1000
    directory: checkpoints/train_unconditional

  logging:
    every_steps: 100
    history_path: outputs/train_unconditional/history.csv

### Preparazione della baseline unconditional

In questa fase vengono creati i dataloader, la U-Net senza conditioning, la schedule DDPM, l'optimizer e il Trainer generale senza text encoder. Prima del training eseguiamo controlli espliciti per assicurarci che la baseline sia realmente unconditional.

In [ ]:
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from src.diffusion.schedule import (
    DDPMSchedule,
    load_diffusion_config,
)
from src.models.conditional_unet import (
    ConditionalUNet,
    load_unet_config,
)
from src.training.trainer import (
    Trainer,
    load_training_config,
    set_training_seed,
)

# ---------------------------------------------------------
# 0. Configurazioni e split
# ---------------------------------------------------------

assert "train" in datasets, "Split train non disponibile."
assert "validation" in datasets, "Split validation non disponibile."
assert len(datasets["train"]) > 0, "Split train vuoto."
assert len(datasets["validation"]) > 0, "Split validation vuoto."

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

unconditional_config = load_training_config(
    Path("configs/train_unconditional.yaml")
)

unet_config = load_unet_config(Path("configs/unet.yaml"))

diffusion_config = load_diffusion_config(
    Path("configs/diffusion.yaml")
)

training_config = unconditional_config["training"]

set_training_seed(int(training_config["seed"]))

batch_size = int(training_config["batch_size"])

assert batch_size > 0
assert "unet" in training_config["learning_rate"]

assert "text_encoder" not in training_config["learning_rate"], (
    "La configurazione unconditional non deve definire "
    "un learning rate per il text encoder."
)

assert len(datasets["train"]) >= batch_size, (
    "Il train split è più piccolo del batch size."
)

# ---------------------------------------------------------
# 1. DataLoader
# ---------------------------------------------------------

loader_kwargs = {
    "num_workers": 0,
    "pin_memory": device.type == "cuda",
}

train_loader_unconditional = DataLoader(
    datasets["train"],
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    **loader_kwargs,
)

validation_loader_unconditional = DataLoader(
    datasets["validation"],
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    **loader_kwargs,
)

assert len(train_loader_unconditional) > 0, (
    "Il DataLoader train non ha batch disponibili."
)

assert len(validation_loader_unconditional) > 0, (
    "Il DataLoader validation non ha batch disponibili."
)

# ---------------------------------------------------------
# 2. Modello, schedule e optimizer
# ---------------------------------------------------------

unconditional_unet = ConditionalUNet.from_config(
    unet_config,
    conditional=False,
).to(device)

unconditional_schedule = DDPMSchedule.from_config(
    diffusion_config,
).to(device)

unconditional_optimizer = torch.optim.AdamW(
    unconditional_unet.parameters(),
    lr=float(training_config["learning_rate"]["unet"]),
    weight_decay=float(training_config["weight_decay"]),
)

trainer_unconditional = Trainer(
    conditional_unet=unconditional_unet,
    text_encoder=None,
    schedule=unconditional_schedule,
    optimizer=unconditional_optimizer,
    train_dataloader=train_loader_unconditional,
    validation_dataloader=validation_loader_unconditional,
    device=device,
    config=unconditional_config,
)

unconditional_checkpoint_dir = Path(
    training_config["checkpoint"]["directory"]
)

unconditional_history_path = Path(
    training_config["logging"]["history_path"]
)

unconditional_checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True,
)

unconditional_history_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

assert unconditional_unet.conditional is False
assert trainer_unconditional.is_conditional is False
assert trainer_unconditional.text_encoder is None

print("--- BASELINE NEURAL UNCONDITIONAL ---")
print(f"Device: {device}")
print(f"Batch size: {batch_size}")
print(f"Batch train: {len(train_loader_unconditional)}")
print(
    f"Batch validation: "
    f"{len(validation_loader_unconditional)}"
)
print(
    "Checkpoint directory: "
    f"{unconditional_checkpoint_dir}"
)
print(f"History path: {unconditional_history_path}")
print(f"Conditional mode: {trainer_unconditional.is_conditional}")

### Smoke test unconditional

Prima del training completo eseguiamo uno smoke test breve e isolato della baseline unconditional. Questo controllo verifica che la U-Net senza text conditioning, il Trainer generale, l'evaluation, il checkpointing e la history funzionino correttamente anche senza text encoder.

In [ ]:
import math
import shutil
from pathlib import Path

# ---------------------------------------------------------
# 0. Configurazione dello smoke test
# ---------------------------------------------------------

SMOKE_STEPS = 20
EVALUATION_EVERY = 10
CHECKPOINT_EVERY = 10
LOGGING_EVERY = 5

smoke_checkpoint_dir = Path(
    "checkpoints/smoke_unconditional"
)

smoke_history_path = Path(
    "outputs/smoke_unconditional/history.csv"
)

assert SMOKE_STEPS > 0
assert 0 < EVALUATION_EVERY <= SMOKE_STEPS
assert 0 < CHECKPOINT_EVERY <= SMOKE_STEPS
assert 0 < LOGGING_EVERY <= SMOKE_STEPS

assert trainer_unconditional.is_conditional is False, (
    "Il trainer unconditional non è realmente unconditional."
)

assert trainer_unconditional.text_encoder is None, (
    "Lo smoke test unconditional non deve usare un text encoder."
)

assert trainer_unconditional.step == 0, (
    "Il trainer unconditional è già avanzato. "
    "Ricrearlo prima di eseguire uno smoke test pulito."
)

assert trainer_unconditional.validation_dataloader is not None, (
    "Il validation DataLoader è necessario per lo smoke test."
)

# ---------------------------------------------------------
# 1. Pulizia preventiva degli artefatti
# ---------------------------------------------------------

if smoke_checkpoint_dir.exists():
    shutil.rmtree(smoke_checkpoint_dir)

if smoke_history_path.exists():
    smoke_history_path.unlink()

smoke_checkpoint_dir.mkdir(parents=True, exist_ok=True)
smoke_history_path.parent.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 2. Esecuzione dello smoke test
# ---------------------------------------------------------

initial_unconditional_smoke_loss = float(
    trainer_unconditional.evaluate()
)

assert math.isfinite(initial_unconditional_smoke_loss)

print("--- AVVIO SMOKE TEST UNCONDITIONAL ---")
print(f"Device: {trainer_unconditional.device}")
print(f"Step target: {SMOKE_STEPS}")
print(
    "Validation loss iniziale: "
    f"{initial_unconditional_smoke_loss:.6f}"
)

trainer_unconditional.fit(
    max_steps=SMOKE_STEPS,
    evaluation_every=EVALUATION_EVERY,
    checkpoint_every=CHECKPOINT_EVERY,
    checkpoint_directory=smoke_checkpoint_dir,
    history_path=smoke_history_path,
    logging_every=LOGGING_EVERY,
)

final_unconditional_smoke_loss = float(
    trainer_unconditional.evaluate()
)

latest_checkpoint = smoke_checkpoint_dir / "latest.pt"
best_checkpoint = smoke_checkpoint_dir / "best.pt"

recorded_evaluation_losses = [
    float(metrics["evaluation_loss"])
    for metrics in trainer_unconditional.history
    if metrics.get("evaluation_loss") is not None
]

assert trainer_unconditional.step == SMOKE_STEPS
assert len(trainer_unconditional.history) == SMOKE_STEPS
assert math.isfinite(final_unconditional_smoke_loss)
assert math.isfinite(trainer_unconditional.best_evaluation_loss)

assert latest_checkpoint.exists()
assert best_checkpoint.exists()
assert smoke_history_path.exists()

assert recorded_evaluation_losses, (
    "Nessuna evaluation loss registrata nello smoke test."
)

assert all(
    math.isfinite(loss)
    for loss in recorded_evaluation_losses
), "Trovata almeno una evaluation loss non finita."

best_recorded_evaluation_loss = min(
    recorded_evaluation_losses
)

assert math.isclose(
    trainer_unconditional.best_evaluation_loss,
    best_recorded_evaluation_loss,
    rel_tol=1e-7,
    abs_tol=1e-8,
), (
    "best_evaluation_loss non coincide con il minimo "
    "delle evaluation loss registrate."
)

assert (
    trainer_unconditional.best_evaluation_loss
    <= final_unconditional_smoke_loss + 1e-7
), (
    "La best evaluation loss non può essere maggiore "
    "della validation loss finale ricalcolata."
)

print("--- ESITO SMOKE TEST UNCONDITIONAL ---")
print(f"Step finale: {trainer_unconditional.step}")
print(
    "Validation loss iniziale: "
    f"{initial_unconditional_smoke_loss:.6f}"
)
print(
    "Validation loss finale: "
    f"{final_unconditional_smoke_loss:.6f}"
)
print(
    "Best validation loss: "
    f"{trainer_unconditional.best_evaluation_loss:.6f}"
)
print(
    "Numero di evaluation registrate: "
    f"{len(recorded_evaluation_losses)}"
)
print(
    "Minima evaluation loss registrata: "
    f"{best_recorded_evaluation_loss:.6f}"
)
print(f"Checkpoint latest presente: {latest_checkpoint.exists()}")
print(f"Checkpoint best presente: {best_checkpoint.exists()}")
print(f"History presente: {smoke_history_path.exists()}")
print("Smoke test unconditional superato.")

### Verifica degli artefatti dello smoke test unconditional

Verifichiamo che lo smoke test unconditional abbia prodotto una history coerente e checkpoint ripristinabili. I controlli verificano inoltre che il trainer ricostruito mantenga effettivamente la modalità unconditional, senza text encoder.

In [ ]:
import math
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.diffusion.schedule import (
    DDPMSchedule,
    load_diffusion_config,
)
from src.models.conditional_unet import (
    ConditionalUNet,
    load_unet_config,
)
from src.training.trainer import (
    Trainer,
    load_training_config,
    set_training_seed,
)

# ---------------------------------------------------------
# 0. Percorsi e history dello smoke test
# ---------------------------------------------------------

SMOKE_STEPS = 20

smoke_checkpoint_dir = Path(
    "checkpoints/smoke_unconditional"
)

smoke_history_path = Path(
    "outputs/smoke_unconditional/history.csv"
)

latest_checkpoint = smoke_checkpoint_dir / "latest.pt"
best_checkpoint = smoke_checkpoint_dir / "best.pt"

for path in [
    latest_checkpoint,
    best_checkpoint,
    smoke_history_path,
]:
    assert path.is_file(), f"Artefatto mancante: {path}"
    print(f"{path} ({path.stat().st_size / 1024**2:.2f} MiB)")

history = pd.read_csv(smoke_history_path)

required_columns = {
    "step",
    "epoch",
    "train_loss",
    "evaluation_loss",
    "gradient_norm",
    "amp_scale",
}

missing_columns = required_columns - set(history.columns)

assert not missing_columns, (
    "Colonne mancanti nella history: "
    f"{sorted(missing_columns)}"
)

assert len(history) == SMOKE_STEPS, (
    "Numero righe history inatteso: "
    f"{len(history)} invece di {SMOKE_STEPS}."
)

assert int(history["step"].iloc[0]) == 1
assert int(history["step"].iloc[-1]) == SMOKE_STEPS
assert history["step"].is_monotonic_increasing
assert history["step"].is_unique
assert history["train_loss"].notna().all()
assert history["train_loss"].map(math.isfinite).all()

evaluated_history = history.dropna(
    subset=["evaluation_loss"]
).copy()

assert not evaluated_history.empty, (
    "Nessuna evaluation loss valida trovata nella history."
)

assert evaluated_history["evaluation_loss"].map(
    math.isfinite
).all()

best_history_row = evaluated_history.loc[
    evaluated_history["evaluation_loss"].idxmin()
]

best_step_from_history = int(best_history_row["step"])
best_loss_from_history = float(
    best_history_row["evaluation_loss"]
)

print("--- HISTORY SMOKE UNCONDITIONAL ---")
print(f"Righe history: {len(history)}")
print(f"Step finale: {int(history['step'].max())}")
print(f"Evaluation registrate: {len(evaluated_history)}")
print(
    "Miglior evaluation loss: "
    f"{best_loss_from_history:.6f} "
    f"(step {best_step_from_history})"
)

# ---------------------------------------------------------
# 1. Ricostruzione del trainer unconditional
# ---------------------------------------------------------

assert "train" in datasets, "Split train non disponibile."
assert "validation" in datasets, "Split validation non disponibile."

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

unconditional_config = load_training_config(
    Path("configs/train_unconditional.yaml")
)

training_config = unconditional_config["training"]

unet_config = load_unet_config(Path("configs/unet.yaml"))

diffusion_config = load_diffusion_config(
    Path("configs/diffusion.yaml")
)

set_training_seed(int(training_config["seed"]))

batch_size = int(training_config["batch_size"])

loader_kwargs = {
    "num_workers": 0,
    "pin_memory": device.type == "cuda",
}

train_loader_unconditional = DataLoader(
    datasets["train"],
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    **loader_kwargs,
)

validation_loader_unconditional = DataLoader(
    datasets["validation"],
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    **loader_kwargs,
)


def build_unconditional_trainer() -> Trainer:
    unconditional_unet = ConditionalUNet.from_config(
        unet_config,
        conditional=False,
    ).to(device)

    unconditional_schedule = DDPMSchedule.from_config(
        diffusion_config,
    ).to(device)

    unconditional_optimizer = torch.optim.AdamW(
        unconditional_unet.parameters(),
        lr=float(training_config["learning_rate"]["unet"]),
        weight_decay=float(training_config["weight_decay"]),
    )

    trainer = Trainer(
        conditional_unet=unconditional_unet,
        text_encoder=None,
        schedule=unconditional_schedule,
        optimizer=unconditional_optimizer,
        train_dataloader=train_loader_unconditional,
        validation_dataloader=validation_loader_unconditional,
        device=device,
        config=unconditional_config,
    )

    assert trainer.is_conditional is False
    assert trainer.text_encoder is None
    assert trainer.conditional_unet.conditional is False

    return trainer


# ---------------------------------------------------------
# 2. Verifica di latest.pt
# ---------------------------------------------------------

latest_trainer = build_unconditional_trainer()
latest_trainer.load(latest_checkpoint)

latest_evaluation_loss = float(latest_trainer.evaluate())

assert latest_trainer.step == SMOKE_STEPS
assert len(latest_trainer.history) == SMOKE_STEPS
assert latest_trainer.is_conditional is False
assert latest_trainer.text_encoder is None
assert math.isfinite(latest_trainer.best_evaluation_loss)
assert math.isfinite(latest_evaluation_loss)

assert math.isclose(
    latest_trainer.best_evaluation_loss,
    best_loss_from_history,
    rel_tol=1e-7,
    abs_tol=1e-8,
), (
    "Il best_evaluation_loss di latest.pt non coincide "
    "con il minimo della history."
)

print("--- VERIFICA LATEST.PT ---")
print(f"Step ripristinato: {latest_trainer.step}")
print(
    "Best evaluation loss salvata: "
    f"{latest_trainer.best_evaluation_loss:.6f}"
)
print(
    "Evaluation loss ricalcolata: "
    f"{latest_evaluation_loss:.6f}"
)

# ---------------------------------------------------------
# 3. Verifica di best.pt
# ---------------------------------------------------------

best_trainer = build_unconditional_trainer()
best_trainer.load(best_checkpoint)

best_evaluation_loss = float(best_trainer.evaluate())

assert best_trainer.step == best_step_from_history, (
    "Lo step in best.pt non coincide con il miglior step "
    "ricavato dalla history."
)

assert len(best_trainer.history) == best_step_from_history
assert best_trainer.is_conditional is False
assert best_trainer.text_encoder is None
assert math.isfinite(best_trainer.best_evaluation_loss)
assert math.isfinite(best_evaluation_loss)

assert math.isclose(
    best_trainer.best_evaluation_loss,
    best_loss_from_history,
    rel_tol=1e-7,
    abs_tol=1e-8,
), (
    "Il best_evaluation_loss di best.pt non coincide "
    "con il minimo della history."
)

assert (
    best_trainer.best_evaluation_loss
    <= best_evaluation_loss + 1e-7
), (
    "La evaluation loss ricalcolata da best.pt è "
    "incompatibile con la migliore loss salvata."
)

print("--- VERIFICA BEST.PT ---")
print(f"Step ripristinato: {best_trainer.step}")
print(
    "Best evaluation loss salvata: "
    f"{best_trainer.best_evaluation_loss:.6f}"
)
print(
    "Evaluation loss ricalcolata: "
    f"{best_evaluation_loss:.6f}"
)

print("--- VERIFICA ARTEFATTI SMOKE COMPLETATA ---")
print("Artefatti unconditional coerenti.")
print(f"latest.pt: {latest_checkpoint}")
print(f"best.pt: {best_checkpoint}")
print(f"history CSV: {smoke_history_path}")

### Training completo unconditional

Dopo il superamento dello smoke test avviamo il training completo della baseline unconditional. Anche in questo caso il processo supporta la ripresa automatica da `latest.pt`, così da rendere l'addestramento robusto a eventuali interruzioni della sessione.

In [ ]:
# ---------------------------------------------------------
# Training ufficiale: baseline DDPM unconditional
# ---------------------------------------------------------

import math
import shutil
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from src.diffusion.schedule import (
    DDPMSchedule,
    load_diffusion_config,
)
from src.models.conditional_unet import (
    ConditionalUNet,
    load_unet_config,
)
from src.training.trainer import (
    Trainer,
    load_training_config,
    set_training_seed,
)


# ---------------------------------------------------------
# 0. Configurazioni e controlli preliminari
# ---------------------------------------------------------

assert "train" in datasets, "Split train non disponibile."
assert "validation" in datasets, "Split validation non disponibile."

assert len(datasets["train"]) > 0
assert len(datasets["validation"]) > 0

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert device.type == "cuda", (
    "Il training ufficiale richiede CUDA."
)

unconditional_config = load_training_config(
    Path("configs/train_unconditional.yaml")
)

unconditional_settings = unconditional_config[
    "training"
]

unet_config = load_unet_config(
    Path("configs/unet.yaml")
)

diffusion_config = load_diffusion_config(
    Path("configs/diffusion.yaml")
)

set_training_seed(
    int(unconditional_settings["seed"])
)

max_steps = int(
    unconditional_settings["max_steps"]
)

batch_size = int(
    unconditional_settings["batch_size"]
)

evaluation_every = int(
    unconditional_settings["evaluation"]["every_steps"]
)

checkpoint_every = int(
    unconditional_settings["checkpoint"]["every_steps"]
)

logging_every = int(
    unconditional_settings["logging"]["every_steps"]
)

assert max_steps > 0
assert batch_size > 0
assert evaluation_every > 0
assert checkpoint_every > 0
assert logging_every > 0

assert "unet" in unconditional_settings[
    "learning_rate"
]

assert "text_encoder" not in unconditional_settings[
    "learning_rate"
]

assert len(datasets["train"]) >= batch_size


# ---------------------------------------------------------
# 1. DataLoader
# ---------------------------------------------------------

loader_kwargs = {
    "num_workers": 0,
    "pin_memory": True,
}

unconditional_train_loader = DataLoader(
    datasets["train"],
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    **loader_kwargs,
)

unconditional_validation_loader = DataLoader(
    datasets["validation"],
    batch_size=batch_size,
    shuffle=False,
    drop_last=False,
    **loader_kwargs,
)

assert len(unconditional_train_loader) > 0
assert len(unconditional_validation_loader) > 0


# ---------------------------------------------------------
# 2. Nuovi modello, schedule e optimizer
# ---------------------------------------------------------

unconditional_unet = ConditionalUNet.from_config(
    unet_config,
    conditional=False,
).to(device)

unconditional_schedule = DDPMSchedule.from_config(
    diffusion_config,
).to(device)

unconditional_optimizer = torch.optim.AdamW(
    unconditional_unet.parameters(),
    lr=float(
        unconditional_settings["learning_rate"]["unet"]
    ),
    weight_decay=float(
        unconditional_settings["weight_decay"]
    ),
)

unconditional_parameters = sum(
    parameter.numel()
    for parameter in unconditional_unet.parameters()
)

assert unconditional_parameters > 5_000_000, (
    "La U-Net unconditional è troppo piccola."
)

print(
    "Parametri U-Net unconditional: "
    f"{unconditional_parameters:,}"
)


# ---------------------------------------------------------
# 3. Sovrascrittura artefatti unconditional precedenti
# ---------------------------------------------------------

unconditional_checkpoint_dir = Path(
    unconditional_settings["checkpoint"]["directory"]
)

unconditional_history_path = Path(
    unconditional_settings["logging"]["history_path"]
)

if unconditional_checkpoint_dir.exists():
    shutil.rmtree(unconditional_checkpoint_dir)
    print(
        "Checkpoint unconditional precedenti eliminati: "
        f"{unconditional_checkpoint_dir}"
    )

if unconditional_history_path.exists():
    unconditional_history_path.unlink()
    print(
        "History unconditional precedente eliminata: "
        f"{unconditional_history_path}"
    )

unconditional_checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True,
)

unconditional_history_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# ---------------------------------------------------------
# 4. Training da zero
# ---------------------------------------------------------

unconditional_trainer = Trainer(
    conditional_unet=unconditional_unet,
    text_encoder=None,
    schedule=unconditional_schedule,
    optimizer=unconditional_optimizer,
    train_dataloader=unconditional_train_loader,
    validation_dataloader=unconditional_validation_loader,
    device=device,
    config=unconditional_config,
)

initial_unconditional_loss = float(
    unconditional_trainer.evaluate()
)

assert math.isfinite(initial_unconditional_loss)

print("--- AVVIO TRAINING UNCONDITIONAL ---")
print(f"Device: {device}")
print(f"Batch size: {batch_size}")
print(
    "Batch train per epoch: "
    f"{len(unconditional_train_loader)}"
)
print(
    "Batch validation: "
    f"{len(unconditional_validation_loader)}"
)
print(f"Step target: {max_steps}")
print(
    "Validation loss iniziale: "
    f"{initial_unconditional_loss:.6f}"
)

unconditional_trainer.fit(
    max_steps=max_steps,
    evaluation_every=evaluation_every,
    checkpoint_every=checkpoint_every,
    checkpoint_directory=unconditional_checkpoint_dir,
    history_path=unconditional_history_path,
    logging_every=logging_every,
)

final_unconditional_loss = float(
    unconditional_trainer.evaluate()
)

unconditional_latest_path = (
    unconditional_checkpoint_dir / "latest.pt"
)

unconditional_best_path = (
    unconditional_checkpoint_dir / "best.pt"
)

assert unconditional_trainer.step == max_steps
assert math.isfinite(final_unconditional_loss)
assert math.isfinite(
    unconditional_trainer.best_evaluation_loss
)

assert unconditional_latest_path.exists()
assert unconditional_best_path.exists()
assert unconditional_history_path.exists()

print("--- TRAINING UNCONDITIONAL COMPLETATO ---")
print(f"Step finale: {unconditional_trainer.step}")
print(
    "Validation loss finale: "
    f"{final_unconditional_loss:.6f}"
)
print(
    "Migliore validation loss: "
    f"{unconditional_trainer.best_evaluation_loss:.6f}"
)
print(f"Checkpoint migliore: {unconditional_best_path}")

In [ ]:
# ---------------------------------------------------------
# Verifica riproducibile del miglior checkpoint unconditional
# ---------------------------------------------------------

import math
from pathlib import Path

import pandas as pd
import torch

from src.diffusion.schedule import DDPMSchedule
from src.models.conditional_unet import ConditionalUNet
from src.training.trainer import Trainer, set_training_seed


assert unconditional_latest_path.exists()
assert unconditional_best_path.exists()
assert unconditional_history_path.exists()

unconditional_history = pd.read_csv(
    unconditional_history_path
)

assert len(unconditional_history) == max_steps
assert unconditional_history["step"].is_unique
assert unconditional_history["step"].is_monotonic_increasing

evaluations = unconditional_history.dropna(
    subset=["evaluation_loss"]
).copy()

assert not evaluations.empty
assert evaluations["evaluation_loss"].map(
    math.isfinite
).all()

best_history_row = evaluations.loc[
    evaluations["evaluation_loss"].idxmin()
]

best_step_from_history = int(
    best_history_row["step"]
)

best_loss_from_history = float(
    best_history_row["evaluation_loss"]
)

# Ricostruzione pulita: nessun riuso del modello rimasto
# in memoria dal training appena concluso.
set_training_seed(
    int(unconditional_settings["seed"])
)

reloaded_unconditional_unet = ConditionalUNet.from_config(
    unet_config,
    conditional=False,
).to(device)

reloaded_unconditional_schedule = (
    DDPMSchedule.from_config(
        diffusion_config
    ).to(device)
)

reloaded_unconditional_optimizer = torch.optim.AdamW(
    reloaded_unconditional_unet.parameters(),
    lr=float(
        unconditional_settings["learning_rate"]["unet"]
    ),
    weight_decay=float(
        unconditional_settings["weight_decay"]
    ),
)

reloaded_unconditional_trainer = Trainer(
    conditional_unet=reloaded_unconditional_unet,
    text_encoder=None,
    schedule=reloaded_unconditional_schedule,
    optimizer=reloaded_unconditional_optimizer,
    train_dataloader=unconditional_train_loader,
    validation_dataloader=unconditional_validation_loader,
    device=device,
    config=unconditional_config,
)

reloaded_unconditional_trainer.load(
    unconditional_best_path
)

reloaded_best_loss = float(
    reloaded_unconditional_trainer.evaluate()
)

assert reloaded_unconditional_trainer.step == (
    best_step_from_history
)

torch.testing.assert_close(
    torch.tensor(reloaded_best_loss),
    torch.tensor(best_loss_from_history),
    rtol=0.0,
    atol=1e-6,
)

print("--- CHECKPOINT UNCONDITIONAL VERIFICATO ---")
print(f"Step best checkpoint: {best_step_from_history}")
print(
    "Best loss dalla history: "
    f"{best_loss_from_history:.6f}"
)
print(
    "Best loss ricalcolata: "
    f"{reloaded_best_loss:.6f}"
)
print(f"Checkpoint: {unconditional_best_path}")

### Verifica degli artefatti unconditional

Concludiamo il blocco verificando gli artefatti del training completo unconditional. I controlli accertano che history, checkpoint più recente e checkpoint migliore siano completi, coerenti tra loro e riferiti a una baseline priva di text encoder.

In [ ]:
import math
from pathlib import Path

import pandas as pd
import torch

# ---------------------------------------------------------
# 0. Percorsi degli artefatti unconditional
# ---------------------------------------------------------

checkpoint_dir = Path("checkpoints/train_unconditional")
history_path = Path(
    "outputs/train_unconditional/history.csv"
)

latest_path = checkpoint_dir / "latest.pt"
best_path = checkpoint_dir / "best.pt"

EXPECTED_MAX_STEPS = 10_000

for path in [latest_path, best_path, history_path]:
    assert path.exists(), f"Artefatto mancante: {path}"
    assert path.stat().st_size > 0, f"Artefatto vuoto: {path}"

print("Artefatti unconditional trovati:")
for path in [latest_path, best_path, history_path]:
    print(f"- {path} ({path.stat().st_size / 1024**2:.2f} MiB)")

# ---------------------------------------------------------
# 1. Verifica della history
# ---------------------------------------------------------

history = pd.read_csv(history_path)

required_columns = {
    "step",
    "epoch",
    "train_loss",
    "evaluation_loss",
    "gradient_norm",
    "amp_scale",
}

missing_columns = required_columns - set(history.columns)

assert not missing_columns, (
    "Colonne mancanti nella history: "
    f"{sorted(missing_columns)}"
)

assert len(history) == EXPECTED_MAX_STEPS, (
    f"History con {len(history)} righe, "
    f"attese {EXPECTED_MAX_STEPS}."
)

assert int(history["step"].iloc[0]) == 1
assert int(history["step"].iloc[-1]) == EXPECTED_MAX_STEPS
assert history["step"].is_monotonic_increasing
assert history["step"].is_unique

assert history["train_loss"].notna().all()
assert history["train_loss"].map(math.isfinite).all()

evaluated = history.dropna(subset=["evaluation_loss"]).copy()

assert not evaluated.empty, (
    "Nessuna evaluation registrata nella history."
)

assert evaluated["evaluation_loss"].map(math.isfinite).all()

best_row = evaluated.loc[
    evaluated["evaluation_loss"].idxmin()
]

best_step_from_history = int(best_row["step"])
best_loss_from_history = float(
    best_row["evaluation_loss"]
)

# ---------------------------------------------------------
# 2. Verifica dei checkpoint serializzati
# ---------------------------------------------------------

latest_checkpoint = torch.load(
    latest_path,
    map_location="cpu",
    weights_only=False,
)

best_checkpoint = torch.load(
    best_path,
    map_location="cpu",
    weights_only=False,
)

required_checkpoint_keys = {
    "format_version",
    "step",
    "epoch",
    "models",
    "optimizer",
    "scaler",
    "rng_state",
    "generator_states",
    "metadata",
    "configs",
}

for checkpoint_name, checkpoint in {
    "latest.pt": latest_checkpoint,
    "best.pt": best_checkpoint,
}.items():
    assert isinstance(checkpoint, dict), (
        f"{checkpoint_name} non contiene un dizionario."
    )

    missing_keys = required_checkpoint_keys - set(checkpoint)

    assert not missing_keys, (
        f"Chiavi mancanti in {checkpoint_name}: "
        f"{sorted(missing_keys)}"
    )

    assert "conditional_unet" in checkpoint["models"]

    assert "text_encoder" not in checkpoint["models"], (
        "Un checkpoint unconditional non deve contenere "
        "un text encoder."
    )

    assert checkpoint["metadata"]["conditional"] is False

assert int(latest_checkpoint["step"]) == EXPECTED_MAX_STEPS

assert int(best_checkpoint["step"]) == best_step_from_history, (
    "Lo step in best.pt non coincide con il miglior step "
    "ricavato dalla history."
)

latest_best_loss = float(
    latest_checkpoint["metadata"]["best_evaluation_loss"]
)

best_best_loss = float(
    best_checkpoint["metadata"]["best_evaluation_loss"]
)

assert math.isfinite(latest_best_loss)
assert math.isfinite(best_best_loss)

assert math.isclose(
    latest_best_loss,
    best_loss_from_history,
    rel_tol=1e-7,
    abs_tol=1e-8,
), (
    "La best evaluation loss nei metadata di latest.pt "
    "non coincide con il minimo della history."
)

assert math.isclose(
    best_best_loss,
    best_loss_from_history,
    rel_tol=1e-7,
    abs_tol=1e-8,
), (
    "La best evaluation loss nei metadata di best.pt "
    "non coincide con il minimo della history."
)

print("--- VERIFICA TRAINING UNCONDITIONAL SUPERATA ---")
print(f"Righe history: {len(history)}")
print(f"Numero evaluation: {len(evaluated)}")
print(f"Step latest.pt: {int(latest_checkpoint['step'])}")
print(f"Step best.pt: {int(best_checkpoint['step'])}")
print(f"Best step dalla history: {best_step_from_history}")
print(f"Best validation loss: {best_loss_from_history:.6f}")
print(f"Latest checkpoint: {latest_path}")
print(f"Best checkpoint: {best_path}")
print(f"History CSV: {history_path}")

Il modello conditional e la baseline unconditional sono stati addestrati con un protocollo coerente, riproducibile e verificato. Entrambi dispongono di checkpoint latest e best, history CSV complete e verifiche di integrità superate. Il progetto è pronto per il Blocco 10, dedicato alla evaluation comparativa su qualità dell’immagine, fedeltà al conditioning, diversità tra seed, numero di parametri, tempo di sampling e memoria CUDA.

## Curve di training/validation e scelta del checkpoint

Prima del confronto finale visualizziamo le curve di training e validation dei due
run ufficiali, conditional e unconditional.

Obiettivi della sezione:
- verificare l'andamento della train loss;
- verificare la validation loss sugli step di evaluation;
- evidenziare lo step del checkpoint migliore `best.pt`;
- salvare una figura PNG riutilizzabile nel report finale.

In [ ]:
# ---------------------------------------------------------
# Curve di training/validation e checkpoint migliore
# ---------------------------------------------------------

import math
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch


# ---------------------------------------------------------
# 0. Percorsi ufficiali dei due run
# ---------------------------------------------------------

conditional_history_path = Path("outputs/train/history.csv")
conditional_best_path = Path("checkpoints/train/best.pt")

unconditional_history_path = Path("outputs/train_unconditional/history.csv")
unconditional_best_path = Path("checkpoints/train_unconditional/best.pt")

for path in [
    conditional_history_path,
    conditional_best_path,
    unconditional_history_path,
    unconditional_best_path,
]:
    assert path.exists(), f"Artefatto mancante: {path}"
    assert path.stat().st_size > 0, f"Artefatto vuoto: {path}"


# ---------------------------------------------------------
# 1. Lettura delle history
# ---------------------------------------------------------

conditional_history = pd.read_csv(conditional_history_path)
unconditional_history = pd.read_csv(unconditional_history_path)

required_columns = {
    "step",
    "train_loss",
    "evaluation_loss",
}

for history_name, history in {
    "conditional": conditional_history,
    "unconditional": unconditional_history,
}.items():
    missing_columns = required_columns - set(history.columns)

    assert not missing_columns, (
        f"Colonne mancanti nella history {history_name}: "
        f"{sorted(missing_columns)}"
    )

    assert len(history) > 0, f"History vuota: {history_name}"
    assert history["step"].is_monotonic_increasing
    assert history["step"].is_unique
    assert history["train_loss"].notna().all()
    assert history["train_loss"].map(math.isfinite).all()


# ---------------------------------------------------------
# 2. Sottoinsiemi di validation
# ---------------------------------------------------------
# La evaluation loss è presente solo negli step in cui il
# trainer ha eseguito la validazione periodica.
# ---------------------------------------------------------

conditional_eval = conditional_history.dropna(
    subset=["evaluation_loss"]
).copy()

unconditional_eval = unconditional_history.dropna(
    subset=["evaluation_loss"]
).copy()

assert not conditional_eval.empty, (
    "Nessuna evaluation loss disponibile per il run conditional."
)
assert not unconditional_eval.empty, (
    "Nessuna evaluation loss disponibile per il run unconditional."
)

assert conditional_eval["evaluation_loss"].map(math.isfinite).all()
assert unconditional_eval["evaluation_loss"].map(math.isfinite).all()


# ---------------------------------------------------------
# 3. Lettura dei best checkpoint
# ---------------------------------------------------------

conditional_best_checkpoint = torch.load(
    conditional_best_path,
    map_location="cpu",
    weights_only=False,
)

unconditional_best_checkpoint = torch.load(
    unconditional_best_path,
    map_location="cpu",
    weights_only=False,
)

for checkpoint_name, checkpoint in {
    "conditional": conditional_best_checkpoint,
    "unconditional": unconditional_best_checkpoint,
}.items():
    assert isinstance(checkpoint, dict), (
        f"Checkpoint non valido: {checkpoint_name}"
    )
    assert "step" in checkpoint, f"Chiave 'step' mancante: {checkpoint_name}"
    assert "metadata" in checkpoint, (
        f"Chiave 'metadata' mancante: {checkpoint_name}"
    )
    assert "best_evaluation_loss" in checkpoint["metadata"], (
        f"Metadata incompleti: {checkpoint_name}"
    )


conditional_best_step = int(conditional_best_checkpoint["step"])
conditional_best_loss = float(
    conditional_best_checkpoint["metadata"]["best_evaluation_loss"]
)

unconditional_best_step = int(unconditional_best_checkpoint["step"])
unconditional_best_loss = float(
    unconditional_best_checkpoint["metadata"]["best_evaluation_loss"]
)

assert math.isfinite(conditional_best_loss)
assert math.isfinite(unconditional_best_loss)


# ---------------------------------------------------------
# 4. Verifica coerenza best checkpoint vs history
# ---------------------------------------------------------

conditional_best_row = conditional_eval.loc[
    conditional_eval["evaluation_loss"].idxmin()
]
unconditional_best_row = unconditional_eval.loc[
    unconditional_eval["evaluation_loss"].idxmin()
]

assert int(conditional_best_row["step"]) == conditional_best_step, (
    "Lo step del best checkpoint conditional non coincide "
    "con il minimo della history."
)

assert int(unconditional_best_row["step"]) == unconditional_best_step, (
    "Lo step del best checkpoint unconditional non coincide "
    "con il minimo della history."
)

assert math.isclose(
    float(conditional_best_row["evaluation_loss"]),
    conditional_best_loss,
    rel_tol=1e-7,
    abs_tol=1e-8,
)

assert math.isclose(
    float(unconditional_best_row["evaluation_loss"]),
    unconditional_best_loss,
    rel_tol=1e-7,
    abs_tol=1e-8,
)


# ---------------------------------------------------------
# 5. Plot finale
# ---------------------------------------------------------

figure, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(16, 6),
    dpi=150,
)

# Conditional
axes[0].plot(
    conditional_history["step"],
    conditional_history["train_loss"],
    label="Train loss",
    linewidth=1.5,
)

axes[0].plot(
    conditional_eval["step"],
    conditional_eval["evaluation_loss"],
    label="Validation loss",
    linewidth=2.0,
    marker="o",
    markersize=3,
)

axes[0].scatter(
    [conditional_best_step],
    [conditional_best_loss],
    color="red",
    s=70,
    zorder=5,
    label=f"Best checkpoint (step {conditional_best_step})",
)

axes[0].set_title("Conditional training")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.25)
axes[0].legend()

# Unconditional
axes[1].plot(
    unconditional_history["step"],
    unconditional_history["train_loss"],
    label="Train loss",
    linewidth=1.5,
)

axes[1].plot(
    unconditional_eval["step"],
    unconditional_eval["evaluation_loss"],
    label="Validation loss",
    linewidth=2.0,
    marker="o",
    markersize=3,
)

axes[1].scatter(
    [unconditional_best_step],
    [unconditional_best_loss],
    color="red",
    s=70,
    zorder=5,
    label=f"Best checkpoint (step {unconditional_best_step})",
)

axes[1].set_title("Unconditional training")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Loss")
axes[1].grid(True, alpha=0.25)
axes[1].legend()

figure.suptitle(
    "Training and validation curves with selected best checkpoints",
    fontsize=15,
    y=1.02,
)

plt.tight_layout()
plt.show()


# ---------------------------------------------------------
# 6. Salvataggio figura finale
# ---------------------------------------------------------

output_dir = Path("outputs/evaluation")
output_dir.mkdir(parents=True, exist_ok=True)

training_curves_path = output_dir / "training_validation_curves.png"

figure.savefig(
    training_curves_path,
    dpi=200,
    bbox_inches="tight",
)

print("--- CURVE TRAINING/VALIDATION SALVATE ---")
print(f"Conditional best step: {conditional_best_step}")
print(f"Conditional best validation loss: {conditional_best_loss:.6f}")
print(f"Unconditional best step: {unconditional_best_step}")
print(f"Unconditional best validation loss: {unconditional_best_loss:.6f}")
print(f"Figura salvata in: {training_curves_path}")

## Inferenza visuale

In questa sezione ricostruiamo i modelli migliori salvati su checkpoint e generiamo
un piccolo pannello qualitativo finale.

Per il modello conditional utilizziamo caption reali provenienti dallo split
`test_ordinary`, così da valutare il comportamento del sistema su combinazioni
osservate durante l'addestramento. La valutazione sullo split composizionale
`test_ood` verrà svolta separatamente nelle celle quantitative successive.

Per coerenza metodologica con il training effettivamente eseguito, il sampling
conditional viene eseguito con `guidance_scale = 1.0`: in questo modo il modello
usa il conditioning testuale senza introdurre classifier-free guidance non
supportata esplicitamente da conditioning dropout nel training.

In [ ]:
# ---------------------------------------------------------
# Inferenza visuale conditional
# ---------------------------------------------------------

import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from src.models.text_encoder import (
    TextTransformerEncoder,
    load_text_encoder_config,
)
from src.models.conditional_unet import (
    ConditionalUNet,
    load_unet_config,
)
from src.diffusion.schedule import (
    DDPMSchedule,
    load_diffusion_config,
)
from src.diffusion.reverse_sampler import (
    ReverseSampler,
    load_sampling_config,
)


# ---------------------------------------------------------
# 0. Check preliminari
# ---------------------------------------------------------

assert "test_ordinary" in datasets, (
    "Split test_ordinary non disponibile."
)
assert len(datasets["test_ordinary"]) > 0, (
    "Split test_ordinary vuoto."
)
assert tokenizer is not None, "Tokenizer non disponibile."

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

conditional_checkpoint_path = Path(
    "checkpoints/train/best.pt"
)

assert conditional_checkpoint_path.exists(), (
    "Checkpoint conditional mancante: "
    f"{conditional_checkpoint_path}"
)


# ---------------------------------------------------------
# 1. Ricostruzione modelli, schedule e sampler
# ---------------------------------------------------------

unet_config = load_unet_config(
    Path("configs/unet.yaml")
)

text_config = load_text_encoder_config(
    Path("configs/text_encoder.yaml")
)

diffusion_config = load_diffusion_config(
    Path("configs/diffusion.yaml")
)

sampling_config = load_sampling_config(
    Path("configs/sampling.yaml")
)

conditional_unet_eval = ConditionalUNet.from_config(
    unet_config,
    conditional=True,
).to(device)

text_encoder_eval = TextTransformerEncoder.from_config(
    text_config
).to(device)

schedule_eval = DDPMSchedule.from_config(
    diffusion_config
).to(device)

checkpoint = torch.load(
    conditional_checkpoint_path,
    map_location=device,
    weights_only=False,
)

assert "models" in checkpoint
assert "conditional_unet" in checkpoint["models"]
assert "text_encoder" in checkpoint["models"]

conditional_unet_eval.load_state_dict(
    checkpoint["models"]["conditional_unet"]
)
text_encoder_eval.load_state_dict(
    checkpoint["models"]["text_encoder"]
)

conditional_unet_eval.eval()
text_encoder_eval.eval()

assert conditional_unet_eval.training is False
assert text_encoder_eval.training is False
assert conditional_unet_eval.conditional is True

# Per coerenza con il training effettivo, disattiviamo la CFG
# impostando guidance_scale = 1.0.
conditional_sampler = ReverseSampler(
    conditional_unet=conditional_unet_eval,
    text_encoder=text_encoder_eval,
    schedule=schedule_eval,
    device=device,
    guidance_scale=1.0,
    clip_denoised=bool(
        sampling_config["sampling"]["clip_denoised"]
    ),
)


# ---------------------------------------------------------
# 2. Prompt di test dallo split ordinary
# ---------------------------------------------------------
# Usiamo 4 caption reali dal test_ordinary per ottenere
# una griglia qualitativa semplice, leggibile e coerente
# con le combinazioni osservate durante il training.
# ---------------------------------------------------------

dataset_ordinary = datasets["test_ordinary"]

num_conditional_samples = 4

test_prompts = []
for sample in dataset_ordinary:
    caption = sample["caption"]
    assert isinstance(caption, str)

    cleaned_caption = caption.strip()

    if cleaned_caption:
        test_prompts.append(cleaned_caption)

    if len(test_prompts) == num_conditional_samples:
        break

assert len(test_prompts) == num_conditional_samples, (
    "Non sono stati trovati abbastanza prompt validi "
    "nel test_ordinary."
)


# ---------------------------------------------------------
# 3. Tokenizzazione dei prompt
# ---------------------------------------------------------
# Riutilizziamo direttamente il tokenizer già costruito
# nel notebook, evitando dipendenze da moduli collate
# non presenti nel progetto.
# ---------------------------------------------------------

encoded_prompts = tokenizer.encode_batch(test_prompts)

input_ids = torch.tensor(
    [encoding.ids for encoding in encoded_prompts],
    dtype=torch.long,
    device=device,
)

attention_mask = torch.tensor(
    [encoding.attention_mask for encoding in encoded_prompts],
    dtype=torch.long,
    device=device,
)

assert input_ids.shape[0] == num_conditional_samples
assert attention_mask.shape == input_ids.shape

max_seq_length = int(
    text_config["text_encoder"]["max_sequence_length"]
)

assert input_ids.shape[1] == max_seq_length, (
    "Lunghezza token inattesa: "
    f"{input_ids.shape[1]} != {max_seq_length}"
)


# ---------------------------------------------------------
# 4. Shape immagine di riferimento
# ---------------------------------------------------------

reference_image = dataset_ordinary[0]["image"]

assert isinstance(reference_image, torch.Tensor)
assert reference_image.ndim == 3

image_shape = tuple(reference_image.shape)


# ---------------------------------------------------------
# 5. Reverse sampling conditional
# ---------------------------------------------------------

conditional_seed = 2026
generator = torch.Generator(device=device)
generator.manual_seed(conditional_seed)

start_time = time.perf_counter()

with torch.inference_mode():
    conditional_generated_images = conditional_sampler.sample(
        input_ids=input_ids,
        attention_mask=attention_mask,
        image_shape=image_shape,
        generator=generator,
        guidance_scale=1.0,
    )

conditional_sampling_time_s = (
    time.perf_counter() - start_time
)

assert conditional_generated_images.shape == (
    num_conditional_samples,
    *image_shape,
)

assert torch.isfinite(
    conditional_generated_images
).all()

assert float(conditional_generated_images.min()) >= -1.0
assert float(conditional_generated_images.max()) <= 1.0


# ---------------------------------------------------------
# 6. Visualizzazione
# ---------------------------------------------------------

conditional_images_to_show = (
    (conditional_generated_images.detach().cpu() + 1.0) / 2.0
).clamp(0.0, 1.0)

figure, axes = plt.subplots(
    nrows=2,
    ncols=2,
    figsize=(10, 10),
    dpi=150,
)

axes = axes.flatten()

for axis, image, prompt in zip(
    axes,
    conditional_images_to_show,
    test_prompts,
):
    axis.imshow(image.permute(1, 2, 0).numpy())
    axis.set_title(prompt[:80], fontsize=10)
    axis.axis("off")

figure.suptitle(
    "Conditional samples from best.pt (test_ordinary prompts)",
    fontsize=16,
    y=0.98,
)

plt.tight_layout()
plt.show()


# ---------------------------------------------------------
# 7. Variabili finali da riusare nelle celle successive
# ---------------------------------------------------------

conditional_grid_images = (
    conditional_generated_images.detach().cpu()
)
conditional_grid_prompts = list(test_prompts)
conditional_grid_sampling_time_s = float(
    conditional_sampling_time_s
)
conditional_grid_split_name = "test_ordinary"
conditional_grid_seed = int(conditional_seed)

print("--- INFERENZA VISUALE CONDITIONAL COMPLETATA ---")
print(f"Split prompt usato: {conditional_grid_split_name}")
print(f"Numero campioni: {num_conditional_samples}")
print(
    f"Sampling time totale: "
    f"{conditional_sampling_time_s:.2f} s"
)
print(f"Seed usato: {conditional_seed}")
print(f"Checkpoint usato: {conditional_checkpoint_path}")

## Inferenza visuale unconditional

Generiamo ora un set di immagini con la baseline unconditional caricata da
`best.pt`, usando seed diversi ma senza conditioning testuale.

Questa sezione serve a:
- verificare che il reverse process funzioni anche senza text encoder;
- ottenere un confronto qualitativo diretto con il modello conditional;
- preparare la grid finale da esportare nel report.

In [ ]:
# ---------------------------------------------------------
# Inferenza visuale unconditional
# ---------------------------------------------------------

import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from src.models.conditional_unet import (
    ConditionalUNet,
    load_unet_config,
)
from src.diffusion.schedule import (
    DDPMSchedule,
    load_diffusion_config,
)


# ---------------------------------------------------------
# 0. Check preliminari
# ---------------------------------------------------------

assert "test_ordinary" in datasets, (
    "Split test_ordinary non disponibile."
)
assert len(datasets["test_ordinary"]) > 0, (
    "Split test_ordinary vuoto."
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

unconditional_checkpoint_path = Path(
    "checkpoints/train_unconditional/best.pt"
)

assert unconditional_checkpoint_path.exists(), (
    "Checkpoint unconditional mancante: "
    f"{unconditional_checkpoint_path}"
)


# ---------------------------------------------------------
# 1. Ricostruzione modello e schedule
# ---------------------------------------------------------

unet_config = load_unet_config(
    Path("configs/unet.yaml")
)

diffusion_config = load_diffusion_config(
    Path("configs/diffusion.yaml")
)

unconditional_unet_eval = ConditionalUNet.from_config(
    unet_config,
    conditional=False,
).to(device)

unconditional_schedule_eval = DDPMSchedule.from_config(
    diffusion_config
).to(device)

checkpoint = torch.load(
    unconditional_checkpoint_path,
    map_location=device,
    weights_only=False,
)

assert "models" in checkpoint
assert "conditional_unet" in checkpoint["models"]

unconditional_unet_eval.load_state_dict(
    checkpoint["models"]["conditional_unet"]
)

unconditional_unet_eval.eval()

assert unconditional_unet_eval.training is False
assert unconditional_unet_eval.conditional is False


# ---------------------------------------------------------
# 2. Shape immagine di riferimento
# ---------------------------------------------------------

dataset_ordinary = datasets["test_ordinary"]

reference_image = dataset_ordinary[0]["image"]

assert isinstance(reference_image, torch.Tensor)
assert reference_image.ndim == 3

image_shape = tuple(reference_image.shape)


# ---------------------------------------------------------
# 3. Noise predictor unconditional
# ---------------------------------------------------------
# In assenza di text conditioning, il modello riceve
# context=None e attention_mask=None.
# ---------------------------------------------------------

def unconditional_noise_predictor(
    noisyimages: torch.Tensor,
    timesteps: torch.Tensor,
) -> torch.Tensor:
    return unconditional_unet_eval(
        noisyimages=noisyimages,
        timesteps=timesteps,
        context=None,
        attention_mask=None,
    )


# ---------------------------------------------------------
# 4. Reverse sampling con seed multipli
# ---------------------------------------------------------

unconditional_seeds = [11, 29, 47, 71, 101, 131, 173, 211]

unconditional_generated_images_list = []
unconditional_sampling_times_s = []

for seed in unconditional_seeds:
    generator = torch.Generator(device=device)
    generator.manual_seed(int(seed))

    x_t = torch.randn(
        1,
        *image_shape,
        device=device,
        dtype=torch.float32,
        generator=generator,
    )

    start_time = time.perf_counter()

    with torch.inference_mode():
        for timestep in reversed(
            range(unconditional_schedule_eval.num_timesteps)
        ):
            timesteps = torch.full(
                (1,),
                fill_value=timestep,
                device=device,
                dtype=torch.long,
            )

            predicted_noise = unconditional_noise_predictor(
                noisyimages=x_t,
                timesteps=timesteps,
            )

            x_t, predicted_x_start = (
                unconditional_schedule_eval.reverse_step(
                    noisyimages=x_t,
                    predicted_noise=predicted_noise,
                    timesteps=timesteps,
                    generator=generator,
                    clip_denoised=True,
                )
            )

    elapsed_s = time.perf_counter() - start_time

    assert x_t.shape == (1, *image_shape)
    assert predicted_x_start.shape == (1, *image_shape)
    assert torch.isfinite(x_t).all()
    assert torch.isfinite(predicted_x_start).all()
    assert float(x_t.min()) >= -1.0
    assert float(x_t.max()) <= 1.0

    unconditional_generated_images_list.append(
        x_t[0].detach().cpu()
    )
    unconditional_sampling_times_s.append(
        float(elapsed_s)
    )

unconditional_generated_images = torch.stack(
    unconditional_generated_images_list,
    dim=0,
)

assert unconditional_generated_images.shape[0] == len(
    unconditional_seeds
)


# ---------------------------------------------------------
# 5. Visualizzazione
# ---------------------------------------------------------

images_to_show = (
    (unconditional_generated_images + 1.0) / 2.0
).clamp(0.0, 1.0)

figure, axes = plt.subplots(
    nrows=2,
    ncols=4,
    figsize=(14, 7),
    dpi=150,
)

axes = axes.flatten()

for axis, image, seed, elapsed_s in zip(
    axes,
    images_to_show,
    unconditional_seeds,
    unconditional_sampling_times_s,
):
    axis.imshow(image.permute(1, 2, 0).numpy())
    axis.set_title(
        f"Seed {seed} | {elapsed_s:.2f} s",
        fontsize=10,
    )
    axis.axis("off")

figure.suptitle(
    "Unconditional samples from best.pt",
    fontsize=16,
    y=0.98,
)

plt.tight_layout()
plt.show()


# ---------------------------------------------------------
# 6. Variabili finali da riusare nelle celle successive
# ---------------------------------------------------------

unconditional_grid_images = (
    unconditional_generated_images.detach().cpu()
)
unconditional_grid_seeds = list(unconditional_seeds)
unconditional_grid_sampling_times_s = list(
    unconditional_sampling_times_s
)
unconditional_grid_split_name = "test_ordinary"

print("--- INFERENZA VISUALE UNCONDITIONAL COMPLETATA ---")
print(f"Numero campioni: {len(unconditional_seeds)}")
print(
    "Sampling medio: "
    f"{sum(unconditional_sampling_times_s) / len(unconditional_sampling_times_s):.2f} s"
)
print(f"Checkpoint usato: {unconditional_checkpoint_path}")

In [ ]:
# ---------------------------------------------------------
# Confronto qualitativo controllato:
# reali, modello conditional e baseline unconditional
# con stessa inizializzazione casuale.
# ---------------------------------------------------------

import time

import matplotlib.pyplot as plt
import torch

from src.diffusion.reverse_sampler import ReverseSampler


assert "test_ordinary" in datasets
assert len(datasets["test_ordinary"]) >= 4
assert "tokenizer" in globals()

# Usiamo i modelli ricaricati dai checkpoint best verificati.
conditional_model = reloaded_conditional_unet.eval()
conditional_encoder = reloaded_conditional_text_encoder.eval()
conditional_schedule_for_sampling = (
    reloaded_conditional_schedule
)

unconditional_model = reloaded_unconditional_unet.eval()
unconditional_schedule_for_sampling = (
    reloaded_unconditional_schedule
)

# Quattro esempi ordinary, quindi combinazioni osservate durante il training.
test_indices = [0, 1, 2, 3]

reference_samples = [
    datasets["test_ordinary"][index]
    for index in test_indices
]

prompts = [
    sample["caption"].strip()
    for sample in reference_samples
]

real_images = torch.stack(
    [sample["image"] for sample in reference_samples]
)

encoded_prompts = tokenizer.encode_batch(prompts)

input_ids = torch.tensor(
    [encoding.ids for encoding in encoded_prompts],
    dtype=torch.long,
    device=device,
)

attention_mask = torch.tensor(
    [encoding.attention_mask for encoding in encoded_prompts],
    dtype=torch.long,
    device=device,
)

assert input_ids.shape == attention_mask.shape
assert input_ids.shape[0] == len(prompts)
assert attention_mask.sum(dim=1).gt(0).all()

image_shape = tuple(real_images.shape[1:])

conditional_sampler = ReverseSampler(
    conditional_unet=conditional_model,
    text_encoder=conditional_encoder,
    schedule=conditional_schedule_for_sampling,
    guidance_scale=1.0,
    clip_denoised=True,
    device=device,
)

# Stesso seed per garantire che conditional e unconditional
# inizino dalla stessa sequenza di rumore casuale.
sampling_seed = 2026

conditional_generator = torch.Generator(
    device=device
).manual_seed(sampling_seed)

start_time = time.perf_counter()

conditional_generated = conditional_sampler.sample(
    input_ids=input_ids,
    attention_mask=attention_mask,
    image_shape=image_shape,
    generator=conditional_generator,
)

conditional_elapsed_s = time.perf_counter() - start_time

assert conditional_generated.shape == real_images.shape
assert torch.isfinite(conditional_generated).all()

# Sampling unconditional manuale, con identica sequenza casuale.
unconditional_generator = torch.Generator(
    device=device
).manual_seed(sampling_seed)

x_t = torch.randn(
    len(prompts),
    *image_shape,
    device=device,
    dtype=torch.float32,
    generator=unconditional_generator,
)

start_time = time.perf_counter()

with torch.inference_mode():
    for timestep in reversed(
        range(unconditional_schedule_for_sampling.num_timesteps)
    ):
        timesteps = torch.full(
            (len(prompts),),
            fill_value=timestep,
            dtype=torch.long,
            device=device,
        )

        predicted_noise = unconditional_model(
            noisyimages=x_t,
            timesteps=timesteps,
            context=None,
            attention_mask=None,
        )

        x_t, _ = unconditional_schedule_for_sampling.reverse_step(
            noisyimages=x_t,
            predicted_noise=predicted_noise,
            timesteps=timesteps,
            generator=unconditional_generator,
            clip_denoised=True,
        )

unconditional_generated = x_t.clamp(-1.0, 1.0)

unconditional_elapsed_s = time.perf_counter() - start_time

assert unconditional_generated.shape == real_images.shape
assert torch.isfinite(unconditional_generated).all()


def to_display_image(image: torch.Tensor):
    # Converte un'immagine da [-1, 1] a formato Matplotlib

    image = image.detach().cpu().float()
    image = ((image + 1.0) / 2.0).clamp(0.0, 1.0)

    return image.permute(1, 2, 0).numpy()


fig, axes = plt.subplots(
    nrows=3,
    ncols=len(prompts),
    figsize=(16, 9),
)

row_labels = [
    "Reale\n(test ordinary)",
    "Conditional\n(stesso rumore)",
    "Unconditional\n(stesso rumore)",
]

image_rows = [
    real_images,
    conditional_generated,
    unconditional_generated,
]

for row_index, images in enumerate(image_rows):
    for column_index in range(len(prompts)):
        axis = axes[row_index, column_index]

        axis.imshow(
            to_display_image(images[column_index])
        )

        axis.axis("off")

        if column_index == 0:
            axis.set_ylabel(
                row_labels[row_index],
                fontsize=11,
                fontweight="bold",
            )

        if row_index == 0:
            axis.set_title(
                f"Prompt {column_index + 1}",
                fontsize=11,
                fontweight="bold",
            )

fig.suptitle(
    "Confronto visuale controllato: reali, conditional e baseline",
    fontsize=15,
    fontweight="bold",
)

fig.tight_layout()

plt.show()

print("--- INFERENZA CONTROLLATA COMPLETATA ---")
print(f"Seed condiviso: {sampling_seed}")
print(
    "Tempo conditional totale: "
    f"{conditional_elapsed_s:.2f} s"
)
print(
    "Tempo unconditional totale: "
    f"{unconditional_elapsed_s:.2f} s"
)

for index, prompt in enumerate(prompts, start=1):
    print(f"\nPrompt {index}: {prompt}")

## Salvataggio delle grid PNG finali

Esportiamo in PNG le grid visuali finali dei campioni conditional e unconditional.
Questi file servono come artefatti riutilizzabili nel report finale e come
evidenza visiva del comportamento dei due modelli.

In [ ]:
# ---------------------------------------------------------
# Salvataggio delle grid PNG finali
# ---------------------------------------------------------

from pathlib import Path

import matplotlib.pyplot as plt


# ---------------------------------------------------------
# 0. Check preliminari
# ---------------------------------------------------------

required_globals = [
    "conditional_grid_images",
    "conditional_grid_prompts",
    "unconditional_grid_images",
    "unconditional_grid_seeds",
    "unconditional_grid_sampling_times_s",
]

for variable_name in required_globals:
    assert variable_name in globals(), (
        f"Variabile mancante: {variable_name}. "
        "Esegui prima le due celle di inferenza visuale."
    )


# ---------------------------------------------------------
# 1. Directory output
# ---------------------------------------------------------

output_dir = Path("outputs/evaluation")
output_dir.mkdir(parents=True, exist_ok=True)

conditional_png_path = output_dir / "conditional_grid.png"
unconditional_png_path = output_dir / "unconditional_grid.png"


# ---------------------------------------------------------
# 2. Salvataggio grid conditional
# ---------------------------------------------------------

conditional_images_to_show = (
    (conditional_grid_images + 1.0) / 2.0
).clamp(0.0, 1.0)

figure_cond, axes_cond = plt.subplots(
    nrows=2,
    ncols=2,
    figsize=(10, 10),
    dpi=150,
)

axes_cond = axes_cond.flatten()

for axis, image, prompt in zip(
    axes_cond,
    conditional_images_to_show,
    conditional_grid_prompts,
):
    axis.imshow(image.permute(1, 2, 0).numpy())
    axis.set_title(prompt[:80], fontsize=10)
    axis.axis("off")

figure_cond.suptitle(
    "Conditional samples from best.pt (test_ordinary prompts)",
    fontsize=16,
    y=0.98,
)

plt.tight_layout()
figure_cond.savefig(
    conditional_png_path,
    dpi=200,
    bbox_inches="tight",
)
plt.show()


# ---------------------------------------------------------
# 3. Salvataggio grid unconditional
# ---------------------------------------------------------

unconditional_images_to_show = (
    (unconditional_grid_images + 1.0) / 2.0
).clamp(0.0, 1.0)

figure_uncond, axes_uncond = plt.subplots(
    nrows=2,
    ncols=4,
    figsize=(14, 7),
    dpi=150,
)

axes_uncond = axes_uncond.flatten()

for axis, image, seed, elapsed_s in zip(
    axes_uncond,
    unconditional_images_to_show,
    unconditional_grid_seeds,
    unconditional_grid_sampling_times_s,
):
    axis.imshow(image.permute(1, 2, 0).numpy())
    axis.set_title(
        f"Seed {seed} | {elapsed_s:.2f} s",
        fontsize=10,
    )
    axis.axis("off")

figure_uncond.suptitle(
    "Unconditional samples from best.pt",
    fontsize=16,
    y=0.98,
)

plt.tight_layout()
figure_uncond.savefig(
    unconditional_png_path,
    dpi=200,
    bbox_inches="tight",
)
plt.show()


# ---------------------------------------------------------
# 4. Riepilogo finale
# ---------------------------------------------------------

print("--- GRID PNG SALVATE ---")
print(f"Conditional grid salvata in: {conditional_png_path}")
print(f"Unconditional grid salvata in: {unconditional_png_path}")

## Valutazione quantitativa su test_ordinary

In questa sezione confrontiamo il modello conditional e la baseline unconditional
sullo split `test_ordinary`, cioè su combinazioni di attributi osservate durante
l'addestramento.

La valutazione quantitativa include:
- una metrica di qualità distributiva tramite KID;
- un confronto di complessità in termini di parametri e memoria dei pesi;
- un profiling del reverse sampling in termini di latenza e picco di memoria CUDA.

Lo split composizionale `test_ood` verrà valutato separatamente nelle celle
successive, insieme alle metriche specifiche di conditioning e diversità.

In [ ]:
# ---------------------------------------------------------
# KID conditional vs unconditional
# ---------------------------------------------------------

import math
from pathlib import Path

import pandas as pd
import torch
from torchmetrics.image.kid import KernelInceptionDistance


# ---------------------------------------------------------
# 0. Check preliminari
# ---------------------------------------------------------

required_globals = [
    "datasets",
    "tokenizer",
    "conditional_sampler",
    "conditional_unet_eval",
    "text_encoder_eval",
    "schedule_eval",
    "unconditional_unet_eval",
    "unconditional_schedule_eval",
]

for variable_name in required_globals:
    assert variable_name in globals(), (
        f"Variabile mancante: {variable_name}. "
        "Esegui prima le celle di inferenza visuale."
    )

assert "test_ordinary" in datasets, (
    "Split test_ordinary non disponibile."
)

dataset_ordinary = datasets["test_ordinary"]
assert len(dataset_ordinary) > 0, (
    "Il test split ordinary non contiene esempi."
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# ---------------------------------------------------------
# 1. Utility KID
# ---------------------------------------------------------

def compute_kid(
    real_batch: torch.Tensor,
    generated_batch: torch.Tensor,
    device: torch.device,
) -> tuple[float, float]:
    subset_size = min(
        len(real_batch),
        len(generated_batch),
        50,
    )

    kid_metric = KernelInceptionDistance(
        subset_size=subset_size,
        normalize=True,
    ).to(device)

    with torch.inference_mode():
        kid_metric.update(
            real_batch.to(device),
            real=True,
        )
        kid_metric.update(
            generated_batch.to(device),
            real=False,
        )
        kid_mean, kid_std = kid_metric.compute()

    kid_mean = float(kid_mean.detach().cpu().item())
    kid_std = float(kid_std.detach().cpu().item())

    assert math.isfinite(kid_mean)
    assert math.isfinite(kid_std)

    return kid_mean, kid_std


# ---------------------------------------------------------
# 2. Campione reale di riferimento
# ---------------------------------------------------------
# Usiamo un campione reale non minuscolo ma ancora ragionevole
# per tempi Colab e costo del reverse sampling DDPM.
# ---------------------------------------------------------

num_kid_samples = 64

assert len(dataset_ordinary) >= num_kid_samples, (
    "test_ordinary non contiene abbastanza esempi per il KID."
)

# Usiamo un sottoinsieme deterministico ma distribuito
# lungo lo split, invece di prendere banalmente i primi N.
selected_indices = torch.linspace(
    0,
    len(dataset_ordinary) - 1,
    steps=num_kid_samples,
).round().to(torch.long).tolist()

real_images = []
conditional_prompts_for_kid = []

for sample_index in selected_indices:
    sample = dataset_ordinary[int(sample_index)]

    image = sample["image"]
    caption = sample["caption"]

    assert isinstance(image, torch.Tensor)
    assert image.ndim == 3
    assert torch.isfinite(image).all()
    assert isinstance(caption, str)
    assert caption.strip()

    real_images.append(image.detach().cpu())
    conditional_prompts_for_kid.append(caption.strip())

real_images = torch.stack(real_images, dim=0)

assert real_images.shape == (
    num_kid_samples,
    3,
    32,
    32,
)
assert len(conditional_prompts_for_kid) == num_kid_samples


# ---------------------------------------------------------
# 3. Campione generato conditional
# ---------------------------------------------------------

encoded_prompts = tokenizer.encode_batch(
    conditional_prompts_for_kid
)

kid_input_ids = torch.tensor(
    [encoding.ids for encoding in encoded_prompts],
    dtype=torch.long,
    device=device,
)

kid_attention_mask = torch.tensor(
    [encoding.attention_mask for encoding in encoded_prompts],
    dtype=torch.long,
    device=device,
)

reference_image = dataset_ordinary[0]["image"]
image_shape = tuple(reference_image.shape)

kid_generator_conditional = torch.Generator(
    device=device
).manual_seed(3107)

with torch.inference_mode():
    conditional_generated_for_kid = conditional_sampler.sample(
        input_ids=kid_input_ids,
        attention_mask=kid_attention_mask,
        image_shape=image_shape,
        generator=kid_generator_conditional,
        guidance_scale=1.0,
    )

assert conditional_generated_for_kid.shape == (
    num_kid_samples,
    *image_shape,
)

assert torch.isfinite(conditional_generated_for_kid).all()


# ---------------------------------------------------------
# 4. Campione generato unconditional
# ---------------------------------------------------------

kid_generator_unconditional = torch.Generator(
    device=device
).manual_seed(3107)

x_t = torch.randn(
    num_kid_samples,
    *image_shape,
    device=device,
    dtype=torch.float32,
    generator=kid_generator_unconditional,
)

with torch.inference_mode():
    for timestep in reversed(
        range(unconditional_schedule_eval.num_timesteps)
    ):
        timesteps = torch.full(
            (num_kid_samples,),
            fill_value=timestep,
            device=device,
            dtype=torch.long,
        )

        predicted_noise = unconditional_unet_eval(
            noisyimages=x_t,
            timesteps=timesteps,
            context=None,
            attention_mask=None,
        )

        x_t, predicted_x_start = (
            unconditional_schedule_eval.reverse_step(
                noisyimages=x_t,
                predicted_noise=predicted_noise,
                timesteps=timesteps,
                generator=kid_generator_unconditional,
                clip_denoised=True,
            )
        )

unconditional_generated_for_kid = x_t

assert unconditional_generated_for_kid.shape == (
    num_kid_samples,
    *image_shape,
)
assert predicted_x_start.shape == (
    num_kid_samples,
    *image_shape,
)
assert torch.isfinite(unconditional_generated_for_kid).all()


# ---------------------------------------------------------
# 5. Conversione da [-1, 1] a [0, 1]
# ---------------------------------------------------------

real_images_01 = (
    (real_images + 1.0) / 2.0
).clamp(0.0, 1.0)

conditional_images_01 = (
    (conditional_generated_for_kid.detach().cpu() + 1.0) / 2.0
).clamp(0.0, 1.0)

unconditional_images_01 = (
    (unconditional_generated_for_kid.detach().cpu() + 1.0) / 2.0
).clamp(0.0, 1.0)

assert real_images_01.ndim == 4
assert conditional_images_01.ndim == 4
assert unconditional_images_01.ndim == 4


# ---------------------------------------------------------
# 6. Calcolo KID
# ---------------------------------------------------------

kid_conditional_mean, kid_conditional_std = compute_kid(
    real_batch=real_images_01,
    generated_batch=conditional_images_01,
    device=device,
)

kid_unconditional_mean, kid_unconditional_std = compute_kid(
    real_batch=real_images_01,
    generated_batch=unconditional_images_01,
    device=device,
)


# ---------------------------------------------------------
# 7. Salvataggio tabella KID
# ---------------------------------------------------------

kid_table_path = Path("outputs/evaluation/kid_comparison.csv")
kid_table_path.parent.mkdir(parents=True, exist_ok=True)

kid_table = pd.DataFrame(
    [
        {
            "model": "Conditional",
            "kid_mean": kid_conditional_mean,
            "kid_std": kid_conditional_std,
            "num_real_images": num_kid_samples,
            "num_generated_images": num_kid_samples,
            "reference_split": "test_ordinary",
        },
        {
            "model": "Unconditional",
            "kid_mean": kid_unconditional_mean,
            "kid_std": kid_unconditional_std,
            "num_real_images": num_kid_samples,
            "num_generated_images": num_kid_samples,
            "reference_split": "test_ordinary",
        },
    ]
)

kid_table.to_csv(kid_table_path, index=False)

display(kid_table)

print("--- KID CONDITIONAL VS UNCONDITIONAL ---")
print(f"Numero immagini usate per modello: {num_kid_samples}")
print(f"CSV salvato in: {kid_table_path}")

In [ ]:
# ---------------------------------------------------------
# Complessità, latenza e memoria
# ---------------------------------------------------------

import gc
import statistics
import time
from pathlib import Path

import pandas as pd
import torch


# ---------------------------------------------------------
# 0. Check preliminari
# ---------------------------------------------------------

required_globals = [
    "conditional_sampler",
    "conditional_unet_eval",
    "text_encoder_eval",
    "unconditional_unet_eval",
    "schedule_eval",
    "unconditional_schedule_eval",
    "tokenizer",
    "datasets",
]

for variable_name in required_globals:
    assert variable_name in globals(), (
        f"Variabile mancante: {variable_name}."
    )

assert conditional_unet_eval.training is False
assert text_encoder_eval.training is False
assert unconditional_unet_eval.training is False

assert conditional_unet_eval.conditional is True
assert unconditional_unet_eval.conditional is False

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

assert "test_ordinary" in datasets, (
    "Split test_ordinary non disponibile."
)

dataset_ordinary = datasets["test_ordinary"]
assert len(dataset_ordinary) > 0, (
    "Il test split ordinary non contiene esempi."
)


# ---------------------------------------------------------
# 1. Utility di profiling
# ---------------------------------------------------------

def synchronize_if_cuda() -> None:
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def clear_cuda_cache() -> None:
    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)


def format_millions(value: int) -> float:
    return value / 1_000_000


def model_size_mib(*modules: torch.nn.Module) -> float:
    total_bytes = 0

    for module in modules:
        for tensor in list(module.parameters()) + list(module.buffers()):
            total_bytes += (
                tensor.numel() * tensor.element_size()
            )

    return total_bytes / 1024**2


def profile_unconditional_sampling(
    model: torch.nn.Module,
    schedule: torch.nn.Module,
    image_shape: tuple[int, int, int],
    seeds: list[int],
    warmup_runs: int,
) -> dict[str, object]:
    for warmup_seed in range(warmup_runs):
        generator = torch.Generator(device=device)
        generator.manual_seed(90000 + warmup_seed)

        x_t = torch.randn(
            1,
            *image_shape,
            device=device,
            dtype=torch.float32,
            generator=generator,
        )

        with torch.inference_mode():
            for timestep in reversed(
                range(schedule.num_timesteps)
            ):
                timesteps = torch.full(
                    (1,),
                    fill_value=timestep,
                    device=device,
                    dtype=torch.long,
                )

                predicted_noise = model(
                    noisyimages=x_t,
                    timesteps=timesteps,
                    context=None,
                    attention_mask=None,
                )

                x_t, predicted_x_start = schedule.reverse_step(
                    noisyimages=x_t,
                    predicted_noise=predicted_noise,
                    timesteps=timesteps,
                    generator=generator,
                    clip_denoised=True,
                )

        assert x_t.shape == (1, *image_shape)
        assert predicted_x_start.shape == (1, *image_shape)

        del x_t
        gc.collect()

    synchronize_if_cuda()
    clear_cuda_cache()
    synchronize_if_cuda()

    elapsed_seconds = []
    peak_allocated_mib = []
    peak_reserved_mib = []

    for seed in seeds:
        generator = torch.Generator(device=device)
        generator.manual_seed(seed)

        clear_cuda_cache()
        synchronize_if_cuda()

        x_t = torch.randn(
            1,
            *image_shape,
            device=device,
            dtype=torch.float32,
            generator=generator,
        )

        start_time = time.perf_counter()

        with torch.inference_mode():
            for timestep in reversed(
                range(schedule.num_timesteps)
            ):
                timesteps = torch.full(
                    (1,),
                    fill_value=timestep,
                    device=device,
                    dtype=torch.long,
                )

                predicted_noise = model(
                    noisyimages=x_t,
                    timesteps=timesteps,
                    context=None,
                    attention_mask=None,
                )

                x_t, predicted_x_start = schedule.reverse_step(
                    noisyimages=x_t,
                    predicted_noise=predicted_noise,
                    timesteps=timesteps,
                    generator=generator,
                    clip_denoised=True,
                )

        synchronize_if_cuda()
        elapsed_s = time.perf_counter() - start_time

        assert x_t.shape == (1, *image_shape)
        assert predicted_x_start.shape == (1, *image_shape)
        assert torch.isfinite(x_t).all()
        assert torch.isfinite(predicted_x_start).all()
        assert float(x_t.min()) >= -1.0
        assert float(x_t.max()) <= 1.0

        elapsed_seconds.append(float(elapsed_s))

        if device.type == "cuda":
            peak_allocated_mib.append(
                torch.cuda.max_memory_allocated(device) / 1024**2
            )
            peak_reserved_mib.append(
                torch.cuda.max_memory_reserved(device) / 1024**2
            )

        del x_t
        gc.collect()

    result = {
        "samples": len(seeds),
        "seconds_per_image": elapsed_seconds,
        "mean_seconds_per_image": statistics.mean(
            elapsed_seconds
        ),
        "std_seconds_per_image": (
            statistics.stdev(elapsed_seconds)
            if len(elapsed_seconds) > 1
            else 0.0
        ),
        "peak_allocated_mib": peak_allocated_mib,
        "peak_reserved_mib": peak_reserved_mib,
        "mean_peak_allocated_mib": (
            statistics.mean(peak_allocated_mib)
            if peak_allocated_mib
            else None
        ),
        "max_peak_allocated_mib": (
            max(peak_allocated_mib)
            if peak_allocated_mib
            else None
        ),
        "mean_peak_reserved_mib": (
            statistics.mean(peak_reserved_mib)
            if peak_reserved_mib
            else None
        ),
        "max_peak_reserved_mib": (
            max(peak_reserved_mib)
            if peak_reserved_mib
            else None
        ),
    }

    return result


def profile_conditional_sampling(
    sampler: object,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    image_shape: tuple[int, int, int],
    seeds: list[int],
    warmup_runs: int,
) -> dict[str, object]:
    for warmup_seed in range(warmup_runs):
        generator = torch.Generator(device=device)
        generator.manual_seed(90000 + warmup_seed)

        with torch.inference_mode():
            warmup_images = sampler.sample(
                input_ids=input_ids,
                attention_mask=attention_mask,
                image_shape=image_shape,
                generator=generator,
                guidance_scale=1.0,
            )

        assert warmup_images.shape == (1, *image_shape)
        assert torch.isfinite(warmup_images).all()

        del warmup_images
        gc.collect()

    synchronize_if_cuda()
    clear_cuda_cache()
    synchronize_if_cuda()

    elapsed_seconds = []
    peak_allocated_mib = []
    peak_reserved_mib = []

    for seed in seeds:
        generator = torch.Generator(device=device)
        generator.manual_seed(seed)

        clear_cuda_cache()
        synchronize_if_cuda()

        start_time = time.perf_counter()

        with torch.inference_mode():
            generated_images = sampler.sample(
                input_ids=input_ids,
                attention_mask=attention_mask,
                image_shape=image_shape,
                generator=generator,
                guidance_scale=1.0,
            )

        synchronize_if_cuda()
        elapsed_s = time.perf_counter() - start_time

        assert generated_images.shape == (1, *image_shape)
        assert torch.isfinite(generated_images).all()
        assert float(generated_images.min()) >= -1.0
        assert float(generated_images.max()) <= 1.0

        elapsed_seconds.append(float(elapsed_s))

        if device.type == "cuda":
            peak_allocated_mib.append(
                torch.cuda.max_memory_allocated(device) / 1024**2
            )
            peak_reserved_mib.append(
                torch.cuda.max_memory_reserved(device) / 1024**2
            )

        del generated_images
        gc.collect()

    result = {
        "samples": len(seeds),
        "seconds_per_image": elapsed_seconds,
        "mean_seconds_per_image": statistics.mean(
            elapsed_seconds
        ),
        "std_seconds_per_image": (
            statistics.stdev(elapsed_seconds)
            if len(elapsed_seconds) > 1
            else 0.0
        ),
        "peak_allocated_mib": peak_allocated_mib,
        "peak_reserved_mib": peak_reserved_mib,
        "mean_peak_allocated_mib": (
            statistics.mean(peak_allocated_mib)
            if peak_allocated_mib
            else None
        ),
        "max_peak_allocated_mib": (
            max(peak_allocated_mib)
            if peak_allocated_mib
            else None
        ),
        "mean_peak_reserved_mib": (
            statistics.mean(peak_reserved_mib)
            if peak_reserved_mib
            else None
        ),
        "max_peak_reserved_mib": (
            max(peak_reserved_mib)
            if peak_reserved_mib
            else None
        ),
    }

    return result


# ---------------------------------------------------------
# 2. Parametri e memoria dei pesi
# ---------------------------------------------------------

conditional_unet_parameters = sum(
    parameter.numel()
    for parameter in conditional_unet_eval.parameters()
)

conditional_text_parameters = sum(
    parameter.numel()
    for parameter in text_encoder_eval.parameters()
)

conditional_total_parameters = (
    conditional_unet_parameters
    + conditional_text_parameters
)

unconditional_total_parameters = sum(
    parameter.numel()
    for parameter in unconditional_unet_eval.parameters()
)

conditional_unet_size_mib = model_size_mib(
    conditional_unet_eval
)

conditional_text_size_mib = model_size_mib(
    text_encoder_eval
)

conditional_total_size_mib = (
    conditional_unet_size_mib
    + conditional_text_size_mib
)

unconditional_total_size_mib = model_size_mib(
    unconditional_unet_eval
)

print("--- PARAMETRI E FOOTPRINT DEI PESI ---")
print(
    "Conditional U-Net:",
    conditional_unet_parameters,
    f"({format_millions(conditional_unet_parameters):.3f} M)",
)
print(
    "Conditional text encoder:",
    conditional_text_parameters,
    f"({format_millions(conditional_text_parameters):.3f} M)",
)
print(
    "Conditional totale:",
    conditional_total_parameters,
    f"({format_millions(conditional_total_parameters):.3f} M)",
)
print(
    "Unconditional U-Net:",
    unconditional_total_parameters,
    f"({format_millions(unconditional_total_parameters):.3f} M)",
)
print()
print(
    f"Conditional U-Net, pesi e buffer: {conditional_unet_size_mib:.2f} MiB"
)
print(
    f"Conditional text encoder, pesi e buffer: {conditional_text_size_mib:.2f} MiB"
)
print(
    f"Conditional totale, pesi e buffer: {conditional_total_size_mib:.2f} MiB"
)
print(
    f"Unconditional, pesi e buffer: {unconditional_total_size_mib:.2f} MiB"
)


# ---------------------------------------------------------
# 3. Prompt benchmark per il profiling conditional
# ---------------------------------------------------------
# Usiamo un solo prompt reale, codificato una volta sola,
# così il benchmark misura soprattutto il costo del reverse sampling.
# ---------------------------------------------------------

benchmark_prompt = dataset_ordinary[0]["caption"]
assert isinstance(benchmark_prompt, str)
assert benchmark_prompt.strip()

benchmark_encoding = tokenizer.encode(
    benchmark_prompt.strip()
)

benchmark_input_ids = torch.tensor(
    [benchmark_encoding.ids],
    dtype=torch.long,
    device=device,
)

benchmark_attention_mask = torch.tensor(
    [benchmark_encoding.attention_mask],
    dtype=torch.long,
    device=device,
)

reference_image = dataset_ordinary[0]["image"]
image_shape = tuple(reference_image.shape)


# ---------------------------------------------------------
# 4. Profiling
# ---------------------------------------------------------

BENCHMARK_SEEDS = [42, 100, 2026, 9999, 31415]
WARMUP_RUNS = 1

assert len(BENCHMARK_SEEDS) >= 3
assert len(set(BENCHMARK_SEEDS)) == len(BENCHMARK_SEEDS)
assert WARMUP_RUNS >= 1

conditional_profile = profile_conditional_sampling(
    sampler=conditional_sampler,
    input_ids=benchmark_input_ids,
    attention_mask=benchmark_attention_mask,
    image_shape=image_shape,
    seeds=BENCHMARK_SEEDS,
    warmup_runs=WARMUP_RUNS,
)

unconditional_profile = profile_unconditional_sampling(
    model=unconditional_unet_eval,
    schedule=unconditional_schedule_eval,
    image_shape=image_shape,
    seeds=BENCHMARK_SEEDS,
    warmup_runs=WARMUP_RUNS,
)

assert conditional_profile["samples"] == len(BENCHMARK_SEEDS)
assert unconditional_profile["samples"] == len(BENCHMARK_SEEDS)

print("--- PROFILING DEL SAMPLING ---")
print(f"Device: {device}")
print(f"Immagine generata per run: {(1, *image_shape)}")
print(f"Numero di seed misurati: {len(BENCHMARK_SEEDS)}")
print(f"Warm-up esclusi: {WARMUP_RUNS}")
print()
print(
    "Conditional tempo medio:",
    f"{conditional_profile['mean_seconds_per_image']:.3f} s",
    f"+- {conditional_profile['std_seconds_per_image']:.3f} s",
)
print(
    "Unconditional tempo medio:",
    f"{unconditional_profile['mean_seconds_per_image']:.3f} s",
    f"+- {unconditional_profile['std_seconds_per_image']:.3f} s",
)

if device.type == "cuda":
    print()
    print(
        "Conditional picco CUDA allocato:",
        f"{conditional_profile['max_peak_allocated_mib']:.2f} MiB",
    )
    print(
        "Conditional picco CUDA riservato:",
        f"{conditional_profile['max_peak_reserved_mib']:.2f} MiB",
    )
    print(
        "Unconditional picco CUDA allocato:",
        f"{unconditional_profile['max_peak_allocated_mib']:.2f} MiB",
    )
    print(
        "Unconditional picco CUDA riservato:",
        f"{unconditional_profile['max_peak_reserved_mib']:.2f} MiB",
    )
else:
    print()
    print(
        "CUDA non disponibile: la memoria GPU non è misurabile in questa esecuzione."
    )

# ---------------------------------------------------------
# 5. Salvataggio tabella finale di complessità
# ---------------------------------------------------------

complexity_table_path = Path(
    "outputs/evaluation/complexity_latency_memory.csv"
)
complexity_table_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

complexity_table = pd.DataFrame(
    [
        {
            "model": "Conditional",
            "parameters_total": conditional_total_parameters,
            "parameters_millions": format_millions(
                conditional_total_parameters
            ),
            "weights_and_buffers_mib": conditional_total_size_mib,
            "mean_seconds_per_image": conditional_profile[
                "mean_seconds_per_image"
            ],
            "std_seconds_per_image": conditional_profile[
                "std_seconds_per_image"
            ],
            "max_peak_allocated_mib": conditional_profile[
                "max_peak_allocated_mib"
            ],
            "max_peak_reserved_mib": conditional_profile[
                "max_peak_reserved_mib"
            ],
        },
        {
            "model": "Unconditional",
            "parameters_total": unconditional_total_parameters,
            "parameters_millions": format_millions(
                unconditional_total_parameters
            ),
            "weights_and_buffers_mib": unconditional_total_size_mib,
            "mean_seconds_per_image": unconditional_profile[
                "mean_seconds_per_image"
            ],
            "std_seconds_per_image": unconditional_profile[
                "std_seconds_per_image"
            ],
            "max_peak_allocated_mib": unconditional_profile[
                "max_peak_allocated_mib"
            ],
            "max_peak_reserved_mib": unconditional_profile[
                "max_peak_reserved_mib"
            ],
        },
    ]
)

complexity_table.to_csv(
    complexity_table_path,
    index=False,
)

display(complexity_table)

print()
print("--- TABELLA COMPLESSITÀ / LATENZA / MEMORIA SALVATA ---")
print(f"CSV salvato in: {complexity_table_path}")

### Tabella di confronto finale

La tabella riunisce i risultati quantitativi di qualità, complessità e costo di sampling. Un valore KID più basso indica una maggiore vicinanza, nello spazio delle feature Inception, alla distribuzione delle immagini reali del test split ordinary. I tempi sono riportati in secondi per immagine dopo warm-up; il picco CUDA riguarda il reverse sampling di una immagine alla volta.

In [ ]:
# ---------------------------------------------------------
# Tabella di confronto finale
# ---------------------------------------------------------

from pathlib import Path

import pandas as pd


# ---------------------------------------------------------
# 0. Check preliminari
# ---------------------------------------------------------

required_globals = [
    "kid_conditional_mean",
    "kid_conditional_std",
    "kid_unconditional_mean",
    "kid_unconditional_std",
    "conditional_total_parameters",
    "unconditional_total_parameters",
    "conditional_total_size_mib",
    "unconditional_total_size_mib",
    "conditional_profile",
    "unconditional_profile",
]

for variable_name in required_globals:
    assert variable_name in globals(), (
        f"Variabile mancante: {variable_name}."
    )


# ---------------------------------------------------------
# 1. Costruzione tabella numerica
# ---------------------------------------------------------

comparison_rows = [
    {
        "model": "Conditional",
        "parameters_total_m": format_millions(
            conditional_total_parameters
        ),
        "parameters_trainable_m": format_millions(
            conditional_total_parameters
        ),
        "weights_buffers_mib": conditional_total_size_mib,
        "kid_mean": kid_conditional_mean,
        "kid_std": kid_conditional_std,
        "sampling_mean_s": conditional_profile[
            "mean_seconds_per_image"
        ],
        "sampling_std_s": conditional_profile[
            "std_seconds_per_image"
        ],
        "peak_cuda_allocated_mib": conditional_profile[
            "max_peak_allocated_mib"
        ],
        "peak_cuda_reserved_mib": conditional_profile[
            "max_peak_reserved_mib"
        ],
        "evaluation_split": "test_ordinary",
    },
    {
        "model": "Unconditional",
        "parameters_total_m": format_millions(
            unconditional_total_parameters
        ),
        "parameters_trainable_m": format_millions(
            unconditional_total_parameters
        ),
        "weights_buffers_mib": unconditional_total_size_mib,
        "kid_mean": kid_unconditional_mean,
        "kid_std": kid_unconditional_std,
        "sampling_mean_s": unconditional_profile[
            "mean_seconds_per_image"
        ],
        "sampling_std_s": unconditional_profile[
            "std_seconds_per_image"
        ],
        "peak_cuda_allocated_mib": unconditional_profile[
            "max_peak_allocated_mib"
        ],
        "peak_cuda_reserved_mib": unconditional_profile[
            "max_peak_reserved_mib"
        ],
        "evaluation_split": "test_ordinary",
    },
]

comparison_table = pd.DataFrame(comparison_rows)

assert len(comparison_table) == 2
assert set(comparison_table["model"]) == {
    "Conditional",
    "Unconditional",
}


# ---------------------------------------------------------
# 2. Tabella formattata per lettura
# ---------------------------------------------------------

display_columns = [
    "model",
    "parameters_total_m",
    "parameters_trainable_m",
    "weights_buffers_mib",
    "kid_mean",
    "kid_std",
    "sampling_mean_s",
    "sampling_std_s",
    "peak_cuda_allocated_mib",
    "peak_cuda_reserved_mib",
    "evaluation_split",
]

formatted_table = comparison_table[
    display_columns
].copy()

formatted_table["parameters_total_m"] = (
    formatted_table["parameters_total_m"]
    .map(lambda value: f"{value:.3f} M")
)

formatted_table["parameters_trainable_m"] = (
    formatted_table["parameters_trainable_m"]
    .map(lambda value: f"{value:.3f} M")
)

for column_name in [
    "weights_buffers_mib",
    "kid_mean",
    "kid_std",
    "sampling_mean_s",
    "sampling_std_s",
]:
    formatted_table[column_name] = (
        formatted_table[column_name]
        .map(lambda value: f"{value:.4f}")
    )

for column_name in [
    "peak_cuda_allocated_mib",
    "peak_cuda_reserved_mib",
]:
    formatted_table[column_name] = (
        formatted_table[column_name]
        .map(
            lambda value: (
                f"{value:.2f}"
                if value is not None
                else "N/A CPU"
            )
        )
    )

formatted_table = formatted_table.rename(
    columns={
        "model": "Modello",
        "parameters_total_m": "Parametri totali",
        "parameters_trainable_m": "Parametri trainable",
        "weights_buffers_mib": "Pesi+buffer MiB",
        "kid_mean": "KID mean",
        "kid_std": "KID std",
        "sampling_mean_s": "Sampling medio s/img",
        "sampling_std_s": "Sampling std s/img",
        "peak_cuda_allocated_mib": "Picco CUDA allocato MiB",
        "peak_cuda_reserved_mib": "Picco CUDA riservato MiB",
        "evaluation_split": "Split",
    }
)


# ---------------------------------------------------------
# 3. Salvataggio CSV
# ---------------------------------------------------------

output_dir = Path("outputs/evaluation")
output_dir.mkdir(parents=True, exist_ok=True)

comparison_output_path = (
    output_dir / "comparison_table.csv"
)

formatted_output_path = (
    output_dir / "comparison_table_formatted.csv"
)

comparison_table.to_csv(
    comparison_output_path,
    index=False,
)

formatted_table.to_csv(
    formatted_output_path,
    index=False,
)


# ---------------------------------------------------------
# 4. Output finale
# ---------------------------------------------------------

print("--- CONFRONTO QUANTITATIVO FINALE ---")
display(formatted_table)
print()
print(f"Tabella numerica salvata in: {comparison_output_path}")
print(f"Tabella formattata salvata in: {formatted_output_path}")

## Lettura dei risultati

Il confronto deve essere interpretato separando capacità generativa, controllo testuale e costo computazionale.

Il KID misura la distanza tra distribuzioni di feature delle immagini reali e generate: valori minori sono preferibili. La diversità a prompt fissato verifica che seed differenti producano campioni visivamente differenti senza cambiare il testo richiesto. La baseline unconditional è utile per separare la qualità generativa non condizionata dal contributo del text conditioning, ma non può soddisfare un prompt specifico perché non riceve testo in input.

Il modello conditional richiede inevitabilmente più parametri e memoria dei pesi, poiché include il text encoder oltre alla U-Net. Nel profiling, il tempo di reverse sampling misura il costo dell'intero processo DDPM con identico numero di timestep per entrambi i modelli; il conditioning viene codificato prima del timer per isolare il costo del sampler, mentre il footprint totale del text encoder viene comunque incluso nelle misure di complessità.

Il KID è calcolato su 64 immagini reali e 64 immagini generate per modello:
va quindi interpretato come confronto interno tra conditional e unconditional,
che condividono dataset, schedule e protocollo di sampling, non come stima
definitiva della qualità assoluta. I picchi CUDA eventualmente riportati
descrivono il processo di benchmark nel notebook e non la sola memoria
intrinseca di ciascun modello.

## Generalizzazione composizionale e valutazioni complementari

La valutazione finale combina campioni qualitativi OOD, una proxy di
conditioning nearest-neighbor e una misura di diversity a prompt fisso.

La proxy nearest-neighbor usa un reference bank comune composto da immagini
`test_ordinary` e `test_ood`. Vengono confrontati due protocolli:

- **reference included**, interpretato come condizione di retrieval rispetto a un bank che contiene il riferimento del prompt e usato anche per verificare la copertura del bank;
- **leave-one-out**, in cui l'immagine reale da cui deriva il prompt viene
  esclusa dal confronto.

Il protocollo leave-one-out viene inoltre applicato alle immagini reali. Tale
calibrazione consente di separare i limiti del valutatore nearest-neighbor dai
limiti effettivi delle immagini generate.

Poiché una combinazione completa di otto attributi può essere unica nel
reference bank, il match completo non viene interpretato isolatamente. Per lo
split composizionale OOD viene quindi riportato anche il match congiunto della
coppia `face_shape + hair_color`, che costituisce l'oggetto specifico della
generalizzazione richiesta.

In [ ]:
# ---------------------------------------------------------
# Campioni qualitativi su test_ood
# ---------------------------------------------------------

import matplotlib.pyplot as plt
import torch


# ---------------------------------------------------------
# 0. Check preliminari
# ---------------------------------------------------------

required_globals = [
    "datasets",
    "tokenizer",
    "conditional_sampler",
]

for variable_name in required_globals:
    assert variable_name in globals(), (
        f"Variabile mancante: {variable_name}."
    )

assert "test_ood" in datasets, (
    "Split test_ood non disponibile."
)

dataset_ood = datasets["test_ood"]

assert len(dataset_ood) > 0, (
    "Lo split test_ood è vuoto."
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# ---------------------------------------------------------
# 1. Selezione di prompt reali OOD
# ---------------------------------------------------------

num_ood_samples = 4

ood_prompts = []
ood_conditioning_values = []

for sample in dataset_ood:
    caption = sample["caption"]
    conditioning_values = sample["conditioning_values"]

    assert isinstance(caption, str)
    assert isinstance(conditioning_values, torch.Tensor)

    cleaned_caption = caption.strip()

    if cleaned_caption:
        ood_prompts.append(cleaned_caption)
        ood_conditioning_values.append(
            conditioning_values.detach().cpu()
        )

    if len(ood_prompts) == num_ood_samples:
        break

assert len(ood_prompts) == num_ood_samples, (
    "Non sono stati trovati abbastanza prompt validi in test_ood."
)


# ---------------------------------------------------------
# 2. Tokenizzazione dei prompt OOD
# ---------------------------------------------------------

encoded_ood_prompts = tokenizer.encode_batch(
    ood_prompts
)

ood_input_ids = torch.tensor(
    [encoding.ids for encoding in encoded_ood_prompts],
    dtype=torch.long,
    device=device,
)

ood_attention_mask = torch.tensor(
    [encoding.attention_mask for encoding in encoded_ood_prompts],
    dtype=torch.long,
    device=device,
)

reference_image = dataset_ood[0]["image"]
image_shape = tuple(reference_image.shape)


# ---------------------------------------------------------
# 3. Sampling conditional sullo split OOD
# ---------------------------------------------------------

ood_seed = 4242
ood_generator = torch.Generator(device=device)
ood_generator.manual_seed(ood_seed)

with torch.inference_mode():
    conditional_generated_ood = conditional_sampler.sample(
        input_ids=ood_input_ids,
        attention_mask=ood_attention_mask,
        image_shape=image_shape,
        generator=ood_generator,
        guidance_scale=1.0,
    )

assert conditional_generated_ood.shape == (
    num_ood_samples,
    *image_shape,
)

assert torch.isfinite(conditional_generated_ood).all()


# ---------------------------------------------------------
# 4. Visualizzazione OOD
# ---------------------------------------------------------

import textwrap

ood_images_to_show = (
    (conditional_generated_ood.detach().cpu() + 1.0) / 2.0
).clamp(0.0, 1.0)

figure, axes = plt.subplots(
    nrows=2,
    ncols=2,
    figsize=(10, 8),
    dpi=160,
)

axes = axes.flatten()

for axis, image, prompt in zip(
    axes,
    ood_images_to_show,
    ood_prompts,
):
    axis.imshow(image.permute(1, 2, 0).numpy())

    axis.set_title(
        textwrap.fill(prompt, width=40),
        fontsize=8,
        pad=8,
        loc="center",
    )

    axis.axis("off")

figure.suptitle(
    "Campioni conditional OOD",
    fontsize=16,
    fontweight="bold",
    y=0.98,
)

figure.tight_layout(
    rect=(0.03, 0.03, 0.97, 0.92),
    h_pad=1.4,
    w_pad=0.8,
)

plt.show()


# ---------------------------------------------------------
# 5. Variabili da riusare
# ---------------------------------------------------------

ood_grid_images = conditional_generated_ood.detach().cpu()
ood_grid_prompts = list(ood_prompts)
ood_grid_conditioning_values = torch.stack(
    ood_conditioning_values,
    dim=0,
)
ood_grid_split_name = "test_ood"
ood_grid_seed = int(ood_seed)

print("--- CAMPIONI OOD GENERATI ---")
print(f"Split: {ood_grid_split_name}")
print(f"Numero campioni: {num_ood_samples}")
print(f"Seed usato: {ood_grid_seed}")

In [ ]:
# ---------------------------------------------------------
# Valutazione finale della proxy di conditioning
# ---------------------------------------------------------
# Il protocollo confronta:
# - generated / reference included;
# - generated / leave-one-out;
# - real / leave-one-out, come calibrazione del valutatore.
#
# Inoltre verifica:
# - se il match completo resta possibile dopo l'esclusione;
# - il match congiunto face_shape + hair_color;
# - il dettaglio separato per attributo.
# ---------------------------------------------------------

from pathlib import Path

import pandas as pd
import torch


# ---------------------------------------------------------
# 0. Check preliminari
# ---------------------------------------------------------

required_globals = [
    "datasets",
    "conditional_sampler",
    "tokenizer",
]

for variable_name in required_globals:
    assert variable_name in globals(), (
        f"Variabile mancante: {variable_name}."
    )

assert "test_ordinary" in datasets
assert "test_ood" in datasets

dataset_ordinary = datasets["test_ordinary"]
dataset_ood = datasets["test_ood"]

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

conditioning_attributes = [
    "face_shape",
    "face_color",
    "hair",
    "hair_color",
    "eye_color",
    "chin_length",
    "facial_hair",
    "glasses",
]

ood_pair_attributes = [
    "face_shape",
    "hair_color",
]

ood_pair_indices = [
    conditioning_attributes.index(attribute)
    for attribute in ood_pair_attributes
]


# ---------------------------------------------------------
# 1. Reference bank comune
# ---------------------------------------------------------

def build_real_reference_bank(
    dataset,
    split_name: str,
    max_items: int,
) -> dict[str, object]:
    # Costruisce un banco di immagini reali e attributi

    images = []
    values = []
    image_ids = []
    split_names = []

    for index in range(min(len(dataset), max_items)):
        sample = dataset[index]

        image = sample["image"]
        conditioning_values = sample[
            "conditioning_values"
        ]

        assert isinstance(image, torch.Tensor)
        assert isinstance(conditioning_values, torch.Tensor)
        assert image.ndim == 3
        assert conditioning_values.shape == (
            len(conditioning_attributes),
        )

        images.append(image.detach().cpu())
        values.append(conditioning_values.detach().cpu())
        image_ids.append(sample["image_id"])
        split_names.append(split_name)

    assert images, "Reference bank vuoto."

    return {
        "images": torch.stack(images, dim=0),
        "conditioning_values": torch.stack(values, dim=0),
        "image_ids": image_ids,
        "splits": split_names,
    }


def merge_reference_banks(
    *banks: dict[str, object],
) -> dict[str, object]:
    # Unisce banchi reali di split differenti
    assert banks, "Nessun bank fornito."

    return {
        "images": torch.cat(
            [bank["images"] for bank in banks],
            dim=0,
        ),
        "conditioning_values": torch.cat(
            [
                bank["conditioning_values"]
                for bank in banks
            ],
            dim=0,
        ),
        "image_ids": [
            image_id
            for bank in banks
            for image_id in bank["image_ids"]
        ],
        "splits": [
            split_name
            for bank in banks
            for split_name in bank["splits"]
        ],
    }


ordinary_reference_bank = build_real_reference_bank(
    dataset=dataset_ordinary,
    split_name="test_ordinary",
    max_items=512,
)

ood_reference_bank = build_real_reference_bank(
    dataset=dataset_ood,
    split_name="test_ood",
    max_items=len(dataset_ood),
)

common_reference_bank = merge_reference_banks(
    ordinary_reference_bank,
    ood_reference_bank,
)

assert common_reference_bank["images"].shape[0] == 656


# ---------------------------------------------------------
# 2. Query: immagini reali e immagini generate
# ---------------------------------------------------------

def build_evaluation_batch(
    dataset,
    num_samples: int,
    seed: int,
) -> dict[str, object]:
    # Prepara prompt, immagini reali e generazioni conditional

    prompts = []
    target_values = []
    target_image_ids = []
    real_images = []

    for sample in dataset:
        prompt = sample["caption"].strip()

        if prompt:
            prompts.append(prompt)
            target_values.append(
                sample["conditioning_values"].detach().cpu()
            )
            target_image_ids.append(sample["image_id"])
            real_images.append(sample["image"].detach().cpu())

        if len(prompts) == num_samples:
            break

    assert len(prompts) == num_samples

    encoded_prompts = tokenizer.encode_batch(prompts)

    input_ids = torch.tensor(
        [encoding.ids for encoding in encoded_prompts],
        dtype=torch.long,
        device=device,
    )

    attention_mask = torch.tensor(
        [
            encoding.attention_mask
            for encoding in encoded_prompts
        ],
        dtype=torch.long,
        device=device,
    )

    image_shape = tuple(real_images[0].shape)

    generator = torch.Generator(device=device)
    generator.manual_seed(int(seed))

    with torch.inference_mode():
        generated_images = conditional_sampler.sample(
            input_ids=input_ids,
            attention_mask=attention_mask,
            image_shape=image_shape,
            generator=generator,
            guidance_scale=1.0,
        )

    assert generated_images.shape == (
        num_samples,
        *image_shape,
    )

    assert torch.isfinite(generated_images).all()

    return {
        "prompts": prompts,
        "target_values": torch.stack(target_values, dim=0),
        "target_image_ids": target_image_ids,
        "real_images": torch.stack(real_images, dim=0),
        "generated_images": generated_images.detach().cpu(),
    }


num_eval_samples = 64

ordinary_batch = build_evaluation_batch(
    dataset=dataset_ordinary,
    num_samples=num_eval_samples,
    seed=5101,
)

ood_batch = build_evaluation_batch(
    dataset=dataset_ood,
    num_samples=num_eval_samples,
    seed=6101,
)


# ---------------------------------------------------------
# 3. Proxy nearest-neighbor e copertura del reference bank
# ---------------------------------------------------------

def safe_mean(
    values: torch.Tensor,
    mask: torch.Tensor,
) -> float:
    # Media definita solo dove il target è rappresentabile

    if not bool(mask.any()):
        return float("nan")

    return float(values[mask].float().mean().item())


def nearest_neighbor_proxy(
    query_images: torch.Tensor,
    target_values: torch.Tensor,
    target_image_ids: list,
    reference_bank: dict[str, object],
    leave_one_out: bool,
) -> dict[str, object]:
    # Valuta una query tramite il nearest neighbor in pixel space.

    # Con leave_one_out=True viene esclusa dal bank l'immagine
    # reale associata al prompt della query.

    reference_images = reference_bank["images"]
    reference_values = reference_bank[
        "conditioning_values"
    ]
    reference_image_ids = reference_bank["image_ids"]

    query_flat = query_images.flatten(
        start_dim=1
    ).float()

    reference_flat = reference_images.flatten(
        start_dim=1
    ).float()

    distance_matrix = torch.cdist(
        query_flat,
        reference_flat,
        p=2.0,
    )

    predicted_values = []
    nearest_distances = []
    full_target_available = []
    pair_target_available = []

    for query_index, target_image_id in enumerate(
        target_image_ids
    ):
        valid_mask = torch.ones(
            len(reference_image_ids),
            dtype=torch.bool,
        )

        if leave_one_out:
            valid_mask = torch.tensor(
                [
                    reference_id != target_image_id
                    for reference_id in reference_image_ids
                ],
                dtype=torch.bool,
            )

        assert valid_mask.any(), (
            "Reference bank vuoto dopo il filtraggio."
        )

        candidate_values = reference_values[valid_mask]

        target = target_values[query_index]

        full_target_available.append(
            bool(
                (
                    candidate_values == target
                )
                .all(dim=1)
                .any()
                .item()
            )
        )

        pair_target_available.append(
            bool(
                (
                    candidate_values[:, ood_pair_indices]
                    == target[ood_pair_indices]
                )
                .all(dim=1)
                .any()
                .item()
            )
        )

        candidate_distances = distance_matrix[
            query_index
        ].clone()

        candidate_distances[~valid_mask] = float("inf")

        nearest_index = int(
            candidate_distances.argmin().item()
        )

        predicted_values.append(
            reference_values[nearest_index]
        )

        nearest_distances.append(
            float(candidate_distances[nearest_index].item())
        )

    predicted_values = torch.stack(
        predicted_values,
        dim=0,
    )

    exact_match_per_attribute = (
        predicted_values == target_values
    )

    all_attributes_match = (
        exact_match_per_attribute
        .all(dim=1)
    )

    ood_pair_match = (
        exact_match_per_attribute[:, ood_pair_indices]
        .all(dim=1)
    )

    return {
        "exact_match_per_attribute": (
            exact_match_per_attribute
        ),
        "all_attributes_match": all_attributes_match,
        "ood_pair_match": ood_pair_match,
        "full_target_available": torch.tensor(
            full_target_available,
            dtype=torch.bool,
        ),
        "pair_target_available": torch.tensor(
            pair_target_available,
            dtype=torch.bool,
        ),
        "nearest_distances": nearest_distances,
    }


# ---------------------------------------------------------
# 4. Valutazioni: generated e calibrazione real leave-one-out
# ---------------------------------------------------------

evaluation_rows = []
attribute_rows = []
ood_pair_rows = []

evaluation_inputs = [
    (
        "test_ordinary",
        ordinary_batch,
    ),
    (
        "test_ood",
        ood_batch,
    ),
]

protocols = [
    (
        "generated_reference_included",
        "generated",
        False,
    ),
    (
        "generated_leave_one_out",
        "generated",
        True,
    ),
    (
        "real_leave_one_out_calibration",
        "real",
        True,
    ),
]

for split_name, batch in evaluation_inputs:
    for protocol_name, query_type, leave_one_out in protocols:
        query_images = (
            batch["generated_images"]
            if query_type == "generated"
            else batch["real_images"]
        )

        proxy = nearest_neighbor_proxy(
            query_images=query_images,
            target_values=batch["target_values"],
            target_image_ids=batch["target_image_ids"],
            reference_bank=common_reference_bank,
            leave_one_out=leave_one_out,
        )

        exact_match = proxy[
            "exact_match_per_attribute"
        ]

        all_match = proxy["all_attributes_match"]

        pair_match = proxy["ood_pair_match"]

        full_available = proxy[
            "full_target_available"
        ]

        pair_available = proxy[
            "pair_target_available"
        ]

        evaluation_rows.append(
            {
                "split": split_name,
                "protocol": protocol_name,
                "query_type": query_type,
                "leave_one_out": leave_one_out,
                "num_queries": num_eval_samples,
                "reference_bank_size": int(
                    common_reference_bank["images"].shape[0]
                ),
                "attribute_match_rate": float(
                    exact_match.float().mean().item()
                ),
                "all_attributes_match_rate": float(
                    all_match.float().mean().item()
                ),
                "full_target_available_rate": float(
                    full_available.float().mean().item()
                ),
                "all_attributes_match_rate_when_possible": (
                    safe_mean(
                        all_match,
                        full_available,
                    )
                ),
                "ood_pair_match_rate": float(
                    pair_match.float().mean().item()
                ),
                "ood_pair_available_rate": float(
                    pair_available.float().mean().item()
                ),
                "ood_pair_match_rate_when_possible": (
                    safe_mean(
                        pair_match,
                        pair_available,
                    )
                ),
                "mean_nearest_neighbor_distance": sum(
                    proxy["nearest_distances"]
                ) / len(proxy["nearest_distances"]),
            }
        )

        for attribute_index, attribute_name in enumerate(
            conditioning_attributes
        ):
            attribute_rows.append(
                {
                    "split": split_name,
                    "protocol": protocol_name,
                    "query_type": query_type,
                    "attribute": attribute_name,
                    "match_rate": float(
                        exact_match[
                            :,
                            attribute_index,
                        ]
                        .float()
                        .mean()
                        .item()
                    ),
                }
            )

        ood_pair_rows.append(
            {
                "split": split_name,
                "protocol": protocol_name,
                "query_type": query_type,
                "pair": "face_shape + hair_color",
                "pair_match_rate": float(
                    pair_match.float().mean().item()
                ),
                "pair_available_rate": float(
                    pair_available.float().mean().item()
                ),
                "pair_match_rate_when_possible": safe_mean(
                    pair_match,
                    pair_available,
                ),
            }
        )


# ---------------------------------------------------------
# 5. Tabelle e salvataggio
# ---------------------------------------------------------

conditioning_summary = pd.DataFrame(
    evaluation_rows
)

conditioning_per_attribute = pd.DataFrame(
    attribute_rows
)

conditioning_ood_pair = pd.DataFrame(
    ood_pair_rows
)

output_dir = Path("outputs/evaluation")
output_dir.mkdir(parents=True, exist_ok=True)

summary_path = (
    output_dir / "conditioning_scores.csv"
)

per_attribute_path = (
    output_dir / "conditioning_scores_per_attribute.csv"
)

ood_pair_path = (
    output_dir / "conditioning_scores_ood_pair.csv"
)

conditioning_summary.to_csv(
    summary_path,
    index=False,
)

conditioning_per_attribute.to_csv(
    per_attribute_path,
    index=False,
)

conditioning_ood_pair.to_csv(
    ood_pair_path,
    index=False,
)


# ---------------------------------------------------------
# 6. Output leggibile
# ---------------------------------------------------------

generated_summary = conditioning_summary.loc[
    conditioning_summary["query_type"] == "generated"
].reset_index(drop=True)

real_calibration_summary = conditioning_summary.loc[
    conditioning_summary["protocol"]
    == "real_leave_one_out_calibration"
].reset_index(drop=True)

ood_pair_generated = conditioning_ood_pair.loc[
    (
        conditioning_ood_pair["split"] == "test_ood"
    )
    & (
        conditioning_ood_pair["query_type"] == "generated"
    )
].reset_index(drop=True)

print("--- VALUTAZIONE CONDITIONING COMPLETATA ---")

print("\nGenerazioni conditional:")
display(generated_summary)

print("\nCalibrazione del proxy su immagini reali:")
display(real_calibration_summary)

print("\nCoppia composizionale OOD:")
display(ood_pair_generated)

print("\nDettaglio per attributo sul protocollo leave-one-out:")
display(
    conditioning_per_attribute.loc[
        conditioning_per_attribute["protocol"]
        == "generated_leave_one_out"
    ].reset_index(drop=True)
)

print(f"\nCSV riepilogo: {summary_path}")
print(f"CSV per attributo: {per_attribute_path}")
print(f"CSV coppia OOD: {ood_pair_path}")

In [ ]:
# ---------------------------------------------------------
# Diversity a prompt fisso su ordinary e OOD
# ---------------------------------------------------------
#
# Per ciascun prompt generiamo più immagini con seed diversi.
# Le distanze sono calcolate soltanto fra immagini originate
# dallo stesso prompt: confrontare prompt diversi misurerebbe
# invece differenze semantiche, non diversity condizionata.
# ---------------------------------------------------------

from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch


# ---------------------------------------------------------
# 0. Check preliminari
# ---------------------------------------------------------

required_globals = [
    "datasets",
    "conditional_sampler",
    "tokenizer",
]

for variable_name in required_globals:
    assert variable_name in globals(), (
        f"Variabile mancante: {variable_name}."
    )

assert "test_ordinary" in datasets
assert "test_ood" in datasets

dataset_ordinary = datasets["test_ordinary"]
dataset_ood = datasets["test_ood"]

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# ---------------------------------------------------------
# 1. Selezione deterministica dei prompt
# ---------------------------------------------------------

def select_prompts(
    dataset,
    split_name: str,
    num_prompts: int,
) -> list[dict[str, str]]:
    # Seleziona caption non vuote in modo deterministico

    selected = []

    for sample in dataset:
        prompt = sample["caption"].strip()

        if prompt:
            selected.append(
                {
                    "split": split_name,
                    "prompt": prompt,
                    "image_id": sample["image_id"],
                }
            )

        if len(selected) == num_prompts:
            break

    assert len(selected) == num_prompts, (
        f"Prompt insufficienti nello split {split_name}."
    )

    return selected


num_prompts_per_split = 2

diversity_prompt_records = (
    select_prompts(
        dataset=dataset_ordinary,
        split_name="test_ordinary",
        num_prompts=num_prompts_per_split,
    )
    + select_prompts(
        dataset=dataset_ood,
        split_name="test_ood",
        num_prompts=num_prompts_per_split,
    )
)

assert len(diversity_prompt_records) == 4


# ---------------------------------------------------------
# 2. Sampling a seed multipli per ogni prompt
# ---------------------------------------------------------

diversity_seeds = [7, 21, 42, 84, 168, 336]

reference_image = dataset_ordinary[0]["image"]

assert isinstance(reference_image, torch.Tensor)
assert reference_image.ndim == 3

image_shape = tuple(reference_image.shape)

diversity_results = []

for prompt_index, prompt_record in enumerate(
    diversity_prompt_records
):
    prompt = prompt_record["prompt"]

    encoded_prompt = tokenizer.encode(prompt)

    fixed_input_ids = torch.tensor(
        [encoded_prompt.ids],
        dtype=torch.long,
        device=device,
    )

    fixed_attention_mask = torch.tensor(
        [encoded_prompt.attention_mask],
        dtype=torch.long,
        device=device,
    )

    generated_samples = []

    # Ogni campione usa esplicitamente il seed riportato
    # nella griglia e nelle tabelle finali.
    for seed in diversity_seeds:
        generator = torch.Generator(device=device)
        generator.manual_seed(int(seed))

        with torch.inference_mode():
            sample = conditional_sampler.sample(
                input_ids=fixed_input_ids,
                attention_mask=fixed_attention_mask,
                image_shape=image_shape,
                generator=generator,
                guidance_scale=1.0,
            )

        assert sample.shape == (1, *image_shape)
        assert torch.isfinite(sample).all()

        generated_samples.append(
            sample[0].detach().cpu()
        )

    generated_images = torch.stack(
        generated_samples,
        dim=0,
    )

    assert generated_images.shape == (
        len(diversity_seeds),
        *image_shape,
    )

    diversity_results.append(
        {
            **prompt_record,
            "images": generated_images,
        }
    )

# ---------------------------------------------------------
# 3. Distanze pairwise nello stesso prompt
# ---------------------------------------------------------

summary_rows = []
pair_rows = []

for prompt_index, result in enumerate(diversity_results):
    images = result["images"]

    pairwise_rmse_values = []
    pairwise_mae_values = []

    for first_index, second_index in combinations(
        range(len(diversity_seeds)),
        2,
    ):
        first_image = images[first_index]
        second_image = images[second_index]

        delta = first_image - second_image

        # RMSE pixel-wise: non è una norma L2 non normalizzata.
        pairwise_rmse = torch.sqrt(
            delta.pow(2).mean()
        ).item()

        pairwise_mae = delta.abs().mean().item()

        pairwise_rmse_values.append(
            float(pairwise_rmse)
        )

        pairwise_mae_values.append(
            float(pairwise_mae)
        )

        pair_rows.append(
            {
                "prompt_index": prompt_index + 1,
                "split": result["split"],
                "prompt": result["prompt"],
                "seed_a": diversity_seeds[first_index],
                "seed_b": diversity_seeds[second_index],
                "pairwise_rmse": float(pairwise_rmse),
                "pairwise_mae": float(pairwise_mae),
            }
        )

    assert pairwise_rmse_values
    assert pairwise_mae_values

    summary_rows.append(
        {
            "prompt_index": prompt_index + 1,
            "split": result["split"],
            "prompt": result["prompt"],
            "image_id": result["image_id"],
            "num_samples": len(diversity_seeds),
            "num_pairs": len(pairwise_rmse_values),
            "mean_pairwise_rmse": sum(
                pairwise_rmse_values
            ) / len(pairwise_rmse_values),
            "mean_pairwise_mae": sum(
                pairwise_mae_values
            ) / len(pairwise_mae_values),
            "min_pairwise_rmse": min(
                pairwise_rmse_values
            ),
            "max_pairwise_rmse": max(
                pairwise_rmse_values
            ),
            "min_pairwise_mae": min(
                pairwise_mae_values
            ),
            "max_pairwise_mae": max(
                pairwise_mae_values
            ),
        }
    )


# ---------------------------------------------------------
# 4. Tabelle aggregate
# ---------------------------------------------------------

diversity_summary = pd.DataFrame(summary_rows)

diversity_pairs = pd.DataFrame(pair_rows)

diversity_by_split = (
    diversity_summary
    .groupby("split", as_index=False)
    .agg(
        num_prompts=("prompt_index", "count"),
        mean_pairwise_rmse=(
            "mean_pairwise_rmse",
            "mean",
        ),
        mean_pairwise_mae=(
            "mean_pairwise_mae",
            "mean",
        ),
    )
)

overall_diversity_summary = pd.DataFrame(
    [
        {
            "num_prompts": len(diversity_summary),
            "num_samples_per_prompt": len(
                diversity_seeds
            ),
            "num_within_prompt_pairs": len(
                diversity_pairs
            ),
            "mean_pairwise_rmse": float(
                diversity_pairs["pairwise_rmse"].mean()
            ),
            "mean_pairwise_mae": float(
                diversity_pairs["pairwise_mae"].mean()
            ),
        }
    ]
)


# ---------------------------------------------------------
# 5. Salvataggio delle metriche
# ---------------------------------------------------------

output_dir = Path("outputs/evaluation")
output_dir.mkdir(parents=True, exist_ok=True)

diversity_summary_path = (
    output_dir / "diversity_summary.csv"
)

diversity_pairs_path = (
    output_dir / "diversity_pairs.csv"
)

diversity_by_split_path = (
    output_dir / "diversity_by_split.csv"
)

diversity_summary.to_csv(
    diversity_summary_path,
    index=False,
)

diversity_pairs.to_csv(
    diversity_pairs_path,
    index=False,
)

diversity_by_split.to_csv(
    diversity_by_split_path,
    index=False,
)


# ---------------------------------------------------------
# 6. Grid qualitativa
# ---------------------------------------------------------

figure, axes = plt.subplots(
    nrows=len(diversity_results),
    ncols=len(diversity_seeds),
    figsize=(15, 10),
    dpi=150,
)

for row_index, result in enumerate(diversity_results):
    images_to_show = (
        (result["images"] + 1.0) / 2.0
    ).clamp(0.0, 1.0)

    for column_index, seed in enumerate(diversity_seeds):
        axis = axes[row_index, column_index]

        axis.imshow(
            images_to_show[column_index]
            .permute(1, 2, 0)
            .numpy()
        )

        axis.axis("off")

        if row_index == 0:
            axis.set_title(
                f"Seed {seed}",
                fontsize=10,
            )

        if column_index == 0:
            axis.set_ylabel(
                (
                    f"{result['split']}\n"
                    f"Prompt {row_index + 1}"
                ),
                fontsize=10,
                fontweight="bold",
            )

figure.suptitle(
    "Diversity conditional: seed diversi a prompt fisso",
    fontsize=16,
    y=0.995,
)

plt.tight_layout()

diversity_grid_path = (
    output_dir / "diversity_grid.png"
)

figure.savefig(
    diversity_grid_path,
    dpi=200,
    bbox_inches="tight",
)

plt.show()


# ---------------------------------------------------------
# 7. Output finale
# ---------------------------------------------------------

print("--- DIVERSITY SU PIU PROMPT COMPLETATA ---")

print("\nRiepilogo per prompt:")
display(diversity_summary)

print("\nRiepilogo per split:")
display(diversity_by_split)

print("\nRiepilogo complessivo:")
display(overall_diversity_summary)

print(f"\nCSV per prompt: {diversity_summary_path}")
print(f"CSV delle coppie: {diversity_pairs_path}")
print(f"CSV per split: {diversity_by_split_path}")
print(f"Grid diversity: {diversity_grid_path}")

In [ ]:
# ---------------------------------------------------------
# Salvataggio grid OOD
# ---------------------------------------------------------

from pathlib import Path

import matplotlib.pyplot as plt


# ---------------------------------------------------------
# 0. Check preliminari
# ---------------------------------------------------------

required_globals = [
    "ood_grid_images",
    "ood_grid_prompts",
]

for variable_name in required_globals:
    assert variable_name in globals(), (
        f"Variabile mancante: {variable_name}."
    )


# ---------------------------------------------------------
# 1. Output path
# ---------------------------------------------------------

output_dir = Path("outputs/evaluation")
output_dir.mkdir(parents=True, exist_ok=True)

ood_png_path = output_dir / "conditional_grid_ood.png"


# ---------------------------------------------------------
# 2. Salvataggio figura
# ---------------------------------------------------------

import textwrap

ood_images_to_save = (
    (ood_grid_images + 1.0) / 2.0
).clamp(0.0, 1.0)

figure, axes = plt.subplots(
    nrows=2,
    ncols=2,
    figsize=(10, 8),
    dpi=160,
)

axes = axes.flatten()

for axis, image, prompt in zip(
    axes,
    ood_images_to_save,
    ood_grid_prompts,
):
    axis.imshow(image.permute(1, 2, 0).numpy())

    axis.set_title(
        textwrap.fill(prompt, width=40),
        fontsize=8,
        pad=8,
        loc="center",
    )

    axis.axis("off")

figure.suptitle(
    "Campioni conditional OOD",
    fontsize=16,
    fontweight="bold",
    y=0.98,
)

figure.tight_layout(
    rect=(0.03, 0.03, 0.97, 0.92),
    h_pad=1.4,
    w_pad=0.8,
)

figure.savefig(
    ood_png_path,
    dpi=200,
    bbox_inches="tight",
    pad_inches=0.08,
)

plt.close(figure)

print("--- GRID OOD SALVATA ---")
print(f"PNG salvata in: {ood_png_path}")

# Lettura finale di OOD, conditioning e diversity

La valutazione mostra che il modello genera avatar plausibili anche per prompt
dello split `test_ood`, nel quale la combinazione `face_shape = 3` e
`hair_color = 3` non è stata osservata durante il training.

La proxy nearest-neighbor non deve essere interpretata come accuracy diretta
degli attributi. Con il protocollo leave-one-out, nessuno dei prompt valutati
mantiene nel reference bank un'altra immagine con tutti gli otto attributi
identici: di conseguenza, il match completo nullo descrive soprattutto un
limite di copertura del bank e non dimostra un fallimento del generatore.

La calibrazione leave-one-out sulle immagini reali ottiene risultati comparabili
a quelli delle immagini generate. La proxy viene quindi usata come controllo
di coerenza, non come classificatore indipendente degli attributi.

Per la coppia composizionale OOD `face_shape + hair_color`, il match congiunto
leave-one-out è pari a `0.578125` sulle immagini generate e a `0.562500` sulle
immagini reali valutate con lo stesso protocollo. Il risultato costituisce
un'evidenza indiretta ma coerente della capacità del modello di gestire la
combinazione esclusa dal training.

La diversity è stata misurata su quattro prompt, due ordinary e due OOD, con
sei seed distinti per prompt. Il valore medio complessivo è
`pairwise RMSE = 0.034558`, calcolato su 60 coppie intra-prompt. I campioni
mantengono gli attributi principali richiesti e mostrano variazioni locali
contenute: il modello privilegia quindi la stabilità del conditioning rispetto
a una diversity estetica marcata.

Loss, KID, confronto con la baseline unconditional, campioni qualitativi,
proxy di conditioning e diversity devono essere letti insieme. Nessuna
singola metrica è conclusiva, ma il protocollo combinato fornisce una
valutazione trasparente di qualità generativa, controllo testuale e limiti del
modello.